# M1 - Marigold Geometric Diagnostic Experiment

# Step 0 - Mount Google Drive and select compute device

In [ ]:
# ============================================================
# Step 0 — Mount Google Drive and select compute device
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

assert torch.cuda.is_available(), "GPU is required for this notebook."

Mounted at /content/drive
Device: cuda


# Step 1 - Project paths + dataset loader


In [ ]:
# ============================================================
# Step 1A — Define project and dataset paths
# ============================================================

from pathlib import Path

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/MSc_Project_ZeroShot_Counting"
)

DATA_ROOT = PROJECT_ROOT / "datasets"

# ------------------------------------------------------------
# FSC-147
# ------------------------------------------------------------

FSC147_ROOT = DATA_ROOT / "FSC147"
FSC147_RAW = FSC147_ROOT / "raw"

FSC147_IMAGE_DIR = FSC147_RAW / "images_384_VarV2"
FSC147_DENSITY_DIR = (
    FSC147_RAW / "gt_density_map_adaptive_384_VarV2"
)

FSC147_ANNOTATION_FILE = (
    FSC147_RAW / "annotation_FSC147_384.json"
)

FSC147_SPLIT_FILE = (
    FSC147_RAW / "Train_Test_Val_FSC_147.json"
)

FSC147_CLASSES_FILE = (
    FSC147_RAW / "ImageClasses_FSC147.txt"
)

# ------------------------------------------------------------
# OmniCount-191
# ------------------------------------------------------------

OMNICOUNT_ROOT = DATA_ROOT / "OmniCount-191"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("DATA_ROOT    :", DATA_ROOT)

print("\nFSC-147 root :", FSC147_ROOT)
print("OmniCount root:", OMNICOUNT_ROOT)

PROJECT_ROOT : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting
DATA_ROOT    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets

FSC-147 root : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147
OmniCount root: /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/OmniCount-191


In [ ]:
# ============================================================
# Step 1B — Clone project repository and import shared modules
# ============================================================

%cd /content

!rm -rf Surrey-MScProject-Zero-Shot-Object-Counting

!git clone \
    https://github.com/paulmcchan/Surrey-MScProject-Zero-Shot-Object-Counting.git

/content
Cloning into 'Surrey-MScProject-Zero-Shot-Object-Counting'...
remote: Enumerating objects: 47, done.
remote: Counting objects: 100% (47/47), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 47 (delta 1), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (47/47), 65.52 KiB | 3.64 MiB/s, done.
Resolving deltas: 100% (1/1), done.


In [ ]:
# ============================================================
# Step 1C — Import project dataset / evaluation utilities
# ============================================================

import sys

REPO_ROOT = (
    "/content/Surrey-MScProject-Zero-Shot-Object-Counting"
)

if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from src.datasets import (
    FSC147Dataset,
    OmniCount191Dataset,
    collate_keep_dicts,
)

from src.evaluation import (
    evaluate_single_label_counts,
)

print("Project modules imported successfully.")

Project modules imported successfully.


# Step 2 - Install / Load Marigold

In [ ]:
# ============================================================
# Step 2A — Install Marigold dependencies
# Freeze verified working versions for reproducibility
# ============================================================

!pip install -q \
    "accelerate==1.14.0" \
    "diffusers==0.40.0" \
    "transformers==5.16.1" \
    scipy \
    matplotlib

In [ ]:
# ============================================================
# Step 2B — Verify Marigold dependencies and GPU environment
# ============================================================

import torch
import diffusers
import transformers
import accelerate
import scipy

print("PyTorch      :", torch.__version__)
print("Diffusers    :", diffusers.__version__)
print("Transformers :", transformers.__version__)
print("Accelerate   :", accelerate.__version__)

print("\nCUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU           :", torch.cuda.get_device_name(0))
    print("CUDA version  :", torch.version.cuda)

PyTorch      : 2.11.0+cu128
Diffusers    : 0.40.0
Transformers : 5.16.1
Accelerate   : 1.14.0

CUDA available: True
GPU           : Tesla T4
CUDA version  : 12.8


In [ ]:
# ============================================================
# Step 2C — Define Marigold model configuration
# ============================================================

MODEL_ID = "prs-eth/marigold-depth-v1-1"

print("Marigold model ID:", MODEL_ID)
print("Compute device   :", device)

Marigold model ID: prs-eth/marigold-depth-v1-1
Compute device   : cuda


In [ ]:
# ============================================================
# Step 2D — Load pretrained Marigold depth pipeline
# ============================================================

import torch
from diffusers import MarigoldDepthPipeline

print("Loading Marigold model:")
print(MODEL_ID)

pipe = MarigoldDepthPipeline.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
)

pipe = pipe.to(device)

print("\nMarigold pipeline loaded successfully.")
print("Pipeline class :", pipe.__class__.__name__)
print("Device         :", pipe.device)

Loading Marigold model:
prs-eth/marigold-depth-v1-1


model_index.json:   0%|          | 0.00/534 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]


Marigold pipeline loaded successfully.
Pipeline class : MarigoldDepthPipeline
Device         : cuda:0


In [ ]:
# ============================================================
# Step 2E — Verify Marigold pipeline and GPU placement
# ============================================================

print("Pipeline class :", pipe.__class__.__name__)
print("Device         :", pipe.device)

# Main model dtype
if hasattr(pipe, "unet"):
    print("UNet dtype     :", pipe.unet.dtype)
else:
    print("UNet dtype     : not exposed")

print("\nPipeline components:")

for name, component in pipe.components.items():
    if component is None:
        component_name = "None"
    else:
        component_name = component.__class__.__name__

    print(f"  {name:20s}: {component_name}")

print("\nExposed Marigold defaults:")

print(
    "  processing_resolution :",
    getattr(pipe, "default_processing_resolution", "not exposed")
)

print(
    "  denoising_steps        :",
    getattr(pipe, "default_denoising_steps", "not exposed")
)

print(
    "  ensemble_size          :",
    getattr(pipe, "default_ensemble_size", "not exposed")
)

print("\nStep 2E verification complete.")

Pipeline class : MarigoldDepthPipeline
Device         : cuda:0
UNet dtype     : torch.float16

Pipeline components:
  unet                : UNet2DConditionModel
  vae                 : AutoencoderKL
  scheduler           : DDIMScheduler
  text_encoder        : CLIPTextModel
  tokenizer           : CLIPTokenizer

Exposed Marigold defaults:
  processing_resolution : 768
  denoising_steps        : 4
  ensemble_size          : not exposed

Step 2E verification complete.


# Step 3 - Define M1 experiment configuration and output paths


In [ ]:
# ============================================================
# Step 3 — Define M1 experiment configuration and output paths
# ============================================================

from pathlib import Path

# ------------------------------------------------------------
# Frozen FSC-147 development subset
# IMPORTANT: reuse exactly the same dev100 as B1/B2
# ------------------------------------------------------------

DEV100_FILE = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "subsets"
    / "FSC147_val_dev_100.csv"
)

assert DEV100_FILE.exists(), f"Missing frozen dev100 file: {DEV100_FILE}"

# ------------------------------------------------------------
# M1 output structure
# ------------------------------------------------------------

M1_ROOT = (
    PROJECT_ROOT
    / "results"
    / "M1_Marigold_Geometric_Diagnostics"
)

M1_DEPTH_CACHE_DIR = M1_ROOT / "depth_cache"
M1_TABLES_DIR = M1_ROOT / "tables"
M1_METADATA_DIR = M1_ROOT / "metadata"
M1_VIS_DIR = M1_ROOT / "visualisations"

for path in [
    M1_ROOT,
    M1_DEPTH_CACHE_DIR,
    M1_TABLES_DIR,
    M1_METADATA_DIR,
    M1_VIS_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Frozen Marigold inference configuration
#
# These match the practical configuration used in the
# earlier Marigold system-familiarisation notebook:
#   processing resolution = 768
#   denoising steps       = 4
#   ensemble size         = 1
# ------------------------------------------------------------

MARIGOLD_PROCESSING_RESOLUTION = 768
MARIGOLD_DENOISING_STEPS = 4
MARIGOLD_ENSEMBLE_SIZE = 1

# Fix the stochastic generator for reproducible M1 depth maps.
# This is made explicit for M1 even though the earlier
# familiarisation work was primarily qualitative.
MARIGOLD_SEED = 0

# ------------------------------------------------------------
# M1 experimental rules
# ------------------------------------------------------------

M1_SPLIT = "FSC147_val_dev_100"

# M1 is diagnostic:
# - no changes to B0/B1/B2 counting rules
# - no GT-driven inference/routing
# - FSC-147 test set remains untouched
M1_ALLOW_TEST_SET = False
M1_COUNT_RULE_MODIFICATION = False

# ------------------------------------------------------------
# Print frozen configuration
# ------------------------------------------------------------

print("M1 experiment configuration")
print("---------------------------")
print("Dev subset              :", DEV100_FILE)
print("M1 root                 :", M1_ROOT)

print("\nMarigold configuration")
print("----------------------")
print("Model ID                :", MODEL_ID)
print("Processing resolution   :", MARIGOLD_PROCESSING_RESOLUTION)
print("Denoising steps         :", MARIGOLD_DENOISING_STEPS)
print("Ensemble size           :", MARIGOLD_ENSEMBLE_SIZE)
print("Random seed             :", MARIGOLD_SEED)

print("\nExperimental constraints")
print("------------------------")
print("Split                    :", M1_SPLIT)
print("Test-set use allowed?    :", M1_ALLOW_TEST_SET)
print("Count-rule modification? :", M1_COUNT_RULE_MODIFICATION)

print("\nOutput directories")
print("------------------")
print("Depth cache :", M1_DEPTH_CACHE_DIR)
print("Tables      :", M1_TABLES_DIR)
print("Metadata    :", M1_METADATA_DIR)
print("Visuals     :", M1_VIS_DIR)

print("\nStep 3 configuration complete.")

M1 experiment configuration
---------------------------
Dev subset              : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/subsets/FSC147_val_dev_100.csv
M1 root                 : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics

Marigold configuration
----------------------
Model ID                : prs-eth/marigold-depth-v1-1
Processing resolution   : 768
Denoising steps         : 4
Ensemble size           : 1
Random seed             : 0

Experimental constraints
------------------------
Split                    : FSC147_val_dev_100
Test-set use allowed?    : False
Count-rule modification? : False

Output directories
------------------
Depth cache : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/depth_cache
Tables      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables
Metadata    : /content/drive/MyDrive

# Step 4 - Load and verify frozen FSC-147 dev100 subset

In [ ]:
# ============================================================
# Step 4 — Load and verify frozen FSC-147 dev100 subset
# ============================================================

import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# Load frozen dev100 table
# ------------------------------------------------------------

dev100_df = pd.read_csv(DEV100_FILE)

print("Loaded dev100:")
print("  rows    :", len(dev100_df))
print("  columns :", list(dev100_df.columns))

# ------------------------------------------------------------
# Core checks
# ------------------------------------------------------------

assert len(dev100_df) == 100, (
    f"Expected 100 rows in frozen dev100, found {len(dev100_df)}"
)

# Require dataset_index because it is the authoritative FSC-147 identity
assert "dataset_index" in dev100_df.columns, (
    "Frozen dev100 file does not contain 'dataset_index'."
)

# Check uniqueness of dataset_index
num_dup_dataset_index = dev100_df["dataset_index"].duplicated().sum()

print("\nIdentity checks")
print("---------------")
print("Duplicate dataset_index :", num_dup_dataset_index)

assert num_dup_dataset_index == 0, (
    f"Found {num_dup_dataset_index} duplicate dataset_index values."
)

# ------------------------------------------------------------
# Resolve image path column
#
# Prefer an existing image-path column if present.
# Otherwise reconstruct from FSC147_IMAGE_DIR and image name.
# ------------------------------------------------------------

candidate_path_cols = [
    "image_path",
    "resolved_image_path",
    "path",
]

image_path_col = None

for col in candidate_path_cols:
    if col in dev100_df.columns:
        image_path_col = col
        break

if image_path_col is not None:
    resolved_paths = [
        Path(p) for p in dev100_df[image_path_col].astype(str)
    ]
else:
    # Try common image-name columns
    candidate_name_cols = [
        "image_id",
        "image_name",
        "filename",
        "file_name",
    ]

    image_name_col = None

    for col in candidate_name_cols:
        if col in dev100_df.columns:
            image_name_col = col
            break

    assert image_name_col is not None, (
        "Could not find either an image-path column "
        "or an image-name column in dev100."
    )

    resolved_paths = [
        FSC147_IMAGE_DIR / str(name)
        for name in dev100_df[image_name_col]
    ]

# ------------------------------------------------------------
# Verify all images exist
# ------------------------------------------------------------

exists_flags = [p.exists() for p in resolved_paths]

num_missing = sum(not x for x in exists_flags)

print("\nImage-path checks")
print("-----------------")
print("Resolved images :", len(resolved_paths))
print("Missing images  :", num_missing)

if num_missing > 0:
    missing_paths = [
        str(p)
        for p, ok in zip(resolved_paths, exists_flags)
        if not ok
    ]

    print("\nFirst missing paths:")
    for p in missing_paths[:10]:
        print(" ", p)

assert num_missing == 0, (
    f"{num_missing} frozen dev100 images could not be resolved."
)

# Store authoritative resolved path for M1
dev100_df = dev100_df.copy()
dev100_df["resolved_image_path"] = [
    str(p) for p in resolved_paths
]

# ------------------------------------------------------------
# Optional density-stratum verification
# ------------------------------------------------------------

print("\nDensity-stratum check")
print("---------------------")

if "density_stratum" in dev100_df.columns:
    density_counts = (
        dev100_df["density_stratum"]
        .value_counts()
        .sort_index()
    )

    print(density_counts)

    expected_density_counts = {
        "S1_0_20": 20,
        "S2_20_40": 20,
        "S3_40_60": 20,
        "S4_60_80": 20,
        "S5_80_95": 10,
        "S6_95_100": 10,
    }

    actual_density_counts = density_counts.to_dict()

    assert actual_density_counts == expected_density_counts, (
        "Density-stratum counts differ from frozen B1/B2 dev100."
    )

    print("Density-stratum counts: PASS")
else:
    print(
        "'density_stratum' not present in dev100 CSV; "
        "no stratum assertion performed."
    )

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\nStep 4 verification complete.")
print("Frozen dev100 rows       :", len(dev100_df))
print("Unique dataset_index     :", dev100_df["dataset_index"].nunique())
print("All image paths resolved :", num_missing == 0)

Loaded dev100:
  rows    : 100
  columns : ['dataset_index', 'image_id', 'category', 'gt_count', 'density_stratum']

Identity checks
---------------
Duplicate dataset_index : 0

Image-path checks
-----------------
Resolved images : 100
Missing images  : 0

Density-stratum check
---------------------
density_stratum
S1_0_20      20
S2_20_40     20
S3_40_60     20
S4_60_80     20
S5_80_95     10
S6_95_100    10
Name: count, dtype: int64
Density-stratum counts: PASS

Step 4 verification complete.
Frozen dev100 rows       : 100
Unique dataset_index     : 100
All image paths resolved : True


# Step 5 - Marigold dev100 depth cache

In [ ]:
# ============================================================
# Step 5A — Single-image Marigold smoke test
# ============================================================

import numpy as np
import torch
from PIL import Image

# ------------------------------------------------------------
# Select first frozen dev100 image
# ------------------------------------------------------------

smoke_row = dev100_df.iloc[0]

smoke_dataset_index = int(smoke_row["dataset_index"])
smoke_image_id = str(smoke_row["image_id"])
smoke_image_path = Path(smoke_row["resolved_image_path"])

print("Smoke-test image")
print("----------------")
print("dataset_index :", smoke_dataset_index)
print("image_id      :", smoke_image_id)
print("image_path    :", smoke_image_path)

# ------------------------------------------------------------
# Load RGB image
# ------------------------------------------------------------

image = Image.open(smoke_image_path).convert("RGB")

print("RGB size      :", image.size)

# ------------------------------------------------------------
# Reproducible Marigold inference
# ------------------------------------------------------------

generator = torch.Generator(device=device).manual_seed(
    MARIGOLD_SEED
)

with torch.inference_mode():
    depth_output = pipe(
        image,
        num_inference_steps=MARIGOLD_DENOISING_STEPS,
        ensemble_size=MARIGOLD_ENSEMBLE_SIZE,
        processing_resolution=MARIGOLD_PROCESSING_RESOLUTION,
        generator=generator,
    )

# ------------------------------------------------------------
# Extract numerical depth prediction
# ------------------------------------------------------------

depth = np.asarray(depth_output.prediction)
depth = np.squeeze(depth)

print("\nDepth output")
print("------------")
print("Raw prediction shape :", np.asarray(depth_output.prediction).shape)
print("Squeezed depth shape :", depth.shape)
print("Depth dtype          :", depth.dtype)
print("Depth min            :", float(depth.min()))
print("Depth max            :", float(depth.max()))
print("Depth mean           :", float(depth.mean()))
print("Finite values only?  :", bool(np.isfinite(depth).all()))

# ------------------------------------------------------------
# Verify output is aligned to original image resolution
# PIL size = (W, H), NumPy depth shape = (H, W)
# ------------------------------------------------------------

expected_shape = (image.height, image.width)

print("\nAlignment check")
print("---------------")
print("Expected depth shape :", expected_shape)
print("Actual depth shape   :", depth.shape)

assert depth.shape == expected_shape, (
    f"Depth/image shape mismatch: expected {expected_shape}, "
    f"got {depth.shape}"
)

assert np.isfinite(depth).all(), (
    "Depth prediction contains NaN or Inf values."
)

print("\nStep 5A smoke test: PASS")

Smoke-test image
----------------
dataset_index : 576
image_id      : 3873.jpg
image_path    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/datasets/FSC147/raw/images_384_VarV2/3873.jpg
RGB size      : (384, 384)


Marigold predictions...:   0%|          | 0/1 [00:00<?, ?it/s]

Diffusion steps...:   0%|          | 0/4 [00:00<?, ?it/s]


Depth output
------------
Raw prediction shape : (1, 384, 384, 1)
Squeezed depth shape : (384, 384)
Depth dtype          : float32
Depth min            : 0.0
Depth max            : 0.8681640625
Depth mean           : 0.2706884741783142
Finite values only?  : True

Alignment check
---------------
Expected depth shape : (384, 384)
Actual depth shape   : (384, 384)

Step 5A smoke test: PASS


In [ ]:
# ============================================================
# Step 5B — Full dev100 Marigold depth caching
# ============================================================

import json
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image

# ------------------------------------------------------------
# Output files
# ------------------------------------------------------------

M1_DEPTH_INDEX_FILE = M1_TABLES_DIR / "M1_depth_cache_index.csv"
M1_DEPTH_MANIFEST_FILE = M1_METADATA_DIR / "M1_depth_cache_manifest.json"

M1_DEPTH_CACHE_DIR.mkdir(parents=True, exist_ok=True)
M1_TABLES_DIR.mkdir(parents=True, exist_ok=True)
M1_METADATA_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Frozen protocol
# ------------------------------------------------------------

M1_DEPTH_PROTOCOL = {
    "module": "M1_Marigold_Geometric_Diagnostics",
    "model_id": MODEL_ID,
    "processing_resolution": MARIGOLD_PROCESSING_RESOLUTION,
    "denoising_steps": MARIGOLD_DENOISING_STEPS,
    "ensemble_size": MARIGOLD_ENSEMBLE_SIZE,
    "seed": MARIGOLD_SEED,
    "device": str(device),
    "dtype": str(pipe.unet.dtype) if hasattr(pipe, "unet") else "unknown",
    "split": M1_SPLIT,
    "test_set_accessed": bool(M1_ALLOW_TEST_SET),
    "count_rule_modified": bool(M1_COUNT_RULE_MODIFICATION),
    "output_format": "compressed_npz_per_image",
    "depth_key": "depth",
}

print("=" * 120)
print("STEP 5B — FULL DEV100 MARIGOLD DEPTH CACHING")
print("=" * 120)
print("Images to process         :", len(dev100_df))
print("Depth cache directory     :", M1_DEPTH_CACHE_DIR)
print("Index file                :", M1_DEPTH_INDEX_FILE)
print("Manifest file             :", M1_DEPTH_MANIFEST_FILE)
print("Model ID                  :", MODEL_ID)
print("Processing resolution     :", MARIGOLD_PROCESSING_RESOLUTION)
print("Denoising steps           :", MARIGOLD_DENOISING_STEPS)
print("Ensemble size             :", MARIGOLD_ENSEMBLE_SIZE)
print("Random seed               :", MARIGOLD_SEED)
print("Test set accessed         :", "NO")
print()

# ------------------------------------------------------------
# Cache helper
# ------------------------------------------------------------

def make_depth_cache_name(row):
    return f"idx_{int(row['dataset_index']):04d}__{Path(str(row['image_id'])).stem}.npz"

# ------------------------------------------------------------
# Process all images
# ------------------------------------------------------------

cache_rows = []
num_reused = 0
num_generated = 0
total_start = time.time()

for i, (_, row) in enumerate(dev100_df.iterrows(), start=1):
    dataset_index = int(row["dataset_index"])
    image_id = str(row["image_id"])
    category = str(row["category"])
    gt_count = int(row["gt_count"])
    density_stratum = str(row["density_stratum"])
    image_path = Path(row["resolved_image_path"])

    cache_name = make_depth_cache_name(row)
    cache_path = M1_DEPTH_CACHE_DIR / cache_name

    image = Image.open(image_path).convert("RGB")
    width, height = image.size
    expected_shape = (height, width)

    row_start = time.time()

    # --------------------------------------------------------
    # Reuse existing cache if valid
    # --------------------------------------------------------
    reused = False
    if cache_path.exists():
        try:
            existing = np.load(cache_path)
            depth = np.asarray(existing["depth"])
            if depth.shape == expected_shape and np.isfinite(depth).all():
                reused = True
                num_reused += 1
            else:
                cache_path.unlink()
        except Exception:
            cache_path.unlink(missing_ok=True)

    # --------------------------------------------------------
    # Generate depth if cache not reusable
    # --------------------------------------------------------
    if not reused:
        generator = torch.Generator(device=device).manual_seed(MARIGOLD_SEED)

        with torch.inference_mode():
            depth_output = pipe(
                image,
                num_inference_steps=MARIGOLD_DENOISING_STEPS,
                ensemble_size=MARIGOLD_ENSEMBLE_SIZE,
                processing_resolution=MARIGOLD_PROCESSING_RESOLUTION,
                generator=generator,
            )

        depth = np.asarray(depth_output.prediction)
        depth = np.squeeze(depth).astype(np.float32)

        assert depth.shape == expected_shape, (
            f"Depth/image shape mismatch for dataset_index={dataset_index}: "
            f"expected {expected_shape}, got {depth.shape}"
        )
        assert np.isfinite(depth).all(), (
            f"Depth contains NaN/Inf for dataset_index={dataset_index}"
        )

        np.savez_compressed(cache_path, depth=depth)
        num_generated += 1

    runtime_sec = time.time() - row_start
    cache_size_bytes = cache_path.stat().st_size

    cache_rows.append({
        "dataset_index": dataset_index,
        "image_id": image_id,
        "category": category,
        "density_stratum": density_stratum,
        "gt_count": gt_count,
        "image_path": str(image_path),
        "cache_path": str(cache_path),
        "cache_file": cache_name,
        "height": height,
        "width": width,
        "depth_shape_h": int(depth.shape[0]),
        "depth_shape_w": int(depth.shape[1]),
        "depth_min": float(depth.min()),
        "depth_max": float(depth.max()),
        "depth_mean": float(depth.mean()),
        "finite_ok": bool(np.isfinite(depth).all()),
        "reused_cache": reused,
        "runtime_sec": runtime_sec,
        "cache_size_bytes": cache_size_bytes,
    })

    print(
        f"[{i:>3}/{len(dev100_df)}] "
        f"{image_id:<10} | {density_stratum:<8} | GT={gt_count:<3} | "
        f"{'reuse' if reused else 'new':<5} | "
        f"{runtime_sec:>5.2f}s | "
        f"depth=({depth.shape[0]},{depth.shape[1]}) | "
        f"size={cache_size_bytes / (1024**2):.2f} MiB"
    )

# ------------------------------------------------------------
# Build index tables
# ------------------------------------------------------------

depth_index_df = pd.DataFrame(cache_rows).sort_values(
    by=["dataset_index"]
).reset_index(drop=True)

# Integrity checks
assert len(depth_index_df) == len(dev100_df), "Depth index row count mismatch."
assert depth_index_df["dataset_index"].nunique() == len(dev100_df), "Duplicate dataset_index in depth index."
assert depth_index_df["cache_path"].nunique() == len(dev100_df), "Duplicate cache_path in depth index."
assert depth_index_df["finite_ok"].all(), "At least one cached depth map is not finite."
assert (depth_index_df["depth_shape_h"] == depth_index_df["height"]).all(), "Depth height mismatch."
assert (depth_index_df["depth_shape_w"] == depth_index_df["width"]).all(), "Depth width mismatch."

depth_index_df.to_csv(M1_DEPTH_INDEX_FILE, index=False)

# ------------------------------------------------------------
# Density summary
# ------------------------------------------------------------

density_summary_df = (
    depth_index_df
    .groupby("density_stratum", as_index=False)
    .agg(
        num_images=("dataset_index", "count"),
        mean_runtime_sec=("runtime_sec", "mean"),
        mean_cache_size_bytes=("cache_size_bytes", "mean"),
        mean_depth_min=("depth_min", "mean"),
        mean_depth_max=("depth_max", "mean"),
        mean_depth_mean=("depth_mean", "mean"),
    )
    .sort_values("density_stratum")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Manifest
# ------------------------------------------------------------

manifest = {
    "step": "5B_full_dev100_depth_caching",
    "protocol": M1_DEPTH_PROTOCOL,
    "num_images_expected": int(len(dev100_df)),
    "num_index_rows": int(len(depth_index_df)),
    "num_generated": int(num_generated),
    "num_reused": int(num_reused),
    "total_runtime_sec": float(time.time() - total_start),
    "depth_cache_dir": str(M1_DEPTH_CACHE_DIR),
    "depth_index_file": str(M1_DEPTH_INDEX_FILE),
    "test_set_accessed": False,
    "count_rule_modified": False,
}

with open(M1_DEPTH_MANIFEST_FILE, "w") as f:
    json.dump(manifest, f, indent=2)

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("STEP 5B COMPLETE")
print("=" * 120)
print("Expected images          :", len(dev100_df))
print("Depth index rows         :", len(depth_index_df))
print("Generated caches         :", num_generated)
print("Reused caches            :", num_reused)
print("Total runtime (sec)      :", round(time.time() - total_start, 2))
print("Test set accessed        : NO")
print("Count rule modified      : NO")

print("\nFirst 10 depth-index rows:")
display(
    depth_index_df[
        [
            "dataset_index", "image_id", "category", "density_stratum",
            "gt_count", "height", "width",
            "depth_min", "depth_max", "depth_mean",
            "reused_cache", "runtime_sec", "cache_size_bytes"
        ]
    ].head(10)
)

print("\nDepth density summary:")
display(density_summary_df)

print("\nSaved depth index:")
print(M1_DEPTH_INDEX_FILE)

print("\nSaved manifest:")
print(M1_DEPTH_MANIFEST_FILE)

STEP 5B — FULL DEV100 MARIGOLD DEPTH CACHING
Images to process         : 100
Depth cache directory     : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/depth_cache
Index file                : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_depth_cache_index.csv
Manifest file             : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/metadata/M1_depth_cache_manifest.json
Model ID                  : prs-eth/marigold-depth-v1-1
Processing resolution     : 768
Denoising steps           : 4
Ensemble size             : 1
Random seed               : 0
Test set accessed         : NO

[  1/100] 3873.jpg   | S1_0_20  | GT=7   | reuse |  0.83s | depth=(384,384) | size=0.19 MiB
[  2/100] 1964.jpg   | S1_0_20  | GT=8   | reuse |  0.47s | depth=(384,540) | size=0.26 MiB
[  3/100] 774.jpg    | S1_0_20  | GT=8   | reuse |  1.18s | depth=(384,575) | 

,dataset_index,image_id,category,density_stratum,gt_count,height,width,depth_min,depth_max,depth_mean,reused_cache,runtime_sec,cache_size_bytes
0,9,215.jpg,grapes,S6_95_100,259,384,408,0.000537,0.974121,0.187338,True,0.418377,270610
1,14,220.jpg,grapes,S4_60_80,59,384,576,0.000000,0.836914,0.463032,True,0.378207,313611
2,20,228.jpg,grapes,S4_60_80,44,384,631,0.000000,1.000000,0.377331,True,0.384964,315835
3,31,241.jpg,grapes,S3_40_60,35,384,588,0.000000,0.996094,0.310347,True,0.374881,331355
4,40,252.jpg,grapes,S3_40_60,22,384,512,0.000000,1.000000,0.266391,True,0.384882,282256
5,49,556.jpg,flamingos,S2_20_40,14,384,683,0.010674,0.960938,0.439072,True,0.410703,365905
6,52,559.jpg,flamingos,S2_20_40,20,384,575,0.000000,0.980469,0.513709,True,0.453411,231911
7,59,567.jpg,flamingos,S1_0_20,10,384,819,0.000000,0.972168,0.450403,True,0.411965,405488
8,67,575.jpg,flamingos,S1_0_20,12,384,674,0.000000,0.969727,0.771398,True,0.516620,262731
9,87,596.jpg,seagulls,S2_20_40,14,384,537,0.367920,0.838379,0.519969,True,0.422303,223680



Depth density summary:


,density_stratum,num_images,mean_runtime_sec,mean_cache_size_bytes,mean_depth_min,mean_depth_max,mean_depth_mean
0,S1_0_20,20,0.523540,265977.30,0.031505,0.956226,0.476363
1,S2_20_40,20,0.495365,265782.00,0.033197,0.980371,0.474493
2,S3_40_60,20,0.448747,279445.05,0.028717,0.930029,0.422059
3,S4_60_80,20,0.451678,265587.45,0.070812,0.948389,0.519557
4,S5_80_95,10,0.478576,318111.40,0.058621,0.994141,0.445679
5,S6_95_100,10,0.492686,318853.50,0.038445,0.940723,0.417778



Saved depth index:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_depth_cache_index.csv

Saved manifest:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/metadata/M1_depth_cache_manifest.json


In [ ]:
# ============================================================
# Step 5C — Marigold cache integrity + reproducibility check
# ============================================================

import hashlib
import time
import numpy as np
import pandas as pd
import torch
from pathlib import Path
from PIL import Image

print("=" * 120)
print("STEP 5C — MARIGOLD CACHE INTEGRITY + REPRODUCIBILITY")
print("=" * 120)

# ------------------------------------------------------------
# Reload authoritative Step 5B cache index from disk
# ------------------------------------------------------------

depth_index_check_df = pd.read_csv(M1_DEPTH_INDEX_FILE)

assert len(depth_index_check_df) == 100, (
    f"Expected 100 cache-index rows, found {len(depth_index_check_df)}"
)

assert depth_index_check_df["dataset_index"].nunique() == 100, (
    "dataset_index is not unique in Step 5B cache index."
)

# ------------------------------------------------------------
# Helper: SHA256
# ------------------------------------------------------------

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)

    return h.hexdigest()

# ------------------------------------------------------------
# Part A — Full 100-image cache integrity check
# ------------------------------------------------------------

integrity_rows = []

for _, row in depth_index_check_df.iterrows():

    dataset_index = int(row["dataset_index"])
    image_id = str(row["image_id"])
    cache_path = Path(row["cache_path"])

    exists_ok = cache_path.exists()

    shape_ok = False
    finite_ok = False
    dtype_ok = False
    stats_ok = False
    cache_sha256 = None

    if exists_ok:
        with np.load(cache_path) as data:

            assert "depth" in data.files, (
                f"'depth' key missing from {cache_path}"
            )

            depth_cached = np.asarray(data["depth"])

        expected_shape = (
            int(row["depth_shape_h"]),
            int(row["depth_shape_w"]),
        )

        shape_ok = depth_cached.shape == expected_shape
        finite_ok = bool(np.isfinite(depth_cached).all())
        dtype_ok = depth_cached.dtype == np.float32

        # Verify stored summary statistics against the cached data.
        # Small tolerance is used only for CSV floating-point serialization.
        stats_ok = (
            np.isclose(
                float(depth_cached.min()),
                float(row["depth_min"]),
                rtol=0,
                atol=1e-7,
            )
            and
            np.isclose(
                float(depth_cached.max()),
                float(row["depth_max"]),
                rtol=0,
                atol=1e-7,
            )
            and
            np.isclose(
                float(depth_cached.mean()),
                float(row["depth_mean"]),
                rtol=0,
                atol=1e-7,
            )
        )

        cache_sha256 = sha256_file(cache_path)

    integrity_rows.append({
        "dataset_index": dataset_index,
        "image_id": image_id,
        "cache_exists": exists_ok,
        "shape_ok": shape_ok,
        "finite_ok": finite_ok,
        "dtype_float32_ok": dtype_ok,
        "stats_ok": stats_ok,
        "cache_sha256": cache_sha256,
    })

integrity_df = pd.DataFrame(integrity_rows)

print("\nPART A — Full cache integrity")
print("-----------------------------")
print("Cache files expected :", len(integrity_df))
print("Files exist          :", int(integrity_df["cache_exists"].sum()))
print("Shape checks PASS    :", int(integrity_df["shape_ok"].sum()))
print("Finite checks PASS   :", int(integrity_df["finite_ok"].sum()))
print("Float32 checks PASS  :", int(integrity_df["dtype_float32_ok"].sum()))
print("Stats checks PASS    :", int(integrity_df["stats_ok"].sum()))

assert integrity_df["cache_exists"].all(), "At least one depth cache file is missing."
assert integrity_df["shape_ok"].all(), "At least one depth cache has the wrong shape."
assert integrity_df["finite_ok"].all(), "At least one depth cache contains NaN/Inf."
assert integrity_df["dtype_float32_ok"].all(), "At least one depth cache is not float32."
assert integrity_df["stats_ok"].all(), "At least one cached depth map disagrees with its Step 5B index statistics."

# Save integrity table
M1_DEPTH_INTEGRITY_FILE = (
    M1_TABLES_DIR / "M1_depth_cache_integrity.csv"
)

integrity_df.to_csv(M1_DEPTH_INTEGRITY_FILE, index=False)

# ------------------------------------------------------------
# Part B — Fixed-seed regeneration check
#
# Select one example from:
#   S1 = sparse
#   S4 = moderate/high
#   S6 = very dense
#
# This does NOT tune or evaluate counting performance.
# ------------------------------------------------------------

REPRO_STRATA = [
    "S1_0_20",
    "S4_60_80",
    "S6_95_100",
]

repro_rows = []

print("\nPART B — Fixed-seed regeneration")
print("--------------------------------")

for stratum in REPRO_STRATA:

    selected = (
        depth_index_check_df[
            depth_index_check_df["density_stratum"] == stratum
        ]
        .sort_values("dataset_index")
        .iloc[0]
    )

    dataset_index = int(selected["dataset_index"])
    image_id = str(selected["image_id"])
    image_path = Path(selected["image_path"])
    cache_path = Path(selected["cache_path"])

    # Load frozen cached depth
    with np.load(cache_path) as data:
        depth_cached = np.asarray(data["depth"]).astype(np.float32)

    # Reload RGB
    image_repro = Image.open(image_path).convert("RGB")

    # Same explicit seed as Step 5B
    generator = torch.Generator(device=device).manual_seed(
        MARIGOLD_SEED
    )

    start = time.time()

    with torch.inference_mode():
        output_repro = pipe(
            image_repro,
            num_inference_steps=MARIGOLD_DENOISING_STEPS,
            ensemble_size=MARIGOLD_ENSEMBLE_SIZE,
            processing_resolution=MARIGOLD_PROCESSING_RESOLUTION,
            generator=generator,
        )

    depth_repro = np.asarray(
        output_repro.prediction
    ).squeeze().astype(np.float32)

    runtime_sec = time.time() - start

    assert depth_repro.shape == depth_cached.shape, (
        f"Regenerated depth shape mismatch for {image_id}"
    )

    abs_diff = np.abs(depth_repro - depth_cached)

    max_abs_diff = float(abs_diff.max())
    mean_abs_diff = float(abs_diff.mean())

    exact_equal = bool(
        np.array_equal(depth_repro, depth_cached)
    )

    allclose_1e6 = bool(
        np.allclose(
            depth_repro,
            depth_cached,
            rtol=0,
            atol=1e-6,
        )
    )

    allclose_1e5 = bool(
        np.allclose(
            depth_repro,
            depth_cached,
            rtol=0,
            atol=1e-5,
        )
    )

    repro_rows.append({
        "density_stratum": stratum,
        "dataset_index": dataset_index,
        "image_id": image_id,
        "height": depth_cached.shape[0],
        "width": depth_cached.shape[1],
        "exact_equal": exact_equal,
        "allclose_atol_1e-6": allclose_1e6,
        "allclose_atol_1e-5": allclose_1e5,
        "max_abs_diff": max_abs_diff,
        "mean_abs_diff": mean_abs_diff,
        "runtime_sec": runtime_sec,
    })

    print(
        f"{stratum:<10} | "
        f"{image_id:<10} | "
        f"exact={exact_equal!s:<5} | "
        f"allclose(1e-6)={allclose_1e6!s:<5} | "
        f"max_diff={max_abs_diff:.8f} | "
        f"mean_diff={mean_abs_diff:.8f}"
    )

repro_df = pd.DataFrame(repro_rows)

# Save reproducibility table
M1_DEPTH_REPRO_FILE = (
    M1_TABLES_DIR / "M1_depth_reproducibility_check.csv"
)

repro_df.to_csv(M1_DEPTH_REPRO_FILE, index=False)

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 120)
print("STEP 5C COMPLETE")
print("=" * 120)

print("Full cache integrity : PASS")
print("Caches verified      :", len(integrity_df))

print("\nReproducibility summary:")
display(repro_df)

print("\nSaved integrity table:")
print(M1_DEPTH_INTEGRITY_FILE)

print("\nSaved reproducibility table:")
print(M1_DEPTH_REPRO_FILE)

STEP 5C — MARIGOLD CACHE INTEGRITY + REPRODUCIBILITY

PART A — Full cache integrity
-----------------------------
Cache files expected : 100
Files exist          : 100
Shape checks PASS    : 100
Finite checks PASS   : 100
Float32 checks PASS  : 100
Stats checks PASS    : 100

PART B — Fixed-seed regeneration
--------------------------------


Marigold predictions...:   0%|          | 0/1 [00:00<?, ?it/s]

Diffusion steps...:   0%|          | 0/4 [00:00<?, ?it/s]

S1_0_20    | 567.jpg    | exact=True  | allclose(1e-6)=True  | max_diff=0.00000000 | mean_diff=0.00000000


Marigold predictions...:   0%|          | 0/1 [00:00<?, ?it/s]

...

Marigold predictions...:   0%|          | 0/1 [00:00<?, ?it/s]

Diffusion steps...:   0%|          | 0/4 [00:00<?, ?it/s]

S6_95_100  | 215.jpg    | exact=True  | allclose(1e-6)=True  | max_diff=0.00000000 | mean_diff=0.00000000

STEP 5C COMPLETE
Full cache integrity : PASS
Caches verified      : 100

Reproducibility summary:


,density_stratum,dataset_index,image_id,height,width,exact_equal,allclose_atol_1e-6,allclose_atol_1e-5,max_abs_diff,mean_abs_diff,runtime_sec
0,S1_0_20,59,567.jpg,384,819,True,True,True,0.0,0.0,0.898913
1,S4_60_80,14,220.jpg,384,576,True,True,True,0.0,0.0,0.923059
2,S6_95_100,9,215.jpg,384,408,True,True,True,0.0,0.0,1.311371



Saved integrity table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_depth_cache_integrity.csv

Saved reproducibility table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_depth_reproducibility_check.csv


# Step 6 - Geometric Evidence analysis

In [ ]:
# ============================================================
# Step 6A — GT-point local Marigold depth diagnostics
# ============================================================

import ast
import numpy as np
import pandas as pd
from pathlib import Path

print("=" * 120)
print("STEP 6A — GT-POINT LOCAL MARIGOLD DEPTH DIAGNOSTICS")
print("=" * 120)

# ------------------------------------------------------------
# Load the frozen B1 C-candidate table.
#
# We reuse C1 only as the authoritative source of:
#   - gt_points_xy
#   - radius_px
#
# This keeps GT coordinates and scale convention identical
# to the previous B1/B2 experiments.
# ------------------------------------------------------------

B1_C_FILE = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "Group_C"
    / "B1_Group_C_E_candidates_per_image.csv"
)

assert B1_C_FILE.exists(), f"Missing B1 C-candidate file: {B1_C_FILE}"

c_all_df = pd.read_csv(B1_C_FILE)

print("Loaded B1 C table:")
print("  rows    :", len(c_all_df))
print("  columns :", list(c_all_df.columns))

required_cols = {
    "dataset_index",
    "image_id",
    "candidate_id",
    "gt_points_xy",
    "radius_px",
}

missing_cols = required_cols - set(c_all_df.columns)

assert not missing_cols, (
    f"B1 C table is missing required columns: {sorted(missing_cols)}"
)

# ------------------------------------------------------------
# Keep exactly one frozen C1 row per dev100 image
# ------------------------------------------------------------

c1_gt_df = (
    c_all_df[
        c_all_df["candidate_id"].astype(str) == "C1"
    ]
    .copy()
)

# Restrict strictly to frozen dev100
c1_gt_df = c1_gt_df[
    c1_gt_df["dataset_index"].isin(
        dev100_df["dataset_index"]
    )
].copy()

assert len(c1_gt_df) == 100, (
    f"Expected 100 C1 rows for dev100, found {len(c1_gt_df)}"
)

assert c1_gt_df["dataset_index"].nunique() == 100, (
    "C1 GT source does not contain exactly one row per dev100 image."
)

# ------------------------------------------------------------
# Parse stored GT point lists
# ------------------------------------------------------------

def parse_xy_list(value):
    if isinstance(value, list):
        parsed = value
    elif pd.isna(value):
        return []
    else:
        parsed = ast.literal_eval(str(value))

    points = []

    for p in parsed:
        if p is None or len(p) < 2:
            continue

        x = float(p[0])
        y = float(p[1])

        points.append((x, y))

    return points


c1_gt_df["gt_points_parsed"] = c1_gt_df["gt_points_xy"].apply(
    parse_xy_list
)

# ------------------------------------------------------------
# Join Marigold cache metadata
# ------------------------------------------------------------

depth_meta_df = pd.read_csv(M1_DEPTH_INDEX_FILE)

join_cols = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
    "cache_path",
    "height",
    "width",
]

m1_gt_source_df = (
    c1_gt_df[
        [
            "dataset_index",
            "image_id",
            "radius_px",
            "gt_points_parsed",
        ]
    ]
    .merge(
        depth_meta_df[join_cols],
        on=["dataset_index", "image_id"],
        how="inner",
        validate="one_to_one",
    )
)

assert len(m1_gt_source_df) == 100

# ------------------------------------------------------------
# Helper: circular local regions around one GT point
# ------------------------------------------------------------

def local_depth_features(depth, x, y, radius_px):
    """
    Diagnostic local geometry around one GT point.

    Regions:
      centre disk : distance <= 0.5R
      local disk  : distance <= R
      outer ring  : R < distance <= 2R

    Coordinates follow image convention:
      x = column
      y = row
    """

    h, w = depth.shape

    # Nearest valid image pixel for point-depth / gradient.
    xi = int(np.clip(np.rint(x), 0, w - 1))
    yi = int(np.clip(np.rint(y), 0, h - 1))

    R = max(float(radius_px), 1.0)

    # Crop only the required 2R neighbourhood.
    x0 = max(0, int(np.floor(x - 2.0 * R)))
    x1 = min(w - 1, int(np.ceil(x + 2.0 * R)))
    y0 = max(0, int(np.floor(y - 2.0 * R)))
    y1 = min(h - 1, int(np.ceil(y + 2.0 * R)))

    patch = depth[y0:y1 + 1, x0:x1 + 1]

    yy, xx = np.mgrid[y0:y1 + 1, x0:x1 + 1]

    dist = np.sqrt(
        (xx.astype(np.float32) - float(x)) ** 2
        +
        (yy.astype(np.float32) - float(y)) ** 2
    )

    centre_mask = dist <= (0.5 * R)
    local_mask = dist <= R
    annulus_mask = (dist > R) & (dist <= (2.0 * R))

    centre_vals = patch[centre_mask]
    local_vals = patch[local_mask]
    annulus_vals = patch[annulus_mask]

    # Fallback only for extreme border/small-region cases.
    centre_median = (
        float(np.median(centre_vals))
        if centre_vals.size > 0
        else float(depth[yi, xi])
    )

    local_median = (
        float(np.median(local_vals))
        if local_vals.size > 0
        else float(depth[yi, xi])
    )

    local_std = (
        float(np.std(local_vals))
        if local_vals.size > 0
        else 0.0
    )

    annulus_median = (
        float(np.median(annulus_vals))
        if annulus_vals.size > 0
        else np.nan
    )

    point_depth = float(depth[yi, xi])

    # Central finite-difference style gradient via NumPy gradient.
    gy, gx = np.gradient(depth.astype(np.float32))

    grad_mag = float(
        np.sqrt(
            gx[yi, xi] ** 2
            +
            gy[yi, xi] ** 2
        )
    )

    centre_annulus_abs = (
        abs(centre_median - annulus_median)
        if np.isfinite(annulus_median)
        else np.nan
    )

    point_local_abs = abs(point_depth - local_median)

    return {
        "x": float(x),
        "y": float(y),
        "x_pixel": xi,
        "y_pixel": yi,
        "radius_px": R,

        "point_depth": point_depth,

        "centre_median_depth": centre_median,
        "local_median_depth": local_median,
        "local_depth_std": local_std,
        "annulus_median_depth": annulus_median,

        "point_vs_local_abs": point_local_abs,
        "centre_vs_annulus_abs": centre_annulus_abs,

        "point_gradient_mag": grad_mag,

        "centre_num_pixels": int(centre_vals.size),
        "local_num_pixels": int(local_vals.size),
        "annulus_num_pixels": int(annulus_vals.size),
    }

# ------------------------------------------------------------
# Compute GT-level geometric diagnostics for all dev100 images
# ------------------------------------------------------------

gt_geom_rows = []

for _, row in m1_gt_source_df.iterrows():

    dataset_index = int(row["dataset_index"])
    image_id = str(row["image_id"])
    category = str(row["category"])
    density_stratum = str(row["density_stratum"])
    gt_count = int(row["gt_count"])
    radius_px = float(row["radius_px"])

    cache_path = Path(row["cache_path"])

    with np.load(cache_path) as data:
        depth = np.asarray(data["depth"]).astype(np.float32)

    assert depth.shape == (
        int(row["height"]),
        int(row["width"]),
    )

    # Robust within-image depth scale.
    depth_p05 = float(np.percentile(depth, 5))
    depth_p95 = float(np.percentile(depth, 95))

    robust_depth_range = max(
        depth_p95 - depth_p05,
        1e-8,
    )

    gt_points = row["gt_points_parsed"]

    # gt_count and stored GT-point count should agree.
    assert len(gt_points) == gt_count, (
        f"GT-count mismatch for {image_id}: "
        f"gt_count={gt_count}, points={len(gt_points)}"
    )

    for gt_idx, (x, y) in enumerate(gt_points):

        feat = local_depth_features(
            depth=depth,
            x=x,
            y=y,
            radius_px=radius_px,
        )

        feat.update({
            "dataset_index": dataset_index,
            "image_id": image_id,
            "category": category,
            "density_stratum": density_stratum,
            "gt_count": gt_count,
            "gt_point_index": gt_idx,

            "image_depth_p05": depth_p05,
            "image_depth_p95": depth_p95,
            "image_robust_depth_range": robust_depth_range,
        })

        feat["point_vs_local_abs_norm"] = (
            feat["point_vs_local_abs"]
            / robust_depth_range
        )

        if np.isfinite(feat["centre_vs_annulus_abs"]):
            feat["centre_vs_annulus_abs_norm"] = (
                feat["centre_vs_annulus_abs"]
                / robust_depth_range
            )
        else:
            feat["centre_vs_annulus_abs_norm"] = np.nan

        feat["point_gradient_mag_norm"] = (
            feat["point_gradient_mag"]
            / robust_depth_range
        )

        gt_geom_rows.append(feat)

gt_geom_df = pd.DataFrame(gt_geom_rows)

# ------------------------------------------------------------
# Integrity checks
# ------------------------------------------------------------

expected_gt_total = int(
    dev100_df["gt_count"].sum()
)

assert len(gt_geom_df) == expected_gt_total, (
    f"Expected {expected_gt_total} GT points, "
    f"found {len(gt_geom_df)}"
)

assert gt_geom_df["dataset_index"].nunique() == 100

assert np.isfinite(
    gt_geom_df["point_depth"]
).all()

assert np.isfinite(
    gt_geom_df["local_median_depth"]
).all()

# ------------------------------------------------------------
# Summaries
# ------------------------------------------------------------

summary_cols = [
    "point_vs_local_abs_norm",
    "centre_vs_annulus_abs_norm",
    "local_depth_std",
    "point_gradient_mag_norm",
]

overall_summary_df = (
    gt_geom_df[summary_cols]
    .describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
    )
    .T
)

density_geom_summary_df = (
    gt_geom_df
    .groupby("density_stratum", as_index=False)
    .agg(
        num_gt_points=("gt_point_index", "count"),

        median_point_local_abs_norm=(
            "point_vs_local_abs_norm",
            "median",
        ),

        median_centre_annulus_abs_norm=(
            "centre_vs_annulus_abs_norm",
            "median",
        ),

        median_local_depth_std=(
            "local_depth_std",
            "median",
        ),

        median_gradient_norm=(
            "point_gradient_mag_norm",
            "median",
        ),
    )
    .sort_values("density_stratum")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

M1_GT_GEOM_FILE = (
    M1_TABLES_DIR
    / "M1_GT_point_local_depth_diagnostics.csv"
)

M1_GT_GEOM_DENSITY_FILE = (
    M1_TABLES_DIR
    / "M1_GT_point_local_depth_by_density.csv"
)

gt_geom_df.to_csv(
    M1_GT_GEOM_FILE,
    index=False,
)

density_geom_summary_df.to_csv(
    M1_GT_GEOM_DENSITY_FILE,
    index=False,
)

# ------------------------------------------------------------
# Report
# ------------------------------------------------------------

print("\nGT geometry diagnostics complete")
print("--------------------------------")
print("Images analysed       :", gt_geom_df["dataset_index"].nunique())
print("Expected GT points    :", expected_gt_total)
print("Analysed GT points    :", len(gt_geom_df))

print("\nOverall GT-point geometric statistics:")
display(overall_summary_df)

print("\nBy density stratum:")
display(density_geom_summary_df)

print("\nExample rows:")
display(
    gt_geom_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_point_index",
            "radius_px",
            "point_depth",
            "local_depth_std",
            "point_vs_local_abs_norm",
            "centre_vs_annulus_abs_norm",
            "point_gradient_mag_norm",
        ]
    ].head(10)
)

print("\nSaved GT-point diagnostics:")
print(M1_GT_GEOM_FILE)

print("\nSaved density summary:")
print(M1_GT_GEOM_DENSITY_FILE)

print("\nStep 6A: PASS")

STEP 6A — GT-POINT LOCAL MARIGOLD DEPTH DIAGNOSTICS
Loaded B1 C table:
  rows    : 300
  columns : ['candidate_id', 'experiment_id', 'candidate_family', 'description', 'dataset_index', 'image_id', 'category', 'density_stratum', 'width', 'height', 'a3c_percentile', 'point_tolerance_frac', 'radius_px', 'gt_count', 'pred_count', 'signed_error', 'abs_error', 'squared_error', 'pred_gt_ratio', 'tp', 'fp', 'fn', 'precision', 'recall', 'f1', 'gt_points_xy', 'pred_points_xy']

GT geometry diagnostics complete
--------------------------------
Images analysed       : 100
Expected GT points    : 10045
Analysed GT points    : 10045

Overall GT-point geometric statistics:


,count,mean,std,min,25%,50%,75%,90%,95%,max
point_vs_local_abs_norm,10045.0,0.125932,0.261166,0.000000,0.001204,0.004365,0.039762,0.597329,0.794773,2.205128
centre_vs_annulus_abs_norm,10045.0,0.080977,0.169760,0.000000,0.002580,0.012796,0.061587,0.246878,0.446528,1.658120
local_depth_std,10045.0,0.054296,0.078097,0.000189,0.005963,0.013092,0.072825,0.183000,0.211497,0.447200
point_gradient_mag_norm,10045.0,0.033638,0.084666,0.000000,0.001064,0.002828,0.013519,0.106829,0.207431,0.845988



By density stratum:


,density_stratum,num_gt_points,median_point_local_abs_norm,median_centre_annulus_abs_norm,median_local_depth_std,median_gradient_norm
0,S1_0_20,195,0.002704,0.020887,0.014448,0.001815
1,S2_20_40,320,0.005529,0.026129,0.012862,0.002245
2,S3_40_60,537,0.006202,0.024324,0.012590,0.002619
3,S4_60_80,1096,0.004976,0.025842,0.011810,0.002482
4,S5_80_95,1211,0.005058,0.037092,0.017200,0.002004
5,S6_95_100,6686,0.003951,0.006924,0.012236,0.003308



Example rows:


,dataset_index,image_id,category,density_stratum,gt_point_index,radius_px,point_depth,local_depth_std,point_vs_local_abs_norm,centre_vs_annulus_abs_norm,point_gradient_mag_norm
0,576,3873.jpg,shallots,S1_0_20,0,12.0,0.197266,0.002379,0.000604,0.005136,0.000378
1,576,3873.jpg,shallots,S1_0_20,1,12.0,0.147705,0.003871,0.000680,0.004230,0.000755
2,576,3873.jpg,shallots,S1_0_20,2,12.0,0.096985,0.003953,0.000529,0.002266,0.000998
3,576,3873.jpg,shallots,S1_0_20,3,12.0,0.006226,0.003572,0.000227,0.003097,0.000725
4,576,3873.jpg,shallots,S1_0_20,4,12.0,0.068604,0.003056,0.000151,0.004003,0.000537
5,576,3873.jpg,shallots,S1_0_20,5,12.0,0.048462,0.000903,0.000076,0.003059,0.000107
6,576,3873.jpg,shallots,S1_0_20,6,12.0,0.143799,0.018013,0.000604,0.001662,0.000459
7,322,1964.jpg,bottle caps,S1_0_20,0,12.0,0.342285,0.002728,0.001079,0.004046,0.000944
8,322,1964.jpg,bottle caps,S1_0_20,1,12.0,0.177368,0.015790,0.000405,0.003236,0.002495
9,322,1964.jpg,bottle caps,S1_0_20,2,12.0,0.323730,0.001493,0.000809,0.007282,0.000688



Saved GT-point diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_GT_point_local_depth_diagnostics.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_GT_point_local_depth_by_density.csv

Step 6A: PASS


In [ ]:
# ============================================================
# Step 6A.1 — Normalize local depth variability
# ============================================================

gt_geom_df["local_depth_std_norm"] = (
    gt_geom_df["local_depth_std"]
    / gt_geom_df["image_robust_depth_range"]
)

assert np.isfinite(
    gt_geom_df["local_depth_std_norm"]
).all()

# Updated density summary
density_geom_summary_df = (
    gt_geom_df
    .groupby("density_stratum", as_index=False)
    .agg(
        num_gt_points=("gt_point_index", "count"),

        median_point_local_abs_norm=(
            "point_vs_local_abs_norm",
            "median",
        ),

        median_centre_annulus_abs_norm=(
            "centre_vs_annulus_abs_norm",
            "median",
        ),

        median_local_depth_std_norm=(
            "local_depth_std_norm",
            "median",
        ),

        median_gradient_norm=(
            "point_gradient_mag_norm",
            "median",
        ),
    )
    .sort_values("density_stratum")
    .reset_index(drop=True)
)

# Overwrite the point-level table with the added normalized feature
gt_geom_df.to_csv(
    M1_GT_GEOM_FILE,
    index=False,
)

density_geom_summary_df.to_csv(
    M1_GT_GEOM_DENSITY_FILE,
    index=False,
)

print("Updated density summary:")
display(density_geom_summary_df)

print("\nStep 6A.1: PASS")

Updated density summary:


,density_stratum,num_gt_points,median_point_local_abs_norm,median_centre_annulus_abs_norm,median_local_depth_std_norm,median_gradient_norm
0,S1_0_20,195,0.002704,0.020887,0.017723,0.001815
1,S2_20_40,320,0.005529,0.026129,0.016881,0.002245
2,S3_40_60,537,0.006202,0.024324,0.020713,0.002619
3,S4_60_80,1096,0.004976,0.025842,0.028337,0.002482
4,S5_80_95,1211,0.005058,0.037092,0.022503,0.002004
5,S6_95_100,6686,0.003951,0.006924,0.018230,0.003308



Step 6A.1: PASS


### **Finding 1 — Step 6A:**

Local Marigold geometry around true object locations is highly variable; simple point-local geometric thresholds are not supported.

In [ ]:
# ============================================================
# Step 6B — Marigold geometry: B0-covered vs B0-missed GT
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

print("=" * 120)
print("STEP 6B — MARIGOLD GEOMETRY: B0-COVERED VS B0-MISSED GT")
print("=" * 120)

# ------------------------------------------------------------
# 1. Frozen B0 mask-cache paths
# ------------------------------------------------------------

B2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "B2_B0_Anchored_Semantic_Validation"
)

B0_MASK_INDEX_FILE = (
    B2_ROOT
    / "tables"
    / "B0_mask_cache_index.csv"
)

assert B0_MASK_INDEX_FILE.exists(), (
    f"Missing frozen B0 mask-cache index: {B0_MASK_INDEX_FILE}"
)

b0_cache_index_df = pd.read_csv(B0_MASK_INDEX_FILE)

assert len(b0_cache_index_df) == 100
assert b0_cache_index_df["dataset_index"].nunique() == 100
assert b0_cache_index_df["reproduced_b0_count"].sum() == 2298
assert b0_cache_index_df["count_match"].all()

print("Frozen B0 cache index       : PASS")
print("B0 images                   :", len(b0_cache_index_df))
print("B0 masks                    :", int(b0_cache_index_df["reproduced_b0_count"].sum()))

# ------------------------------------------------------------
# 2. Resolve frozen B0 mask-cache paths robustly
#
# cache_file may be stored as either:
#   - an absolute path, or
#   - a relative filename/path.
# ------------------------------------------------------------

def resolve_b0_cache_path(cache_value):
    p = Path(str(cache_value))

    # Case 1: already an absolute path
    if p.is_absolute():
        return p

    # Case 2: relative to B2 root
    candidate = B2_ROOT / p

    if candidate.exists():
        return candidate

    # Case 3: stored only as filename; locate exactly once
    matches = list(
        B2_ROOT.rglob(p.name)
    )

    assert len(matches) == 1, (
        f"Expected exactly one B0 cache match for {p.name}, "
        f"found {len(matches)}"
    )

    return matches[0]


# Resolve every indexed cache file
b0_cache_index_df = b0_cache_index_df.copy()

b0_cache_index_df["resolved_cache_path"] = (
    b0_cache_index_df["cache_file"]
    .apply(resolve_b0_cache_path)
)

# Verify all 100 exist
exists_flags = (
    b0_cache_index_df["resolved_cache_path"]
    .apply(lambda p: Path(p).exists())
)

assert exists_flags.all(), (
    f"{int((~exists_flags).sum())} B0 cache files are missing."
)

print("B0 cache files              : 100 / 100")

# Display one example so we know exactly what was resolved
print(
    "Example resolved cache      :",
    b0_cache_index_df.iloc[0]["resolved_cache_path"],
)

# ------------------------------------------------------------
# 3. Frozen B0 mask loader
#
# This reproduces the lossless bit-packed cache convention
# used in B2.
# ------------------------------------------------------------

def load_b0_masks_from_cache(cache_path):

    with np.load(
        cache_path,
        allow_pickle=False,
    ) as data:

        packed_masks = data["packed_masks"]

        image_height = int(
            data["image_height"]
        )

        image_width = int(
            data["image_width"]
        )

        num_masks = int(
            data["num_masks"]
        )

    num_pixels = (
        image_height
        *
        image_width
    )

    if num_masks > 0:

        unpacked = np.unpackbits(
            packed_masks,
            axis=1,
            bitorder="little",
        )

        unpacked = unpacked[
            :,
            :num_pixels
        ]

        masks = (
            unpacked
            .reshape(
                num_masks,
                image_height,
                image_width,
            )
            .astype(bool)
        )

    else:

        masks = np.empty(
            (
                0,
                image_height,
                image_width,
            ),
            dtype=bool,
        )

    return masks


# ------------------------------------------------------------
# 4. Classify every Step-6A GT point by frozen B0 coverage
#
# covered:
#   GT point falls inside >=1 B0 mask
#
# missed:
#   GT point falls inside no B0 mask
#
# This is diagnostic GT usage only.
# ------------------------------------------------------------

gt_b0_geom_parts = []

for dataset_index, gt_img_df in gt_geom_df.groupby(
    "dataset_index",
    sort=False,
):

    idx = int(dataset_index)

    cache_row = b0_cache_index_df[
        b0_cache_index_df["dataset_index"] == idx
    ]

    assert len(cache_row) == 1

    cache_row = cache_row.iloc[0]

    cache_path = Path(
        cache_row["resolved_cache_path"]
    )

    masks = load_b0_masks_from_cache(
        cache_path
    )

    h = int(cache_row["image_height"])
    w = int(cache_row["image_width"])

    assert masks.shape == (
        int(cache_row["reproduced_b0_count"]),
        h,
        w,
    )

    tmp = gt_img_df.copy()

    coverage_count = []

    for _, gt_row in tmp.iterrows():

        # Same pixel-coordinate convention already used
        # for Step 6A depth sampling.
        x = int(
            np.clip(
                np.rint(gt_row["x"]),
                0,
                w - 1,
            )
        )

        y = int(
            np.clip(
                np.rint(gt_row["y"]),
                0,
                h - 1,
            )
        )

        if masks.shape[0] == 0:
            n_cover = 0
        else:
            n_cover = int(
                masks[:, y, x].sum()
            )

        coverage_count.append(
            n_cover
        )

    tmp["b0_covering_mask_count"] = (
        coverage_count
    )

    tmp["b0_gt_status"] = np.where(
        tmp["b0_covering_mask_count"] > 0,
        "covered",
        "missed",
    )

    gt_b0_geom_parts.append(tmp)

gt_b0_geom_df = pd.concat(
    gt_b0_geom_parts,
    ignore_index=True,
)

# ------------------------------------------------------------
# 5. Exact reproduction of frozen B2 GT coverage
# ------------------------------------------------------------

status_counts = (
    gt_b0_geom_df["b0_gt_status"]
    .value_counts()
)

num_covered = int(
    status_counts.get("covered", 0)
)

num_missed = int(
    status_counts.get("missed", 0)
)

print("\nB0 GT-coverage reproduction")
print("---------------------------")
print("Total GT points             :", len(gt_b0_geom_df))
print("B0-covered GT               :", num_covered)
print("B0-missed GT                :", num_missed)

assert len(gt_b0_geom_df) == 10045
assert num_covered == 4404, (
    f"Expected 4404 B0-covered GT, got {num_covered}"
)
assert num_missed == 5641, (
    f"Expected 5641 B0-missed GT, got {num_missed}"
)

print("Frozen B2 coverage identity : PASS")

# ------------------------------------------------------------
# 6. Geometry features to compare
# ------------------------------------------------------------

GEOM_FEATURES = [
    "point_vs_local_abs_norm",
    "centre_vs_annulus_abs_norm",
    "local_depth_std_norm",
    "point_gradient_mag_norm",
]

for col in GEOM_FEATURES:
    assert col in gt_b0_geom_df.columns
    assert np.isfinite(
        gt_b0_geom_df[col]
    ).all()

# ------------------------------------------------------------
# 7. Descriptive covered-vs-missed comparison
# ------------------------------------------------------------

group_summary_rows = []

for status in [
    "covered",
    "missed",
]:

    g = gt_b0_geom_df[
        gt_b0_geom_df["b0_gt_status"] == status
    ]

    for feature in GEOM_FEATURES:

        vals = g[feature].to_numpy()

        group_summary_rows.append({
            "b0_gt_status": status,
            "feature": feature,
            "n": len(vals),
            "mean": float(np.mean(vals)),
            "median": float(np.median(vals)),
            "p25": float(np.percentile(vals, 25)),
            "p75": float(np.percentile(vals, 75)),
            "p90": float(np.percentile(vals, 90)),
            "p95": float(np.percentile(vals, 95)),
        })

geom_group_summary_df = pd.DataFrame(
    group_summary_rows
)

# ------------------------------------------------------------
# 8. Discrimination diagnostics
#
# Positive class = B0-missed GT.
#
# AUC = 0.5 means no ranking separation.
# Separability AUC = max(AUC, 1-AUC), independent of direction.
#
# Mann-Whitney U is reported descriptively; with >10k points,
# p-values can become very small even for weak effects.
#
# Rank-biserial:
#   +1 => feature larger for missed GT
#   -1 => feature larger for covered GT
# ------------------------------------------------------------

comparison_rows = []

y_missed = (
    gt_b0_geom_df["b0_gt_status"]
    .eq("missed")
    .astype(int)
    .to_numpy()
)

for feature in GEOM_FEATURES:

    covered_vals = (
        gt_b0_geom_df.loc[
            gt_b0_geom_df["b0_gt_status"] == "covered",
            feature,
        ]
        .to_numpy()
    )

    missed_vals = (
        gt_b0_geom_df.loc[
            gt_b0_geom_df["b0_gt_status"] == "missed",
            feature,
        ]
        .to_numpy()
    )

    raw_auc = roc_auc_score(
        y_missed,
        gt_b0_geom_df[feature].to_numpy(),
    )

    separability_auc = max(
        raw_auc,
        1.0 - raw_auc,
    )

    direction = (
        "higher_in_missed"
        if raw_auc > 0.5
        else
        "higher_in_covered"
        if raw_auc < 0.5
        else
        "none"
    )

    # U for missed relative to covered
    mw = mannwhitneyu(
        missed_vals,
        covered_vals,
        alternative="two-sided",
    )

    n_missed = len(missed_vals)
    n_covered = len(covered_vals)

    rank_biserial = (
        2.0
        * float(mw.statistic)
        / (n_missed * n_covered)
        - 1.0
    )

    comparison_rows.append({
        "feature": feature,

        "n_covered": n_covered,
        "n_missed": n_missed,

        "covered_median": float(
            np.median(covered_vals)
        ),

        "missed_median": float(
            np.median(missed_vals)
        ),

        "median_difference_missed_minus_covered": (
            float(np.median(missed_vals))
            -
            float(np.median(covered_vals))
        ),

        "auc_missed_positive": float(
            raw_auc
        ),

        "separability_auc": float(
            separability_auc
        ),

        "direction": direction,

        "mannwhitney_u": float(
            mw.statistic
        ),

        "mannwhitney_p": float(
            mw.pvalue
        ),

        "rank_biserial_missed_vs_covered": float(
            rank_biserial
        ),
    })

geom_comparison_df = pd.DataFrame(
    comparison_rows
).sort_values(
    "separability_auc",
    ascending=False,
).reset_index(drop=True)

# ------------------------------------------------------------
# 9. By-density comparison
#
# Density remains DIAGNOSTIC ONLY.
# It is not used for inference routing.
# ------------------------------------------------------------

density_rows = []

for density_stratum, density_df in (
    gt_b0_geom_df.groupby(
        "density_stratum",
        sort=True,
    )
):

    for feature in GEOM_FEATURES:

        c = (
            density_df.loc[
                density_df["b0_gt_status"] == "covered",
                feature,
            ]
            .to_numpy()
        )

        m = (
            density_df.loc[
                density_df["b0_gt_status"] == "missed",
                feature,
            ]
            .to_numpy()
        )

        # Both groups must exist
        assert len(c) > 0
        assert len(m) > 0

        labels = (
            density_df["b0_gt_status"]
            .eq("missed")
            .astype(int)
            .to_numpy()
        )

        vals = (
            density_df[feature]
            .to_numpy()
        )

        auc = roc_auc_score(
            labels,
            vals,
        )

        density_rows.append({
            "density_stratum": density_stratum,
            "feature": feature,

            "n_covered": len(c),
            "n_missed": len(m),

            "covered_median": float(
                np.median(c)
            ),

            "missed_median": float(
                np.median(m)
            ),

            "median_difference_missed_minus_covered": (
                float(np.median(m))
                -
                float(np.median(c))
            ),

            "auc_missed_positive": float(
                auc
            ),

            "separability_auc": float(
                max(auc, 1.0 - auc)
            ),
        })

geom_density_comparison_df = pd.DataFrame(
    density_rows
)

# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

M1_B0_GT_GEOM_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry.csv"
)

M1_B0_GT_GEOM_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry_summary.csv"
)

M1_B0_GT_GEOM_COMPARISON_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry_discrimination.csv"
)

M1_B0_GT_GEOM_DENSITY_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry_by_density.csv"
)

gt_b0_geom_df.to_csv(
    M1_B0_GT_GEOM_FILE,
    index=False,
)

geom_group_summary_df.to_csv(
    M1_B0_GT_GEOM_SUMMARY_FILE,
    index=False,
)

geom_comparison_df.to_csv(
    M1_B0_GT_GEOM_COMPARISON_FILE,
    index=False,
)

geom_density_comparison_df.to_csv(
    M1_B0_GT_GEOM_DENSITY_FILE,
    index=False,
)

# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print("\nOverall geometric discrimination")
print("--------------------------------")
display(
    geom_comparison_df[
        [
            "feature",
            "covered_median",
            "missed_median",
            "median_difference_missed_minus_covered",
            "auc_missed_positive",
            "separability_auc",
            "direction",
            "rank_biserial_missed_vs_covered",
            "mannwhitney_p",
        ]
    ]
)

print("\nGT counts by density and B0 status")
print("----------------------------------")

coverage_by_density_df = (
    gt_b0_geom_df
    .groupby(
        [
            "density_stratum",
            "b0_gt_status",
        ]
    )
    .size()
    .unstack(fill_value=0)
)

display(
    coverage_by_density_df
)

print("\nGeometry by density:")
display(
    geom_density_comparison_df[
        [
            "density_stratum",
            "feature",
            "n_covered",
            "n_missed",
            "covered_median",
            "missed_median",
            "auc_missed_positive",
            "separability_auc",
        ]
    ]
)

print("\nSaved point-level table:")
print(M1_B0_GT_GEOM_FILE)

print("\nSaved overall discrimination:")
print(M1_B0_GT_GEOM_COMPARISON_FILE)

print("\nSaved density diagnostics:")
print(M1_B0_GT_GEOM_DENSITY_FILE)

print("\nGT used for analysis       : YES")
print("GT used for inference      : NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6B: PASS")

STEP 6B — MARIGOLD GEOMETRY: B0-COVERED VS B0-MISSED GT
Frozen B0 cache index       : PASS
B0 images                   : 100
B0 masks                    : 2298
B0 cache files              : 100 / 100
Example resolved cache      : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/B2_B0_Anchored_Semantic_Validation/cache/B0_masks/00009_215_B0_masks.npz

B0 GT-coverage reproduction
---------------------------
Total GT points             : 10045
B0-covered GT               : 4404
B0-missed GT                : 5641
Frozen B2 coverage identity : PASS

Overall geometric discrimination
--------------------------------


,feature,covered_median,missed_median,median_difference_missed_minus_covered,auc_missed_positive,separability_auc,direction,rank_biserial_missed_vs_covered,mannwhitney_p
0,local_depth_std_norm,0.011532,0.065052,0.053521,0.719263,0.719263,higher_in_missed,0.438526,0.000000e+00
1,point_vs_local_abs_norm,0.002408,0.014747,0.012339,0.705058,0.705058,higher_in_missed,0.410116,2.360014e-273
2,point_gradient_mag_norm,0.002025,0.006889,0.004864,0.683666,0.683666,higher_in_missed,0.367332,1.068325e-219
3,centre_vs_annulus_abs_norm,0.004826,0.022787,0.017961,0.624794,0.624794,higher_in_missed,0.249588,1.629758e-102



GT counts by density and B0 status
----------------------------------


b0_gt_status,covered,missed
density_stratum,,
S1_0_20,107,88
S2_20_40,134,186
S3_40_60,204,333
S4_60_80,645,451
S5_80_95,576,635
S6_95_100,2738,3948



Geometry by density:


,density_stratum,feature,n_covered,n_missed,covered_median,missed_median,auc_missed_positive,separability_auc
0,S1_0_20,point_vs_local_abs_norm,107,88,0.001490,0.005981,0.682137,0.682137
1,S1_0_20,centre_vs_annulus_abs_norm,107,88,0.012297,0.041260,0.693500,0.693500
2,S1_0_20,local_depth_std_norm,107,88,0.009756,0.044174,0.755310,0.755310
3,S1_0_20,point_gradient_mag_norm,107,88,0.001512,0.002617,0.622133,0.622133
4,S2_20_40,point_vs_local_abs_norm,134,186,0.003456,0.007994,0.656395,0.656395
5,S2_20_40,centre_vs_annulus_abs_norm,134,186,0.023958,0.028390,0.593023,0.593023
6,S2_20_40,local_depth_std_norm,134,186,0.013937,0.023922,0.659926,0.659926
7,S2_20_40,point_gradient_mag_norm,134,186,0.002059,0.002620,0.592080,0.592080
8,S3_40_60,point_vs_local_abs_norm,204,333,0.004718,0.007634,0.551088,0.551088
9,S3_40_60,centre_vs_annulus_abs_norm,204,333,0.035460,0.022300,0.466974,0.533026



Saved point-level table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_B0_covered_vs_missed_GT_geometry.csv

Saved overall discrimination:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_B0_covered_vs_missed_GT_geometry_discrimination.csv

Saved density diagnostics:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_B0_covered_vs_missed_GT_geometry_by_density.csv

GT used for analysis       : YES
GT used for inference      : NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6B: PASS


In [ ]:
# ============================================================
# Step 6B.1 — Within-image B0-covered vs B0-missed geometry
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import wilcoxon
from sklearn.metrics import roc_auc_score

print("=" * 120)
print("STEP 6B.1 — WITHIN-IMAGE B0-COVERED VS B0-MISSED GEOMETRY")
print("=" * 120)

# ------------------------------------------------------------
# Features established in Step 6A/6A.1
# ------------------------------------------------------------

GEOM_FEATURES = [
    "point_vs_local_abs_norm",
    "centre_vs_annulus_abs_norm",
    "local_depth_std_norm",
    "point_gradient_mag_norm",
]

# ------------------------------------------------------------
# 1. Identify images containing BOTH covered and missed GT
# ------------------------------------------------------------

status_per_image_df = (
    gt_b0_geom_df
    .groupby(
        [
            "dataset_index",
            "image_id",
            "density_stratum",
            "b0_gt_status",
        ]
    )
    .size()
    .unstack(
        fill_value=0
    )
    .reset_index()
)

for col in ["covered", "missed"]:
    if col not in status_per_image_df.columns:
        status_per_image_df[col] = 0

status_per_image_df["has_both"] = (
    (status_per_image_df["covered"] > 0)
    &
    (status_per_image_df["missed"] > 0)
)

mixed_image_df = status_per_image_df[
    status_per_image_df["has_both"]
].copy()

mixed_dataset_indices = set(
    mixed_image_df["dataset_index"]
    .astype(int)
    .tolist()
)

print("Total dev100 images          :", gt_b0_geom_df["dataset_index"].nunique())
print("Images with covered GT       :", int((status_per_image_df["covered"] > 0).sum()))
print("Images with missed GT        :", int((status_per_image_df["missed"] > 0).sum()))
print("Images with BOTH             :", len(mixed_image_df))

assert len(mixed_image_df) > 0

# ------------------------------------------------------------
# 2. Compute within-image feature comparisons
# ------------------------------------------------------------

within_rows = []

for dataset_index, image_df in (
    gt_b0_geom_df[
        gt_b0_geom_df["dataset_index"].isin(
            mixed_dataset_indices
        )
    ]
    .groupby(
        "dataset_index",
        sort=True,
    )
):

    dataset_index = int(dataset_index)

    image_id = str(
        image_df["image_id"].iloc[0]
    )

    category = str(
        image_df["category"].iloc[0]
    )

    density_stratum = str(
        image_df["density_stratum"].iloc[0]
    )

    covered_df = image_df[
        image_df["b0_gt_status"] == "covered"
    ]

    missed_df = image_df[
        image_df["b0_gt_status"] == "missed"
    ]

    assert len(covered_df) > 0
    assert len(missed_df) > 0

    y = (
        image_df["b0_gt_status"]
        .eq("missed")
        .astype(int)
        .to_numpy()
    )

    for feature in GEOM_FEATURES:

        covered_vals = (
            covered_df[feature]
            .to_numpy()
        )

        missed_vals = (
            missed_df[feature]
            .to_numpy()
        )

        covered_median = float(
            np.median(covered_vals)
        )

        missed_median = float(
            np.median(missed_vals)
        )

        median_diff = (
            missed_median
            -
            covered_median
        )

        values = (
            image_df[feature]
            .to_numpy()
        )

        # AUC is defined because every selected image
        # contains both classes.
        #
        # If all feature values in the image are identical,
        # sklearn still returns AUC = 0.5.
        auc = float(
            roc_auc_score(
                y,
                values,
            )
        )

        within_rows.append({
            "dataset_index": dataset_index,
            "image_id": image_id,
            "category": category,
            "density_stratum": density_stratum,

            "feature": feature,

            "n_covered": int(
                len(covered_vals)
            ),

            "n_missed": int(
                len(missed_vals)
            ),

            "covered_median": covered_median,
            "missed_median": missed_median,

            "median_diff_missed_minus_covered": (
                median_diff
            ),

            "auc_missed_positive": auc,

            "auc_minus_chance": (
                auc - 0.5
            ),
        })

within_image_geom_df = pd.DataFrame(
    within_rows
)

assert (
    within_image_geom_df["dataset_index"]
    .nunique()
    ==
    len(mixed_image_df)
)

# ------------------------------------------------------------
# 3. Image-level overall summary
#
# Each image contributes ONE value per feature.
# This avoids weighting dense images by their GT count.
# ------------------------------------------------------------

summary_rows = []

for feature in GEOM_FEATURES:

    f = within_image_geom_df[
        within_image_geom_df["feature"] == feature
    ].copy()

    diffs = (
        f["median_diff_missed_minus_covered"]
        .to_numpy()
    )

    aucs = (
        f["auc_missed_positive"]
        .to_numpy()
    )

    # Image-level directional proportions
    prop_diff_positive = float(
        np.mean(diffs > 0)
    )

    prop_diff_negative = float(
        np.mean(diffs < 0)
    )

    prop_auc_above_half = float(
        np.mean(aucs > 0.5)
    )

    # Wilcoxon signed-rank on IMAGE-LEVEL median differences.
    #
    # This is more appropriate than treating all 10,045 GT
    # points as independent observations.
    nonzero_diffs = diffs[
        diffs != 0
    ]

    if len(nonzero_diffs) > 0:
        wilcox = wilcoxon(
            nonzero_diffs,
            alternative="two-sided",
            zero_method="wilcox",
        )

        wilcox_stat = float(
            wilcox.statistic
        )

        wilcox_p = float(
            wilcox.pvalue
        )
    else:
        wilcox_stat = np.nan
        wilcox_p = np.nan

    summary_rows.append({
        "feature": feature,

        "num_mixed_images": len(f),

        "median_image_diff": float(
            np.median(diffs)
        ),

        "mean_image_diff": float(
            np.mean(diffs)
        ),

        "p25_image_diff": float(
            np.percentile(diffs, 25)
        ),

        "p75_image_diff": float(
            np.percentile(diffs, 75)
        ),

        "prop_images_diff_positive": (
            prop_diff_positive
        ),

        "prop_images_diff_negative": (
            prop_diff_negative
        ),

        "median_within_image_auc": float(
            np.median(aucs)
        ),

        "mean_within_image_auc": float(
            np.mean(aucs)
        ),

        "prop_images_auc_gt_0_5": (
            prop_auc_above_half
        ),

        "wilcoxon_statistic": (
            wilcox_stat
        ),

        "wilcoxon_p": (
            wilcox_p
        ),
    })

within_image_summary_df = pd.DataFrame(
    summary_rows
).sort_values(
    "median_within_image_auc",
    ascending=False,
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Density-stratified image-level summary
#
# Still diagnostic only.
# No density-based routing is introduced.
# ------------------------------------------------------------

density_rows = []

for (
    density_stratum,
    feature
), group_df in (
    within_image_geom_df
    .groupby(
        [
            "density_stratum",
            "feature",
        ],
        sort=True,
    )
):

    diffs = (
        group_df[
            "median_diff_missed_minus_covered"
        ]
        .to_numpy()
    )

    aucs = (
        group_df[
            "auc_missed_positive"
        ]
        .to_numpy()
    )

    density_rows.append({
        "density_stratum": density_stratum,
        "feature": feature,

        "num_images": int(
            len(group_df)
        ),

        "median_image_diff": float(
            np.median(diffs)
        ),

        "prop_images_diff_positive": float(
            np.mean(diffs > 0)
        ),

        "median_within_image_auc": float(
            np.median(aucs)
        ),

        "mean_within_image_auc": float(
            np.mean(aucs)
        ),

        "prop_images_auc_gt_0_5": float(
            np.mean(aucs > 0.5)
        ),
    })

within_image_density_df = pd.DataFrame(
    density_rows
)

# ------------------------------------------------------------
# 5. Mixed-image counts by density
# ------------------------------------------------------------

mixed_images_by_density_df = (
    mixed_image_df
    .groupby(
        "density_stratum",
        as_index=False,
    )
    .agg(
        num_mixed_images=(
            "dataset_index",
            "count",
        ),

        total_covered_gt=(
            "covered",
            "sum",
        ),

        total_missed_gt=(
            "missed",
            "sum",
        ),
    )
)

# ------------------------------------------------------------
# 6. Save outputs
# ------------------------------------------------------------

M1_B0_GT_WITHIN_IMAGE_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry_within_image.csv"
)

M1_B0_GT_WITHIN_IMAGE_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry_within_image_summary.csv"
)

M1_B0_GT_WITHIN_IMAGE_DENSITY_FILE = (
    M1_TABLES_DIR
    / "M1_B0_covered_vs_missed_GT_geometry_within_image_by_density.csv"
)

within_image_geom_df.to_csv(
    M1_B0_GT_WITHIN_IMAGE_FILE,
    index=False,
)

within_image_summary_df.to_csv(
    M1_B0_GT_WITHIN_IMAGE_SUMMARY_FILE,
    index=False,
)

within_image_density_df.to_csv(
    M1_B0_GT_WITHIN_IMAGE_DENSITY_FILE,
    index=False,
)

# ------------------------------------------------------------
# 7. Report
# ------------------------------------------------------------

print("\nMixed images by density")
print("-----------------------")
display(
    mixed_images_by_density_df
)

print("\nImage-balanced overall geometry comparison")
print("------------------------------------------")
display(
    within_image_summary_df[
        [
            "feature",
            "num_mixed_images",
            "median_image_diff",
            "prop_images_diff_positive",
            "median_within_image_auc",
            "mean_within_image_auc",
            "prop_images_auc_gt_0_5",
            "wilcoxon_p",
        ]
    ]
)

print("\nImage-balanced comparison by density")
print("------------------------------------")
display(
    within_image_density_df[
        [
            "density_stratum",
            "feature",
            "num_images",
            "median_image_diff",
            "prop_images_diff_positive",
            "median_within_image_auc",
            "mean_within_image_auc",
            "prop_images_auc_gt_0_5",
        ]
    ]
)

print("\nSaved within-image table:")
print(M1_B0_GT_WITHIN_IMAGE_FILE)

print("\nSaved overall summary:")
print(M1_B0_GT_WITHIN_IMAGE_SUMMARY_FILE)

print("\nSaved density summary:")
print(M1_B0_GT_WITHIN_IMAGE_DENSITY_FILE)

print("\nGT used for analysis       : YES")
print("GT used for inference      : NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6B.1: PASS")

STEP 6B.1 — WITHIN-IMAGE B0-COVERED VS B0-MISSED GEOMETRY
Total dev100 images          : 100
Images with covered GT       : 84
Images with missed GT        : 82
Images with BOTH             : 66

Mixed images by density
-----------------------


,density_stratum,num_mixed_images,total_covered_gt,total_missed_gt
0,S1_0_20,9,57,27
1,S2_20_40,15,116,130
2,S3_40_60,16,153,278
3,S4_60_80,12,215,393
4,S5_80_95,7,343,548
5,S6_95_100,7,2519,2281



Image-balanced overall geometry comparison
------------------------------------------


,feature,num_mixed_images,median_image_diff,prop_images_diff_positive,median_within_image_auc,mean_within_image_auc,prop_images_auc_gt_0_5,wilcoxon_p
0,local_depth_std_norm,66,0.002452,0.666667,0.635642,0.600135,0.666667,0.008571
1,point_gradient_mag_norm,66,0.000357,0.651515,0.609073,0.584112,0.712121,0.029620
2,centre_vs_annulus_abs_norm,66,0.001403,0.590909,0.562523,0.558631,0.621212,0.076280
3,point_vs_local_abs_norm,66,0.000025,0.515152,0.557458,0.541025,0.590909,0.738096



Image-balanced comparison by density
------------------------------------


,density_stratum,feature,num_images,median_image_diff,prop_images_diff_positive,median_within_image_auc,mean_within_image_auc,prop_images_auc_gt_0_5
0,S1_0_20,centre_vs_annulus_abs_norm,9,-0.000850,0.444444,0.450000,0.571689,0.444444
1,S1_0_20,local_depth_std_norm,9,0.001214,0.666667,0.700000,0.644080,0.666667
2,S1_0_20,point_gradient_mag_norm,9,0.000402,0.555556,0.727273,0.637598,0.666667
3,S1_0_20,point_vs_local_abs_norm,9,-0.000227,0.222222,0.400000,0.449274,0.333333
4,S2_20_40,centre_vs_annulus_abs_norm,15,0.001568,0.733333,0.571429,0.542194,0.666667
5,S2_20_40,local_depth_std_norm,15,0.000290,0.533333,0.500000,0.499805,0.466667
6,S2_20_40,point_gradient_mag_norm,15,0.000028,0.533333,0.510101,0.471337,0.533333
7,S2_20_40,point_vs_local_abs_norm,15,0.000033,0.533333,0.575758,0.515362,0.600000
8,S3_40_60,centre_vs_annulus_abs_norm,16,0.002521,0.625000,0.606127,0.605486,0.750000
9,S3_40_60,local_depth_std_norm,16,0.002452,0.687500,0.684921,0.648674,0.687500



Saved within-image table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_B0_covered_vs_missed_GT_geometry_within_image.csv

Saved overall summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_B0_covered_vs_missed_GT_geometry_within_image_summary.csv

Saved density summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_B0_covered_vs_missed_GT_geometry_within_image_by_density.csv

GT used for analysis       : YES
GT used for inference      : NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6B.1: PASS


### **M1 Finding 2:**

After controlling for scene-level differences using within-image comparisons, B0-missed objects still tend to exhibit stronger local Marigold depth variability and gradients than B0-covered objects.

The effect is moderate and scene-dependent, supporting geometry as a complementary cue rather than an independent counting signal.

In [ ]:
# ============================================================
# Step 6C — Marigold geometry of semantic recovery candidates
#           TP recovery vs non-TP recovery
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

from scipy.stats import mannwhitneyu
from sklearn.metrics import roc_auc_score

print("=" * 120)
print("STEP 6C — MARIGOLD GEOMETRY OF SEMANTIC RECOVERY CANDIDATES")
print("=" * 120)

# ------------------------------------------------------------
# 1. Frozen B2 Step-3B point-level matching table
# ------------------------------------------------------------

B2_ROOT = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "B2_B0_Anchored_Semantic_Validation"
)

B2_RECOVERY_GT_MATCH_FILE = (
    B2_ROOT
    / "tables"
    / "B0_recovery_GT_matching_long.csv"
)

assert B2_RECOVERY_GT_MATCH_FILE.exists(), (
    f"Missing frozen Step-3B table: {B2_RECOVERY_GT_MATCH_FILE}"
)

recovery_match_df = pd.read_csv(
    B2_RECOVERY_GT_MATCH_FILE
)

print("Loaded frozen Step-3B point-level table")
print("----------------------------------------")
print("Rows    :", len(recovery_match_df))
print("Columns :", list(recovery_match_df.columns))

# ------------------------------------------------------------
# 2. Resolve actual Step-3B schema
#
# We do NOT recreate matching here.
# We only identify the existing fields in the frozen table.
# ------------------------------------------------------------

def resolve_existing_column(df, candidates, role):
    for col in candidates:
        if col in df.columns:
            print(f"{role:<26}: {col}")
            return col

    raise RuntimeError(
        f"\nCould not resolve column for '{role}'.\n"
        f"Available columns:\n{list(df.columns)}"
    )


dataset_col = resolve_existing_column(
    recovery_match_df,
    ["dataset_index"],
    "dataset index",
)

image_col = resolve_existing_column(
    recovery_match_df,
    ["image_id"],
    "image id",
)

source_col = resolve_existing_column(
    recovery_match_df,
    [
        "candidate_id",
        "source",
        "recovery_source",
    ],
    "semantic source",
)

x_col = resolve_existing_column(
    recovery_match_df,
    [
        "x",
        "point_x",
        "candidate_x",
        "recovery_x",
        "pred_x",
    ],
    "candidate x",
)

y_col = resolve_existing_column(
    recovery_match_df,
    [
        "y",
        "point_y",
        "candidate_y",
        "recovery_y",
        "pred_y",
    ],
    "candidate y",
)

# ------------------------------------------------------------
# Resolve TP / non-TP label
#
# Prefer an explicit boolean/numeric TP column.
# Otherwise use a non-null matched-GT identifier.
# ------------------------------------------------------------

explicit_tp_candidates = [
    "is_tp",
    "tp",
    "credited_tp",
    "recovery_tp",
    "is_recovered",
]

tp_col = next(
    (
        c
        for c in explicit_tp_candidates
        if c in recovery_match_df.columns
    ),
    None,
)

if tp_col is not None:

    raw_tp = recovery_match_df[tp_col]

    if raw_tp.dtype == bool:
        recovery_match_df["recovery_is_tp"] = raw_tp

    else:
        recovery_match_df["recovery_is_tp"] = (
            pd.to_numeric(
                raw_tp,
                errors="coerce",
            )
            .fillna(0)
            .astype(int)
            .astype(bool)
        )

    print(f"{'TP label':<26}: {tp_col}")

else:

    matched_gt_candidates = [
        "matched_gt_index",
        "matched_gt_idx",
        "gt_match_index",
        "matched_missed_gt_index",
        "matched_gt_point_index",
    ]

    matched_gt_col = next(
        (
            c
            for c in matched_gt_candidates
            if c in recovery_match_df.columns
        ),
        None,
    )

    if matched_gt_col is None:
        raise RuntimeError(
            "\nCould not resolve TP/non-TP label from the frozen "
            "Step-3B table.\nAvailable columns:\n"
            f"{list(recovery_match_df.columns)}"
        )

    recovery_match_df["recovery_is_tp"] = (
        recovery_match_df[matched_gt_col]
        .notna()
    )

    print(
        f"{'TP label':<26}: "
        f"non-null {matched_gt_col}"
    )

# ------------------------------------------------------------
# 3. Restrict to the three frozen B2 recovery sources
# ------------------------------------------------------------

RECOVERY_SOURCES_6C = [
    "C1",
    "D6B-T-native",
    "D6B9-strict",
]

recovery_6c_df = recovery_match_df[
    recovery_match_df[source_col]
    .astype(str)
    .isin(RECOVERY_SOURCES_6C)
].copy()

recovery_6c_df[source_col] = (
    recovery_6c_df[source_col].astype(str)
)

# ------------------------------------------------------------
# 4. Reproduce frozen Step-3B counts EXACTLY
# ------------------------------------------------------------

frozen_expected = {
    "C1": {
        "candidates": 1215,
        "tp": 623,
    },
    "D6B-T-native": {
        "candidates": 8047,
        "tp": 1453,
    },
    "D6B9-strict": {
        "candidates": 1911,
        "tp": 550,
    },
}

identity_rows = []

for source in RECOVERY_SOURCES_6C:

    s = recovery_6c_df[
        recovery_6c_df[source_col] == source
    ]

    n_total = len(s)
    n_tp = int(
        s["recovery_is_tp"].sum()
    )
    n_non_tp = n_total - n_tp

    identity_rows.append({
        "candidate_id": source,
        "recovery_candidates": n_total,
        "tp_recovery": n_tp,
        "non_tp_recovery": n_non_tp,
    })

    assert (
        n_total
        ==
        frozen_expected[source]["candidates"]
    ), (
        f"{source}: expected "
        f"{frozen_expected[source]['candidates']} candidates, "
        f"got {n_total}"
    )

    assert (
        n_tp
        ==
        frozen_expected[source]["tp"]
    ), (
        f"{source}: expected "
        f"{frozen_expected[source]['tp']} TP, "
        f"got {n_tp}"
    )

recovery_identity_df = pd.DataFrame(
    identity_rows
)

print("\nFrozen Step-3B identity reproduction")
print("------------------------------------")
display(recovery_identity_df)

print("Step-3B recovery identity   : PASS")

# ------------------------------------------------------------
# 5. Join frozen Marigold cache metadata
# ------------------------------------------------------------

depth_meta_6c_df = pd.read_csv(
    M1_DEPTH_INDEX_FILE
)

depth_lookup = (
    depth_meta_6c_df
    .set_index("dataset_index")
)

assert depth_lookup.index.is_unique

# ------------------------------------------------------------
# 6. Compute Marigold geometry at each recovery candidate
#
# Reuse the SAME local_depth_features() function from Step 6A.
#
# The candidate radius comes from the frozen per-image C1
# table already loaded as c1_gt_df in Step 6A.
# ------------------------------------------------------------

assert "local_depth_features" in globals(), (
    "Step 6A function local_depth_features() is not available. "
    "Run Step 6A first."
)

radius_lookup = (
    c1_gt_df[
        [
            "dataset_index",
            "radius_px",
        ]
    ]
    .drop_duplicates(
        "dataset_index"
    )
    .set_index(
        "dataset_index"
    )["radius_px"]
    .to_dict()
)

geom_rows = []

# Cache one depth map at a time
for dataset_index, image_candidates in (
    recovery_6c_df
    .groupby(
        dataset_col,
        sort=False,
    )
):

    dataset_index = int(
        dataset_index
    )

    assert dataset_index in depth_lookup.index
    assert dataset_index in radius_lookup

    depth_row = depth_lookup.loc[
        dataset_index
    ]

    cache_path = Path(
        depth_row["cache_path"]
    )

    with np.load(cache_path) as data:
        depth = (
            np.asarray(
                data["depth"]
            )
            .astype(np.float32)
        )

    radius_px = float(
        radius_lookup[
            dataset_index
        ]
    )

    # Same robust image-level normalization used in 6A
    depth_p05 = float(
        np.percentile(
            depth,
            5,
        )
    )

    depth_p95 = float(
        np.percentile(
            depth,
            95,
        )
    )

    robust_depth_range = max(
        depth_p95 - depth_p05,
        1e-8,
    )

    for row_idx, row in (
        image_candidates.iterrows()
    ):

        x = float(
            row[x_col]
        )

        y = float(
            row[y_col]
        )

        feat = local_depth_features(
            depth=depth,
            x=x,
            y=y,
            radius_px=radius_px,
        )

        feat.update({
            "original_row_index": int(
                row_idx
            ),

            "dataset_index": dataset_index,

            "image_id": str(
                row[image_col]
            ),

            "candidate_id": str(
                row[source_col]
            ),

            "recovery_is_tp": bool(
                row["recovery_is_tp"]
            ),

            "recovery_label": (
                "TP"
                if bool(
                    row["recovery_is_tp"]
                )
                else
                "non-TP"
            ),

            "image_depth_p05": (
                depth_p05
            ),

            "image_depth_p95": (
                depth_p95
            ),

            "image_robust_depth_range": (
                robust_depth_range
            ),
        })

        # Normalized features
        feat[
            "point_vs_local_abs_norm"
        ] = (
            feat[
                "point_vs_local_abs"
            ]
            / robust_depth_range
        )

        feat[
            "centre_vs_annulus_abs_norm"
        ] = (
            feat[
                "centre_vs_annulus_abs"
            ]
            / robust_depth_range
            if np.isfinite(
                feat[
                    "centre_vs_annulus_abs"
                ]
            )
            else np.nan
        )

        feat[
            "local_depth_std_norm"
        ] = (
            feat[
                "local_depth_std"
            ]
            / robust_depth_range
        )

        feat[
            "point_gradient_mag_norm"
        ] = (
            feat[
                "point_gradient_mag"
            ]
            / robust_depth_range
        )

        geom_rows.append(
            feat
        )

recovery_geom_df = pd.DataFrame(
    geom_rows
)

assert (
    len(recovery_geom_df)
    ==
    len(recovery_6c_df)
)

# ------------------------------------------------------------
# 7. Attach density/category metadata from frozen dev100
# ------------------------------------------------------------

meta_cols = [
    "dataset_index",
    "category",
    "density_stratum",
    "gt_count",
]

recovery_geom_df = (
    recovery_geom_df
    .merge(
        dev100_df[
            meta_cols
        ],
        on="dataset_index",
        how="left",
        validate="many_to_one",
    )
)

# ------------------------------------------------------------
# 8. Geometry feature integrity
# ------------------------------------------------------------

GEOM_FEATURES_6C = [
    "point_vs_local_abs_norm",
    "centre_vs_annulus_abs_norm",
    "local_depth_std_norm",
    "point_gradient_mag_norm",
]

for feature in GEOM_FEATURES_6C:

    assert (
        recovery_geom_df[
            feature
        ]
        .notna()
        .all()
    ), (
        f"NaN found in {feature}"
    )

    assert np.isfinite(
        recovery_geom_df[
            feature
        ]
    ).all()

print("\nGeometry extraction")
print("-------------------")
print("Recovery candidates analysed :", len(recovery_geom_df))
print("Images represented            :", recovery_geom_df["dataset_index"].nunique())
print("Geometry features             :", GEOM_FEATURES_6C)

# ------------------------------------------------------------
# 9. TP-vs-non-TP discrimination BY SOURCE
#
# Positive class = TP recovery candidate.
# ------------------------------------------------------------

discrimination_rows = []

for source in RECOVERY_SOURCES_6C:

    source_df = recovery_geom_df[
        recovery_geom_df[
            "candidate_id"
        ] == source
    ]

    y_tp = (
        source_df[
            "recovery_is_tp"
        ]
        .astype(int)
        .to_numpy()
    )

    for feature in GEOM_FEATURES_6C:

        tp_vals = (
            source_df.loc[
                source_df[
                    "recovery_is_tp"
                ],
                feature,
            ]
            .to_numpy()
        )

        non_tp_vals = (
            source_df.loc[
                ~source_df[
                    "recovery_is_tp"
                ],
                feature,
            ]
            .to_numpy()
        )

        auc = float(
            roc_auc_score(
                y_tp,
                source_df[
                    feature
                ].to_numpy(),
            )
        )

        mw = mannwhitneyu(
            tp_vals,
            non_tp_vals,
            alternative="two-sided",
        )

        rank_biserial = (
            2.0
            * float(
                mw.statistic
            )
            / (
                len(tp_vals)
                *
                len(non_tp_vals)
            )
            - 1.0
        )

        discrimination_rows.append({
            "candidate_id": source,
            "feature": feature,

            "n_tp": len(tp_vals),
            "n_non_tp": len(
                non_tp_vals
            ),

            "tp_median": float(
                np.median(
                    tp_vals
                )
            ),

            "non_tp_median": float(
                np.median(
                    non_tp_vals
                )
            ),

            "median_difference_tp_minus_non_tp": (
                float(
                    np.median(
                        tp_vals
                    )
                )
                -
                float(
                    np.median(
                        non_tp_vals
                    )
                )
            ),

            "auc_tp_positive": auc,

            "separability_auc": float(
                max(
                    auc,
                    1.0 - auc,
                )
            ),

            "direction": (
                "higher_in_TP"
                if auc > 0.5
                else
                "higher_in_non_TP"
                if auc < 0.5
                else
                "none"
            ),

            "rank_biserial_TP_vs_nonTP": (
                rank_biserial
            ),

            "mannwhitney_p": float(
                mw.pvalue
            ),
        })

recovery_geom_discrimination_df = (
    pd.DataFrame(
        discrimination_rows
    )
    .sort_values(
        [
            "candidate_id",
            "separability_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)

# ------------------------------------------------------------
# 10. Density-stratified descriptive diagnostics
#
# Only compute AUC when BOTH TP and non-TP are present.
# Density is diagnostic only.
# ------------------------------------------------------------

density_rows = []

for (
    density_stratum,
    source,
), group_df in (
    recovery_geom_df
    .groupby(
        [
            "density_stratum",
            "candidate_id",
        ],
        sort=True,
    )
):

    for feature in GEOM_FEATURES_6C:

        tp = group_df[
            group_df[
                "recovery_is_tp"
            ]
        ][feature].to_numpy()

        non_tp = group_df[
            ~group_df[
                "recovery_is_tp"
            ]
        ][feature].to_numpy()

        if (
            len(tp) == 0
            or
            len(non_tp) == 0
        ):
            auc = np.nan
        else:
            auc = float(
                roc_auc_score(
                    group_df[
                        "recovery_is_tp"
                    ].astype(int),
                    group_df[
                        feature
                    ],
                )
            )

        density_rows.append({
            "density_stratum": (
                density_stratum
            ),

            "candidate_id": (
                source
            ),

            "feature": feature,

            "n_tp": int(
                len(tp)
            ),

            "n_non_tp": int(
                len(non_tp)
            ),

            "tp_median": (
                float(
                    np.median(tp)
                )
                if len(tp) > 0
                else np.nan
            ),

            "non_tp_median": (
                float(
                    np.median(
                        non_tp
                    )
                )
                if len(non_tp) > 0
                else np.nan
            ),

            "auc_tp_positive": (
                auc
            ),

            "separability_auc": (
                max(
                    auc,
                    1.0 - auc,
                )
                if np.isfinite(
                    auc
                )
                else np.nan
            ),
        })

recovery_geom_density_df = (
    pd.DataFrame(
        density_rows
    )
)

# ------------------------------------------------------------
# 11. Image-balanced diagnostic
#
# Avoid letting images with many semantic candidates dominate
# interpretation. Compute within-image AUC wherever an image
# has BOTH TP and non-TP for that source.
# ------------------------------------------------------------

within_rows = []

for (
    dataset_index,
    source,
), image_source_df in (
    recovery_geom_df
    .groupby(
        [
            "dataset_index",
            "candidate_id",
        ],
        sort=True,
    )
):

    n_tp = int(
        image_source_df[
            "recovery_is_tp"
        ].sum()
    )

    n_non_tp = int(
        (
            ~image_source_df[
                "recovery_is_tp"
            ]
        ).sum()
    )

    if (
        n_tp == 0
        or
        n_non_tp == 0
    ):
        continue

    y = (
        image_source_df[
            "recovery_is_tp"
        ]
        .astype(int)
        .to_numpy()
    )

    for feature in GEOM_FEATURES_6C:

        auc = float(
            roc_auc_score(
                y,
                image_source_df[
                    feature
                ].to_numpy(),
            )
        )

        within_rows.append({
            "dataset_index": int(
                dataset_index
            ),

            "image_id": str(
                image_source_df[
                    "image_id"
                ].iloc[0]
            ),

            "density_stratum": str(
                image_source_df[
                    "density_stratum"
                ].iloc[0]
            ),

            "candidate_id": (
                source
            ),

            "feature": feature,

            "n_tp": n_tp,
            "n_non_tp": n_non_tp,

            "within_image_auc": (
                auc
            ),
        })

recovery_geom_within_df = (
    pd.DataFrame(
        within_rows
    )
)

within_summary_df = (
    recovery_geom_within_df
    .groupby(
        [
            "candidate_id",
            "feature",
        ],
        as_index=False,
    )
    .agg(
        num_mixed_images=(
            "dataset_index",
            "count",
        ),

        median_within_image_auc=(
            "within_image_auc",
            "median",
        ),

        mean_within_image_auc=(
            "within_image_auc",
            "mean",
        ),

        prop_images_auc_gt_0_5=(
            "within_image_auc",
            lambda s: float(
                np.mean(
                    s > 0.5
                )
            ),
        ),
    )
)

# ------------------------------------------------------------
# 12. Save
# ------------------------------------------------------------

M1_RECOVERY_GEOM_FILE = (
    M1_TABLES_DIR
    / "M1_semantic_recovery_candidate_geometry.csv"
)

M1_RECOVERY_GEOM_DISC_FILE = (
    M1_TABLES_DIR
    / "M1_semantic_recovery_candidate_geometry_discrimination.csv"
)

M1_RECOVERY_GEOM_DENSITY_FILE = (
    M1_TABLES_DIR
    / "M1_semantic_recovery_candidate_geometry_by_density.csv"
)

M1_RECOVERY_GEOM_WITHIN_FILE = (
    M1_TABLES_DIR
    / "M1_semantic_recovery_candidate_geometry_within_image.csv"
)

M1_RECOVERY_GEOM_WITHIN_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_semantic_recovery_candidate_geometry_within_image_summary.csv"
)

recovery_geom_df.to_csv(
    M1_RECOVERY_GEOM_FILE,
    index=False,
)

recovery_geom_discrimination_df.to_csv(
    M1_RECOVERY_GEOM_DISC_FILE,
    index=False,
)

recovery_geom_density_df.to_csv(
    M1_RECOVERY_GEOM_DENSITY_FILE,
    index=False,
)

recovery_geom_within_df.to_csv(
    M1_RECOVERY_GEOM_WITHIN_FILE,
    index=False,
)

within_summary_df.to_csv(
    M1_RECOVERY_GEOM_WITHIN_SUMMARY_FILE,
    index=False,
)

# ------------------------------------------------------------
# 13. Report
# ------------------------------------------------------------

print("\nOverall TP vs non-TP geometry by semantic source")
print("------------------------------------------------")
display(
    recovery_geom_discrimination_df[
        [
            "candidate_id",
            "feature",
            "n_tp",
            "n_non_tp",
            "tp_median",
            "non_tp_median",
            "median_difference_tp_minus_non_tp",
            "auc_tp_positive",
            "separability_auc",
            "direction",
            "rank_biserial_TP_vs_nonTP",
        ]
    ]
)

print("\nImage-balanced within-image AUC")
print("--------------------------------")
display(
    within_summary_df[
        [
            "candidate_id",
            "feature",
            "num_mixed_images",
            "median_within_image_auc",
            "mean_within_image_auc",
            "prop_images_auc_gt_0_5",
        ]
    ]
    .sort_values(
        [
            "candidate_id",
            "median_within_image_auc",
        ],
        ascending=[
            True,
            False,
        ],
    )
)

print("\nDensity diagnostics:")
display(
    recovery_geom_density_df[
        [
            "density_stratum",
            "candidate_id",
            "feature",
            "n_tp",
            "n_non_tp",
            "tp_median",
            "non_tp_median",
            "auc_tp_positive",
            "separability_auc",
        ]
    ]
)

print("\nSaved point-level geometry:")
print(M1_RECOVERY_GEOM_FILE)

print("\nSaved discrimination table:")
print(M1_RECOVERY_GEOM_DISC_FILE)

print("\nSaved within-image summary:")
print(M1_RECOVERY_GEOM_WITHIN_SUMMARY_FILE)

print("\nGT used for analysis       : YES")
print("GT used for candidate label: YES — frozen Step-3B diagnostic only")
print("GT used for inference      : NO")
print("New matching performed     : NO")
print("Threshold tuning           : NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6C: PASS")

STEP 6C — MARIGOLD GEOMETRY OF SEMANTIC RECOVERY CANDIDATES
Loaded frozen Step-3B point-level table
----------------------------------------
Rows    : 11173
Columns : ['branch', 'candidate_id', 'dataset_index', 'image_id', 'category', 'density_stratum', 'gt_count', 'b0_missed_gt_count', 'matching_radius_px', 'point_index', 'x', 'y', 'matched_missed_gt', 'matched_gt_index', 'match_distance_px', 'unsupported_recovery_point']
dataset index             : dataset_index
image id                  : image_id
semantic source           : candidate_id
candidate x               : x
candidate y               : y
TP label                  : non-null matched_gt_index

Frozen Step-3B identity reproduction
------------------------------------


,candidate_id,recovery_candidates,tp_recovery,non_tp_recovery
0,C1,1215,623,592
1,D6B-T-native,8047,1453,6594
2,D6B9-strict,1911,550,1361


Step-3B recovery identity   : PASS

Geometry extraction
-------------------
Recovery candidates analysed : 11173
Images represented            : 95
Geometry features             : ['point_vs_local_abs_norm', 'centre_vs_annulus_abs_norm', 'local_depth_std_norm', 'point_gradient_mag_norm']

Overall TP vs non-TP geometry by semantic source
------------------------------------------------


,candidate_id,feature,n_tp,n_non_tp,tp_median,non_tp_median,median_difference_tp_minus_non_tp,auc_tp_positive,separability_auc,direction,rank_biserial_TP_vs_nonTP
0,C1,point_gradient_mag_norm,623,592,0.015187,0.004162,0.011025,0.694814,0.694814,higher_in_TP,0.389628
1,C1,point_vs_local_abs_norm,623,592,0.040283,0.005288,0.034994,0.693305,0.693305,higher_in_TP,0.386610
2,C1,local_depth_std_norm,623,592,0.096288,0.048152,0.048136,0.615036,0.615036,higher_in_TP,0.230071
3,C1,centre_vs_annulus_abs_norm,623,592,0.039904,0.020142,0.019762,0.604350,0.604350,higher_in_TP,0.208700
4,D6B-T-native,point_vs_local_abs_norm,1453,6594,0.006149,0.004173,0.001976,0.565704,0.565704,higher_in_TP,0.131408
5,D6B-T-native,point_gradient_mag_norm,1453,6594,0.003462,0.002795,0.000667,0.540578,0.540578,higher_in_TP,0.081157
6,D6B-T-native,local_depth_std_norm,1453,6594,0.040871,0.045808,-0.004937,0.485981,0.514019,higher_in_non_TP,-0.028038
7,D6B-T-native,centre_vs_annulus_abs_norm,1453,6594,0.022291,0.022776,-0.000486,0.494419,0.505581,higher_in_non_TP,-0.011162
8,D6B9-strict,point_vs_local_abs_norm,550,1361,0.005531,0.003242,0.002289,0.580421,0.580421,higher_in_TP,0.160842
9,D6B9-strict,centre_vs_annulus_abs_norm,550,1361,0.021669,0.015456,0.006213,0.556214,0.556214,higher_in_TP,0.112428



Image-balanced within-image AUC
--------------------------------


,candidate_id,feature,num_mixed_images,median_within_image_auc,mean_within_image_auc,prop_images_auc_gt_0_5
0,C1,centre_vs_annulus_abs_norm,53,0.500000,0.503955,0.471698
3,C1,point_vs_local_abs_norm,53,0.500000,0.519422,0.433962
2,C1,point_gradient_mag_norm,53,0.404628,0.438028,0.433962
1,C1,local_depth_std_norm,53,0.384615,0.413810,0.377358
4,D6B-T-native,centre_vs_annulus_abs_norm,71,0.475336,0.481948,0.422535
7,D6B-T-native,point_vs_local_abs_norm,71,0.433333,0.471409,0.408451
6,D6B-T-native,point_gradient_mag_norm,71,0.361538,0.420580,0.295775
5,D6B-T-native,local_depth_std_norm,71,0.352679,0.416996,0.352113
8,D6B9-strict,centre_vs_annulus_abs_norm,59,0.486538,0.496780,0.491525
11,D6B9-strict,point_vs_local_abs_norm,59,0.455319,0.492844,0.423729



Density diagnostics:


,density_stratum,candidate_id,feature,n_tp,n_non_tp,tp_median,non_tp_median,auc_tp_positive,separability_auc
0,S1_0_20,C1,point_vs_local_abs_norm,24,68,0.005765,0.006940,0.488358,0.511642
1,S1_0_20,C1,centre_vs_annulus_abs_norm,24,68,0.031266,0.022578,0.621936,0.621936
2,S1_0_20,C1,local_depth_std_norm,24,68,0.052128,0.046954,0.587623,0.587623
3,S1_0_20,C1,point_gradient_mag_norm,24,68,0.004900,0.004127,0.611520,0.611520
4,S1_0_20,D6B-T-native,point_vs_local_abs_norm,59,882,0.006377,0.002881,0.624303,0.624303
...,...,...,...,...,...,...,...,...,...
67,S6_95_100,D6B-T-native,point_gradient_mag_norm,525,763,0.006882,0.004293,0.531018,0.531018
68,S6_95_100,D6B9-strict,point_vs_local_abs_norm,86,99,0.005707,0.003647,0.608527,0.608527
69,S6_95_100,D6B9-strict,centre_vs_annulus_abs_norm,86,99,0.016310,0.012318,0.580456,0.580456
70,S6_95_100,D6B9-strict,local_depth_std_norm,86,99,0.035102,0.013564,0.625440,0.625440



Saved point-level geometry:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_semantic_recovery_candidate_geometry.csv

Saved discrimination table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_semantic_recovery_candidate_geometry_discrimination.csv

Saved within-image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_semantic_recovery_candidate_geometry_within_image_summary.csv

GT used for analysis       : YES
GT used for candidate label: YES — frozen Step-3B diagnostic only
GT used for inference      : NO
New matching performed     : NO
Threshold tuning           : NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6C: PASS


### **M1 Finding 3:**

Although Marigold geometry is associated with whether true objects are missed by B0, simple point-local depth features do not consistently discriminate true from non-TP semantic recovery candidates within the same image.

Stronger pooled discrimination, particularly for C1, is largely attributable to between-image variation.

Therefore, simple global geometric thresholding of semantic recovery points is not supported by the current evidence.

In [ ]:
# ============================================================
# Step 6D — Part A
# Semantic-region representation / artifact audit
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path

print("=" * 120)
print("STEP 6D — PART A: SEMANTIC-REGION REPRESENTATION AUDIT")
print("=" * 120)

# ------------------------------------------------------------
# 1. Existing Group-C table from Step 6A
# ------------------------------------------------------------

assert "c_all_df" in globals(), (
    "c_all_df is not available. Run Step 6A first."
)

assert "B1_C_FILE" in globals(), (
    "B1_C_FILE is not available. Run Step 6A first."
)

print("\nFrozen Group-C candidate table")
print("------------------------------")
print("File    :", B1_C_FILE)
print("Rows    :", len(c_all_df))
print("Columns :", len(c_all_df.columns))

print("\nAll columns:")
for i, col in enumerate(c_all_df.columns):
    print(f"  {i:02d}: {col}")

# ------------------------------------------------------------
# 2. Candidate identities and completeness
# ------------------------------------------------------------

assert "candidate_id" in c_all_df.columns

candidate_counts = (
    c_all_df["candidate_id"]
    .astype(str)
    .value_counts()
    .sort_index()
)

print("\nCandidate rows:")
display(
    candidate_counts
    .rename("rows")
    .to_frame()
)

# ------------------------------------------------------------
# 3. Search table schema for region/mask/spatial representations
# ------------------------------------------------------------

REGION_KEYWORDS = [
    "mask",
    "region",
    "component",
    "pixel",
    "seg",
    "semantic",
    "bbox",
    "box",
    "contour",
    "polygon",
    "area",
    "label",
    "map",
]

region_like_cols = [
    col
    for col in c_all_df.columns
    if any(
        key in col.lower()
        for key in REGION_KEYWORDS
    )
]

print("\nRegion/mask-like columns:")
if region_like_cols:
    for col in region_like_cols:
        print("  -", col)
else:
    print("  NONE")

# ------------------------------------------------------------
# 4. Inspect C1 specifically
# ------------------------------------------------------------

c1_audit_df = (
    c_all_df[
        c_all_df["candidate_id"].astype(str) == "C1"
    ]
    .copy()
)

assert len(c1_audit_df) == 100, (
    f"Expected 100 C1 rows, found {len(c1_audit_df)}"
)

print("\nC1 non-null / type audit:")
audit_rows = []

for col in c1_audit_df.columns:

    series = c1_audit_df[col]

    first_nonnull = (
        series.dropna().iloc[0]
        if series.notna().any()
        else None
    )

    sample_repr = repr(first_nonnull)

    # keep display manageable
    if len(sample_repr) > 180:
        sample_repr = (
            sample_repr[:177]
            + "..."
        )

    audit_rows.append({
        "column": col,
        "dtype": str(series.dtype),
        "non_null_rows": int(
            series.notna().sum()
        ),
        "unique_values": int(
            series.nunique(
                dropna=True
            )
        ),
        "example": sample_repr,
    })

c1_schema_audit_df = pd.DataFrame(
    audit_rows
)

display(
    c1_schema_audit_df
)

# ------------------------------------------------------------
# 5. Look for path-like columns and test whether referenced
#    artifacts actually exist
# ------------------------------------------------------------

path_like_cols = [
    col
    for col in c_all_df.columns
    if (
        "path" in col.lower()
        or
        "file" in col.lower()
        or
        "cache" in col.lower()
    )
]

print("\nPath/file/cache-like columns:")
if path_like_cols:

    for col in path_like_cols:

        vals = (
            c1_audit_df[col]
            .dropna()
            .astype(str)
        )

        print(f"\n{col}:")
        print(
            "  non-null rows :",
            len(vals),
        )

        if len(vals) > 0:

            example = vals.iloc[0]

            print(
                "  example       :",
                example,
            )

            p = Path(example)

            print(
                "  example exists:",
                p.exists()
                if p.is_absolute()
                else "relative/not directly checked",
            )

else:
    print("  NONE")

# ------------------------------------------------------------
# 6. Audit files surrounding the frozen Group-C table
# ------------------------------------------------------------

GROUP_C_DIR = Path(
    B1_C_FILE
).parent

print("\nGroup-C directory:")
print(GROUP_C_DIR)

assert GROUP_C_DIR.exists()

all_files = [
    p
    for p in GROUP_C_DIR.rglob("*")
    if p.is_file()
]

print(
    "Files under Group-C directory:",
    len(all_files),
)

artifact_rows = []

for p in all_files:

    name_lower = p.name.lower()

    likely_region_artifact = any(
        key in name_lower
        for key in [
            "mask",
            "region",
            "semantic",
            "component",
            "seg",
            "cache",
        ]
    )

    artifact_rows.append({
        "relative_path": str(
            p.relative_to(
                GROUP_C_DIR
            )
        ),
        "suffix": p.suffix.lower(),
        "size_kib": round(
            p.stat().st_size / 1024,
            2,
        ),
        "region_related_name": (
            likely_region_artifact
        ),
    })

group_c_artifacts_df = (
    pd.DataFrame(
        artifact_rows
    )
    .sort_values(
        [
            "region_related_name",
            "relative_path",
        ],
        ascending=[
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)

print("\nFiles with region/mask-related names:")
region_artifact_display_df = (
    group_c_artifacts_df[
        group_c_artifacts_df[
            "region_related_name"
        ]
    ]
)

if len(region_artifact_display_df) > 0:
    display(
        region_artifact_display_df
    )
else:
    print("  NONE FOUND")

print("\nAll file types:")
display(
    group_c_artifacts_df[
        "suffix"
    ]
    .value_counts()
    .rename("num_files")
    .to_frame()
)

# ------------------------------------------------------------
# 7. Save audit table
# ------------------------------------------------------------

M1_6D_REGION_AUDIT_FILE = (
    M1_TABLES_DIR
    / "M1_6D_semantic_region_representation_audit.csv"
)

c1_schema_audit_df.to_csv(
    M1_6D_REGION_AUDIT_FILE,
    index=False,
)

print("\nSaved schema audit:")
print(M1_6D_REGION_AUDIT_FILE)

print("\nGT used for analysis       : NO")
print("Marigold used              : NO")
print("Semantic masks modified    : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6D Part A: PASS")

STEP 6D — PART A: SEMANTIC-REGION REPRESENTATION AUDIT

Frozen Group-C candidate table
------------------------------
File    : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_C/B1_Group_C_E_candidates_per_image.csv
Rows    : 300
Columns : 27

All columns:
  00: candidate_id
  01: experiment_id
  02: candidate_family
  03: description
  04: dataset_index
  05: image_id
  06: category
  07: density_stratum
  08: width
  09: height
  10: a3c_percentile
  11: point_tolerance_frac
  12: radius_px
  13: gt_count
  14: pred_count
  15: signed_error
  16: abs_error
  17: squared_error
  18: pred_gt_ratio
  19: tp
  20: fp
  21: fn
  22: precision
  23: recall
  24: f1
  25: gt_points_xy
  26: pred_points_xy

Candidate rows:


,rows
candidate_id,
C1,100
C2b,100
C2c,100



Region/mask-like columns:
  NONE

C1 non-null / type audit:


,column,dtype,non_null_rows,unique_values,example
0,candidate_id,object,100,1,'C1'
1,experiment_id,object,100,1,'B1.C.1'
2,candidate_family,object,100,1,'region_based_semantic'
3,description,object,100,1,'4-connected components'
4,dataset_index,int64,100,100,np.int64(576)
5,image_id,object,100,100,'3873.jpg'
6,category,object,100,21,'shallots'
7,density_stratum,object,100,6,'S1_0_20'
8,width,int64,100,45,np.int64(384)
9,height,int64,100,1,np.int64(384)



Path/file/cache-like columns:
  NONE

Group-C directory:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/Group_C
Files under Group-C directory: 11

Files with region/mask-related names:
  NONE FOUND

All file types:


,num_files
suffix,
.csv,11



Saved schema audit:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_semantic_region_representation_audit.csv

GT used for analysis       : NO
Marigold used              : NO
Semantic masks modified    : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6D Part A: PASS


In [ ]:
# ============================================================
# Step 6D — Part B
# Load + validate frozen A3c semantic-mask artifacts in M1
# ============================================================
#
# Purpose
# -------
# Bring the newly cached A3c binary semantic masks from the
# rerun ABC notebook into M1 and establish that they align
# exactly with:
#
#   1. frozen FSC-147 dev100 identities;
#   2. frozen Marigold depth-cache identities/shapes;
#   3. frozen C1 per-image point/count outputs;
#   4. the ABC A3c reproduction-verification record.
#
# IMPORTANT
# ---------
# This step performs NO Marigold refinement.
# It only establishes an authoritative semantic-mask source
# for subsequent region-level semantic-geometric experiments.
# ============================================================

import ast
import numpy as np
import pandas as pd
from pathlib import Path


print("=" * 120)
print("STEP 6D — PART B: LOAD + VALIDATE FROZEN A3c SEMANTIC MASKS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Locate frozen A3c artifacts created by ABC Steps 8L.1–8L.3
# ------------------------------------------------------------

ABC_ARTIFACT_ROOT = (
    PROJECT_ROOT
    / "results"
    / "B1_experiments"
    / "ABC_artifacts"
)

A3C_MASK_CACHE_DIR = (
    ABC_ARTIFACT_ROOT
    / "A3c_masks"
)

A3C_METADATA_DIR = (
    ABC_ARTIFACT_ROOT
    / "metadata"
)

A3C_MASK_INDEX_FILE = (
    A3C_METADATA_DIR
    / "A3c_mask_cache_index.csv"
)

A3C_REPRO_FILE = (
    A3C_METADATA_DIR
    / "A3c_mask_reproduction_verification.csv"
)


assert ABC_ARTIFACT_ROOT.exists(), (
    f"Missing ABC artifact root: {ABC_ARTIFACT_ROOT}"
)

assert A3C_MASK_INDEX_FILE.exists(), (
    f"Missing A3c mask index: {A3C_MASK_INDEX_FILE}"
)

assert A3C_REPRO_FILE.exists(), (
    f"Missing A3c reproduction verification: {A3C_REPRO_FILE}"
)


print("\nFrozen A3c artifacts")
print("--------------------")
print("Artifact root       :", ABC_ARTIFACT_ROOT)
print("Mask cache          :", A3C_MASK_CACHE_DIR)
print("Mask index          :", A3C_MASK_INDEX_FILE)
print("Reproduction record :", A3C_REPRO_FILE)


# ------------------------------------------------------------
# 2. Load authoritative A3c index + ABC reproduction record
# ------------------------------------------------------------

a3c_index_6d_df = pd.read_csv(
    A3C_MASK_INDEX_FILE
)

a3c_repro_6d_df = pd.read_csv(
    A3C_REPRO_FILE
)


assert len(a3c_index_6d_df) == 100, (
    f"Expected 100 A3c index rows, found {len(a3c_index_6d_df)}"
)

assert (
    a3c_index_6d_df["dataset_index"].nunique()
    == 100
), "A3c index dataset_index is not unique."

assert len(a3c_repro_6d_df) == 100, (
    f"Expected 100 A3c reproduction rows, found {len(a3c_repro_6d_df)}"
)

assert (
    a3c_repro_6d_df["dataset_index"].nunique()
    == 100
), "A3c reproduction dataset_index is not unique."

assert "all_match" in a3c_repro_6d_df.columns

# CSV may reload booleans either as bool or text.
if a3c_repro_6d_df["all_match"].dtype == bool:
    repro_all_match = a3c_repro_6d_df["all_match"]
else:
    repro_all_match = (
        a3c_repro_6d_df["all_match"]
        .astype(str)
        .str.lower()
        .map({
            "true": True,
            "false": False,
        })
    )

assert repro_all_match.notna().all(), (
    "Could not interpret all_match in A3c reproduction table."
)

assert repro_all_match.all(), (
    "ABC A3c reproduction verification is not 100/100 PASS."
)


print("\nABC reproduction verification")
print("-----------------------------")
print("Rows                  :", len(a3c_repro_6d_df))
print("All-match images      :", int(repro_all_match.sum()), "/ 100")
print("ABC reproduction      : PASS")


# ------------------------------------------------------------
# 3. Load frozen M1 Marigold depth index
# ------------------------------------------------------------

assert "M1_DEPTH_INDEX_FILE" in globals(), (
    "M1_DEPTH_INDEX_FILE is unavailable. Run Step 5B first."
)

assert M1_DEPTH_INDEX_FILE.exists(), (
    f"Missing Marigold depth index: {M1_DEPTH_INDEX_FILE}"
)

depth_index_6d_df = pd.read_csv(
    M1_DEPTH_INDEX_FILE
)

assert len(depth_index_6d_df) == 100
assert depth_index_6d_df["dataset_index"].nunique() == 100


# ------------------------------------------------------------
# 4. Obtain frozen C1 rows already loaded in Step 6A
# ------------------------------------------------------------

assert "c_all_df" in globals(), (
    "c_all_df unavailable. Run Step 6A first."
)

required_c_cols = {
    "dataset_index",
    "image_id",
    "candidate_id",
    "pred_count",
    "pred_points_xy",
}

missing_c_cols = (
    required_c_cols
    - set(c_all_df.columns)
)

assert not missing_c_cols, (
    f"Frozen Group-C table missing columns: "
    f"{sorted(missing_c_cols)}"
)


c1_6d_df = (
    c_all_df[
        c_all_df["candidate_id"]
        .astype(str)
        .eq("C1")
    ]
    .copy()
)

c1_6d_df = (
    c1_6d_df[
        c1_6d_df["dataset_index"].isin(
            dev100_df["dataset_index"]
        )
    ]
    .copy()
)

assert len(c1_6d_df) == 100
assert c1_6d_df["dataset_index"].nunique() == 100


# ------------------------------------------------------------
# 5. Parse frozen C1 point lists
# ------------------------------------------------------------

def parse_xy_list_6d(value):

    if isinstance(value, list):
        parsed = value

    elif pd.isna(value):
        return []

    else:
        parsed = ast.literal_eval(
            str(value)
        )

    points = []

    for p in parsed:

        if (
            p is None
            or len(p) < 2
        ):
            continue

        points.append(
            (
                float(p[0]),
                float(p[1]),
            )
        )

    return points


c1_6d_df["pred_points_parsed"] = (
    c1_6d_df["pred_points_xy"]
    .apply(parse_xy_list_6d)
)

c1_6d_df["parsed_point_count"] = (
    c1_6d_df["pred_points_parsed"]
    .apply(len)
)

assert (
    c1_6d_df["parsed_point_count"]
    ==
    c1_6d_df["pred_count"]
).all(), (
    "Frozen C1 pred_points_xy count does not match pred_count."
)


# ------------------------------------------------------------
# 6. Build one authoritative aligned table
# ------------------------------------------------------------

dev_cols = [
    "dataset_index",
    "image_id",
    "category",
    "density_stratum",
    "gt_count",
]

a3c_cols = [
    "dataset_index",
    "image_id",
    "category",
    "a3c_percentile",
    "a3c_threshold",
    "height",
    "width",
    "foreground_pixels",
    "foreground_fraction",
    "num_components",
    "num_reference_points",
    "cache_file",
]

depth_cols = [
    "dataset_index",
    "image_id",
    "height",
    "width",
    "depth_shape_h",
    "depth_shape_w",
    "cache_path",
]

c1_cols = [
    "dataset_index",
    "image_id",
    "pred_count",
    "parsed_point_count",
]


a3c_align_6d_df = (
    dev100_df[
        dev_cols
    ]
    .merge(
        a3c_index_6d_df[
            a3c_cols
        ],
        on=[
            "dataset_index",
            "image_id",
            "category",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=("", "_a3c"),
    )
    .merge(
        depth_index_6d_df[
            depth_cols
        ],
        on=[
            "dataset_index",
            "image_id",
        ],
        how="inner",
        validate="one_to_one",
        suffixes=(
            "_a3c",
            "_depth",
        ),
    )
    .merge(
        c1_6d_df[
            c1_cols
        ],
        on=[
            "dataset_index",
            "image_id",
        ],
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert len(a3c_align_6d_df) == 100, (
    f"Expected 100 fully aligned rows, "
    f"found {len(a3c_align_6d_df)}"
)


# ------------------------------------------------------------
# 7. Validate each saved A3c mask + depth map
# ------------------------------------------------------------

validation_rows = []


for _, row in a3c_align_6d_df.iterrows():

    dataset_index = int(
        row["dataset_index"]
    )

    image_id = str(
        row["image_id"]
    )

    a3c_path = Path(
        row["cache_file"]
    )

    depth_path = Path(
        row["cache_path"]
    )


    # --------------------------------------------------------
    # Existence
    # --------------------------------------------------------

    a3c_exists = a3c_path.exists()
    depth_exists = depth_path.exists()

    assert a3c_exists, (
        f"Missing A3c mask: {a3c_path}"
    )

    assert depth_exists, (
        f"Missing depth cache: {depth_path}"
    )


    # --------------------------------------------------------
    # Load A3c mask
    # --------------------------------------------------------

    with np.load(
        a3c_path,
        allow_pickle=False,
    ) as data:

        assert "mask" in data.files

        a3c_mask = (
            np.asarray(
                data["mask"]
            )
            .astype(bool)
        )

        stored_percentile = float(
            data["percentile"]
        )


    # --------------------------------------------------------
    # Load Marigold depth
    # --------------------------------------------------------

    with np.load(
        depth_path,
        allow_pickle=False,
    ) as data:

        assert "depth" in data.files

        depth = (
            np.asarray(
                data["depth"]
            )
            .astype(np.float32)
        )


    # --------------------------------------------------------
    # Basic structural checks
    # --------------------------------------------------------

    expected_a3c_shape = (
        int(
            row[
                "height_a3c"
            ]
        ),
        int(
            row[
                "width_a3c"
            ]
        ),
    )

    expected_depth_shape = (
        int(
            row[
                "depth_shape_h"
            ]
        ),
        int(
            row[
                "depth_shape_w"
            ]
        ),
    )


    mask_shape_match = (
        a3c_mask.shape
        ==
        expected_a3c_shape
    )

    depth_shape_match = (
        depth.shape
        ==
        expected_depth_shape
    )

    mask_depth_shape_match = (
        a3c_mask.shape
        ==
        depth.shape
    )

    mask_finite = bool(
        np.isfinite(
            a3c_mask.astype(
                np.float32
            )
        ).all()
    )

    depth_finite = bool(
        np.isfinite(
            depth
        ).all()
    )


    # --------------------------------------------------------
    # A3c metadata checks
    # --------------------------------------------------------

    foreground_pixels_match = (
        int(
            a3c_mask.sum()
        )
        ==
        int(
            row[
                "foreground_pixels"
            ]
        )
    )

    foreground_fraction_match = bool(
        np.isclose(
            float(
                a3c_mask.mean()
            ),
            float(
                row[
                    "foreground_fraction"
                ]
            ),
            rtol=0.0,
            atol=1e-12,
        )
    )

    percentile_match = bool(
        np.isclose(
            stored_percentile,
            90.0,
            rtol=0.0,
            atol=0.0,
        )
        and
        np.isclose(
            float(
                row[
                    "a3c_percentile"
                ]
            ),
            90.0,
            rtol=0.0,
            atol=0.0,
        )
    )


    # --------------------------------------------------------
    # C1 count identity
    #
    # A3c index num_components was produced using the frozen
    # connected-component procedure in the ABC notebook.
    # C1 is the 4-connected-component route.
    # --------------------------------------------------------

    c1_component_count_match = (
        int(
            row[
                "num_components"
            ]
        )
        ==
        int(
            row[
                "pred_count"
            ]
        )
    )

    c1_point_count_match = (
        int(
            row[
                "num_reference_points"
            ]
        )
        ==
        int(
            row[
                "parsed_point_count"
            ]
        )
    )


    all_match = all(
        [
            a3c_exists,
            depth_exists,
            mask_shape_match,
            depth_shape_match,
            mask_depth_shape_match,
            mask_finite,
            depth_finite,
            foreground_pixels_match,
            foreground_fraction_match,
            percentile_match,
            c1_component_count_match,
            c1_point_count_match,
        ]
    )


    validation_rows.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "mask_shape_match":
                mask_shape_match,

            "depth_shape_match":
                depth_shape_match,

            "mask_depth_shape_match":
                mask_depth_shape_match,

            "mask_finite":
                mask_finite,

            "depth_finite":
                depth_finite,

            "foreground_pixels_match":
                foreground_pixels_match,

            "foreground_fraction_match":
                foreground_fraction_match,

            "percentile_match":
                percentile_match,

            "c1_component_count_match":
                c1_component_count_match,

            "c1_point_count_match":
                c1_point_count_match,

            "all_match":
                all_match,
        }
    )


a3c_validation_6d_df = pd.DataFrame(
    validation_rows
)


# ------------------------------------------------------------
# 8. Summaries / frozen identities
# ------------------------------------------------------------

print("\nAlignment summary")
print("-----------------")
print(
    "Fully aligned images       :",
    len(a3c_align_6d_df),
)

print(
    "A3c foreground pixels      :",
    int(
        a3c_align_6d_df[
            "foreground_pixels"
        ].sum()
    ),
)

print(
    "A3c connected components   :",
    int(
        a3c_align_6d_df[
            "num_components"
        ].sum()
    ),
)

print(
    "Frozen C1 predicted points :",
    int(
        a3c_align_6d_df[
            "pred_count"
        ].sum()
    ),
)


# Frozen identity expected from the rerun ABC notebook
assert (
    int(
        a3c_align_6d_df[
            "num_components"
        ].sum()
    )
    == 2554
), (
    "Expected frozen A3c/C1 total of 2554 components."
)

assert (
    int(
        a3c_align_6d_df[
            "pred_count"
        ].sum()
    )
    == 2554
), (
    "Expected frozen C1 total of 2554 points."
)


# ------------------------------------------------------------
# 9. Check table
# ------------------------------------------------------------

check_cols = [
    "mask_shape_match",
    "depth_shape_match",
    "mask_depth_shape_match",
    "mask_finite",
    "depth_finite",
    "foreground_pixels_match",
    "foreground_fraction_match",
    "percentile_match",
    "c1_component_count_match",
    "c1_point_count_match",
    "all_match",
]

check_summary_rows = []

for col in check_cols:

    check_summary_rows.append(
        {
            "check":
                col,

            "passed_images":
                int(
                    a3c_validation_6d_df[
                        col
                    ].sum()
                ),

            "total_images":
                len(
                    a3c_validation_6d_df
                ),

            "passed":
                bool(
                    a3c_validation_6d_df[
                        col
                    ].all()
                ),
        }
    )


a3c_validation_summary_6d_df = (
    pd.DataFrame(
        check_summary_rows
    )
)

display(
    a3c_validation_summary_6d_df
)


# ------------------------------------------------------------
# 10. Save M1 alignment records
# ------------------------------------------------------------

M1_6D_A3C_ALIGNMENT_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_semantic_mask_alignment.csv"
)

M1_6D_A3C_VALIDATION_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_semantic_mask_validation.csv"
)

a3c_align_6d_df.to_csv(
    M1_6D_A3C_ALIGNMENT_FILE,
    index=False,
)

a3c_validation_6d_df.to_csv(
    M1_6D_A3C_VALIDATION_FILE,
    index=False,
)


# ------------------------------------------------------------
# 11. Hard freeze assertion
# ------------------------------------------------------------

failed_6d_df = (
    a3c_validation_6d_df[
        ~a3c_validation_6d_df[
            "all_match"
        ]
    ]
)

if len(failed_6d_df) > 0:

    print("\nFAILED IMAGE(S):")

    display(
        failed_6d_df
    )


assert (
    a3c_validation_6d_df[
        "all_match"
    ]
    .all()
), (
    "Step 6D Part B validation FAILED. "
    "Inspect failed_6d_df before proceeding."
)


# ------------------------------------------------------------
# 12. Final protocol report
# ------------------------------------------------------------

print("\nSaved alignment table:")
print(M1_6D_A3C_ALIGNMENT_FILE)

print("\nSaved validation table:")
print(M1_6D_A3C_VALIDATION_FILE)

print("\nABC A3c reproduction       : PASS")
print("M1 A3c/depth alignment     : PASS")
print("A3c -> frozen C1 count     : PASS")

print("\nGT used for analysis       : NO")
print("Marigold depth modified    : NO")
print("Semantic masks modified    : NO")
print("Threshold tuning           : NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6D Part B: PASS")

STEP 6D — PART B: LOAD + VALIDATE FROZEN A3c SEMANTIC MASKS

Frozen A3c artifacts
--------------------
Artifact root       : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/ABC_artifacts
Mask cache          : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/ABC_artifacts/A3c_masks
Mask index          : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/ABC_artifacts/metadata/A3c_mask_cache_index.csv
Reproduction record : /content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/B1_experiments/ABC_artifacts/metadata/A3c_mask_reproduction_verification.csv

ABC reproduction verification
-----------------------------
Rows                  : 100
All-match images      : 100 / 100
ABC reproduction      : PASS

Alignment summary
-----------------
Fully aligned images       : 100
A3c foreground pixels      : 2105914
A3c connected components   : 2554
Frozen C1 predicted points : 2554


,check,passed_images,total_images,passed
0,mask_shape_match,100,100,True
1,depth_shape_match,100,100,True
2,mask_depth_shape_match,100,100,True
3,mask_finite,100,100,True
4,depth_finite,100,100,True
5,foreground_pixels_match,100,100,True
6,foreground_fraction_match,100,100,True
7,percentile_match,100,100,True
8,c1_component_count_match,100,100,True
9,c1_point_count_match,100,100,True



Saved alignment table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_A3c_semantic_mask_alignment.csv

Saved validation table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_A3c_semantic_mask_validation.csv

ABC A3c reproduction       : PASS
M1 A3c/depth alignment     : PASS
A3c -> frozen C1 count     : PASS

GT used for analysis       : NO
Marigold depth modified    : NO
Semantic masks modified    : NO
Threshold tuning           : NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6D Part B: PASS


In [ ]:
# ============================================================
# Step 6D — Part C
# Region-level Marigold geometry of frozen A3c components
# ============================================================
#
# Purpose
# -------
# Characterise Marigold geometry at the level of the frozen
# A3c connected components, WITHOUT modifying the masks.
#
# For every A3c component:
#
#   - reproduce the original 4-connected C1 component
#   - compute component-level Marigold depth statistics
#   - measure internal depth heterogeneity
#   - measure component-vs-local-surrounding depth contrast
#   - count how many GT points fall inside the component
#     (diagnostic only)
#
# Main diagnostic questions:
#
#   1. Do components containing multiple GT objects have
#      stronger depth heterogeneity than single-GT components?
#
#   2. Do zero-GT semantic components have different geometry
#      from GT-supported components?
#
# IMPORTANT
# ---------
#   - no semantic mask modification
#   - no geometric threshold selection
#   - no density routing
#   - no count-rule modification
#   - GT is diagnostic only
#   - test set remains untouched
# ============================================================

import ast
import numpy as np
import pandas as pd

from pathlib import Path
from scipy import ndimage
from sklearn.metrics import roc_auc_score


print("=" * 120)
print("STEP 6D — PART C: REGION-LEVEL MARIGOLD GEOMETRY OF A3c COMPONENTS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "a3c_align_6d_df" in globals(), (
    "a3c_align_6d_df unavailable. Run Step 6D Part B first."
)

assert len(a3c_align_6d_df) == 100

assert "c1_gt_df" in globals(), (
    "c1_gt_df unavailable. Run Step 6A first."
)

assert len(c1_gt_df) == 100


# ------------------------------------------------------------
# 2. Authoritative GT-point lookup
#
# GT is used ONLY to diagnose component occupancy:
#
#   0 GT points  -> unsupported semantic component
#   1 GT point   -> single-object-supported component
#   2+ GT points -> potentially merged semantic component
#
# No GT information will be used for inference.
# ------------------------------------------------------------

def parse_gt_points_6dc(value):

    if isinstance(value, list):
        parsed = value

    elif pd.isna(value):
        return []

    else:
        parsed = ast.literal_eval(
            str(value)
        )

    points = []

    for p in parsed:

        if (
            p is None
            or len(p) < 2
        ):
            continue

        points.append(
            (
                float(p[0]),
                float(p[1]),
            )
        )

    return points


gt_lookup_6dc = {}

for _, row in c1_gt_df.iterrows():

    dataset_index = int(
        row["dataset_index"]
    )

    if "gt_points_parsed" in c1_gt_df.columns:

        points = row[
            "gt_points_parsed"
        ]

    else:

        points = parse_gt_points_6dc(
            row["gt_points_xy"]
        )

    gt_lookup_6dc[
        dataset_index
    ] = points


assert len(gt_lookup_6dc) == 100


# ------------------------------------------------------------
# 3. Component-level geometry helper
# ------------------------------------------------------------

def component_depth_features_6dc(
    component_mask,
    depth,
    image_robust_depth_range,
):
    """
    Compute geometry descriptors for one A3c component.

    All depth variability measures are normalized by the
    within-image P95-P05 Marigold depth range.

    Surrounding depth uses a one-pixel 4-connected dilation
    ring. This is descriptive only; no threshold is applied.
    """

    component_mask = np.asarray(
        component_mask,
        dtype=bool,
    )

    depth = np.asarray(
        depth,
        dtype=np.float32,
    )

    ys, xs = np.where(
        component_mask
    )

    assert len(xs) > 0

    area = int(
        component_mask.sum()
    )

    centroid_x = float(
        xs.mean()
    )

    centroid_y = float(
        ys.mean()
    )

    bbox_xmin = int(
        xs.min()
    )

    bbox_xmax = int(
        xs.max()
    )

    bbox_ymin = int(
        ys.min()
    )

    bbox_ymax = int(
        ys.max()
    )


    # --------------------------------------------------------
    # Component depth distribution
    # --------------------------------------------------------

    component_depth = depth[
        component_mask
    ]

    depth_mean = float(
        np.mean(
            component_depth
        )
    )

    depth_median = float(
        np.median(
            component_depth
        )
    )

    depth_std = float(
        np.std(
            component_depth
        )
    )

    depth_p05 = float(
        np.percentile(
            component_depth,
            5
        )
    )

    depth_p95 = float(
        np.percentile(
            component_depth,
            95
        )
    )

    depth_robust_range = (
        depth_p95
        -
        depth_p05
    )


    # --------------------------------------------------------
    # Boundary / interior geometry
    #
    # Use the same default 2-D 4-connected structure as the
    # original scipy.ndimage.label() C1 implementation.
    # --------------------------------------------------------

    structure_4 = (
        ndimage.generate_binary_structure(
            rank=2,
            connectivity=1,
        )
    )

    eroded = ndimage.binary_erosion(
        component_mask,
        structure=structure_4,
        iterations=1,
        border_value=0,
    )

    boundary_mask = (
        component_mask
        &
        ~eroded
    )

    interior_mask = eroded


    boundary_depth = depth[
        boundary_mask
    ]

    interior_depth = depth[
        interior_mask
    ]


    boundary_median = (
        float(
            np.median(
                boundary_depth
            )
        )
        if boundary_depth.size > 0
        else np.nan
    )

    interior_median = (
        float(
            np.median(
                interior_depth
            )
        )
        if interior_depth.size > 0
        else np.nan
    )

    boundary_vs_interior_abs = (
        abs(
            boundary_median
            -
            interior_median
        )
        if (
            np.isfinite(
                boundary_median
            )
            and
            np.isfinite(
                interior_median
            )
        )
        else np.nan
    )


    # --------------------------------------------------------
    # Immediate 4-connected surrounding ring
    # --------------------------------------------------------

    dilated = ndimage.binary_dilation(
        component_mask,
        structure=structure_4,
        iterations=1,
    )

    ring_mask = (
        dilated
        &
        ~component_mask
    )

    ring_depth = depth[
        ring_mask
    ]

    ring_median = (
        float(
            np.median(
                ring_depth
            )
        )
        if ring_depth.size > 0
        else np.nan
    )

    component_vs_ring_abs = (
        abs(
            depth_median
            -
            ring_median
        )
        if np.isfinite(
            ring_median
        )
        else np.nan
    )


    # --------------------------------------------------------
    # Mean Marigold gradient magnitude inside component
    # --------------------------------------------------------

    gy, gx = np.gradient(
        depth.astype(
            np.float32
        )
    )

    gradient_mag = np.sqrt(
        gx ** 2
        +
        gy ** 2
    )

    mean_gradient = float(
        np.mean(
            gradient_mag[
                component_mask
            ]
        )
    )


    # --------------------------------------------------------
    # Normalize by within-image robust depth range
    # --------------------------------------------------------

    scale = max(
        float(
            image_robust_depth_range
        ),
        1e-8,
    )

    return {
        "area_pixels":
            area,

        "centroid_x":
            centroid_x,

        "centroid_y":
            centroid_y,

        "bbox_xmin":
            bbox_xmin,

        "bbox_ymin":
            bbox_ymin,

        "bbox_xmax":
            bbox_xmax,

        "bbox_ymax":
            bbox_ymax,

        "component_depth_mean":
            depth_mean,

        "component_depth_median":
            depth_median,

        "component_depth_std":
            depth_std,

        "component_depth_p05":
            depth_p05,

        "component_depth_p95":
            depth_p95,

        "component_depth_robust_range":
            depth_robust_range,

        "component_depth_std_norm":
            depth_std
            / scale,

        "component_depth_robust_range_norm":
            depth_robust_range
            / scale,

        "boundary_median_depth":
            boundary_median,

        "interior_median_depth":
            interior_median,

        "boundary_vs_interior_abs":
            boundary_vs_interior_abs,

        "boundary_vs_interior_abs_norm":
            (
                boundary_vs_interior_abs
                / scale
                if np.isfinite(
                    boundary_vs_interior_abs
                )
                else np.nan
            ),

        "ring_median_depth":
            ring_median,

        "component_vs_ring_abs":
            component_vs_ring_abs,

        "component_vs_ring_abs_norm":
            (
                component_vs_ring_abs
                / scale
                if np.isfinite(
                    component_vs_ring_abs
                )
                else np.nan
            ),

        "mean_gradient_mag":
            mean_gradient,

        "mean_gradient_mag_norm":
            mean_gradient
            / scale,

        "boundary_pixels":
            int(
                boundary_mask.sum()
            ),

        "interior_pixels":
            int(
                interior_mask.sum()
            ),

        "ring_pixels":
            int(
                ring_mask.sum()
            ),
    }


# ------------------------------------------------------------
# 4. Extract all frozen A3c components
# ------------------------------------------------------------

component_rows = []


for _, row in a3c_align_6d_df.iterrows():

    dataset_index = int(
        row["dataset_index"]
    )

    image_id = str(
        row["image_id"]
    )

    category = str(
        row["category"]
    )

    density_stratum = str(
        row["density_stratum"]
    )

    gt_count = int(
        row["gt_count"]
    )


    # --------------------------------------------------------
    # Load frozen A3c mask
    # --------------------------------------------------------

    with np.load(
        Path(
            row["cache_file"]
        ),
        allow_pickle=False,
    ) as data:

        a3c_mask = (
            np.asarray(
                data["mask"]
            )
            .astype(bool)
        )


    # --------------------------------------------------------
    # Load aligned Marigold depth
    # --------------------------------------------------------

    with np.load(
        Path(
            row["cache_path"]
        ),
        allow_pickle=False,
    ) as data:

        depth = (
            np.asarray(
                data["depth"]
            )
            .astype(
                np.float32
            )
        )


    assert (
        a3c_mask.shape
        ==
        depth.shape
    )


    # --------------------------------------------------------
    # Within-image robust Marigold scale
    # --------------------------------------------------------

    image_depth_p05 = float(
        np.percentile(
            depth,
            5
        )
    )

    image_depth_p95 = float(
        np.percentile(
            depth,
            95
        )
    )

    image_robust_depth_range = max(
        image_depth_p95
        -
        image_depth_p05,
        1e-8,
    )


    # --------------------------------------------------------
    # Reproduce frozen C1 4-connected components
    #
    # scipy.ndimage.label() default = 4-connectivity in 2-D,
    # matching the original ABC Step 8C implementation.
    # --------------------------------------------------------

    labelled_components, num_components = (
        ndimage.label(
            a3c_mask
        )
    )


    assert (
        int(
            num_components
        )
        ==
        int(
            row[
                "num_components"
            ]
        )
    ), (
        f"C1 component mismatch for {image_id}: "
        f"{num_components} vs "
        f"{row['num_components']}"
    )


    # --------------------------------------------------------
    # Map GT points to component labels
    #
    # Exact pixel containment only.
    # No radius matching and no recovery matching.
    # --------------------------------------------------------

    gt_points = (
        gt_lookup_6dc[
            dataset_index
        ]
    )

    assert (
        len(
            gt_points
        )
        ==
        gt_count
    )

    component_gt_counts = {
        component_id: 0
        for component_id
        in range(
            1,
            num_components + 1
        )
    }

    h, w = a3c_mask.shape

    num_gt_inside_any_component = 0


    for x, y in gt_points:

        xi = int(
            np.clip(
                np.rint(x),
                0,
                w - 1,
            )
        )

        yi = int(
            np.clip(
                np.rint(y),
                0,
                h - 1,
            )
        )

        component_id = int(
            labelled_components[
                yi,
                xi
            ]
        )

        if component_id > 0:

            component_gt_counts[
                component_id
            ] += 1

            num_gt_inside_any_component += 1


    # --------------------------------------------------------
    # Component-level records
    # --------------------------------------------------------

    for component_id in range(
        1,
        num_components + 1,
    ):

        component_mask = (
            labelled_components
            ==
            component_id
        )

        feat = (
            component_depth_features_6dc(
                component_mask=
                    component_mask,

                depth=
                    depth,

                image_robust_depth_range=
                    image_robust_depth_range,
            )
        )

        component_gt_count = int(
            component_gt_counts[
                component_id
            ]
        )

        if component_gt_count == 0:

            gt_occupancy = (
                "GT0"
            )

        elif component_gt_count == 1:

            gt_occupancy = (
                "GT1"
            )

        else:

            gt_occupancy = (
                "GT2plus"
            )


        feat.update(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "category":
                    category,

                "density_stratum":
                    density_stratum,

                "image_gt_count":
                    gt_count,

                "component_id":
                    int(
                        component_id
                    ),

                "num_components_image":
                    int(
                        num_components
                    ),

                "component_gt_count":
                    component_gt_count,

                "gt_occupancy":
                    gt_occupancy,

                "gt_supported":
                    bool(
                        component_gt_count
                        >= 1
                    ),

                "potentially_merged":
                    bool(
                        component_gt_count
                        >= 2
                    ),

                "image_gt_inside_a3c":
                    int(
                        num_gt_inside_any_component
                    ),

                "image_depth_p05":
                    image_depth_p05,

                "image_depth_p95":
                    image_depth_p95,

                "image_robust_depth_range":
                    image_robust_depth_range,
            }
        )

        component_rows.append(
            feat
        )


a3c_component_geom_df = (
    pd.DataFrame(
        component_rows
    )
)


# ------------------------------------------------------------
# 5. Integrity checks
# ------------------------------------------------------------

assert len(
    a3c_component_geom_df
) == 2554, (
    f"Expected 2554 frozen C1 components, "
    f"found {len(a3c_component_geom_df)}"
)

assert (
    a3c_component_geom_df[
        [
            "dataset_index",
            "component_id",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    ==
    2554
)

assert (
    a3c_component_geom_df[
        "dataset_index"
    ]
    .nunique()
    ==
    100
)


# ------------------------------------------------------------
# 6. Define region-level geometry features
# ------------------------------------------------------------

REGION_GEOM_FEATURES_6DC = [
    "component_depth_std_norm",
    "component_depth_robust_range_norm",
    "boundary_vs_interior_abs_norm",
    "component_vs_ring_abs_norm",
    "mean_gradient_mag_norm",
]


# ------------------------------------------------------------
# 7. GT-occupancy summary
# ------------------------------------------------------------

occupancy_summary_6dc_df = (
    a3c_component_geom_df
    .groupby(
        "gt_occupancy",
        as_index=False,
    )
    .agg(
        num_components=(
            "component_id",
            "count",
        ),

        median_area_pixels=(
            "area_pixels",
            "median",
        ),

        median_depth_std_norm=(
            "component_depth_std_norm",
            "median",
        ),

        median_depth_range_norm=(
            "component_depth_robust_range_norm",
            "median",
        ),

        median_boundary_interior_norm=(
            "boundary_vs_interior_abs_norm",
            "median",
        ),

        median_component_ring_norm=(
            "component_vs_ring_abs_norm",
            "median",
        ),

        median_gradient_norm=(
            "mean_gradient_mag_norm",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 8. Pooled discrimination helper
# ------------------------------------------------------------

def pooled_auc_6dc(
    df,
    label_col,
    feature,
):

    tmp = (
        df[
            [
                label_col,
                feature,
            ]
        ]
        .dropna()
        .copy()
    )

    if (
        tmp[label_col]
        .nunique()
        < 2
    ):
        return {
            "n_positive": 0,
            "n_negative": 0,
            "positive_median": np.nan,
            "negative_median": np.nan,
            "auc_positive": np.nan,
            "separability_auc": np.nan,
            "direction": "undefined",
        }


    y = (
        tmp[
            label_col
        ]
        .astype(int)
        .to_numpy()
    )

    x = (
        tmp[
            feature
        ]
        .astype(float)
        .to_numpy()
    )


    auc = float(
        roc_auc_score(
            y,
            x,
        )
    )


    positive_median = float(
        np.median(
            x[
                y == 1
            ]
        )
    )

    negative_median = float(
        np.median(
            x[
                y == 0
            ]
        )
    )


    return {
        "n_positive":
            int(
                np.sum(
                    y == 1
                )
            ),

        "n_negative":
            int(
                np.sum(
                    y == 0
                )
            ),

        "positive_median":
            positive_median,

        "negative_median":
            negative_median,

        "auc_positive":
            auc,

        "separability_auc":
            max(
                auc,
                1.0 - auc,
            ),

        "direction":
            (
                "higher_in_positive"
                if auc >= 0.5
                else "higher_in_negative"
            ),
    }


# ------------------------------------------------------------
# 9. Pooled discrimination:
#
# A. GT-supported vs GT0
# B. potentially merged (2+) vs single-GT
# ------------------------------------------------------------

discrimination_rows = []


# A. supported vs unsupported
for feature in REGION_GEOM_FEATURES_6DC:

    result = pooled_auc_6dc(
        df=
            a3c_component_geom_df,

        label_col=
            "gt_supported",

        feature=
            feature,
    )

    result.update(
        {
            "comparison":
                "GT_supported_vs_GT0",

            "feature":
                feature,
        }
    )

    discrimination_rows.append(
        result
    )


# B. merged vs single
single_or_merged_df = (
    a3c_component_geom_df[
        a3c_component_geom_df[
            "component_gt_count"
        ]
        >= 1
    ]
    .copy()
)

single_or_merged_df[
    "merged_vs_single"
] = (
    single_or_merged_df[
        "component_gt_count"
    ]
    >= 2
)


for feature in REGION_GEOM_FEATURES_6DC:

    result = pooled_auc_6dc(
        df=
            single_or_merged_df,

        label_col=
            "merged_vs_single",

        feature=
            feature,
    )

    result.update(
        {
            "comparison":
                "GT2plus_vs_GT1",

            "feature":
                feature,
        }
    )

    discrimination_rows.append(
        result
    )


region_discrimination_6dc_df = (
    pd.DataFrame(
        discrimination_rows
    )
)


# ------------------------------------------------------------
# 10. Image-balanced within-image AUC
#
# This guards against the scene-level confounding seen in
# Step 6C.
# ------------------------------------------------------------

within_rows = []


for comparison in [
    "GT_supported_vs_GT0",
    "GT2plus_vs_GT1",
]:

    for feature in REGION_GEOM_FEATURES_6DC:

        for (
            dataset_index,
            image_df
        ) in (
            a3c_component_geom_df
            .groupby(
                "dataset_index"
            )
        ):

            if (
                comparison
                ==
                "GT_supported_vs_GT0"
            ):

                tmp = (
                    image_df[
                        [
                            "gt_supported",
                            feature,
                        ]
                    ]
                    .dropna()
                    .copy()
                )

                label_col = (
                    "gt_supported"
                )

            else:

                tmp = (
                    image_df[
                        image_df[
                            "component_gt_count"
                        ]
                        >= 1
                    ][
                        [
                            "component_gt_count",
                            feature,
                        ]
                    ]
                    .dropna()
                    .copy()
                )

                if len(tmp) == 0:
                    continue

                tmp[
                    "merged_vs_single"
                ] = (
                    tmp[
                        "component_gt_count"
                    ]
                    >= 2
                )

                label_col = (
                    "merged_vs_single"
                )


            if (
                len(tmp) == 0
                or
                tmp[
                    label_col
                ]
                .nunique()
                < 2
            ):
                continue


            y = (
                tmp[
                    label_col
                ]
                .astype(int)
                .to_numpy()
            )

            x = (
                tmp[
                    feature
                ]
                .astype(float)
                .to_numpy()
            )


            auc = float(
                roc_auc_score(
                    y,
                    x,
                )
            )


            within_rows.append(
                {
                    "dataset_index":
                        int(
                            dataset_index
                        ),

                    "comparison":
                        comparison,

                    "feature":
                        feature,

                    "num_components":
                        int(
                            len(tmp)
                        ),

                    "auc_positive":
                        auc,

                    "separability_auc":
                        max(
                            auc,
                            1.0 - auc,
                        ),
                }
            )


within_region_auc_6dc_df = (
    pd.DataFrame(
        within_rows
    )
)


within_region_summary_6dc_df = (
    within_region_auc_6dc_df
    .groupby(
        [
            "comparison",
            "feature",
        ],
        as_index=False,
    )
    .agg(
        num_mixed_images=(
            "dataset_index",
            "count",
        ),

        median_within_image_auc=(
            "auc_positive",
            "median",
        ),

        mean_within_image_auc=(
            "auc_positive",
            "mean",
        ),

        prop_images_auc_gt_0_5=(
            "auc_positive",
            lambda x:
                float(
                    np.mean(
                        np.asarray(
                            x
                        )
                        > 0.5
                    )
                ),
        ),
    )
)


# ------------------------------------------------------------
# 11. Density-stratified descriptive summary
#
# Density remains diagnostic only.
# ------------------------------------------------------------

density_region_summary_6dc_df = (
    a3c_component_geom_df
    .groupby(
        [
            "density_stratum",
            "gt_occupancy",
        ],
        as_index=False,
    )
    .agg(
        num_components=(
            "component_id",
            "count",
        ),

        median_area_pixels=(
            "area_pixels",
            "median",
        ),

        median_depth_std_norm=(
            "component_depth_std_norm",
            "median",
        ),

        median_depth_range_norm=(
            "component_depth_robust_range_norm",
            "median",
        ),

        median_component_ring_norm=(
            "component_vs_ring_abs_norm",
            "median",
        ),

        median_gradient_norm=(
            "mean_gradient_mag_norm",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 12. Save outputs
# ------------------------------------------------------------

M1_6DC_COMPONENT_GEOM_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_component_geometry.csv"
)

M1_6DC_OCCUPANCY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_component_geometry_by_GT_occupancy.csv"
)

M1_6DC_DISCRIMINATION_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_component_geometry_discrimination.csv"
)

M1_6DC_WITHIN_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_component_geometry_within_image.csv"
)

M1_6DC_WITHIN_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_component_geometry_within_image_summary.csv"
)

M1_6DC_DENSITY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_A3c_component_geometry_by_density.csv"
)


a3c_component_geom_df.to_csv(
    M1_6DC_COMPONENT_GEOM_FILE,
    index=False,
)

occupancy_summary_6dc_df.to_csv(
    M1_6DC_OCCUPANCY_FILE,
    index=False,
)

region_discrimination_6dc_df.to_csv(
    M1_6DC_DISCRIMINATION_FILE,
    index=False,
)

within_region_auc_6dc_df.to_csv(
    M1_6DC_WITHIN_FILE,
    index=False,
)

within_region_summary_6dc_df.to_csv(
    M1_6DC_WITHIN_SUMMARY_FILE,
    index=False,
)

density_region_summary_6dc_df.to_csv(
    M1_6DC_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 13. Report
# ------------------------------------------------------------

print("\nA3c component inventory")
print("-----------------------")

print(
    "Images                         :",
    a3c_component_geom_df[
        "dataset_index"
    ].nunique(),
)

print(
    "Total A3c components           :",
    len(
        a3c_component_geom_df
    ),
)

print(
    "GT0 components                 :",
    int(
        (
            a3c_component_geom_df[
                "component_gt_count"
            ]
            == 0
        ).sum()
    ),
)

print(
    "GT1 components                 :",
    int(
        (
            a3c_component_geom_df[
                "component_gt_count"
            ]
            == 1
        ).sum()
    ),
)

print(
    "GT2+ components                :",
    int(
        (
            a3c_component_geom_df[
                "component_gt_count"
            ]
            >= 2
        ).sum()
    ),
)

print(
    "GT points inside A3c components:",
    int(
        a3c_component_geom_df[
            "component_gt_count"
        ].sum()
    ),
)

print(
    "Total GT points                :",
    int(
        dev100_df[
            "gt_count"
        ].sum()
    ),
)


print("\nGeometry by GT occupancy")
print("------------------------")

display(
    occupancy_summary_6dc_df
)


print("\nPooled geometry discrimination")
print("------------------------------")

display(
    region_discrimination_6dc_df[
        [
            "comparison",
            "feature",
            "n_positive",
            "n_negative",
            "positive_median",
            "negative_median",
            "auc_positive",
            "separability_auc",
            "direction",
        ]
    ]
)


print("\nImage-balanced within-image AUC")
print("--------------------------------")

display(
    within_region_summary_6dc_df[
        [
            "comparison",
            "feature",
            "num_mixed_images",
            "median_within_image_auc",
            "mean_within_image_auc",
            "prop_images_auc_gt_0_5",
        ]
    ]
)


print("\nDensity diagnostic summary")
print("--------------------------")

display(
    density_region_summary_6dc_df
)


print("\nSaved component-level geometry:")
print(M1_6DC_COMPONENT_GEOM_FILE)

print("\nSaved GT-occupancy summary:")
print(M1_6DC_OCCUPANCY_FILE)

print("\nSaved pooled discrimination:")
print(M1_6DC_DISCRIMINATION_FILE)

print("\nSaved within-image summary:")
print(M1_6DC_WITHIN_SUMMARY_FILE)


# ------------------------------------------------------------
# 14. Protocol guards
# ------------------------------------------------------------

print("\nGT used for analysis       : YES — component occupancy diagnostic only")
print("GT used for inference      : NO")
print("Marigold depth modified    : NO")
print("Semantic masks modified    : NO")
print("Geometry threshold selected: NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6D Part C: PASS")

STEP 6D — PART C: REGION-LEVEL MARIGOLD GEOMETRY OF A3c COMPONENTS

A3c component inventory
-----------------------
Images                         : 100
Total A3c components           : 2554
GT0 components                 : 1476
GT1 components                 : 760
GT2+ components                : 318
GT points inside A3c components: 2860
Total GT points                : 10045

Geometry by GT occupancy
------------------------


,gt_occupancy,num_components,median_area_pixels,median_depth_std_norm,median_depth_range_norm,median_boundary_interior_norm,median_component_ring_norm,median_gradient_norm
0,GT0,1476,50.0,0.009801,0.029106,0.003068,0.003117,0.005387
1,GT1,760,346.0,0.032279,0.094270,0.013130,0.015876,0.006899
2,GT2plus,318,1068.5,0.076382,0.234089,0.021776,0.024105,0.014241



Pooled geometry discrimination
------------------------------


,comparison,feature,n_positive,n_negative,positive_median,negative_median,auc_positive,separability_auc,direction
0,GT_supported_vs_GT0,component_depth_std_norm,1078,1476,0.043688,0.009801,0.738756,0.738756,higher_in_positive
1,GT_supported_vs_GT0,component_depth_robust_range_norm,1078,1476,0.128176,0.029106,0.742840,0.742840,higher_in_positive
2,GT_supported_vs_GT0,boundary_vs_interior_abs_norm,1074,1181,0.015022,0.003068,0.722078,0.722078,higher_in_positive
3,GT_supported_vs_GT0,component_vs_ring_abs_norm,1078,1476,0.017653,0.003117,0.729809,0.729809,higher_in_positive
4,GT_supported_vs_GT0,mean_gradient_mag_norm,1078,1476,0.008506,0.005387,0.602360,0.602360,higher_in_positive
5,GT2plus_vs_GT1,component_depth_std_norm,318,760,0.076382,0.032279,0.662537,0.662537,higher_in_positive
6,GT2plus_vs_GT1,component_depth_robust_range_norm,318,760,0.234089,0.094270,0.674944,0.674944,higher_in_positive
7,GT2plus_vs_GT1,boundary_vs_interior_abs_norm,318,756,0.021776,0.013130,0.566354,0.566354,higher_in_positive
8,GT2plus_vs_GT1,component_vs_ring_abs_norm,318,760,0.024105,0.015876,0.562243,0.562243,higher_in_positive
9,GT2plus_vs_GT1,mean_gradient_mag_norm,318,760,0.014241,0.006899,0.586279,0.586279,higher_in_positive



Image-balanced within-image AUC
--------------------------------


,comparison,feature,num_mixed_images,median_within_image_auc,mean_within_image_auc,prop_images_auc_gt_0_5
0,GT2plus_vs_GT1,boundary_vs_interior_abs_norm,63,0.700000,0.662001,0.714286
1,GT2plus_vs_GT1,component_depth_robust_range_norm,63,0.900000,0.817113,0.920635
2,GT2plus_vs_GT1,component_depth_std_norm,63,0.875000,0.791854,0.873016
3,GT2plus_vs_GT1,component_vs_ring_abs_norm,63,0.704159,0.664524,0.714286
4,GT2plus_vs_GT1,mean_gradient_mag_norm,63,0.509091,0.580935,0.507937
5,GT_supported_vs_GT0,boundary_vs_interior_abs_norm,90,0.830736,0.779744,0.900000
6,GT_supported_vs_GT0,component_depth_robust_range_norm,91,0.875000,0.847550,0.967033
7,GT_supported_vs_GT0,component_depth_std_norm,91,0.872549,0.842499,0.967033
8,GT_supported_vs_GT0,component_vs_ring_abs_norm,91,0.814798,0.786269,0.912088
9,GT_supported_vs_GT0,mean_gradient_mag_norm,91,0.558974,0.579336,0.571429



Density diagnostic summary
--------------------------


,density_stratum,gt_occupancy,num_components,median_area_pixels,median_depth_std_norm,median_depth_range_norm,median_component_ring_norm,median_gradient_norm
0,S1_0_20,GT0,158,101.0,0.008503,0.027369,0.002101,0.003799
1,S1_0_20,GT1,70,2324.5,0.045598,0.142301,0.017145,0.004664
2,S1_0_20,GT2plus,12,9391.0,0.075353,0.257506,0.040553,0.006758
3,S2_20_40,GT0,282,78.0,0.010291,0.030555,0.003123,0.005867
4,S2_20_40,GT1,81,860.0,0.042585,0.135111,0.025869,0.007051
5,S2_20_40,GT2plus,29,5972.0,0.062855,0.174877,0.031909,0.007382
6,S3_40_60,GT0,282,58.0,0.012462,0.038804,0.003690,0.006828
7,S3_40_60,GT1,79,928.0,0.038351,0.127852,0.030485,0.009681
8,S3_40_60,GT2plus,40,3055.0,0.081306,0.235739,0.043399,0.011307
9,S4_60_80,GT0,309,48.0,0.006018,0.017315,0.002008,0.003884



Saved component-level geometry:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_A3c_component_geometry.csv

Saved GT-occupancy summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_A3c_component_geometry_by_GT_occupancy.csv

Saved pooled discrimination:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_A3c_component_geometry_discrimination.csv

Saved within-image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_A3c_component_geometry_within_image_summary.csv

GT used for analysis       : YES — component occupancy diagnostic only
GT used for inference      : NO
Marigold depth modified    : NO
Semantic masks modified    : NO
Geometry threshold selected: NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed      

### Finding 4 (upto Step 6D-Part C):

**Region-level Marigold geometry** provides substantially stronger and more consistent discrimination than point-level geometry.

A3c components containing multiple GT objects exhibit greater normalized within-component depth variability than single-object components, with median within-image AUCs of 0.900 for robust depth range and 0.875 for depth standard deviation.

Similar discrimination is observed between GT-supported and zero-GT components.

However, component size differs substantially between these groups and may confound the geometric signal; therefore, **geometry should not yet be converted into a refinement threshold without controlling for component area.**

In [23]:
# ============================================================
# Step 6D — Part D
# Component-area confound control
# ============================================================
#
# Purpose
# -------
# Step 6D-C found strong region-level Marigold discrimination,
# especially for:
#
#   - component_depth_std_norm
#   - component_depth_robust_range_norm
#
# However, GT0 / GT1 / GT2+ components also differed greatly
# in component area.
#
# This step asks:
#
#   Does Marigold geometry provide information beyond
#   component size?
#
# Two complementary diagnostics are used:
#
#   A. Grouped cross-validated prediction
#        Area only
#        vs
#        Area + geometry
#
#      GroupKFold by image prevents components from the same
#      image appearing in both train and validation folds.
#
#   B. Area-controlled geometry residuals
#      Regress each geometry feature on:
#
#        log(component area) + image identity
#
#      and evaluate whether the residual geometry still
#      separates the diagnostic GT groups.
#
# IMPORTANT
# ---------
#   - diagnostic only
#   - no refinement threshold
#   - no mask modification
#   - no count-rule modification
#   - GT used only for analysis
#   - test set untouched
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


print("=" * 120)
print("STEP 6D — PART D: COMPONENT-AREA CONFOUND CONTROL")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "a3c_component_geom_df" in globals(), (
    "a3c_component_geom_df unavailable. "
    "Run Step 6D Part C first."
)

assert len(a3c_component_geom_df) == 2554

assert "REGION_GEOM_FEATURES_6DC" in globals()


# ------------------------------------------------------------
# 2. Working table
# ------------------------------------------------------------

area_control_6dd_df = (
    a3c_component_geom_df
    .copy()
)

area_control_6dd_df[
    "log_area"
] = np.log1p(
    area_control_6dd_df[
        "area_pixels"
    ].astype(float)
)


assert np.isfinite(
    area_control_6dd_df[
        "log_area"
    ]
).all()


# ------------------------------------------------------------
# 3. Diagnostic comparisons
# ------------------------------------------------------------

comparison_specs_6dd = {
    "GT_supported_vs_GT0": {
        "df":
            area_control_6dd_df.copy(),

        "label":
            "gt_supported",
    },

    "GT2plus_vs_GT1": {
        "df":
            area_control_6dd_df[
                area_control_6dd_df[
                    "component_gt_count"
                ]
                >= 1
            ].copy(),

        "label":
            "merged_vs_single",
    },
}


comparison_specs_6dd[
    "GT2plus_vs_GT1"
]["df"][
    "merged_vs_single"
] = (
    comparison_specs_6dd[
        "GT2plus_vs_GT1"
    ]["df"][
        "component_gt_count"
    ]
    >= 2
)


# ------------------------------------------------------------
# 4. Grouped CV helper
#
# Compare:
#
#   Area only
#   Area + one Marigold feature
#
# Groups = dataset_index (image)
# ------------------------------------------------------------

def grouped_cv_auc_6dd(
    df,
    label_col,
    geometry_feature=None,
    n_splits=5,
):

    cols = [
        "dataset_index",
        "log_area",
        label_col,
    ]

    if geometry_feature is not None:
        cols.append(
            geometry_feature
        )

    tmp = (
        df[
            cols
        ]
        .dropna()
        .copy()
    )

    y = (
        tmp[
            label_col
        ]
        .astype(int)
        .to_numpy()
    )

    groups = (
        tmp[
            "dataset_index"
        ]
        .astype(int)
        .to_numpy()
    )


    if geometry_feature is None:

        feature_cols = [
            "log_area"
        ]

    else:

        feature_cols = [
            "log_area",
            geometry_feature,
        ]


    X = (
        tmp[
            feature_cols
        ]
        .astype(float)
        .to_numpy()
    )


    gkf = GroupKFold(
        n_splits=n_splits
    )


    fold_rows = []


    for fold_id, (
        train_idx,
        valid_idx,
    ) in enumerate(
        gkf.split(
            X,
            y,
            groups=groups,
        ),
        start=1,
    ):

        y_train = y[
            train_idx
        ]

        y_valid = y[
            valid_idx
        ]


        # AUC is undefined if a validation fold contains
        # only one class.
        if (
            np.unique(
                y_train
            ).size < 2
            or
            np.unique(
                y_valid
            ).size < 2
        ):
            continue


        model = Pipeline(
            steps=[
                (
                    "scale",
                    StandardScaler(),
                ),

                (
                    "logistic",
                    LogisticRegression(
                        solver="liblinear",
                        max_iter=2000,
                        random_state=0,
                    ),
                ),
            ]
        )


        model.fit(
            X[
                train_idx
            ],
            y_train,
        )


        prob = (
            model.predict_proba(
                X[
                    valid_idx
                ]
            )[:, 1]
        )


        auc = float(
            roc_auc_score(
                y_valid,
                prob,
            )
        )


        fold_rows.append(
            {
                "fold":
                    int(
                        fold_id
                    ),

                "auc":
                    auc,

                "n_train":
                    int(
                        len(
                            train_idx
                        )
                    ),

                "n_valid":
                    int(
                        len(
                            valid_idx
                        )
                    ),
            }
        )


    fold_df = pd.DataFrame(
        fold_rows
    )


    return {
        "num_valid_folds":
            int(
                len(
                    fold_df
                )
            ),

        "mean_auc":
            float(
                fold_df[
                    "auc"
                ].mean()
            )
            if len(fold_df) > 0
            else np.nan,

        "median_auc":
            float(
                fold_df[
                    "auc"
                ].median()
            )
            if len(fold_df) > 0
            else np.nan,

        "std_auc":
            float(
                fold_df[
                    "auc"
                ].std(
                    ddof=0
                )
            )
            if len(fold_df) > 0
            else np.nan,

        "fold_df":
            fold_df,
    }


# ------------------------------------------------------------
# 5. Grouped CV:
#    area-only versus area + Marigold feature
# ------------------------------------------------------------

cv_summary_rows = []
cv_fold_rows = []


for comparison_name, spec in comparison_specs_6dd.items():

    df_cmp = spec[
        "df"
    ]

    label_col = spec[
        "label"
    ]


    # --------------------------------------------------------
    # Area-only baseline
    # --------------------------------------------------------

    area_result = grouped_cv_auc_6dd(
        df=
            df_cmp,

        label_col=
            label_col,

        geometry_feature=
            None,
    )


    area_mean_auc = (
        area_result[
            "mean_auc"
        ]
    )


    cv_summary_rows.append(
        {
            "comparison":
                comparison_name,

            "model":
                "area_only",

            "geometry_feature":
                None,

            "num_valid_folds":
                area_result[
                    "num_valid_folds"
                ],

            "mean_auc":
                area_result[
                    "mean_auc"
                ],

            "median_auc":
                area_result[
                    "median_auc"
                ],

            "std_auc":
                area_result[
                    "std_auc"
                ],

            "delta_auc_vs_area":
                0.0,
        }
    )


    for _, fold_row in (
        area_result[
            "fold_df"
        ].iterrows()
    ):

        cv_fold_rows.append(
            {
                "comparison":
                    comparison_name,

                "model":
                    "area_only",

                "geometry_feature":
                    None,

                **fold_row.to_dict(),
            }
        )


    # --------------------------------------------------------
    # Area + each geometry feature
    # --------------------------------------------------------

    for feature in REGION_GEOM_FEATURES_6DC:

        geom_result = (
            grouped_cv_auc_6dd(
                df=
                    df_cmp,

                label_col=
                    label_col,

                geometry_feature=
                    feature,
            )
        )


        cv_summary_rows.append(
            {
                "comparison":
                    comparison_name,

                "model":
                    "area_plus_geometry",

                "geometry_feature":
                    feature,

                "num_valid_folds":
                    geom_result[
                        "num_valid_folds"
                    ],

                "mean_auc":
                    geom_result[
                        "mean_auc"
                    ],

                "median_auc":
                    geom_result[
                        "median_auc"
                    ],

                "std_auc":
                    geom_result[
                        "std_auc"
                    ],

                "delta_auc_vs_area":
                    (
                        geom_result[
                            "mean_auc"
                        ]
                        -
                        area_mean_auc
                    ),
            }
        )


        for _, fold_row in (
            geom_result[
                "fold_df"
            ].iterrows()
        ):

            cv_fold_rows.append(
                {
                    "comparison":
                        comparison_name,

                    "model":
                        "area_plus_geometry",

                    "geometry_feature":
                        feature,

                    **fold_row.to_dict(),
                }
            )


area_geometry_cv_6dd_df = pd.DataFrame(
    cv_summary_rows
)

area_geometry_cv_folds_6dd_df = pd.DataFrame(
    cv_fold_rows
)


# ------------------------------------------------------------
# 6. Residualise geometry against:
#
#      log(area) + image identity
#
# This controls both component size and image-level depth
# characteristics.
# ------------------------------------------------------------

def geometry_residuals_6dd(
    df,
    geometry_feature,
):

    tmp = (
        df[
            [
                "dataset_index",
                "log_area",
                geometry_feature,
            ]
        ]
        .dropna()
        .copy()
    )


    X = tmp[
        [
            "dataset_index",
            "log_area",
        ]
    ]

    y = tmp[
        geometry_feature
    ].astype(float)


    preprocessor = ColumnTransformer(
        transformers=[
            (
                "image",
                OneHotEncoder(
                    handle_unknown="ignore",
                ),
                [
                    "dataset_index"
                ],
            ),

            (
                "area",
                StandardScaler(),
                [
                    "log_area"
                ],
            ),
        ],
        remainder="drop",
    )


    residual_model = Pipeline(
        steps=[
            (
                "preprocess",
                preprocessor,
            ),

            (
                "linear",
                LinearRegression(),
            ),
        ]
    )


    predicted = (
        residual_model.fit(
            X,
            y,
        )
        .predict(
            X
        )
    )


    tmp[
        "geometry_residual"
    ] = (
        y.to_numpy()
        -
        predicted
    )


    return tmp[
        [
            "geometry_residual"
        ]
    ].set_axis(
        tmp.index
    )


# ------------------------------------------------------------
# 7. Compute residual geometry features
# ------------------------------------------------------------

for feature in REGION_GEOM_FEATURES_6DC:

    residual_series = (
        geometry_residuals_6dd(
            df=
                area_control_6dd_df,

            geometry_feature=
                feature,
        )
    )

    residual_col = (
        f"{feature}_area_image_residual"
    )

    area_control_6dd_df[
        residual_col
    ] = np.nan

    area_control_6dd_df.loc[
        residual_series.index,
        residual_col,
    ] = (
        residual_series[
            "geometry_residual"
        ]
    )


# ------------------------------------------------------------
# 8. Residual discrimination:
#
# Does geometry still separate groups after component area
# and image-level offsets have been removed?
# ------------------------------------------------------------

residual_auc_rows = []


for comparison_name, spec in comparison_specs_6dd.items():

    label_col = spec[
        "label"
    ]


    if (
        comparison_name
        ==
        "GT_supported_vs_GT0"
    ):

        cmp_df = (
            area_control_6dd_df
            .copy()
        )

        cmp_df[
            label_col
        ] = (
            cmp_df[
                "gt_supported"
            ]
        )

    else:

        cmp_df = (
            area_control_6dd_df[
                area_control_6dd_df[
                    "component_gt_count"
                ]
                >= 1
            ]
            .copy()
        )

        cmp_df[
            label_col
        ] = (
            cmp_df[
                "component_gt_count"
            ]
            >= 2
        )


    for feature in REGION_GEOM_FEATURES_6DC:

        residual_col = (
            f"{feature}_area_image_residual"
        )

        tmp = (
            cmp_df[
                [
                    "dataset_index",
                    label_col,
                    residual_col,
                ]
            ]
            .dropna()
            .copy()
        )


        y = (
            tmp[
                label_col
            ]
            .astype(int)
            .to_numpy()
        )

        x = (
            tmp[
                residual_col
            ]
            .astype(float)
            .to_numpy()
        )


        pooled_auc = float(
            roc_auc_score(
                y,
                x,
            )
        )


        # ----------------------------------------------------
        # Within-image AUC after area control
        # ----------------------------------------------------

        image_aucs = []


        for (
            dataset_index,
            image_df
        ) in (
            tmp.groupby(
                "dataset_index"
            )
        ):

            if (
                image_df[
                    label_col
                ]
                .nunique()
                < 2
            ):
                continue


            yi = (
                image_df[
                    label_col
                ]
                .astype(int)
                .to_numpy()
            )

            xi = (
                image_df[
                    residual_col
                ]
                .astype(float)
                .to_numpy()
            )


            image_auc = float(
                roc_auc_score(
                    yi,
                    xi,
                )
            )


            image_aucs.append(
                image_auc
            )


        residual_auc_rows.append(
            {
                "comparison":
                    comparison_name,

                "feature":
                    feature,

                "n_positive":
                    int(
                        np.sum(
                            y == 1
                        )
                    ),

                "n_negative":
                    int(
                        np.sum(
                            y == 0
                        )
                    ),

                "pooled_residual_auc":
                    pooled_auc,

                "pooled_residual_separability":
                    max(
                        pooled_auc,
                        1.0 - pooled_auc,
                    ),

                "num_mixed_images":
                    int(
                        len(
                            image_aucs
                        )
                    ),

                "median_within_image_residual_auc":
                    (
                        float(
                            np.median(
                                image_aucs
                            )
                        )
                        if len(
                            image_aucs
                        ) > 0
                        else np.nan
                    ),

                "mean_within_image_residual_auc":
                    (
                        float(
                            np.mean(
                                image_aucs
                            )
                        )
                        if len(
                            image_aucs
                        ) > 0
                        else np.nan
                    ),

                "prop_images_residual_auc_gt_0_5":
                    (
                        float(
                            np.mean(
                                np.asarray(
                                    image_aucs
                                )
                                > 0.5
                            )
                        )
                        if len(
                            image_aucs
                        ) > 0
                        else np.nan
                    ),
            }
        )


geometry_residual_auc_6dd_df = (
    pd.DataFrame(
        residual_auc_rows
    )
)


# ------------------------------------------------------------
# 9. Area-only descriptive AUC
#
# This tells us how much of Part C can already be explained
# by region size alone.
# ------------------------------------------------------------

area_auc_rows = []


for comparison_name, spec in comparison_specs_6dd.items():

    df_cmp = spec[
        "df"
    ].copy()

    label_col = spec[
        "label"
    ]


    tmp = (
        df_cmp[
            [
                label_col,
                "log_area",
            ]
        ]
        .dropna()
    )


    y = (
        tmp[
            label_col
        ]
        .astype(int)
        .to_numpy()
    )

    x = (
        tmp[
            "log_area"
        ]
        .astype(float)
        .to_numpy()
    )


    auc = float(
        roc_auc_score(
            y,
            x,
        )
    )


    area_auc_rows.append(
        {
            "comparison":
                comparison_name,

            "area_auc":
                auc,

            "area_separability_auc":
                max(
                    auc,
                    1.0 - auc,
                ),
        }
    )


area_only_auc_6dd_df = pd.DataFrame(
    area_auc_rows
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

M1_6DD_AREA_ONLY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_component_area_only_auc.csv"
)

M1_6DD_CV_FILE = (
    M1_TABLES_DIR
    / "M1_6D_area_plus_geometry_grouped_cv.csv"
)

M1_6DD_CV_FOLDS_FILE = (
    M1_TABLES_DIR
    / "M1_6D_area_plus_geometry_grouped_cv_folds.csv"
)

M1_6DD_RESIDUAL_FILE = (
    M1_TABLES_DIR
    / "M1_6D_area_controlled_geometry_residual_auc.csv"
)

M1_6DD_COMPONENT_FILE = (
    M1_TABLES_DIR
    / "M1_6D_component_geometry_with_area_control_residuals.csv"
)


area_only_auc_6dd_df.to_csv(
    M1_6DD_AREA_ONLY_FILE,
    index=False,
)

area_geometry_cv_6dd_df.to_csv(
    M1_6DD_CV_FILE,
    index=False,
)

area_geometry_cv_folds_6dd_df.to_csv(
    M1_6DD_CV_FOLDS_FILE,
    index=False,
)

geometry_residual_auc_6dd_df.to_csv(
    M1_6DD_RESIDUAL_FILE,
    index=False,
)

area_control_6dd_df.to_csv(
    M1_6DD_COMPONENT_FILE,
    index=False,
)


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print("\nArea-only discrimination")
print("------------------------")

display(
    area_only_auc_6dd_df
)


print("\nGrouped 5-fold CV: area only vs area + geometry")
print("------------------------------------------------")

display(
    area_geometry_cv_6dd_df[
        [
            "comparison",
            "model",
            "geometry_feature",
            "num_valid_folds",
            "mean_auc",
            "median_auc",
            "std_auc",
            "delta_auc_vs_area",
        ]
    ]
    .sort_values(
        [
            "comparison",
            "delta_auc_vs_area",
        ],
        ascending=[
            True,
            False,
        ],
    )
)


print("\nArea + image controlled residual geometry")
print("-----------------------------------------")

display(
    geometry_residual_auc_6dd_df[
        [
            "comparison",
            "feature",
            "pooled_residual_auc",
            "pooled_residual_separability",
            "num_mixed_images",
            "median_within_image_residual_auc",
            "mean_within_image_residual_auc",
            "prop_images_residual_auc_gt_0_5",
        ]
    ]
)


print("\nSaved area-only results:")
print(M1_6DD_AREA_ONLY_FILE)

print("\nSaved grouped-CV results:")
print(M1_6DD_CV_FILE)

print("\nSaved area-controlled residual results:")
print(M1_6DD_RESIDUAL_FILE)


# ------------------------------------------------------------
# 12. Protocol report
# ------------------------------------------------------------

print("\nGT used for analysis       : YES — diagnostic labels only")
print("GT used for inference      : NO")
print("Component area controlled  : YES")
print("Image identity controlled  : YES")
print("Geometry threshold selected: NO")
print("Marigold depth modified    : NO")
print("Semantic masks modified    : NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6D Part D: PASS")

STEP 6D — PART D: COMPONENT-AREA CONFOUND CONTROL

Area-only discrimination
------------------------


,comparison,area_auc,area_separability_auc
0,GT_supported_vs_GT0,0.820903,0.820903
1,GT2plus_vs_GT1,0.741725,0.741725



Grouped 5-fold CV: area only vs area + geometry
------------------------------------------------


,comparison,model,geometry_feature,num_valid_folds,mean_auc,median_auc,std_auc,delta_auc_vs_area
11,GT2plus_vs_GT1,area_plus_geometry,mean_gradient_mag_norm,5,0.806394,0.819337,0.074661,0.050944
8,GT2plus_vs_GT1,area_plus_geometry,component_depth_robust_range_norm,5,0.797035,0.814870,0.070407,0.041585
7,GT2plus_vs_GT1,area_plus_geometry,component_depth_std_norm,5,0.793926,0.806532,0.068230,0.038476
10,GT2plus_vs_GT1,area_plus_geometry,component_vs_ring_abs_norm,5,0.760601,0.785785,0.059790,0.005151
6,GT2plus_vs_GT1,area_only,None,5,0.755450,0.773824,0.061868,0.000000
9,GT2plus_vs_GT1,area_plus_geometry,boundary_vs_interior_abs_norm,5,0.754403,0.789280,0.050811,-0.001047
5,GT_supported_vs_GT0,area_plus_geometry,mean_gradient_mag_norm,5,0.843654,0.849886,0.024636,0.024392
1,GT_supported_vs_GT0,area_plus_geometry,component_depth_std_norm,5,0.841704,0.844917,0.024759,0.022441
4,GT_supported_vs_GT0,area_plus_geometry,component_vs_ring_abs_norm,5,0.840826,0.842950,0.023060,0.021564
2,GT_supported_vs_GT0,area_plus_geometry,component_depth_robust_range_norm,5,0.840586,0.844541,0.024471,0.021324



Area + image controlled residual geometry
-----------------------------------------


,comparison,feature,pooled_residual_auc,pooled_residual_separability,num_mixed_images,median_within_image_residual_auc,mean_within_image_residual_auc,prop_images_residual_auc_gt_0_5
0,GT_supported_vs_GT0,component_depth_std_norm,0.482501,0.517499,91,0.260110,0.377033,0.307692
1,GT_supported_vs_GT0,component_depth_robust_range_norm,0.478176,0.521824,91,0.260870,0.377172,0.307692
2,GT_supported_vs_GT0,boundary_vs_interior_abs_norm,0.494519,0.505481,90,0.372418,0.414263,0.388889
3,GT_supported_vs_GT0,component_vs_ring_abs_norm,0.463520,0.536480,91,0.311765,0.386617,0.351648
4,GT_supported_vs_GT0,mean_gradient_mag_norm,0.613222,0.613222,91,0.727273,0.723782,0.824176
5,GT2plus_vs_GT1,component_depth_std_norm,0.557349,0.557349,63,0.542510,0.561660,0.539683
6,GT2plus_vs_GT1,component_depth_robust_range_norm,0.573407,0.573407,63,0.583333,0.559788,0.523810
7,GT2plus_vs_GT1,boundary_vs_interior_abs_norm,0.427224,0.572776,63,0.430303,0.441300,0.428571
8,GT2plus_vs_GT1,component_vs_ring_abs_norm,0.411830,0.588170,63,0.400000,0.419893,0.349206
9,GT2plus_vs_GT1,mean_gradient_mag_norm,0.561379,0.561379,63,0.750000,0.667997,0.650794



Saved area-only results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_component_area_only_auc.csv

Saved grouped-CV results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_area_plus_geometry_grouped_cv.csv

Saved area-controlled residual results:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_area_controlled_geometry_residual_auc.csv

GT used for analysis       : YES — diagnostic labels only
GT used for inference      : NO
Component area controlled  : YES
Image identity controlled  : YES
Geometry threshold selected: NO
Marigold depth modified    : NO
Semantic masks modified    : NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6D Part D: PASS


### **Finding 4 (updated after 6D-Part D):**

Region-level Marigold geometry is more informative than point-level geometry, but much of the strong depth-variability signal is explained by component area.

Area alone strongly discriminates GT-supported from GT0 components and GT2+ from GT1 components.

After controlling for area and image identity, depth standard deviation and robust depth range retain only weak independent discrimination, although grouped cross-validation shows small additional predictive gains when these features are added to area.

Mean gradient retains a more consistent residual signal.

Therefore, geometry appears complementary to component size rather than independently decisive, and a **direct refinement threshold is still not yet justified.**

In [24]:
# ============================================================
# Step 6D — Part E
# Area-stratified / size-matched geometry robustness audit
# ============================================================
#
# Purpose
# -------
# Step 6D-D showed that component area is a major confound:
#
#   - area alone strongly discriminates the diagnostic groups
#   - adding Marigold geometry gives modest CV improvements
#   - residualisation reduces much of the depth-range/std signal
#
# However, residualisation assumes a particular functional form.
#
# This step therefore performs a non-parametric robustness check:
#
#   Compare geometry ONLY among components of similar size.
#
# We use area quantile bins at several resolutions:
#
#   q = 5, 10, 20
#
# Two analyses:
#
#   A. Area-stratified pooled analysis
#      Compare positive vs negative components within the same
#      area bin.
#
#   B. Image + area-stratified analysis
#      Compare positive vs negative components only when they
#      occur:
#
#          - in the same image
#          - in the same area bin
#
# This controls component area without fitting a regression
# model and also controls scene/image effects.
#
# IMPORTANT
# ---------
#   - diagnostic only
#   - no geometry threshold selection
#   - no mask modification
#   - no count-rule modification
#   - GT used only for analysis
#   - test set untouched
# ============================================================

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score


print("=" * 120)
print("STEP 6D — PART E: AREA-STRATIFIED / SIZE-MATCHED GEOMETRY ROBUSTNESS AUDIT")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "area_control_6dd_df" in globals(), (
    "area_control_6dd_df unavailable. "
    "Run Step 6D Part D first."
)

assert len(area_control_6dd_df) == 2554

assert "REGION_GEOM_FEATURES_6DC" in globals()

assert "log_area" in area_control_6dd_df.columns


# ------------------------------------------------------------
# 2. Diagnostic comparison datasets
# ------------------------------------------------------------

comparison_dfs_6de = {}


# ------------------------------------------------------------
# A. GT-supported vs GT0
# ------------------------------------------------------------

supported_df_6de = (
    area_control_6dd_df
    .copy()
)

supported_df_6de[
    "diagnostic_label"
] = (
    supported_df_6de[
        "gt_supported"
    ]
    .astype(bool)
)

comparison_dfs_6de[
    "GT_supported_vs_GT0"
] = supported_df_6de


# ------------------------------------------------------------
# B. GT2+ vs GT1
# ------------------------------------------------------------

merged_df_6de = (
    area_control_6dd_df[
        area_control_6dd_df[
            "component_gt_count"
        ]
        >= 1
    ]
    .copy()
)

merged_df_6de[
    "diagnostic_label"
] = (
    merged_df_6de[
        "component_gt_count"
    ]
    >= 2
)

comparison_dfs_6de[
    "GT2plus_vs_GT1"
] = merged_df_6de


# ------------------------------------------------------------
# 3. Area-bin resolutions
#
# qcut gives approximately equal-frequency size strata.
#
# Multiple resolutions are used deliberately so that the
# conclusion is not dependent on one arbitrary binning choice.
# ------------------------------------------------------------

AREA_BIN_COUNTS_6DE = [
    5,
    10,
    20,
]


# ------------------------------------------------------------
# 4. Helper:
#    build area quantile bins
# ------------------------------------------------------------

def add_area_bins_6de(
    df,
    q,
):

    out = df.copy()

    # qcut can drop bins if many identical component areas occur.
    out[
        "area_bin"
    ] = pd.qcut(
        out[
            "log_area"
        ],
        q=q,
        duplicates="drop",
    )

    return out


# ------------------------------------------------------------
# 5. Helper:
#    AUC for one mixed group
# ------------------------------------------------------------

def group_auc_6de(
    df,
    feature,
):

    tmp = (
        df[
            [
                "diagnostic_label",
                feature,
            ]
        ]
        .dropna()
        .copy()
    )

    if (
        len(tmp) == 0
        or
        tmp[
            "diagnostic_label"
        ]
        .nunique()
        < 2
    ):
        return None


    y = (
        tmp[
            "diagnostic_label"
        ]
        .astype(int)
        .to_numpy()
    )

    x = (
        tmp[
            feature
        ]
        .astype(float)
        .to_numpy()
    )


    auc = float(
        roc_auc_score(
            y,
            x,
        )
    )

    n_positive = int(
        np.sum(
            y == 1
        )
    )

    n_negative = int(
        np.sum(
            y == 0
        )
    )

    # Number of positive-negative pairs contributing to AUC.
    pair_weight = int(
        n_positive
        *
        n_negative
    )


    return {
        "auc_positive":
            auc,

        "separability_auc":
            max(
                auc,
                1.0 - auc,
            ),

        "n_positive":
            n_positive,

        "n_negative":
            n_negative,

        "num_components":
            int(
                len(tmp)
            ),

        "pair_weight":
            pair_weight,
    }


# ------------------------------------------------------------
# 6. Area-stratified pooled analysis
# ------------------------------------------------------------

area_bin_detail_rows = []
area_bin_summary_rows = []


for comparison_name, base_df in comparison_dfs_6de.items():

    for q in AREA_BIN_COUNTS_6DE:

        df_q = add_area_bins_6de(
            base_df,
            q=q,
        )

        actual_num_bins = int(
            df_q[
                "area_bin"
            ]
            .nunique()
        )


        for feature in REGION_GEOM_FEATURES_6DC:

            feature_bin_rows = []


            for area_bin, bin_df in (
                df_q.groupby(
                    "area_bin",
                    observed=True,
                )
            ):

                result = group_auc_6de(
                    bin_df,
                    feature,
                )

                if result is None:
                    continue


                row = {
                    "comparison":
                        comparison_name,

                    "requested_num_bins":
                        q,

                    "actual_num_bins":
                        actual_num_bins,

                    "area_bin":
                        str(
                            area_bin
                        ),

                    "feature":
                        feature,

                    **result,
                }


                area_bin_detail_rows.append(
                    row
                )

                feature_bin_rows.append(
                    row
                )


            feature_bin_df = pd.DataFrame(
                feature_bin_rows
            )


            if len(
                feature_bin_df
            ) == 0:

                continue


            total_pair_weight = float(
                feature_bin_df[
                    "pair_weight"
                ].sum()
            )


            weighted_auc = float(
                np.average(
                    feature_bin_df[
                        "auc_positive"
                    ],
                    weights=
                        feature_bin_df[
                            "pair_weight"
                        ],
                )
            )


            area_bin_summary_rows.append(
                {
                    "comparison":
                        comparison_name,

                    "requested_num_bins":
                        q,

                    "actual_num_bins":
                        actual_num_bins,

                    "feature":
                        feature,

                    "num_mixed_area_bins":
                        int(
                            len(
                                feature_bin_df
                            )
                        ),

                    "pair_weighted_auc":
                        weighted_auc,

                    "median_area_bin_auc":
                        float(
                            feature_bin_df[
                                "auc_positive"
                            ]
                            .median()
                        ),

                    "mean_area_bin_auc":
                        float(
                            feature_bin_df[
                                "auc_positive"
                            ]
                            .mean()
                        ),

                    "prop_area_bins_auc_gt_0_5":
                        float(
                            np.mean(
                                feature_bin_df[
                                    "auc_positive"
                                ]
                                .to_numpy()
                                > 0.5
                            )
                        ),

                    "total_pair_weight":
                        int(
                            total_pair_weight
                        ),
                }
            )


area_bin_detail_6de_df = pd.DataFrame(
    area_bin_detail_rows
)

area_bin_summary_6de_df = pd.DataFrame(
    area_bin_summary_rows
)


# ------------------------------------------------------------
# 7. Image + area-bin stratified analysis
#
# Stronger control:
#
#   positive and negative components must occur
#   in the SAME image and SAME approximate size stratum.
# ------------------------------------------------------------

image_area_detail_rows = []
image_area_summary_rows = []


for comparison_name, base_df in comparison_dfs_6de.items():

    for q in AREA_BIN_COUNTS_6DE:

        df_q = add_area_bins_6de(
            base_df,
            q=q,
        )

        actual_num_bins = int(
            df_q[
                "area_bin"
            ]
            .nunique()
        )


        for feature in REGION_GEOM_FEATURES_6DC:

            feature_group_rows = []


            grouped = df_q.groupby(
                [
                    "dataset_index",
                    "area_bin",
                ],
                observed=True,
            )


            for (
                dataset_index,
                area_bin,
            ), group_df in grouped:

                result = group_auc_6de(
                    group_df,
                    feature,
                )

                if result is None:
                    continue


                row = {
                    "comparison":
                        comparison_name,

                    "requested_num_bins":
                        q,

                    "actual_num_bins":
                        actual_num_bins,

                    "dataset_index":
                        int(
                            dataset_index
                        ),

                    "area_bin":
                        str(
                            area_bin
                        ),

                    "feature":
                        feature,

                    **result,
                }


                image_area_detail_rows.append(
                    row
                )

                feature_group_rows.append(
                    row
                )


            feature_group_df = pd.DataFrame(
                feature_group_rows
            )


            if len(
                feature_group_df
            ) == 0:

                continue


            weighted_auc = float(
                np.average(
                    feature_group_df[
                        "auc_positive"
                    ],
                    weights=
                        feature_group_df[
                            "pair_weight"
                        ],
                )
            )


            image_area_summary_rows.append(
                {
                    "comparison":
                        comparison_name,

                    "requested_num_bins":
                        q,

                    "actual_num_bins":
                        actual_num_bins,

                    "feature":
                        feature,

                    "num_mixed_image_area_groups":
                        int(
                            len(
                                feature_group_df
                            )
                        ),

                    "num_contributing_images":
                        int(
                            feature_group_df[
                                "dataset_index"
                            ]
                            .nunique()
                        ),

                    "pair_weighted_auc":
                        weighted_auc,

                    "median_image_area_auc":
                        float(
                            feature_group_df[
                                "auc_positive"
                            ]
                            .median()
                        ),

                    "mean_image_area_auc":
                        float(
                            feature_group_df[
                                "auc_positive"
                            ]
                            .mean()
                        ),

                    "prop_groups_auc_gt_0_5":
                        float(
                            np.mean(
                                feature_group_df[
                                    "auc_positive"
                                ]
                                .to_numpy()
                                > 0.5
                            )
                        ),

                    "total_pair_weight":
                        int(
                            feature_group_df[
                                "pair_weight"
                            ]
                            .sum()
                        ),
                }
            )


image_area_detail_6de_df = pd.DataFrame(
    image_area_detail_rows
)

image_area_summary_6de_df = pd.DataFrame(
    image_area_summary_rows
)


# ------------------------------------------------------------
# 8. Size-distribution sanity check
#
# Report median component area for each diagnostic class.
# This is descriptive only.
# ------------------------------------------------------------

size_summary_rows = []


for comparison_name, df_cmp in comparison_dfs_6de.items():

    for label_value, label_name in [
        (False, "negative"),
        (True, "positive"),
    ]:

        class_df = (
            df_cmp[
                df_cmp[
                    "diagnostic_label"
                ]
                ==
                label_value
            ]
        )


        size_summary_rows.append(
            {
                "comparison":
                    comparison_name,

                "class":
                    label_name,

                "num_components":
                    int(
                        len(
                            class_df
                        )
                    ),

                "median_area_pixels":
                    float(
                        class_df[
                            "area_pixels"
                        ]
                        .median()
                    ),

                "median_log_area":
                    float(
                        class_df[
                            "log_area"
                        ]
                        .median()
                    ),
            }
        )


size_summary_6de_df = pd.DataFrame(
    size_summary_rows
)


# ------------------------------------------------------------
# 9. Save outputs
# ------------------------------------------------------------

M1_6DE_AREA_BIN_DETAIL_FILE = (
    M1_TABLES_DIR
    / "M1_6D_area_stratified_geometry_detail.csv"
)

M1_6DE_AREA_BIN_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_area_stratified_geometry_summary.csv"
)

M1_6DE_IMAGE_AREA_DETAIL_FILE = (
    M1_TABLES_DIR
    / "M1_6D_image_area_stratified_geometry_detail.csv"
)

M1_6DE_IMAGE_AREA_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_image_area_stratified_geometry_summary.csv"
)

M1_6DE_SIZE_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_6D_area_stratified_size_summary.csv"
)


area_bin_detail_6de_df.to_csv(
    M1_6DE_AREA_BIN_DETAIL_FILE,
    index=False,
)

area_bin_summary_6de_df.to_csv(
    M1_6DE_AREA_BIN_SUMMARY_FILE,
    index=False,
)

image_area_detail_6de_df.to_csv(
    M1_6DE_IMAGE_AREA_DETAIL_FILE,
    index=False,
)

image_area_summary_6de_df.to_csv(
    M1_6DE_IMAGE_AREA_SUMMARY_FILE,
    index=False,
)

size_summary_6de_df.to_csv(
    M1_6DE_SIZE_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 10. Report
# ------------------------------------------------------------

print("\nOriginal size difference")
print("------------------------")

display(
    size_summary_6de_df
)


print("\nArea-stratified pooled geometry")
print("-------------------------------")

display(
    area_bin_summary_6de_df[
        [
            "comparison",
            "requested_num_bins",
            "feature",
            "num_mixed_area_bins",
            "pair_weighted_auc",
            "median_area_bin_auc",
            "mean_area_bin_auc",
            "prop_area_bins_auc_gt_0_5",
        ]
    ]
    .sort_values(
        [
            "comparison",
            "requested_num_bins",
            "pair_weighted_auc",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)


print("\nImage + area-stratified geometry")
print("--------------------------------")

display(
    image_area_summary_6de_df[
        [
            "comparison",
            "requested_num_bins",
            "feature",
            "num_mixed_image_area_groups",
            "num_contributing_images",
            "pair_weighted_auc",
            "median_image_area_auc",
            "mean_image_area_auc",
            "prop_groups_auc_gt_0_5",
        ]
    ]
    .sort_values(
        [
            "comparison",
            "requested_num_bins",
            "pair_weighted_auc",
        ],
        ascending=[
            True,
            True,
            False,
        ],
    )
)


print("\nSaved area-stratified summary:")
print(M1_6DE_AREA_BIN_SUMMARY_FILE)

print("\nSaved image + area-stratified summary:")
print(M1_6DE_IMAGE_AREA_SUMMARY_FILE)


# ------------------------------------------------------------
# 11. Protocol report
# ------------------------------------------------------------

print("\nGT used for analysis       : YES — diagnostic labels only")
print("GT used for inference      : NO")
print("Component area controlled  : YES — non-parametric stratification")
print("Image identity controlled  : YES — second analysis")
print("Multiple area resolutions  : YES — 5 / 10 / 20 bins")
print("Geometry threshold selected: NO")
print("Marigold depth modified    : NO")
print("Semantic masks modified    : NO")
print("Density used for routing   : NO")
print("Count rule modified        : NO")
print("Test set accessed          : NO")

print("\nStep 6D Part E: PASS")

STEP 6D — PART E: AREA-STRATIFIED / SIZE-MATCHED GEOMETRY ROBUSTNESS AUDIT

Original size difference
------------------------


,comparison,class,num_components,median_area_pixels,median_log_area
0,GT_supported_vs_GT0,negative,1476,50.0,3.931826
1,GT_supported_vs_GT0,positive,1078,479.0,6.173786
2,GT2plus_vs_GT1,negative,760,346.0,5.849321
3,GT2plus_vs_GT1,positive,318,1068.5,6.974941



Area-stratified pooled geometry
-------------------------------


,comparison,requested_num_bins,feature,num_mixed_area_bins,pair_weighted_auc,median_area_bin_auc,mean_area_bin_auc,prop_area_bins_auc_gt_0_5
16,GT2plus_vs_GT1,5,component_depth_robust_range_norm,5,0.668189,0.689165,0.672168,1.000000
15,GT2plus_vs_GT1,5,component_depth_std_norm,5,0.659524,0.687412,0.664318,1.000000
19,GT2plus_vs_GT1,5,mean_gradient_mag_norm,5,0.657080,0.679005,0.658918,1.000000
17,GT2plus_vs_GT1,5,boundary_vs_interior_abs_norm,5,0.528248,0.581270,0.554462,0.600000
18,GT2plus_vs_GT1,5,component_vs_ring_abs_norm,5,0.522337,0.571852,0.548235,0.600000
21,GT2plus_vs_GT1,10,component_depth_robust_range_norm,10,0.675683,0.676581,0.678062,1.000000
24,GT2plus_vs_GT1,10,mean_gradient_mag_norm,10,0.671120,0.684133,0.673140,0.900000
20,GT2plus_vs_GT1,10,component_depth_std_norm,10,0.662953,0.650789,0.664986,1.000000
22,GT2plus_vs_GT1,10,boundary_vs_interior_abs_norm,10,0.524094,0.554786,0.548865,0.600000
23,GT2plus_vs_GT1,10,component_vs_ring_abs_norm,10,0.519285,0.546506,0.545314,0.500000



Image + area-stratified geometry
--------------------------------


,comparison,requested_num_bins,feature,num_mixed_image_area_groups,num_contributing_images,pair_weighted_auc,median_image_area_auc,mean_image_area_auc,prop_groups_auc_gt_0_5
16,GT2plus_vs_GT1,5,component_depth_robust_range_norm,60,43,0.611947,0.718750,0.670604,0.700000
15,GT2plus_vs_GT1,5,component_depth_std_norm,60,43,0.608850,0.666667,0.657963,0.650000
18,GT2plus_vs_GT1,5,component_vs_ring_abs_norm,60,43,0.565929,0.570175,0.556496,0.566667
17,GT2plus_vs_GT1,5,boundary_vs_interior_abs_norm,60,43,0.554338,0.642424,0.585734,0.566667
19,GT2plus_vs_GT1,5,mean_gradient_mag_norm,60,43,0.529204,0.666667,0.631615,0.633333
21,GT2plus_vs_GT1,10,component_depth_robust_range_norm,57,32,0.542440,0.577778,0.591490,0.543860
23,GT2plus_vs_GT1,10,component_vs_ring_abs_norm,57,32,0.534483,0.500000,0.510132,0.456140
20,GT2plus_vs_GT1,10,component_depth_std_norm,57,32,0.527851,0.577778,0.580525,0.508772
24,GT2plus_vs_GT1,10,mean_gradient_mag_norm,57,32,0.517241,0.666667,0.619938,0.578947
22,GT2plus_vs_GT1,10,boundary_vs_interior_abs_norm,57,32,0.491957,0.500000,0.507298,0.456140



Saved area-stratified summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_area_stratified_geometry_summary.csv

Saved image + area-stratified summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_6D_image_area_stratified_geometry_summary.csv

GT used for analysis       : YES — diagnostic labels only
GT used for inference      : NO
Component area controlled  : YES — non-parametric stratification
Image identity controlled  : YES — second analysis
Multiple area resolutions  : YES — 5 / 10 / 20 bins
Geometry threshold selected: NO
Marigold depth modified    : NO
Semantic masks modified    : NO
Density used for routing   : NO
Count rule modified        : NO
Test set accessed          : NO

Step 6D Part E: PASS


### **Finding 4 — Step 6D:**

Region-level Marigold geometry initially shows stronger discrimination than point-level geometry, but component area explains a substantial proportion of this effect.

After controlling for component size and image identity, geometry provides little reliable information for distinguishing GT-supported from unsupported A3c components.

A weaker residual signal remains for distinguishing multi-object from single-object components, suggesting that geometry may be more useful for identifying or refining potentially merged semantic regions than for filtering semantic regions outright.

### **Step 6 Summary — Geometric Evidence Analysis**

Step 6 investigated how Marigold depth could contribute geometric evidence to the zero-shot counting pipeline, progressing from local point-level diagnostics to semantic-region analysis.

**Step 6A — Local GT-point geometry**
- Marigold showed measurable local depth variation around true object locations.
- However, the feature distributions were broad and heavy-tailed.
- Simple point-local rules such as “large depth contrast = object” were therefore not supported.

**Finding 1**
Local Marigold geometry around true object locations is highly variable; simple point-local geometric thresholds are not supported.

---

**Step 6B — B0-covered vs B0-missed GT objects**
- Objects missed by the B0 SAM2 baseline tended to exhibit stronger local depth variability and gradients than objects already covered by B0.
- The effect remained moderate after controlling for image-level differences.
- This suggested that Marigold may provide complementary information about B0 failure modes.

**Finding 2**
B0-missed objects tend to exhibit stronger local depth variability than B0-covered objects, supporting Marigold as a complementary cue for identifying B0 failure rather than as an independent counting signal.

---

**Step 6C — Geometry of semantic recovery points**
- Point-local Marigold features were tested on C1, D6B-T and D6B9 semantic recovery candidates.
- Stronger pooled discrimination, especially for C1, largely disappeared or reversed when candidates were compared within the same image.
- This showed that much of the apparent point-level signal was due to between-image variation rather than reliable point-level discrimination.

**Finding 3**
Point-local Marigold features do not consistently distinguish true from non-TP semantic recovery candidates within the same image. Therefore, simple global geometric thresholding of semantic recovery points is not supported.

**Architectural implication**
The evidence motivated moving from point-level geometry to region-level semantic-mask analysis.

---

**Step 6D — Region-level A3c geometry**
- The frozen A3c semantic masks were recovered, cached and validated exactly against the original C1 outputs.
- Initial region-level analysis showed substantially stronger geometric discrimination than point-level analysis.
- Multi-GT components exhibited greater within-component depth variability than single-GT components.
- However, component area differed strongly between GT0, GT1 and GT2+ groups.

**Component-area confound control**
- Component area alone was already strongly discriminative.
- Adding Marigold geometry to area gave modest grouped cross-validation improvements.
- After controlling for component area and image identity, much of the apparent depth-variability signal weakened substantially.

**Non-parametric size-matched robustness check**
- When components were compared within similar area ranges, pooled geometry remained moderately informative.
- Under the stronger same-image + similar-area comparison, geometry provided little reliable information for distinguishing GT-supported from GT0 components.
- A weaker residual signal remained for distinguishing GT2+ from GT1 components, particularly through within-component depth variability.

**Finding 4**
Region-level Marigold geometry is more informative than point-level geometry, but much of the apparent signal is explained by component size and scene differences. Geometry is therefore not supported as a strong semantic-region filtering cue. A weaker residual signal remains for potentially merged multi-object regions, suggesting that geometry may be more useful for component splitting/refinement than for rejecting semantic components outright.

---

### **Overall Step 6 Conclusion**

Across the Step 6 diagnostics, Marigold does not emerge as an independent counter or as a reliable global point/region rejection signal.

Its most defensible role is as a **complementary geometric cue applied to semantic regions**, particularly for identifying or refining potentially merged components.

### Implication for Step 7

The next phase should move from diagnosis to intervention:

**Step 7 — Geometry-Guided Semantic Component Refinement**

The first controlled experiment should test whether Marigold can help split or refine potentially merged A3c semantic components while keeping the original A3c mask/C1 representation as the baseline.

No global geometric rejection threshold should be introduced at this stage.

# **Step 7 - Geometry-Guided Semantic Component Refinement**

In [25]:
# ============================================================
# Step 7A — Define and audit candidate components for
#            geometry-guided splitting
# ============================================================
#
# Purpose
# -------
# Define an inference-available candidate set of A3c/C1
# components that are eligible for a later Marigold-based
# splitting attempt.
#
# IMPORTANT DESIGN DECISION
# -------------------------
# Step 6 showed that component area is already a strong signal
# of potentially merged semantic regions, while Marigold
# geometry provides only modest additional discrimination
# after controlling for area/image effects.
#
# Therefore Step 7 separates:
#
#   candidate selection  -> component size only
#   splitting mechanism  -> Marigold geometry (Step 7B)
#
# This prevents geometry from being used both to select and
# split components, giving a cleaner ablation.
#
# Primary candidate rule
# ----------------------
# A component is a split candidate if its area is in the
# upper quartile (>= 75th percentile) of A3c component areas
# WITHIN THE SAME IMAGE.
#
# This rule:
#   - uses no GT labels
#   - uses no density routing
#   - uses no global pixel-area threshold
#   - adapts to object/image scale
#
# We additionally audit 50%, 75%, and 90% area-percentile
# rules to understand intervention burden.
#
# The PRIMARY rule is frozen here as 75%.
#
# No splitting occurs in this step.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 7A — DEFINE + AUDIT CANDIDATE COMPONENTS FOR GEOMETRY-GUIDED SPLITTING")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "a3c_component_geom_df" in globals(), (
    "a3c_component_geom_df unavailable. "
    "Run Step 6D Part C first."
)

assert len(a3c_component_geom_df) == 2554

assert (
    a3c_component_geom_df[
        ["dataset_index", "component_id"]
    ]
    .drop_duplicates()
    .shape[0]
    == 2554
)


# ------------------------------------------------------------
# 2. Freeze candidate-selection protocol
# ------------------------------------------------------------

PRIMARY_SPLIT_AREA_PERCENTILE_7A = 75

AUDIT_AREA_PERCENTILES_7A = [
    50,
    75,
    90,
]

assert (
    PRIMARY_SPLIT_AREA_PERCENTILE_7A
    in AUDIT_AREA_PERCENTILES_7A
)


# ------------------------------------------------------------
# 3. Build inference-only working table
#
# Candidate selection below uses ONLY:
#
#   dataset_index
#   component_id
#   area_pixels
#
# Geometry and GT columns remain available only for later
# diagnostics/evaluation and DO NOT determine candidacy.
# ------------------------------------------------------------

split_candidates_7a_df = (
    a3c_component_geom_df
    .copy()
    .sort_values(
        [
            "dataset_index",
            "component_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 4. Compute image-relative component area percentile
#
# Percentile rank is based only on components from the same
# image.
#
# rank(method="average", pct=True):
#   smallest component -> near 0
#   largest component  -> 1
# ------------------------------------------------------------

split_candidates_7a_df[
    "area_percentile_within_image"
] = (
    split_candidates_7a_df
    .groupby(
        "dataset_index"
    )[
        "area_pixels"
    ]
    .rank(
        method="average",
        pct=True,
    )
)


assert (
    split_candidates_7a_df[
        "area_percentile_within_image"
    ]
    .between(
        0.0,
        1.0,
        inclusive="both",
    )
    .all()
)


# ------------------------------------------------------------
# 5. Define candidate flags for audit thresholds
# ------------------------------------------------------------

for percentile in AUDIT_AREA_PERCENTILES_7A:

    threshold_fraction = (
        percentile
        / 100.0
    )

    col = (
        f"candidate_area_p{percentile}"
    )

    split_candidates_7a_df[
        col
    ] = (
        split_candidates_7a_df[
            "area_percentile_within_image"
        ]
        >= threshold_fraction
    )


# ------------------------------------------------------------
# 6. Freeze PRIMARY candidate flag
# ------------------------------------------------------------

PRIMARY_SPLIT_CANDIDATE_COL_7A = (
    f"candidate_area_p"
    f"{PRIMARY_SPLIT_AREA_PERCENTILE_7A}"
)

split_candidates_7a_df[
    "split_candidate"
] = (
    split_candidates_7a_df[
        PRIMARY_SPLIT_CANDIDATE_COL_7A
    ]
    .astype(bool)
)


# ------------------------------------------------------------
# 7. Explicit guard:
#    candidate selection must not depend on GT or Marigold
# ------------------------------------------------------------

CANDIDATE_SELECTION_INPUTS_7A = [
    "dataset_index",
    "component_id",
    "area_pixels",
]

print("\nCandidate-selection inputs:")
print(CANDIDATE_SELECTION_INPUTS_7A)

print(
    "\nGT used in candidate rule       : NO"
)

print(
    "Marigold used in candidate rule : NO"
)


# ------------------------------------------------------------
# 8. Audit candidate burden across area-percentile variants
# ------------------------------------------------------------

audit_rows_7a = []


total_components_7a = len(
    split_candidates_7a_df
)

total_component_pixels_7a = int(
    split_candidates_7a_df[
        "area_pixels"
    ].sum()
)


for percentile in AUDIT_AREA_PERCENTILES_7A:

    candidate_col = (
        f"candidate_area_p{percentile}"
    )

    candidate_df = (
        split_candidates_7a_df[
            split_candidates_7a_df[
                candidate_col
            ]
        ]
        .copy()
    )


    per_image_counts = (
        candidate_df
        .groupby(
            "dataset_index"
        )
        .size()
        .reindex(
            split_candidates_7a_df[
                "dataset_index"
            ].unique(),
            fill_value=0,
        )
    )


    candidate_pixels = int(
        candidate_df[
            "area_pixels"
        ].sum()
    )


    audit_rows_7a.append(
        {
            "area_percentile_rule":
                int(
                    percentile
                ),

            "num_candidate_components":
                int(
                    len(
                        candidate_df
                    )
                ),

            "candidate_fraction_components":
                float(
                    len(
                        candidate_df
                    )
                    /
                    total_components_7a
                ),

            "num_images_with_candidate":
                int(
                    candidate_df[
                        "dataset_index"
                    ]
                    .nunique()
                ),

            "median_candidates_per_image":
                float(
                    per_image_counts
                    .median()
                ),

            "mean_candidates_per_image":
                float(
                    per_image_counts
                    .mean()
                ),

            "max_candidates_per_image":
                int(
                    per_image_counts
                    .max()
                ),

            "candidate_pixels":
                candidate_pixels,

            "candidate_fraction_foreground_pixels":
                float(
                    candidate_pixels
                    /
                    total_component_pixels_7a
                ),

            "median_candidate_area":
                float(
                    candidate_df[
                        "area_pixels"
                    ]
                    .median()
                ),

            "primary_rule":
                bool(
                    percentile
                    ==
                    PRIMARY_SPLIT_AREA_PERCENTILE_7A
                ),
        }
    )


candidate_rule_audit_7a_df = (
    pd.DataFrame(
        audit_rows_7a
    )
)


# ------------------------------------------------------------
# 9. Primary candidate inventory
# ------------------------------------------------------------

primary_candidates_7a_df = (
    split_candidates_7a_df[
        split_candidates_7a_df[
            "split_candidate"
        ]
    ]
    .copy()
)


primary_non_candidates_7a_df = (
    split_candidates_7a_df[
        ~split_candidates_7a_df[
            "split_candidate"
        ]
    ]
    .copy()
)


assert (
    len(
        primary_candidates_7a_df
    )
    +
    len(
        primary_non_candidates_7a_df
    )
    ==
    2554
)


# ------------------------------------------------------------
# 10. Inference-available descriptive geometry
#
# Geometry is NOT used for candidate selection here.
# We simply record whether the selected large components have
# different geometric characteristics before Step 7B.
# ------------------------------------------------------------

GEOM_AUDIT_FEATURES_7A = [
    "component_depth_std_norm",
    "component_depth_robust_range_norm",
    "boundary_vs_interior_abs_norm",
    "component_vs_ring_abs_norm",
    "mean_gradient_mag_norm",
]


geometry_audit_rows_7a = []


for candidate_value, candidate_name in [
    (False, "non_candidate"),
    (True, "candidate"),
]:

    subset = (
        split_candidates_7a_df[
            split_candidates_7a_df[
                "split_candidate"
            ]
            ==
            candidate_value
        ]
    )


    row = {
        "group":
            candidate_name,

        "num_components":
            int(
                len(
                    subset
                )
            ),

        "median_area_pixels":
            float(
                subset[
                    "area_pixels"
                ]
                .median()
            ),
    }


    for feature in GEOM_AUDIT_FEATURES_7A:

        row[
            f"median_{feature}"
        ] = float(
            subset[
                feature
            ]
            .median()
        )


    geometry_audit_rows_7a.append(
        row
    )


candidate_geometry_audit_7a_df = (
    pd.DataFrame(
        geometry_audit_rows_7a
    )
)


# ------------------------------------------------------------
# 11. Diagnostic-only GT audit
#
# IMPORTANT:
# The candidate rule above is already frozen.
#
# GT is examined only AFTER candidacy has been defined.
# These quantities are developmental diagnostics and must not
# be used to change the candidate rule in this step.
# ------------------------------------------------------------

gt_audit_rows_7a = []


for candidate_value, candidate_name in [
    (False, "non_candidate"),
    (True, "candidate"),
]:

    subset = (
        split_candidates_7a_df[
            split_candidates_7a_df[
                "split_candidate"
            ]
            ==
            candidate_value
        ]
    )


    gt0 = int(
        (
            subset[
                "component_gt_count"
            ]
            == 0
        ).sum()
    )

    gt1 = int(
        (
            subset[
                "component_gt_count"
            ]
            == 1
        ).sum()
    )

    gt2plus = int(
        (
            subset[
                "component_gt_count"
            ]
            >= 2
        ).sum()
    )


    gt_audit_rows_7a.append(
        {
            "group":
                candidate_name,

            "num_components":
                int(
                    len(
                        subset
                    )
                ),

            "GT0_components":
                gt0,

            "GT1_components":
                gt1,

            "GT2plus_components":
                gt2plus,

            "GT2plus_fraction":
                float(
                    gt2plus
                    /
                    len(
                        subset
                    )
                )
                if len(
                    subset
                ) > 0
                else np.nan,
        }
    )


candidate_gt_audit_7a_df = (
    pd.DataFrame(
        gt_audit_rows_7a
    )
)


# ------------------------------------------------------------
# 12. GT2+ capture diagnostic
#
# Again: evaluation only.
# ------------------------------------------------------------

total_gt2plus_components_7a = int(
    (
        split_candidates_7a_df[
            "component_gt_count"
        ]
        >= 2
    ).sum()
)

candidate_gt2plus_components_7a = int(
    (
        primary_candidates_7a_df[
            "component_gt_count"
        ]
        >= 2
    ).sum()
)

gt2plus_candidate_recall_7a = float(
    candidate_gt2plus_components_7a
    /
    total_gt2plus_components_7a
)


# ------------------------------------------------------------
# 13. Density-stratified diagnostic only
# ------------------------------------------------------------

candidate_density_7a_df = (
    split_candidates_7a_df
    .groupby(
        "density_stratum",
        as_index=False,
    )
    .agg(
        total_components=(
            "component_id",
            "count",
        ),

        candidate_components=(
            "split_candidate",
            "sum",
        ),
    )
)


candidate_density_7a_df[
    "candidate_fraction"
] = (
    candidate_density_7a_df[
        "candidate_components"
    ]
    /
    candidate_density_7a_df[
        "total_components"
    ]
)


# ------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------

M1_7A_COMPONENT_FILE = (
    M1_TABLES_DIR
    / "M1_7A_split_candidate_components.csv"
)

M1_7A_RULE_AUDIT_FILE = (
    M1_TABLES_DIR
    / "M1_7A_split_candidate_rule_audit.csv"
)

M1_7A_GEOMETRY_AUDIT_FILE = (
    M1_TABLES_DIR
    / "M1_7A_split_candidate_geometry_audit.csv"
)

M1_7A_GT_AUDIT_FILE = (
    M1_TABLES_DIR
    / "M1_7A_split_candidate_GT_diagnostic.csv"
)

M1_7A_DENSITY_FILE = (
    M1_TABLES_DIR
    / "M1_7A_split_candidate_by_density.csv"
)


split_candidates_7a_df.to_csv(
    M1_7A_COMPONENT_FILE,
    index=False,
)

candidate_rule_audit_7a_df.to_csv(
    M1_7A_RULE_AUDIT_FILE,
    index=False,
)

candidate_geometry_audit_7a_df.to_csv(
    M1_7A_GEOMETRY_AUDIT_FILE,
    index=False,
)

candidate_gt_audit_7a_df.to_csv(
    M1_7A_GT_AUDIT_FILE,
    index=False,
)

candidate_density_7a_df.to_csv(
    M1_7A_DENSITY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 15. Report
# ------------------------------------------------------------

print("\nCandidate-rule burden audit")
print("---------------------------")

display(
    candidate_rule_audit_7a_df
)


print("\nPRIMARY candidate rule")
print("----------------------")

print(
    "Within-image area percentile : >=",
    PRIMARY_SPLIT_AREA_PERCENTILE_7A,
)

print(
    "Total components             :",
    total_components_7a,
)

print(
    "Primary candidate components :",
    len(
        primary_candidates_7a_df
    ),
)

print(
    "Candidate fraction           :",
    round(
        len(
            primary_candidates_7a_df
        )
        /
        total_components_7a,
        4,
    ),
)

print(
    "Images with candidate        :",
    primary_candidates_7a_df[
        "dataset_index"
    ].nunique(),
    "/ 100",
)

print(
    "Foreground pixels affected   :",
    int(
        primary_candidates_7a_df[
            "area_pixels"
        ].sum()
    ),
)

print(
    "Foreground pixel fraction    :",
    round(
        primary_candidates_7a_df[
            "area_pixels"
        ].sum()
        /
        total_component_pixels_7a,
        4,
    ),
)


print("\nCandidate vs non-candidate geometry")
print("-----------------------------------")

display(
    candidate_geometry_audit_7a_df
)


print("\nGT diagnostic — NOT used for candidate definition")
print("--------------------------------------------------")

display(
    candidate_gt_audit_7a_df
)


print(
    "\nGT2+ components captured by primary candidates:",
    candidate_gt2plus_components_7a,
    "/",
    total_gt2plus_components_7a,
)

print(
    "GT2+ candidate recall:",
    round(
        gt2plus_candidate_recall_7a,
        4,
    ),
)


print("\nDensity diagnostic — NOT used for routing")
print("------------------------------------------")

display(
    candidate_density_7a_df
)


print("\nSaved candidate-component table:")
print(M1_7A_COMPONENT_FILE)

print("\nSaved candidate-rule audit:")
print(M1_7A_RULE_AUDIT_FILE)


# ------------------------------------------------------------
# 16. Protocol report
# ------------------------------------------------------------

print("\nPrimary candidate rule frozen : YES — within-image area >= 75th percentile")
print("GT used for candidate rule    : NO")
print("Marigold used for candidate rule: NO")
print("GT used for analysis          : YES — post-hoc diagnostic only")
print("Density used for routing      : NO")
print("Geometry threshold selected   : NO")
print("Semantic masks modified       : NO")
print("Components split              : NO")
print("Count rule modified           : NO")
print("Test set accessed             : NO")

print("\nStep 7A: PASS")

STEP 7A — DEFINE + AUDIT CANDIDATE COMPONENTS FOR GEOMETRY-GUIDED SPLITTING

Candidate-selection inputs:
['dataset_index', 'component_id', 'area_pixels']

GT used in candidate rule       : NO
Marigold used in candidate rule : NO

Candidate-rule burden audit
---------------------------


,area_percentile_rule,num_candidate_components,candidate_fraction_components,num_images_with_candidate,median_candidates_per_image,mean_candidates_per_image,max_candidates_per_image,candidate_pixels,candidate_fraction_foreground_pixels,median_candidate_area,primary_rule
0,50,1352,0.529366,100,11.0,13.52,54,2012950,0.955856,521.5,False
1,75,699,0.273688,100,6.0,6.99,27,1765578,0.838390,1131.0,True
2,90,309,0.120987,100,3.0,3.09,11,1358681,0.645174,2584.0,False



PRIMARY candidate rule
----------------------
Within-image area percentile : >= 75
Total components             : 2554
Primary candidate components : 699
Candidate fraction           : 0.2737
Images with candidate        : 100 / 100
Foreground pixels affected   : 1765578
Foreground pixel fraction    : 0.8384

Candidate vs non-candidate geometry
-----------------------------------


,group,num_components,median_area_pixels,median_component_depth_std_norm,median_component_depth_robust_range_norm,median_boundary_vs_interior_abs_norm,median_component_vs_ring_abs_norm,median_mean_gradient_mag_norm
0,non_candidate,1855,71.0,0.011930,0.035684,0.004158,0.004157,0.005844
1,candidate,699,1131.0,0.049196,0.140509,0.016792,0.020426,0.007384



GT diagnostic — NOT used for candidate definition
--------------------------------------------------


,group,num_components,GT0_components,GT1_components,GT2plus_components,GT2plus_fraction
0,non_candidate,1855,1294,512,49,0.026415
1,candidate,699,182,248,269,0.384835



GT2+ components captured by primary candidates: 269 / 318
GT2+ candidate recall: 0.8459

Density diagnostic — NOT used for routing
------------------------------------------


,density_stratum,total_components,candidate_components,candidate_fraction
0,S1_0_20,240,74,0.308333
1,S2_20_40,392,110,0.280612
2,S3_40_60,401,111,0.276808
3,S4_60_80,566,153,0.270318
4,S5_80_95,465,123,0.264516
5,S6_95_100,490,128,0.261224



Saved candidate-component table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_7A_split_candidate_components.csv

Saved candidate-rule audit:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_7A_split_candidate_rule_audit.csv

Primary candidate rule frozen : YES — within-image area >= 75th percentile
GT used for candidate rule    : NO
Marigold used for candidate rule: NO
GT used for analysis          : YES — post-hoc diagnostic only
Density used for routing      : NO
Geometry threshold selected   : NO
Semantic masks modified       : NO
Components split              : NO
Count rule modified           : NO
Test set accessed             : NO

Step 7A: PASS


### **Finding 5 — Step 7A:**

An inference-only within-image component-area rule provides a practical screening mechanism for geometry-guided refinement.

The frozen 75th-percentile rule selects 27.4% of A3c components while capturing 84.6% of GT2+ components in the post-hoc diagnostic.

However, most selected candidates are GT0 or GT1, so component size is suitable for defining eligibility for refinement, not as evidence that a component should actually be split.

In [26]:
# ============================================================
# Step 7B — Generate Marigold depth-partition split proposals
# ============================================================
#
# Purpose
# -------
# Apply one simple, controlled geometry-based splitting
# mechanism to the candidate A3c/C1 components frozen in
# Step 7A.
#
# Candidate selection:
#   - unchanged from Step 7A
#   - within-image component-area >= 75th percentile
#
# Splitting mechanism:
#
#   1. Take the Marigold depth values INSIDE one candidate
#      A3c component.
#
#   2. Partition those depth values into two groups using
#      deterministic 1-D k-means (k=2).
#
#   3. For each depth group, recover its 4-connected spatial
#      components.
#
#   4. Each resulting connected region becomes a RAW split
#      proposal subcomponent.
#
# Non-candidate A3c components are preserved unchanged.
#
# IMPORTANT
# ---------
# This is deliberately a RAW proposal-generation experiment.
#
# We do NOT:
#   - use GT to decide whether to split
#   - tune a depth threshold
#   - remove small fragments
#   - merge fragments
#   - use density routing
#   - modify the count rule
#
# Step 7C will evaluate whether these proposals are useful
# or harmful before any count-level decision is made.
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy import ndimage
from sklearn.cluster import KMeans


print("=" * 120)
print("STEP 7B — MARIGOLD DEPTH-PARTITION SPLIT PROPOSALS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "split_candidates_7a_df" in globals(), (
    "split_candidates_7a_df unavailable. "
    "Run Step 7A first."
)

assert "a3c_align_6d_df" in globals(), (
    "a3c_align_6d_df unavailable. "
    "Run Step 6D Part B first."
)

assert len(split_candidates_7a_df) == 2554
assert len(a3c_align_6d_df) == 100

assert "split_candidate" in split_candidates_7a_df.columns

assert (
    split_candidates_7a_df[
        "split_candidate"
    ].sum()
    == 699
), (
    "Expected frozen Step-7A primary candidate count = 699."
)


# ------------------------------------------------------------
# 2. Freeze Step-7B splitting protocol
# ------------------------------------------------------------

DEPTH_SPLIT_K_7B = 2

DEPTH_SPLIT_RANDOM_STATE_7B = 0

DEPTH_SPLIT_N_INIT_7B = 10


# Same 4-connectivity used by frozen C1
STRUCTURE_4_7B = (
    ndimage.generate_binary_structure(
        rank=2,
        connectivity=1,
    )
)


print("\nFrozen Step-7B protocol")
print("-----------------------")

print(
    "Candidate rule          : Step-7A frozen area >= 75th percentile"
)

print(
    "Depth partition         : 1-D k-means"
)

print(
    "k                       :",
    DEPTH_SPLIT_K_7B,
)

print(
    "random_state            :",
    DEPTH_SPLIT_RANDOM_STATE_7B,
)

print(
    "n_init                  :",
    DEPTH_SPLIT_N_INIT_7B,
)

print(
    "Spatial connectivity    : 4-connected"
)

print(
    "Small-fragment filtering: NONE"
)

print(
    "GT used                 : NO"
)


# ------------------------------------------------------------
# 3. Output directories
# ------------------------------------------------------------

M1_7B_ROOT = (
    M1_TABLES_DIR.parent
    / "step7_geometry_refinement"
)

M1_7B_LABEL_DIR = (
    M1_7B_ROOT
    / "raw_split_label_maps"
)

M1_7B_METADATA_DIR = (
    M1_7B_ROOT
    / "metadata"
)


for directory in [
    M1_7B_ROOT,
    M1_7B_LABEL_DIR,
    M1_7B_METADATA_DIR,
]:

    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ------------------------------------------------------------
# 4. Candidate lookup
# ------------------------------------------------------------

candidate_lookup_7b = {}

for (
    dataset_index,
    image_df
) in (
    split_candidates_7a_df
    .groupby(
        "dataset_index"
    )
):

    candidate_lookup_7b[
        int(dataset_index)
    ] = {
        int(component_id)
        for component_id
        in image_df.loc[
            image_df[
                "split_candidate"
            ],
            "component_id",
        ].tolist()
    }


assert len(candidate_lookup_7b) == 100

assert (
    sum(
        len(v)
        for v in candidate_lookup_7b.values()
    )
    == 699
)


# ------------------------------------------------------------
# 5. Storage
# ------------------------------------------------------------

proposal_rows_7b = []

candidate_summary_rows_7b = []

image_summary_rows_7b = []


# ------------------------------------------------------------
# 6. Process all 100 images
# ------------------------------------------------------------

for _, image_row in (
    a3c_align_6d_df
    .sort_values(
        "dataset_index"
    )
    .iterrows()
):

    dataset_index = int(
        image_row[
            "dataset_index"
        ]
    )

    image_id = str(
        image_row[
            "image_id"
        ]
    )


    # --------------------------------------------------------
    # Load frozen A3c semantic mask
    # --------------------------------------------------------

    with np.load(
        Path(
            image_row[
                "cache_file"
            ]
        ),
        allow_pickle=False,
    ) as data:

        a3c_mask = (
            np.asarray(
                data["mask"]
            )
            .astype(bool)
        )


    # --------------------------------------------------------
    # Load aligned Marigold depth
    # --------------------------------------------------------

    with np.load(
        Path(
            image_row[
                "cache_path"
            ]
        ),
        allow_pickle=False,
    ) as data:

        depth = (
            np.asarray(
                data["depth"]
            )
            .astype(
                np.float32
            )
        )


    assert (
        a3c_mask.shape
        ==
        depth.shape
    )


    # --------------------------------------------------------
    # Reconstruct frozen C1 component map
    # --------------------------------------------------------

    c1_labels, num_c1_components = (
        ndimage.label(
            a3c_mask,
            structure=
                STRUCTURE_4_7B,
        )
    )


    assert (
        int(
            num_c1_components
        )
        ==
        int(
            image_row[
                "num_components"
            ]
        )
    )


    candidate_ids = (
        candidate_lookup_7b[
            dataset_index
        ]
    )


    # --------------------------------------------------------
    # Full-image raw split proposal label map
    #
    # Every final proposal subcomponent receives a unique
    # positive integer label.
    # --------------------------------------------------------

    proposal_label_map = np.zeros(
        a3c_mask.shape,
        dtype=np.int32,
    )

    next_proposal_label = 1


    image_candidate_count = 0

    image_original_components = int(
        num_c1_components
    )

    image_raw_subcomponents = 0

    image_candidate_subcomponents = 0


    # --------------------------------------------------------
    # Process each frozen C1 component
    # --------------------------------------------------------

    for component_id in range(
        1,
        num_c1_components + 1,
    ):

        component_mask = (
            c1_labels
            ==
            component_id
        )

        component_area = int(
            component_mask.sum()
        )

        is_candidate = (
            component_id
            in
            candidate_ids
        )


        # ====================================================
        # NON-CANDIDATE:
        # preserve exactly as one proposal component
        # ====================================================

        if not is_candidate:

            proposal_label = (
                next_proposal_label
            )

            next_proposal_label += 1


            proposal_label_map[
                component_mask
            ] = (
                proposal_label
            )


            proposal_rows_7b.append(
                {
                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_id,

                    "original_component_id":
                        int(
                            component_id
                        ),

                    "split_candidate":
                        False,

                    "split_attempted":
                        False,

                    "split_status":
                        "non_candidate_preserved",

                    "depth_cluster":
                        -1,

                    "spatial_component_within_cluster":
                        1,

                    "proposal_subcomponent_id":
                        int(
                            proposal_label
                        ),

                    "original_component_area":
                        component_area,

                    "proposal_area":
                        component_area,

                    "proposal_area_fraction_of_original":
                        1.0,

                    "cluster_depth_center":
                        np.nan,

                    "num_unique_depth_values":
                        int(
                            np.unique(
                                depth[
                                    component_mask
                                ]
                            ).size
                        ),
                }
            )


            image_raw_subcomponents += 1

            continue


        # ====================================================
        # CANDIDATE COMPONENT
        # ====================================================

        image_candidate_count += 1


        component_depth = (
            depth[
                component_mask
            ]
            .reshape(
                -1,
                1,
            )
        )


        unique_depth_values = int(
            np.unique(
                component_depth
            ).size
        )


        # ----------------------------------------------------
        # Degenerate geometry:
        # fewer than two distinct depth values
        #
        # Preserve unchanged rather than invent a split.
        # ----------------------------------------------------

        if (
            unique_depth_values
            <
            DEPTH_SPLIT_K_7B
        ):

            proposal_label = (
                next_proposal_label
            )

            next_proposal_label += 1


            proposal_label_map[
                component_mask
            ] = (
                proposal_label
            )


            proposal_rows_7b.append(
                {
                    "dataset_index":
                        dataset_index,

                    "image_id":
                        image_id,

                    "original_component_id":
                        int(
                            component_id
                        ),

                    "split_candidate":
                        True,

                    "split_attempted":
                        False,

                    "split_status":
                        "degenerate_depth_preserved",

                    "depth_cluster":
                        -1,

                    "spatial_component_within_cluster":
                        1,

                    "proposal_subcomponent_id":
                        int(
                            proposal_label
                        ),

                    "original_component_area":
                        component_area,

                    "proposal_area":
                        component_area,

                    "proposal_area_fraction_of_original":
                        1.0,

                    "cluster_depth_center":
                        float(
                            np.mean(
                                component_depth
                            )
                        ),

                    "num_unique_depth_values":
                        unique_depth_values,
                }
            )


            image_raw_subcomponents += 1

            image_candidate_subcomponents += 1

            continue


        # ----------------------------------------------------
        # Deterministic 1-D k-means depth partition
        # ----------------------------------------------------

        kmeans = KMeans(
            n_clusters=
                DEPTH_SPLIT_K_7B,

            random_state=
                DEPTH_SPLIT_RANDOM_STATE_7B,

            n_init=
                DEPTH_SPLIT_N_INIT_7B,
        )


        depth_cluster_vector = (
            kmeans.fit_predict(
                component_depth
            )
        )


        cluster_centres = (
            kmeans.cluster_centers_
            .reshape(-1)
        )


        # ----------------------------------------------------
        # Put cluster assignment back into image coordinates
        # ----------------------------------------------------

        component_y, component_x = (
            np.where(
                component_mask
            )
        )


        depth_cluster_map = np.full(
            a3c_mask.shape,
            fill_value=-1,
            dtype=np.int8,
        )


        depth_cluster_map[
            component_y,
            component_x,
        ] = (
            depth_cluster_vector
            .astype(
                np.int8
            )
        )


        candidate_proposal_count = 0


        # ----------------------------------------------------
        # Within each depth cluster:
        # recover 4-connected spatial pieces
        # ----------------------------------------------------

        for depth_cluster_id in range(
            DEPTH_SPLIT_K_7B
        ):

            cluster_mask = (
                component_mask
                &
                (
                    depth_cluster_map
                    ==
                    depth_cluster_id
                )
            )


            spatial_labels, num_spatial = (
                ndimage.label(
                    cluster_mask,
                    structure=
                        STRUCTURE_4_7B,
                )
            )


            for spatial_id in range(
                1,
                num_spatial + 1,
            ):

                proposal_mask = (
                    spatial_labels
                    ==
                    spatial_id
                )

                proposal_area = int(
                    proposal_mask.sum()
                )


                assert (
                    proposal_area
                    > 0
                )


                proposal_label = (
                    next_proposal_label
                )

                next_proposal_label += 1


                proposal_label_map[
                    proposal_mask
                ] = (
                    proposal_label
                )


                candidate_proposal_count += 1

                image_raw_subcomponents += 1

                image_candidate_subcomponents += 1


                proposal_rows_7b.append(
                    {
                        "dataset_index":
                            dataset_index,

                        "image_id":
                            image_id,

                        "original_component_id":
                            int(
                                component_id
                            ),

                        "split_candidate":
                            True,

                        "split_attempted":
                            True,

                        "split_status":
                            "raw_depth_partition",

                        "depth_cluster":
                            int(
                                depth_cluster_id
                            ),

                        "spatial_component_within_cluster":
                            int(
                                spatial_id
                            ),

                        "proposal_subcomponent_id":
                            int(
                                proposal_label
                            ),

                        "original_component_area":
                            component_area,

                        "proposal_area":
                            proposal_area,

                        "proposal_area_fraction_of_original":
                            float(
                                proposal_area
                                /
                                component_area
                            ),

                        "cluster_depth_center":
                            float(
                                cluster_centres[
                                    depth_cluster_id
                                ]
                            ),

                        "num_unique_depth_values":
                            unique_depth_values,
                    }
                )


        # ----------------------------------------------------
        # Candidate-level summary
        # ----------------------------------------------------

        candidate_summary_rows_7b.append(
            {
                "dataset_index":
                    dataset_index,

                "image_id":
                    image_id,

                "original_component_id":
                    int(
                        component_id
                    ),

                "original_component_area":
                    component_area,

                "num_unique_depth_values":
                    unique_depth_values,

                "split_attempted":
                    True,

                "num_raw_subcomponents":
                    int(
                        candidate_proposal_count
                    ),

                "raw_count_increment":
                    int(
                        candidate_proposal_count
                        -
                        1
                    ),

                "cluster_center_0":
                    float(
                        cluster_centres[
                            0
                        ]
                    ),

                "cluster_center_1":
                    float(
                        cluster_centres[
                            1
                        ]
                    ),

                "cluster_center_abs_difference":
                    float(
                        abs(
                            cluster_centres[
                                1
                            ]
                            -
                            cluster_centres[
                                0
                            ]
                        )
                    ),
            }
        )


    # --------------------------------------------------------
    # Proposal-map integrity
    #
    # Union of proposal regions MUST exactly equal A3c.
    # No pixels may be added or removed.
    # --------------------------------------------------------

    proposal_foreground = (
        proposal_label_map
        >
        0
    )


    exact_foreground_identity = bool(
        np.array_equal(
            proposal_foreground,
            a3c_mask,
        )
    )


    assert exact_foreground_identity, (
        f"Proposal foreground differs from A3c for {image_id}"
    )


    # --------------------------------------------------------
    # Save full-image proposal label map
    # --------------------------------------------------------

    proposal_map_file = (
        M1_7B_LABEL_DIR
        /
        (
            f"{dataset_index:05d}_"
            f"{Path(image_id).stem}_"
            f"raw_depth_split_labels.npz"
        )
    )


    np.savez_compressed(
        proposal_map_file,

        proposal_labels=
            proposal_label_map,

        dataset_index=
            np.int32(
                dataset_index
            ),

        image_id=
            np.array(
                image_id
            ),

        original_num_components=
            np.int32(
                image_original_components
            ),

        raw_num_subcomponents=
            np.int32(
                image_raw_subcomponents
            ),

        num_candidate_components=
            np.int32(
                image_candidate_count
            ),

        height=
            np.int32(
                a3c_mask.shape[0]
            ),

        width=
            np.int32(
                a3c_mask.shape[1]
            ),
    )


    # --------------------------------------------------------
    # Image-level summary
    # --------------------------------------------------------

    image_summary_rows_7b.append(
        {
            "dataset_index":
                dataset_index,

            "image_id":
                image_id,

            "original_num_components":
                image_original_components,

            "num_candidate_components":
                image_candidate_count,

            "raw_num_subcomponents":
                image_raw_subcomponents,

            "raw_count_increment":
                int(
                    image_raw_subcomponents
                    -
                    image_original_components
                ),

            "candidate_raw_subcomponents":
                image_candidate_subcomponents,

            "foreground_identity":
                exact_foreground_identity,

            "proposal_map_file":
                str(
                    proposal_map_file
                ),
        }
    )


# ------------------------------------------------------------
# 7. Build output tables
# ------------------------------------------------------------

raw_split_proposals_7b_df = (
    pd.DataFrame(
        proposal_rows_7b
    )
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


candidate_split_summary_7b_df = (
    pd.DataFrame(
        candidate_summary_rows_7b
    )
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


image_split_summary_7b_df = (
    pd.DataFrame(
        image_summary_rows_7b
    )
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 8. Integrity checks
# ------------------------------------------------------------

assert len(
    image_split_summary_7b_df
) == 100


assert (
    image_split_summary_7b_df[
        "foreground_identity"
    ]
    .all()
)


assert (
    image_split_summary_7b_df[
        "original_num_components"
    ]
    .sum()
    == 2554
)


assert (
    image_split_summary_7b_df[
        "num_candidate_components"
    ]
    .sum()
    == 699
)


# Every original component must appear in proposal table
assert (
    raw_split_proposals_7b_df[
        [
            "dataset_index",
            "original_component_id",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    == 2554
)


# Proposal labels must be unique within each image
proposal_uniqueness_check_7b = (
    raw_split_proposals_7b_df
    .groupby(
        "dataset_index"
    )[
        "proposal_subcomponent_id"
    ]
    .apply(
        lambda s:
            s.nunique()
            ==
            len(s)
    )
)

assert (
    proposal_uniqueness_check_7b
    .all()
)


# ------------------------------------------------------------
# 9. Mechanical fragmentation summary
#
# IMPORTANT:
# This is NOT count performance.
# It only describes how aggressively the raw geometry
# partition fragments the candidate components.
# ------------------------------------------------------------

fragmentation_summary_7b = {
    "original_components":
        int(
            image_split_summary_7b_df[
                "original_num_components"
            ].sum()
        ),

    "candidate_components":
        int(
            image_split_summary_7b_df[
                "num_candidate_components"
            ].sum()
        ),

    "raw_proposal_subcomponents":
        int(
            image_split_summary_7b_df[
                "raw_num_subcomponents"
            ].sum()
        ),

    "raw_component_increment":
        int(
            image_split_summary_7b_df[
                "raw_count_increment"
            ].sum()
        ),

    "median_raw_subcomponents_per_candidate":
        float(
            candidate_split_summary_7b_df[
                "num_raw_subcomponents"
            ].median()
        ),

    "mean_raw_subcomponents_per_candidate":
        float(
            candidate_split_summary_7b_df[
                "num_raw_subcomponents"
            ].mean()
        ),

    "max_raw_subcomponents_per_candidate":
        int(
            candidate_split_summary_7b_df[
                "num_raw_subcomponents"
            ].max()
        ),
}


fragmentation_summary_7b_df = (
    pd.DataFrame(
        [
            fragmentation_summary_7b
        ]
    )
)


# ------------------------------------------------------------
# 10. Candidate fragmentation distribution
# ------------------------------------------------------------

candidate_fragmentation_distribution_7b_df = (
    candidate_split_summary_7b_df[
        "num_raw_subcomponents"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "num_raw_subcomponents"
    )
    .reset_index(
        name="num_candidate_components"
    )
)


candidate_fragmentation_distribution_7b_df[
    "fraction_candidates"
] = (
    candidate_fragmentation_distribution_7b_df[
        "num_candidate_components"
    ]
    /
    len(
        candidate_split_summary_7b_df
    )
)


# ------------------------------------------------------------
# 11. Save tables
# ------------------------------------------------------------

M1_7B_PROPOSALS_FILE = (
    M1_TABLES_DIR
    / "M1_7B_raw_depth_split_proposals.csv"
)

M1_7B_CANDIDATE_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_7B_candidate_split_summary.csv"
)

M1_7B_IMAGE_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_7B_image_split_summary.csv"
)

M1_7B_FRAGMENTATION_FILE = (
    M1_TABLES_DIR
    / "M1_7B_raw_fragmentation_summary.csv"
)

M1_7B_FRAGMENTATION_DISTRIBUTION_FILE = (
    M1_TABLES_DIR
    / "M1_7B_raw_fragmentation_distribution.csv"
)


raw_split_proposals_7b_df.to_csv(
    M1_7B_PROPOSALS_FILE,
    index=False,
)

candidate_split_summary_7b_df.to_csv(
    M1_7B_CANDIDATE_SUMMARY_FILE,
    index=False,
)

image_split_summary_7b_df.to_csv(
    M1_7B_IMAGE_SUMMARY_FILE,
    index=False,
)

fragmentation_summary_7b_df.to_csv(
    M1_7B_FRAGMENTATION_FILE,
    index=False,
)

candidate_fragmentation_distribution_7b_df.to_csv(
    M1_7B_FRAGMENTATION_DISTRIBUTION_FILE,
    index=False,
)


# ------------------------------------------------------------
# 12. Report
# ------------------------------------------------------------

print("\nRaw fragmentation summary")
print("-------------------------")

display(
    fragmentation_summary_7b_df
)


print("\nCandidate fragmentation distribution")
print("------------------------------------")

display(
    candidate_fragmentation_distribution_7b_df
)


print("\nImage-level raw split summary")
print("-----------------------------")

display(
    image_split_summary_7b_df[
        [
            "dataset_index",
            "original_num_components",
            "num_candidate_components",
            "raw_num_subcomponents",
            "raw_count_increment",
            "foreground_identity",
        ]
    ]
    .head(20)
)


print("\nSaved raw proposal table:")
print(M1_7B_PROPOSALS_FILE)

print("\nSaved candidate summary:")
print(M1_7B_CANDIDATE_SUMMARY_FILE)

print("\nSaved image summary:")
print(M1_7B_IMAGE_SUMMARY_FILE)

print("\nSaved proposal label maps:")
print(M1_7B_LABEL_DIR)


# ------------------------------------------------------------
# 13. Protocol report
# ------------------------------------------------------------

print("\nStep-7A candidate rule reused : YES — unchanged")
print("GT used                       : NO")
print("Marigold used                 : YES — splitting mechanism only")
print("Global depth threshold used   : NO")
print("Small-fragment filtering      : NO")
print("Semantic foreground added     : NO")
print("Semantic foreground removed   : NO")
print("A3c foreground preserved      : YES — exact")
print("Density used for routing      : NO")
print("Count performance evaluated   : NO")
print("Count rule modified           : NO")
print("Test set accessed             : NO")

print("\nStep 7B: PASS")

STEP 7B — MARIGOLD DEPTH-PARTITION SPLIT PROPOSALS

Frozen Step-7B protocol
-----------------------
Candidate rule          : Step-7A frozen area >= 75th percentile
Depth partition         : 1-D k-means
k                       : 2
random_state            : 0
n_init                  : 10
Spatial connectivity    : 4-connected
Small-fragment filtering: NONE
GT used                 : NO

Raw fragmentation summary
-------------------------


,original_components,candidate_components,raw_proposal_subcomponents,raw_component_increment,median_raw_subcomponents_per_candidate,mean_raw_subcomponents_per_candidate,max_raw_subcomponents_per_candidate
0,2554,699,8377,5823,5.0,9.330472,359



Candidate fragmentation distribution
------------------------------------


,num_raw_subcomponents,num_candidate_components,fraction_candidates
0,2,141,0.201717
1,3,110,0.157368
2,4,90,0.128755
3,5,63,0.090129
4,6,33,0.047210
5,7,44,0.062947
6,8,29,0.041488
7,9,20,0.028612
8,10,19,0.027182
9,11,18,0.025751



Image-level raw split summary
-----------------------------


,dataset_index,original_num_components,num_candidate_components,raw_num_subcomponents,raw_count_increment,foreground_identity
0,9,6,2,28,22,True
1,14,32,9,99,67,True
2,20,26,7,67,41,True
3,31,7,2,11,4,True
4,40,1,1,5,4,True
5,49,8,3,78,70,True
6,52,18,5,37,19,True
7,59,8,3,28,20,True
8,67,9,3,35,26,True
9,87,10,3,55,45,True



Saved raw proposal table:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_7B_raw_depth_split_proposals.csv

Saved candidate summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_7B_candidate_split_summary.csv

Saved image summary:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/tables/M1_7B_image_split_summary.csv

Saved proposal label maps:
/content/drive/MyDrive/MSc_Project_ZeroShot_Counting/results/M1_Marigold_Geometric_Diagnostics/step7_geometry_refinement/raw_split_label_maps

Step-7A candidate rule reused : YES — unchanged
GT used                       : NO
Marigold used                 : YES — splitting mechanism only
Global depth threshold used   : NO
Small-fragment filtering      : NO
Semantic foreground added     : NO
Semantic foreground removed   : NO
A3c foreground preserved      : YES — exact
Density used for r

### **Finding 6 — Step 7B:**

Naïve binary Marigold depth partitioning preserves the A3c semantic foreground exactly but causes substantial spatial over-fragmentation.

Although depth is partitioned into only two clusters, spatial disconnectedness within each depth cluster produces a median of five and a mean of 9.33 subcomponents per candidate, with extreme cases reaching hundreds of fragments.

Therefore, raw depth partition components cannot be interpreted directly as object instances.

In [27]:
# ============================================================
# Step 7C Part A — Structural diagnosis of raw Marigold
#                  split proposals
# ============================================================
#
# Purpose
# -------
# Diagnose WHY the raw Step-7B depth partition produced heavy
# fragmentation before introducing any filtering, merging, or
# split-acceptance rule.
#
# Questions:
#
#   1. How does fragmentation differ between original
#      GT0 / GT1 / GT2+ A3c components?
#
#   2. Are most extra pieces very small relative to the
#      original semantic component?
#
#   3. How much original area is retained by the largest
#      one / two raw fragments?
#
#   4. How many raw fragments are needed to explain 90%
#      of the original component area?
#
#   5. Is fragmentation largely related to original
#      component size?
#
# IMPORTANT
# ---------
# GT is used ONLY for post-hoc diagnosis.
#
# No:
#   - filtering
#   - fragment deletion
#   - fragment merging
#   - threshold selection
#   - count modification
#   - density routing
#
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import spearmanr


print("=" * 120)
print("STEP 7C PART A — STRUCTURAL DIAGNOSIS OF RAW DEPTH-SPLIT PROPOSALS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "raw_split_proposals_7b_df" in globals(), (
    "raw_split_proposals_7b_df unavailable. "
    "Run Step 7B first."
)

assert "candidate_split_summary_7b_df" in globals(), (
    "candidate_split_summary_7b_df unavailable. "
    "Run Step 7B first."
)

assert "a3c_component_geom_df" in globals(), (
    "a3c_component_geom_df unavailable. "
    "Run Step 6D Part C first."
)


# Step 7C concerns ONLY the 699 frozen Step-7A candidates
candidate_proposals_7c_df = (
    raw_split_proposals_7b_df[
        raw_split_proposals_7b_df[
            "split_candidate"
        ]
    ]
    .copy()
)


assert (
    candidate_proposals_7c_df[
        [
            "dataset_index",
            "original_component_id",
        ]
    ]
    .drop_duplicates()
    .shape[0]
    == 699
)


# ------------------------------------------------------------
# 2. Attach frozen GT diagnostic label of ORIGINAL component
#
# component_gt_count was produced during Step 6D from exact
# FSC-147 GT-point containment inside the original A3c
# component.
#
# GT is NOT used to generate the proposals.
# ------------------------------------------------------------

gt_lookup_7c_df = (
    a3c_component_geom_df[
        [
            "dataset_index",
            "component_id",
            "component_gt_count",
        ]
    ]
    .copy()
    .rename(
        columns={
            "component_id":
                "original_component_id"
        }
    )
)


candidate_proposals_7c_df = (
    candidate_proposals_7c_df
    .merge(
        gt_lookup_7c_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    candidate_proposals_7c_df[
        "component_gt_count"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 3. GT grouping — diagnostic only
# ------------------------------------------------------------

candidate_proposals_7c_df[
    "gt_group"
] = np.select(
    [
        candidate_proposals_7c_df[
            "component_gt_count"
        ] == 0,

        candidate_proposals_7c_df[
            "component_gt_count"
        ] == 1,

        candidate_proposals_7c_df[
            "component_gt_count"
        ] >= 2,
    ],
    [
        "GT0",
        "GT1",
        "GT2+",
    ],
    default="UNKNOWN",
)


assert (
    candidate_proposals_7c_df[
        "gt_group"
    ]
    != "UNKNOWN"
).all()


# ------------------------------------------------------------
# 4. Build one row per ORIGINAL candidate component
#
# For each component calculate:
#
#   - number of raw fragments
#   - largest-fragment area fraction
#   - top-2 area fraction
#   - tiny-fragment burden
#   - number of fragments required to cover 90% area
#
# Relative fragment-size definitions are DIAGNOSTIC ONLY.
#
# We inspect:
#   <= 1% original area
#   <= 5% original area
#
# These are NOT proposed filtering thresholds.
# ------------------------------------------------------------

component_diag_rows_7c = []


for (
    dataset_index,
    original_component_id
), group_df in (
    candidate_proposals_7c_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ],
        sort=True,
    )
):

    group_df = (
        group_df
        .sort_values(
            "proposal_area",
            ascending=False,
        )
        .reset_index(
            drop=True
        )
    )


    original_area = int(
        group_df[
            "original_component_area"
        ].iloc[0]
    )

    component_gt_count = int(
        group_df[
            "component_gt_count"
        ].iloc[0]
    )

    gt_group = str(
        group_df[
            "gt_group"
        ].iloc[0]
    )


    fragment_areas = (
        group_df[
            "proposal_area"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    fragment_fracs = (
        fragment_areas
        /
        original_area
    )


    num_fragments = int(
        len(
            fragment_areas
        )
    )


    # --------------------------------------------------------
    # Integrity:
    # raw split fragments must exactly repartition original
    # semantic component area.
    # --------------------------------------------------------

    assert (
        fragment_areas.sum()
        ==
        original_area
    ), (
        f"Area mismatch for dataset_index={dataset_index}, "
        f"component={original_component_id}"
    )


    largest_fraction = float(
        fragment_fracs[0]
    )


    top2_fraction = float(
        fragment_fracs[
            :min(
                2,
                num_fragments,
            )
        ].sum()
    )


    # --------------------------------------------------------
    # Tiny-island diagnostics
    #
    # Purely descriptive.
    # Do NOT interpret these as filtering thresholds yet.
    # --------------------------------------------------------

    tiny_1pct_mask = (
        fragment_fracs
        <= 0.01
    )

    tiny_5pct_mask = (
        fragment_fracs
        <= 0.05
    )


    num_tiny_1pct = int(
        tiny_1pct_mask.sum()
    )

    num_tiny_5pct = int(
        tiny_5pct_mask.sum()
    )


    area_fraction_tiny_1pct = float(
        fragment_fracs[
            tiny_1pct_mask
        ].sum()
    )


    area_fraction_tiny_5pct = float(
        fragment_fracs[
            tiny_5pct_mask
        ].sum()
    )


    # --------------------------------------------------------
    # Number of largest fragments required to explain >=90%
    # of original semantic area.
    # --------------------------------------------------------

    cumulative_area_fraction = (
        np.cumsum(
            fragment_fracs
        )
    )


    num_fragments_for_90pct = int(
        np.searchsorted(
            cumulative_area_fraction,
            0.90,
            side="left",
        )
        + 1
    )


    # --------------------------------------------------------
    # Fragment-count excess
    # --------------------------------------------------------

    raw_fragment_increment = int(
        num_fragments
        - 1
    )


    component_diag_rows_7c.append(
        {
            "dataset_index":
                int(
                    dataset_index
                ),

            "original_component_id":
                int(
                    original_component_id
                ),

            "original_component_area":
                original_area,

            "component_gt_count":
                component_gt_count,

            "gt_group":
                gt_group,

            "num_raw_fragments":
                num_fragments,

            "raw_fragment_increment":
                raw_fragment_increment,

            "largest_fragment_fraction":
                largest_fraction,

            "top2_fragment_fraction":
                top2_fraction,

            "num_fragments_le_1pct":
                num_tiny_1pct,

            "fraction_fragments_le_1pct":
                float(
                    num_tiny_1pct
                    /
                    num_fragments
                ),

            "area_fraction_in_le_1pct_fragments":
                area_fraction_tiny_1pct,

            "num_fragments_le_5pct":
                num_tiny_5pct,

            "fraction_fragments_le_5pct":
                float(
                    num_tiny_5pct
                    /
                    num_fragments
                ),

            "area_fraction_in_le_5pct_fragments":
                area_fraction_tiny_5pct,

            "num_fragments_for_90pct_area":
                num_fragments_for_90pct,
        }
    )


component_fragment_diag_7c_df = (
    pd.DataFrame(
        component_diag_rows_7c
    )
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    component_fragment_diag_7c_df
) == 699


# ------------------------------------------------------------
# 5. Check frozen Step-7A GT-group inventory
# ------------------------------------------------------------

gt_inventory_7c_df = (
    component_fragment_diag_7c_df[
        "gt_group"
    ]
    .value_counts()
    .reindex(
        [
            "GT0",
            "GT1",
            "GT2+",
        ]
    )
    .rename_axis(
        "gt_group"
    )
    .reset_index(
        name="num_candidate_components"
    )
)


# Expected from Step 7A:
# GT0 = 182
# GT1 = 248
# GT2+ = 269

assert (
    gt_inventory_7c_df
    .set_index(
        "gt_group"
    )
    .loc[
        "GT0",
        "num_candidate_components"
    ]
    == 182
)

assert (
    gt_inventory_7c_df
    .set_index(
        "gt_group"
    )
    .loc[
        "GT1",
        "num_candidate_components"
    ]
    == 248
)

assert (
    gt_inventory_7c_df
    .set_index(
        "gt_group"
    )
    .loc[
        "GT2+",
        "num_candidate_components"
    ]
    == 269
)


# ------------------------------------------------------------
# 6. Summarise fragmentation by ORIGINAL GT group
# ------------------------------------------------------------

def q25(series):
    return series.quantile(
        0.25
    )


def q75(series):
    return series.quantile(
        0.75
    )


fragmentation_by_gt_7c_df = (
    component_fragment_diag_7c_df
    .groupby(
        "gt_group",
        observed=True,
    )
    .agg(
        num_components=(
            "original_component_id",
            "count",
        ),

        median_original_area=(
            "original_component_area",
            "median",
        ),

        median_raw_fragments=(
            "num_raw_fragments",
            "median",
        ),

        mean_raw_fragments=(
            "num_raw_fragments",
            "mean",
        ),

        q25_raw_fragments=(
            "num_raw_fragments",
            q25,
        ),

        q75_raw_fragments=(
            "num_raw_fragments",
            q75,
        ),

        max_raw_fragments=(
            "num_raw_fragments",
            "max",
        ),

        median_largest_fragment_fraction=(
            "largest_fragment_fraction",
            "median",
        ),

        median_top2_fragment_fraction=(
            "top2_fragment_fraction",
            "median",
        ),

        median_fraction_fragments_le_1pct=(
            "fraction_fragments_le_1pct",
            "median",
        ),

        median_area_fraction_le_1pct=(
            "area_fraction_in_le_1pct_fragments",
            "median",
        ),

        median_fraction_fragments_le_5pct=(
            "fraction_fragments_le_5pct",
            "median",
        ),

        median_area_fraction_le_5pct=(
            "area_fraction_in_le_5pct_fragments",
            "median",
        ),

        median_fragments_for_90pct_area=(
            "num_fragments_for_90pct_area",
            "median",
        ),

        mean_fragments_for_90pct_area=(
            "num_fragments_for_90pct_area",
            "mean",
        ),
    )
    .reindex(
        [
            "GT0",
            "GT1",
            "GT2+",
        ]
    )
    .reset_index()
)


# ------------------------------------------------------------
# 7. Global tiny-fragment burden
#
# Calculate separately by NUMBER and by PIXEL AREA.
# This distinction is important:
#
# many fragments can be tiny numerically while representing
# only a small percentage of foreground pixels.
# ------------------------------------------------------------

candidate_proposals_7c_df[
    "fragment_fraction_original"
] = (
    candidate_proposals_7c_df[
        "proposal_area"
    ]
    /
    candidate_proposals_7c_df[
        "original_component_area"
    ]
)


total_candidate_fragments_7c = int(
    len(
        candidate_proposals_7c_df
    )
)

total_candidate_fragment_pixels_7c = int(
    candidate_proposals_7c_df[
        "proposal_area"
    ].sum()
)


tiny_global_rows_7c = []


for threshold in [
    0.01,
    0.05,
]:

    tiny_mask = (
        candidate_proposals_7c_df[
            "fragment_fraction_original"
        ]
        <= threshold
    )


    num_tiny = int(
        tiny_mask.sum()
    )

    pixels_tiny = int(
        candidate_proposals_7c_df.loc[
            tiny_mask,
            "proposal_area",
        ].sum()
    )


    tiny_global_rows_7c.append(
        {
            "relative_size_definition":
                f"<= {int(threshold * 100)}% original area",

            "num_fragments":
                num_tiny,

            "fraction_of_all_fragments":
                float(
                    num_tiny
                    /
                    total_candidate_fragments_7c
                ),

            "pixels_in_fragments":
                pixels_tiny,

            "fraction_of_candidate_foreground_pixels":
                float(
                    pixels_tiny
                    /
                    total_candidate_fragment_pixels_7c
                ),
        }
    )


tiny_fragment_global_7c_df = (
    pd.DataFrame(
        tiny_global_rows_7c
    )
)


# ------------------------------------------------------------
# 8. Relationship between original component size and
#    fragmentation
#
# Spearman rank correlation is used because both distributions
# are strongly non-Gaussian / heavy-tailed.
# ------------------------------------------------------------

rho_area_fragments_7c, p_area_fragments_7c = (
    spearmanr(
        component_fragment_diag_7c_df[
            "original_component_area"
        ],
        component_fragment_diag_7c_df[
            "num_raw_fragments"
        ],
    )
)


rho_area_increment_7c, p_area_increment_7c = (
    spearmanr(
        component_fragment_diag_7c_df[
            "original_component_area"
        ],
        component_fragment_diag_7c_df[
            "raw_fragment_increment"
        ],
    )
)


fragmentation_size_relation_7c_df = (
    pd.DataFrame(
        [
            {
                "comparison":
                    "original area vs number raw fragments",

                "spearman_rho":
                    float(
                        rho_area_fragments_7c
                    ),

                "p_value":
                    float(
                        p_area_fragments_7c
                    ),
            },

            {
                "comparison":
                    "original area vs raw fragment increment",

                "spearman_rho":
                    float(
                        rho_area_increment_7c
                    ),

                "p_value":
                    float(
                        p_area_increment_7c
                    ),
            },
        ]
    )
)


# ------------------------------------------------------------
# 9. Extreme fragmentation cases
#
# Diagnostic only — useful later for visual inspection.
# ------------------------------------------------------------

extreme_fragmentation_7c_df = (
    component_fragment_diag_7c_df
    .sort_values(
        [
            "num_raw_fragments",
            "original_component_area",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .head(
        20
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 10. Save outputs
# ------------------------------------------------------------

M1_7C_COMPONENT_DIAG_FILE = (
    M1_TABLES_DIR
    / "M1_7C_A_component_fragmentation_diagnostic.csv"
)

M1_7C_GT_SUMMARY_FILE = (
    M1_TABLES_DIR
    / "M1_7C_A_fragmentation_by_GT_group.csv"
)

M1_7C_TINY_FRAGMENT_FILE = (
    M1_TABLES_DIR
    / "M1_7C_A_tiny_fragment_burden.csv"
)

M1_7C_SIZE_RELATION_FILE = (
    M1_TABLES_DIR
    / "M1_7C_A_fragmentation_size_relation.csv"
)

M1_7C_EXTREME_FILE = (
    M1_TABLES_DIR
    / "M1_7C_A_extreme_fragmentation_cases.csv"
)


component_fragment_diag_7c_df.to_csv(
    M1_7C_COMPONENT_DIAG_FILE,
    index=False,
)

fragmentation_by_gt_7c_df.to_csv(
    M1_7C_GT_SUMMARY_FILE,
    index=False,
)

tiny_fragment_global_7c_df.to_csv(
    M1_7C_TINY_FRAGMENT_FILE,
    index=False,
)

fragmentation_size_relation_7c_df.to_csv(
    M1_7C_SIZE_RELATION_FILE,
    index=False,
)

extreme_fragmentation_7c_df.to_csv(
    M1_7C_EXTREME_FILE,
    index=False,
)


# ------------------------------------------------------------
# 11. Report
# ------------------------------------------------------------

print("\nFrozen candidate GT inventory")
print("-----------------------------")

display(
    gt_inventory_7c_df
)


print("\nFragmentation by original GT group")
print("----------------------------------")

display(
    fragmentation_by_gt_7c_df
)


print("\nGlobal tiny-fragment burden")
print("---------------------------")

display(
    tiny_fragment_global_7c_df
)


print("\nRelationship between component size and fragmentation")
print("-----------------------------------------------------")

display(
    fragmentation_size_relation_7c_df
)


print("\n20 most heavily fragmented candidate components")
print("-----------------------------------------------")

display(
    extreme_fragmentation_7c_df[
        [
            "dataset_index",
            "original_component_id",
            "gt_group",
            "component_gt_count",
            "original_component_area",
            "num_raw_fragments",
            "largest_fragment_fraction",
            "top2_fragment_fraction",
            "num_fragments_for_90pct_area",
            "fraction_fragments_le_1pct",
            "fraction_fragments_le_5pct",
        ]
    ]
)


# ------------------------------------------------------------
# 12. Global descriptive summary
# ------------------------------------------------------------

print("\nGlobal structural summary")
print("-------------------------")

print(
    "Candidate components:",
    len(
        component_fragment_diag_7c_df
    ),
)

print(
    "Raw candidate fragments:",
    total_candidate_fragments_7c,
)

print(
    "Median fragments / candidate:",
    round(
        component_fragment_diag_7c_df[
            "num_raw_fragments"
        ].median(),
        4,
    ),
)

print(
    "Median largest-fragment area fraction:",
    round(
        component_fragment_diag_7c_df[
            "largest_fragment_fraction"
        ].median(),
        4,
    ),
)

print(
    "Median top-2 area fraction:",
    round(
        component_fragment_diag_7c_df[
            "top2_fragment_fraction"
        ].median(),
        4,
    ),
)

print(
    "Median fragments needed for 90% area:",
    round(
        component_fragment_diag_7c_df[
            "num_fragments_for_90pct_area"
        ].median(),
        4,
    ),
)


# ------------------------------------------------------------
# 13. Protocol report
# ------------------------------------------------------------

print("\nGT used to generate split proposals : NO")
print("GT used for diagnostic grouping      : YES")
print("Fragment-size threshold selected     : NO")
print("Fragments removed                    : NO")
print("Fragments merged                     : NO")
print("Split accepted/rejected              : NO")
print("Semantic foreground modified         : NO")
print("Count rule modified                  : NO")
print("Density routing used                 : NO")
print("Test set accessed                    : NO")

print("\nStep 7C Part A: PASS")

STEP 7C PART A — STRUCTURAL DIAGNOSIS OF RAW DEPTH-SPLIT PROPOSALS

Frozen candidate GT inventory
-----------------------------


,gt_group,num_candidate_components
0,GT0,182
1,GT1,248
2,GT2+,269



Fragmentation by original GT group
----------------------------------


,gt_group,num_components,median_original_area,median_raw_fragments,mean_raw_fragments,q25_raw_fragments,q75_raw_fragments,max_raw_fragments,median_largest_fragment_fraction,median_top2_fragment_fraction,median_fraction_fragments_le_1pct,median_area_fraction_le_1pct,median_fraction_fragments_le_5pct,median_area_fraction_le_5pct,median_fragments_for_90pct_area,mean_fragments_for_90pct_area
0,GT0,182,901.5,4.0,6.362637,2.0,7.0,73,0.613073,0.982541,0.250000,0.000808,0.366667,0.007008,2.0,2.373626
1,GT1,248,1033.5,4.0,5.209677,3.0,6.0,47,0.656782,0.990863,0.333333,0.001526,0.500000,0.006506,2.0,2.048387
2,GT2+,269,1442.0,8.0,15.137546,4.0,15.0,359,0.596542,0.906254,0.500000,0.006627,0.636364,0.025120,2.0,3.434944



Global tiny-fragment burden
---------------------------


...


Relationship between component size and fragmentation
-----------------------------------------------------


,comparison,spearman_rho,p_value
0,original area vs number raw fragments,0.302416,3.000962e-16
1,original area vs raw fragment increment,0.302416,3.000962e-16



20 most heavily fragmented candidate components
-----------------------------------------------


,dataset_index,original_component_id,gt_group,component_gt_count,original_component_area,num_raw_fragments,largest_fragment_fraction,top2_fragment_fraction,num_fragments_for_90pct_area,fraction_fragments_le_1pct,fraction_fragments_le_5pct
0,253,90,GT2+,3,6526,359,0.375881,0.550107,31,0.963788,0.991643
1,1192,13,GT2+,197,12367,173,0.596345,0.677610,44,0.959538,0.988439
2,253,64,GT2+,6,1616,141,0.324876,0.599629,21,0.943262,0.971631
3,253,65,GT2+,9,1501,115,0.181213,0.320453,19,0.878261,0.956522
4,862,12,GT2+,3,6955,83,0.265996,0.526096,4,0.939759,0.951807
5,862,1,GT0,0,1037,73,0.365477,0.620058,10,0.904110,0.945205
6,253,100,GT2+,3,969,72,0.367389,0.575851,13,0.902778,0.944444
7,609,2,GT2+,18,18000,70,0.755278,0.867222,3,0.928571,0.957143
8,1269,4,GT2+,26,22229,68,0.493859,0.901165,2,0.955882,0.955882
9,253,84,GT2+,13,1000,67,0.593000,0.682000,8,0.880597,0.955224



Global structural summary
-------------------------
Candidate components: 699
Raw candidate fragments: 6522
Median fragments / candidate: 5.0
Median largest-fragment area fraction: 0.6279
Median top-2 area fraction: 0.9688
Median fragments needed for 90% area: 2.0

GT used to generate split proposals : NO
GT used for diagnostic grouping      : YES
Fragment-size threshold selected     : NO
Fragments removed                    : NO
Fragments merged                     : NO
Split accepted/rejected              : NO
Semantic foreground modified         : NO
Count rule modified                  : NO
Density routing used                 : NO
Test set accessed                    : NO

Step 7C Part A: PASS


### **Finding 7 — Step 7C Part A:**

The apparent over-fragmentation produced by binary Marigold depth partitioning is dominated numerically by very small disconnected islands.

Although candidate components produce a median of five raw fragments, the two largest fragments contain a median of 96.9% of the original semantic area, and two fragments are sufficient to cover 90% of the area for the median candidate.

GT2+ components show substantially greater fragmentation than GT0/GT1 components, but component size remains a partial confound.

These results suggest that the **raw depth partition may contain a useful dominant split obscured by small spatial fragments.**

In [29]:
# ============================================================
# Step 7C Part B — GT-point mapping onto raw Marigold
#                  split proposal fragments
# ============================================================
#
# Purpose
# -------
# Determine whether the raw Step-7B Marigold depth partition
# actually separates annotated FSC-147 objects into different
# spatial proposal fragments.
#
# Questions
# ---------
# 1. For each original candidate component, how many distinct
#    raw proposal fragments contain GT points?
#
# 2. For GT2+ components, are multiple GT points separated
#    across different proposal fragments?
#
# 3. Are GT-bearing fragments usually among the dominant
#    area fragments?
#
# 4. For GT1 components, does the single GT point remain in
#    one of the dominant fragments?
#
# 5. Do very small raw fragments contain meaningful GT
#    support, or are they mostly connectivity noise?
#
# IMPORTANT
# ---------
# GT is used ONLY for post-hoc diagnostic evaluation.
#
# No:
#   - fragment filtering
#   - fragment merging
#   - split acceptance/rejection
#   - threshold selection
#   - count modification
#   - density routing
#
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy import ndimage


print("=" * 120)
print("STEP 7C PART B — GT-POINT MAPPING ONTO RAW DEPTH-SPLIT PROPOSALS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "raw_split_proposals_7b_df" in globals(), (
    "raw_split_proposals_7b_df unavailable. "
    "Run Step 7B first."
)

assert "image_split_summary_7b_df" in globals(), (
    "image_split_summary_7b_df unavailable. "
    "Run Step 7B first."
)

assert "a3c_component_geom_df" in globals(), (
    "a3c_component_geom_df unavailable. "
    "Run Step 6D Part C first."
)

assert "component_fragment_diag_7c_df" in globals(), (
    "component_fragment_diag_7c_df unavailable. "
    "Run Step 7C Part A first."
)

assert "a3c_align_6d_df" in globals(), (
    "a3c_align_6d_df unavailable. "
    "Run Step 6D Part B first."
)

assert "gt_lookup_6dc" in globals(), (
    "gt_lookup_6dc unavailable. "
    "Run Step 6D Part C first."
)


assert len(
    component_fragment_diag_7c_df
) == 699


assert len(
    gt_lookup_6dc
) == 100


# ------------------------------------------------------------
# 2. Reuse EXACT authoritative GT source from Step 6D
#
# Step 6D Part C constructed:
#
#   gt_lookup_6dc[dataset_index] = [(x, y), ...]
#
# from the frozen C1 GT annotations.
#
# These are the SAME GT points used to derive
# `component_gt_count`.
#
# No GT reloading, resizing, rescaling, or reinterpretation.
# ------------------------------------------------------------

gt_point_rows_7c = []


for dataset_index, points in (
    gt_lookup_6dc.items()
):

    for gt_point_index, point in enumerate(
        points
    ):

        x = float(
            point[0]
        )

        y = float(
            point[1]
        )


        gt_point_rows_7c.append(
            {
                "dataset_index":
                    int(
                        dataset_index
                    ),

                "gt_point_index":
                    int(
                        gt_point_index
                    ),

                "x":
                    x,

                "y":
                    y,
            }
        )


gt_points_7c_df = (
    pd.DataFrame(
        gt_point_rows_7c
    )
    .sort_values(
        [
            "dataset_index",
            "gt_point_index",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    gt_points_7c_df
) > 0


assert (
    gt_points_7c_df[
        "dataset_index"
    ]
    .nunique()
    ==
    100
)


print("\nAuthoritative GT source")
print("-----------------------")

print(
    "Source                  : gt_lookup_6dc"
)

print(
    "Original source         : frozen C1 GT annotations"
)

print(
    "Images                  :",
    gt_points_7c_df[
        "dataset_index"
    ].nunique(),
)

print(
    "Total GT points         :",
    len(
        gt_points_7c_df
    ),
)

print(
    "Coordinate handling     : same nearest-pixel + clipping rule as Step 6D"
)


# ------------------------------------------------------------
# 3. Frozen Step-7A candidate keys
# ------------------------------------------------------------

candidate_keys_7c_df = (
    component_fragment_diag_7c_df[
        [
            "dataset_index",
            "original_component_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


assert len(
    candidate_keys_7c_df
) == 699


candidate_lookup_7c = (
    candidate_keys_7c_df
    .groupby(
        "dataset_index"
    )[
        "original_component_id"
    ]
    .apply(
        lambda s:
            set(
                s.astype(int)
            )
    )
    .to_dict()
)


# ------------------------------------------------------------
# 4. Raw proposal metadata
# ------------------------------------------------------------

proposal_meta_7c_df = (
    raw_split_proposals_7b_df[
        raw_split_proposals_7b_df[
            "split_candidate"
        ]
    ]
    [
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
            "proposal_area",
            "original_component_area",
            "proposal_area_fraction_of_original",
        ]
    ]
    .copy()
)


# Rank proposal fragments by area WITHIN each original
# candidate component.
#
# Rank 1 = largest fragment.
# Rank 2 = second-largest fragment.
proposal_meta_7c_df[
    "proposal_area_rank"
] = (
    proposal_meta_7c_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ]
    )[
        "proposal_area"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


assert (
    proposal_meta_7c_df[
        [
            "dataset_index",
            "proposal_subcomponent_id",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)


# ------------------------------------------------------------
# 5. Map authoritative GT points to:
#
#    A. frozen original A3c component
#    B. Step-7B raw proposal fragment
#
# IMPORTANT:
# Coordinate conversion exactly follows Step 6D:
#
#   xi = round(x) -> clip [0, W-1]
#   yi = round(y) -> clip [0, H-1]
# ------------------------------------------------------------

mapped_gt_rows_7c = []


for _, image_row in (
    image_split_summary_7b_df
    .sort_values(
        "dataset_index"
    )
    .iterrows()
):

    dataset_index = int(
        image_row[
            "dataset_index"
        ]
    )


    # --------------------------------------------------------
    # Frozen candidate IDs for this image
    # --------------------------------------------------------

    candidate_component_ids = (
        candidate_lookup_7c.get(
            dataset_index,
            set(),
        )
    )


    assert len(
        candidate_component_ids
    ) > 0, (
        f"No Step-7A candidates found for "
        f"dataset_index={dataset_index}"
    )


    # --------------------------------------------------------
    # Load raw Step-7B proposal label map
    # --------------------------------------------------------

    proposal_map_file = Path(
        image_row[
            "proposal_map_file"
        ]
    )


    assert proposal_map_file.exists(), (
        f"Missing Step-7B proposal map: "
        f"{proposal_map_file}"
    )


    with np.load(
        proposal_map_file,
        allow_pickle=False,
    ) as data:

        proposal_labels = (
            np.asarray(
                data[
                    "proposal_labels"
                ]
            )
            .astype(
                np.int32
            )
        )


    H, W = (
        proposal_labels.shape
    )


    # --------------------------------------------------------
    # Recover exact frozen A3c semantic mask
    # --------------------------------------------------------

    align_rows = (
        a3c_align_6d_df[
            a3c_align_6d_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
    )


    assert len(
        align_rows
    ) == 1, (
        f"Expected exactly one A3c alignment row for "
        f"dataset_index={dataset_index}; "
        f"found {len(align_rows)}"
    )


    align_row = (
        align_rows.iloc[0]
    )


    a3c_cache_file = Path(
        align_row[
            "cache_file"
        ]
    )


    assert a3c_cache_file.exists(), (
        f"Missing A3c cache file: "
        f"{a3c_cache_file}"
    )


    with np.load(
        a3c_cache_file,
        allow_pickle=False,
    ) as data:

        a3c_mask = (
            np.asarray(
                data[
                    "mask"
                ]
            )
            .astype(bool)
        )


    assert (
        a3c_mask.shape
        ==
        proposal_labels.shape
    )


    # --------------------------------------------------------
    # Reconstruct frozen C1/A3c component labels using the
    # same 4-connectivity convention.
    # --------------------------------------------------------

    structure_4_7c = (
        ndimage.generate_binary_structure(
            rank=2,
            connectivity=1,
        )
    )


    original_labels, num_original_components = (
        ndimage.label(
            a3c_mask,
            structure=
                structure_4_7c,
        )
    )


    assert (
        int(
            num_original_components
        )
        ==
        int(
            align_row[
                "num_components"
            ]
        )
    )


    # --------------------------------------------------------
    # Exact GT points for this image
    # --------------------------------------------------------

    gt_points_this_image = (
        gt_lookup_6dc[
            dataset_index
        ]
    )


    # --------------------------------------------------------
    # Map every GT point
    # --------------------------------------------------------

    for gt_point_index, (
        x,
        y,
    ) in enumerate(
        gt_points_this_image
    ):

        # EXACT Step-6D convention:
        #
        # nearest pixel + image-bound clipping
        xi = int(
            np.clip(
                np.rint(
                    float(x)
                ),
                0,
                W - 1,
            )
        )

        yi = int(
            np.clip(
                np.rint(
                    float(y)
                ),
                0,
                H - 1,
            )
        )


        original_component_id = int(
            original_labels[
                yi,
                xi,
            ]
        )


        # GT point outside all A3c foreground.
        #
        # This is expected for many GT points because A3c does
        # not cover the full FSC-147 GT set.
        if (
            original_component_id
            ==
            0
        ):

            continue


        # We analyse only frozen Step-7A split candidates.
        if (
            original_component_id
            not in
            candidate_component_ids
        ):

            continue


        proposal_subcomponent_id = int(
            proposal_labels[
                yi,
                xi,
            ]
        )


        # Step 7B proved proposal foreground exactly equals A3c.
        # Therefore a GT point inside an A3c candidate must map
        # to a positive raw proposal label.
        assert (
            proposal_subcomponent_id
            >
            0
        ), (
            f"GT point lies in A3c but not in Step-7B proposal "
            f"foreground: dataset={dataset_index}, "
            f"component={original_component_id}, "
            f"x={xi}, y={yi}"
        )


        mapped_gt_rows_7c.append(
            {
                "dataset_index":
                    dataset_index,

                "gt_point_index":
                    int(
                        gt_point_index
                    ),

                "x":
                    float(
                        x
                    ),

                "y":
                    float(
                        y
                    ),

                "pixel_x":
                    xi,

                "pixel_y":
                    yi,

                "original_component_id":
                    original_component_id,

                "proposal_subcomponent_id":
                    proposal_subcomponent_id,
            }
        )


mapped_gt_points_7c_df = (
    pd.DataFrame(
        mapped_gt_rows_7c
    )
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
            "gt_point_index",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    mapped_gt_points_7c_df
) > 0


# ------------------------------------------------------------
# 6. Attach raw-fragment metadata to each mapped GT point
# ------------------------------------------------------------

mapped_gt_points_7c_df = (
    mapped_gt_points_7c_df
    .merge(
        proposal_meta_7c_df,
        on=[
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    mapped_gt_points_7c_df[
        "proposal_area"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 7. Count GT points within each raw proposal fragment
# ------------------------------------------------------------

proposal_gt_counts_7c_df = (
    mapped_gt_points_7c_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
        ],
        as_index=False,
    )
    .agg(
        gt_points_in_fragment=(
            "gt_point_index",
            "count",
        ),

        proposal_area=(
            "proposal_area",
            "first",
        ),

        original_component_area=(
            "original_component_area",
            "first",
        ),

        proposal_area_fraction_of_original=(
            "proposal_area_fraction_of_original",
            "first",
        ),

        proposal_area_rank=(
            "proposal_area_rank",
            "first",
        ),
    )
)


# ------------------------------------------------------------
# 8. One diagnostic row per ORIGINAL Step-7A candidate
# ------------------------------------------------------------

component_gt_summary_rows_7c = []


for _, comp_row in (
    component_fragment_diag_7c_df
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .iterrows()
):

    dataset_index = int(
        comp_row[
            "dataset_index"
        ]
    )

    original_component_id = int(
        comp_row[
            "original_component_id"
        ]
    )

    original_gt_count = int(
        comp_row[
            "component_gt_count"
        ]
    )

    gt_group = str(
        comp_row[
            "gt_group"
        ]
    )


    fragment_gt_df = (
        proposal_gt_counts_7c_df[
            (
                proposal_gt_counts_7c_df[
                    "dataset_index"
                ]
                ==
                dataset_index
            )
            &
            (
                proposal_gt_counts_7c_df[
                    "original_component_id"
                ]
                ==
                original_component_id
            )
        ]
        .copy()
    )


    num_gt_bearing_fragments = int(
        len(
            fragment_gt_df
        )
    )


    gt_points_mapped = int(
        fragment_gt_df[
            "gt_points_in_fragment"
        ]
        .sum()
    )


    # --------------------------------------------------------
    # CRITICAL consistency check:
    #
    # Step 7C must exactly reproduce component_gt_count from
    # Step 6D because:
    #
    #   same GT source
    #   same A3c component map
    #   same coordinate rounding/clipping rule
    # --------------------------------------------------------

    assert (
        gt_points_mapped
        ==
        original_gt_count
    ), (
        f"GT occupancy mismatch:\n"
        f"dataset_index={dataset_index}\n"
        f"component_id={original_component_id}\n"
        f"Step6D component_gt_count={original_gt_count}\n"
        f"Step7C mapped GT count={gt_points_mapped}"
    )


    # --------------------------------------------------------
    # GT0
    # --------------------------------------------------------

    if (
        original_gt_count
        ==
        0
    ):

        max_gt_in_one_fragment = 0

        separated_gt_fraction = np.nan

        all_gt_separated = False

        num_gt_in_top1 = 0

        num_gt_in_top2 = 0

        gt_points_in_small_1pct = 0

        gt_points_in_small_5pct = 0


    # --------------------------------------------------------
    # GT1 / GT2+
    # --------------------------------------------------------

    else:

        max_gt_in_one_fragment = int(
            fragment_gt_df[
                "gt_points_in_fragment"
            ]
            .max()
        )


        # Fraction of original GT points lying outside the most
        # GT-populated fragment.
        #
        # Example:
        #
        # 3 GT points distributed [2,1] ->
        # separated fraction = 1 - 2/3 = 0.333
        separated_gt_fraction = float(
            1.0
            -
            (
                max_gt_in_one_fragment
                /
                original_gt_count
            )
        )


        # Strong criterion:
        # every GT point is placed in its own distinct raw
        # proposal fragment.
        all_gt_separated = bool(
            num_gt_bearing_fragments
            ==
            original_gt_count
        )


        # ----------------------------------------------------
        # GT support in dominant AREA fragments
        # ----------------------------------------------------

        num_gt_in_top1 = int(
            fragment_gt_df.loc[
                fragment_gt_df[
                    "proposal_area_rank"
                ]
                <= 1,
                "gt_points_in_fragment",
            ]
            .sum()
        )


        num_gt_in_top2 = int(
            fragment_gt_df.loc[
                fragment_gt_df[
                    "proposal_area_rank"
                ]
                <= 2,
                "gt_points_in_fragment",
            ]
            .sum()
        )


        # ----------------------------------------------------
        # Diagnostic only:
        # GT support in very small fragments
        #
        # 1% and 5% remain diagnostic probes.
        # They are NOT filtering thresholds.
        # ----------------------------------------------------

        gt_points_in_small_1pct = int(
            fragment_gt_df.loc[
                fragment_gt_df[
                    "proposal_area_fraction_of_original"
                ]
                <= 0.01,
                "gt_points_in_fragment",
            ]
            .sum()
        )


        gt_points_in_small_5pct = int(
            fragment_gt_df.loc[
                fragment_gt_df[
                    "proposal_area_fraction_of_original"
                ]
                <= 0.05,
                "gt_points_in_fragment",
            ]
            .sum()
        )


    component_gt_summary_rows_7c.append(
        {
            "dataset_index":
                dataset_index,

            "original_component_id":
                original_component_id,

            "gt_group":
                gt_group,

            "original_gt_count":
                original_gt_count,

            "num_raw_fragments":
                int(
                    comp_row[
                        "num_raw_fragments"
                    ]
                ),

            "num_gt_bearing_fragments":
                num_gt_bearing_fragments,

            "gt_points_mapped":
                gt_points_mapped,

            "max_gt_in_one_fragment":
                max_gt_in_one_fragment,

            "separated_gt_fraction":
                separated_gt_fraction,

            "all_gt_separated":
                all_gt_separated,

            "gt_points_in_largest_fragment":
                num_gt_in_top1,

            "gt_points_in_top2_fragments":
                num_gt_in_top2,

            "gt_fraction_in_largest_fragment":
                (
                    float(
                        num_gt_in_top1
                        /
                        original_gt_count
                    )
                    if
                    original_gt_count
                    >
                    0
                    else
                    np.nan
                ),

            "gt_fraction_in_top2_fragments":
                (
                    float(
                        num_gt_in_top2
                        /
                        original_gt_count
                    )
                    if
                    original_gt_count
                    >
                    0
                    else
                    np.nan
                ),

            "gt_points_in_le_1pct_fragments":
                gt_points_in_small_1pct,

            "gt_points_in_le_5pct_fragments":
                gt_points_in_small_5pct,

            "gt_fraction_in_le_1pct_fragments":
                (
                    float(
                        gt_points_in_small_1pct
                        /
                        original_gt_count
                    )
                    if
                    original_gt_count
                    >
                    0
                    else
                    np.nan
                ),

            "gt_fraction_in_le_5pct_fragments":
                (
                    float(
                        gt_points_in_small_5pct
                        /
                        original_gt_count
                    )
                    if
                    original_gt_count
                    >
                    0
                    else
                    np.nan
                ),
        }
    )


component_gt_split_diag_7c_df = (
    pd.DataFrame(
        component_gt_summary_rows_7c
    )
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    component_gt_split_diag_7c_df
) == 699


# ------------------------------------------------------------
# 9. Global consistency checks
# ------------------------------------------------------------

expected_candidate_gt_total_7c = int(
    component_gt_split_diag_7c_df[
        "original_gt_count"
    ]
    .sum()
)


actual_candidate_gt_total_7c = int(
    len(
        mapped_gt_points_7c_df
    )
)


assert (
    actual_candidate_gt_total_7c
    ==
    expected_candidate_gt_total_7c
), (
    f"Candidate GT total mismatch: "
    f"mapped={actual_candidate_gt_total_7c}, "
    f"expected={expected_candidate_gt_total_7c}"
)


# ------------------------------------------------------------
# 10. GT1 diagnostic
#
# For GT1 components:
# Where does the one annotated object lie after partitioning?
# ------------------------------------------------------------

gt1_diag_7c_df = (
    component_gt_split_diag_7c_df[
        component_gt_split_diag_7c_df[
            "gt_group"
        ]
        ==
        "GT1"
    ]
    .copy()
)


assert len(
    gt1_diag_7c_df
) == 248


gt1_summary_7c_df = (
    pd.DataFrame(
        [
            {
                "num_GT1_components":
                    int(
                        len(
                            gt1_diag_7c_df
                        )
                    ),

                "GT_in_largest_fragment_fraction":
                    float(
                        (
                            gt1_diag_7c_df[
                                "gt_points_in_largest_fragment"
                            ]
                            ==
                            1
                        )
                        .mean()
                    ),

                "GT_in_top2_fragments_fraction":
                    float(
                        (
                            gt1_diag_7c_df[
                                "gt_points_in_top2_fragments"
                            ]
                            ==
                            1
                        )
                        .mean()
                    ),

                "GT_in_le_1pct_fragment_fraction":
                    float(
                        (
                            gt1_diag_7c_df[
                                "gt_points_in_le_1pct_fragments"
                            ]
                            >
                            0
                        )
                        .mean()
                    ),

                "GT_in_le_5pct_fragment_fraction":
                    float(
                        (
                            gt1_diag_7c_df[
                                "gt_points_in_le_5pct_fragments"
                            ]
                            >
                            0
                        )
                        .mean()
                    ),

                "median_raw_fragments":
                    float(
                        gt1_diag_7c_df[
                            "num_raw_fragments"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


# ------------------------------------------------------------
# 11. GT2+ diagnostic
#
# Main question:
# Does the raw Marigold partition separate multiple
# annotated objects?
# ------------------------------------------------------------

gt2_diag_7c_df = (
    component_gt_split_diag_7c_df[
        component_gt_split_diag_7c_df[
            "gt_group"
        ]
        ==
        "GT2+"
    ]
    .copy()
)


assert len(
    gt2_diag_7c_df
) == 269


gt2_summary_7c_df = (
    pd.DataFrame(
        [
            {
                "num_GT2plus_components":
                    int(
                        len(
                            gt2_diag_7c_df
                        )
                    ),

                "median_GT_count":
                    float(
                        gt2_diag_7c_df[
                            "original_gt_count"
                        ]
                        .median()
                    ),

                "mean_GT_count":
                    float(
                        gt2_diag_7c_df[
                            "original_gt_count"
                        ]
                        .mean()
                    ),

                "median_GT_bearing_fragments":
                    float(
                        gt2_diag_7c_df[
                            "num_gt_bearing_fragments"
                        ]
                        .median()
                    ),

                "mean_GT_bearing_fragments":
                    float(
                        gt2_diag_7c_df[
                            "num_gt_bearing_fragments"
                        ]
                        .mean()
                    ),

                "fraction_with_GT_in_2plus_fragments":
                    float(
                        (
                            gt2_diag_7c_df[
                                "num_gt_bearing_fragments"
                            ]
                            >=
                            2
                        )
                        .mean()
                    ),

                "fraction_with_GT_in_3plus_fragments":
                    float(
                        (
                            gt2_diag_7c_df[
                                "num_gt_bearing_fragments"
                            ]
                            >=
                            3
                        )
                        .mean()
                    ),

                "fraction_all_GT_separated":
                    float(
                        gt2_diag_7c_df[
                            "all_gt_separated"
                        ]
                        .mean()
                    ),

                "median_separated_GT_fraction":
                    float(
                        gt2_diag_7c_df[
                            "separated_gt_fraction"
                        ]
                        .median()
                    ),

                "median_GT_fraction_in_largest_fragment":
                    float(
                        gt2_diag_7c_df[
                            "gt_fraction_in_largest_fragment"
                        ]
                        .median()
                    ),

                "median_GT_fraction_in_top2_fragments":
                    float(
                        gt2_diag_7c_df[
                            "gt_fraction_in_top2_fragments"
                        ]
                        .median()
                    ),

                "median_GT_fraction_in_le_1pct_fragments":
                    float(
                        gt2_diag_7c_df[
                            "gt_fraction_in_le_1pct_fragments"
                        ]
                        .median()
                    ),

                "median_GT_fraction_in_le_5pct_fragments":
                    float(
                        gt2_diag_7c_df[
                            "gt_fraction_in_le_5pct_fragments"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


# ------------------------------------------------------------
# 12. Distribution:
#
# original number of GT points
# versus
# number of raw proposal fragments carrying those GT points
# ------------------------------------------------------------

gt_separation_distribution_7c_df = (
    gt2_diag_7c_df
    .groupby(
        [
            "original_gt_count",
            "num_gt_bearing_fragments",
        ],
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size":
                "num_components"
        }
    )
    .sort_values(
        [
            "original_gt_count",
            "num_gt_bearing_fragments",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 13. GT occupancy of ALL candidate raw fragments
#
# Fragment GT group:
#
#   GT0  = no GT point
#   GT1  = exactly one GT point
#   GT2+ = two or more GT points
# ------------------------------------------------------------

all_candidate_fragments_7c_df = (
    proposal_meta_7c_df
    .merge(
        proposal_gt_counts_7c_df[
            [
                "dataset_index",
                "original_component_id",
                "proposal_subcomponent_id",
                "gt_points_in_fragment",
            ]
        ],
        on=[
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


all_candidate_fragments_7c_df[
    "gt_points_in_fragment"
] = (
    all_candidate_fragments_7c_df[
        "gt_points_in_fragment"
    ]
    .fillna(0)
    .astype(int)
)


all_candidate_fragments_7c_df[
    "fragment_gt_group"
] = np.select(
    [
        all_candidate_fragments_7c_df[
            "gt_points_in_fragment"
        ]
        ==
        0,

        all_candidate_fragments_7c_df[
            "gt_points_in_fragment"
        ]
        ==
        1,

        all_candidate_fragments_7c_df[
            "gt_points_in_fragment"
        ]
        >=
        2,
    ],
    [
        "GT0",
        "GT1",
        "GT2+",
    ],
    default="UNKNOWN",
)


assert (
    all_candidate_fragments_7c_df[
        "fragment_gt_group"
    ]
    !=
    "UNKNOWN"
).all()


fragment_gt_occupancy_7c_df = (
    all_candidate_fragments_7c_df
    .groupby(
        "fragment_gt_group",
        as_index=False,
    )
    .agg(
        num_fragments=(
            "proposal_subcomponent_id",
            "count",
        ),

        total_pixels=(
            "proposal_area",
            "sum",
        ),

        median_area_fraction_original=(
            "proposal_area_fraction_of_original",
            "median",
        ),

        mean_area_fraction_original=(
            "proposal_area_fraction_of_original",
            "mean",
        ),
    )
)


fragment_gt_occupancy_7c_df[
    "fraction_all_candidate_fragments"
] = (
    fragment_gt_occupancy_7c_df[
        "num_fragments"
    ]
    /
    len(
        all_candidate_fragments_7c_df
    )
)


# ------------------------------------------------------------
# 14. GT support by fragment-area rank
#
# This helps determine whether GT-bearing pieces are usually
# among the dominant spatial fragments.
# ------------------------------------------------------------

gt_fragment_rank_distribution_7c_df = (
    mapped_gt_points_7c_df
    .groupby(
        "proposal_area_rank",
        as_index=False,
    )
    .agg(
        num_GT_points=(
            "gt_point_index",
            "count",
        )
    )
    .sort_values(
        "proposal_area_rank"
    )
    .reset_index(
        drop=True
    )
)


gt_fragment_rank_distribution_7c_df[
    "fraction_mapped_GT_points"
] = (
    gt_fragment_rank_distribution_7c_df[
        "num_GT_points"
    ]
    /
    len(
        mapped_gt_points_7c_df
    )
)


# Cumulative GT support by area rank
gt_fragment_rank_distribution_7c_df[
    "cumulative_fraction_mapped_GT_points"
] = (
    gt_fragment_rank_distribution_7c_df[
        "fraction_mapped_GT_points"
    ]
    .cumsum()
)


# ------------------------------------------------------------
# 15. GT support in diagnostic small-fragment bands
#
# Again: 1% and 5% are descriptive probes ONLY.
# ------------------------------------------------------------

small_fragment_gt_summary_7c_df = (
    pd.DataFrame(
        [
            {
                "relative_fragment_definition":
                    "<=1% original component",

                "num_GT_points":
                    int(
                        (
                            mapped_gt_points_7c_df[
                                "proposal_area_fraction_of_original"
                            ]
                            <=
                            0.01
                        )
                        .sum()
                    ),

                "fraction_mapped_GT_points":
                    float(
                        (
                            mapped_gt_points_7c_df[
                                "proposal_area_fraction_of_original"
                            ]
                            <=
                            0.01
                        )
                        .mean()
                    ),
            },

            {
                "relative_fragment_definition":
                    "<=5% original component",

                "num_GT_points":
                    int(
                        (
                            mapped_gt_points_7c_df[
                                "proposal_area_fraction_of_original"
                            ]
                            <=
                            0.05
                        )
                        .sum()
                    ),

                "fraction_mapped_GT_points":
                    float(
                        (
                            mapped_gt_points_7c_df[
                                "proposal_area_fraction_of_original"
                            ]
                            <=
                            0.05
                        )
                        .mean()
                    ),
            },
        ]
    )
)


# ------------------------------------------------------------
# 16. Save outputs
# ------------------------------------------------------------

M1_7C_B_MAPPED_GT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_mapped_GT_points_to_raw_fragments.csv"
)

M1_7C_B_COMPONENT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_component_GT_split_diagnostic.csv"
)

M1_7C_B_GT1_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_GT1_summary.csv"
)

M1_7C_B_GT2_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_GT2plus_summary.csv"
)

M1_7C_B_SEPARATION_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_GT_separation_distribution.csv"
)

M1_7C_B_FRAGMENT_OCCUPANCY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_fragment_GT_occupancy.csv"
)

M1_7C_B_GT_RANK_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_GT_fragment_area_rank_distribution.csv"
)

M1_7C_B_SMALL_FRAGMENT_GT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_B_small_fragment_GT_summary.csv"
)


mapped_gt_points_7c_df.to_csv(
    M1_7C_B_MAPPED_GT_FILE,
    index=False,
)

component_gt_split_diag_7c_df.to_csv(
    M1_7C_B_COMPONENT_FILE,
    index=False,
)

gt1_summary_7c_df.to_csv(
    M1_7C_B_GT1_FILE,
    index=False,
)

gt2_summary_7c_df.to_csv(
    M1_7C_B_GT2_FILE,
    index=False,
)

gt_separation_distribution_7c_df.to_csv(
    M1_7C_B_SEPARATION_FILE,
    index=False,
)

fragment_gt_occupancy_7c_df.to_csv(
    M1_7C_B_FRAGMENT_OCCUPANCY_FILE,
    index=False,
)

gt_fragment_rank_distribution_7c_df.to_csv(
    M1_7C_B_GT_RANK_FILE,
    index=False,
)

small_fragment_gt_summary_7c_df.to_csv(
    M1_7C_B_SMALL_FRAGMENT_GT_FILE,
    index=False,
)


# ------------------------------------------------------------
# 17. Report — GT1
# ------------------------------------------------------------

print("\nGT1 diagnostic")
print("--------------")

display(
    gt1_summary_7c_df
)


# ------------------------------------------------------------
# 18. Report — GT2+
# ------------------------------------------------------------

print("\nGT2+ separation diagnostic")
print("--------------------------")

display(
    gt2_summary_7c_df
)


# ------------------------------------------------------------
# 19. Report — fragment GT occupancy
# ------------------------------------------------------------

print("\nRaw candidate-fragment GT occupancy")
print("-----------------------------------")

display(
    fragment_gt_occupancy_7c_df
)


# ------------------------------------------------------------
# 20. Report — small-fragment GT support
# ------------------------------------------------------------

print("\nGT support in small raw fragments")
print("---------------------------------")

display(
    small_fragment_gt_summary_7c_df
)


# ------------------------------------------------------------
# 21. Report — GT support by fragment area rank
# ------------------------------------------------------------

print("\nGT support by raw-fragment area rank")
print("------------------------------------")

display(
    gt_fragment_rank_distribution_7c_df
    .head(
        20
    )
)


# ------------------------------------------------------------
# 22. Report — GT separation distribution
# ------------------------------------------------------------

print("\nGT2+ separation distribution")
print("----------------------------")

display(
    gt_separation_distribution_7c_df
    .head(
        50
    )
)


# ------------------------------------------------------------
# 23. Informative GT2+ cases
# ------------------------------------------------------------

print("\nGT2+ cases with highest number of GT-bearing fragments")
print("------------------------------------------------------")

display(
    gt2_diag_7c_df
    .sort_values(
        [
            "num_gt_bearing_fragments",
            "original_gt_count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    [
        [
            "dataset_index",
            "original_component_id",
            "original_gt_count",
            "num_raw_fragments",
            "num_gt_bearing_fragments",
            "max_gt_in_one_fragment",
            "separated_gt_fraction",
            "gt_fraction_in_largest_fragment",
            "gt_fraction_in_top2_fragments",
            "gt_fraction_in_le_1pct_fragments",
            "gt_fraction_in_le_5pct_fragments",
        ]
    ]
    .head(
        20
    )
)


# ------------------------------------------------------------
# 24. Final consistency summary
# ------------------------------------------------------------

print("\nConsistency checks")
print("------------------")

print(
    "Candidate components analysed        :",
    len(
        component_gt_split_diag_7c_df
    ),
)

print(
    "GT0 candidate components             :",
    int(
        (
            component_gt_split_diag_7c_df[
                "gt_group"
            ]
            ==
            "GT0"
        )
        .sum()
    ),
)

print(
    "GT1 candidate components             :",
    int(
        (
            component_gt_split_diag_7c_df[
                "gt_group"
            ]
            ==
            "GT1"
        )
        .sum()
    ),
)

print(
    "GT2+ candidate components            :",
    int(
        (
            component_gt_split_diag_7c_df[
                "gt_group"
            ]
            ==
            "GT2+"
        )
        .sum()
    ),
)

print(
    "Mapped GT points inside candidates   :",
    actual_candidate_gt_total_7c,
)

print(
    "Expected Step-6D candidate GT points :",
    expected_candidate_gt_total_7c,
)

print(
    "Exact GT occupancy reproduction      : YES"
)


# ------------------------------------------------------------
# 25. Protocol report
# ------------------------------------------------------------

print("\nGT source reused from Step 6D       : YES — gt_lookup_6dc")
print("GT coordinate convention reused      : YES — round + clip")
print("GT used to generate raw proposals     : NO")
print("GT used for diagnostic evaluation     : YES")
print("Fragment filtering                    : NO")
print("Fragment merging                      : NO")
print("Split accepted/rejected               : NO")
print("Size threshold selected               : NO")
print("Count rule modified                   : NO")
print("Density routing used                  : NO")
print("Test set accessed                     : NO")

print("\nStep 7C Part B: PASS")

STEP 7C PART B — GT-POINT MAPPING ONTO RAW DEPTH-SPLIT PROPOSALS

Authoritative GT source
-----------------------
Source                  : gt_lookup_6dc
Original source         : frozen C1 GT annotations
Images                  : 100
Total GT points         : 10045
Coordinate handling     : same nearest-pixel + clipping rule as Step 6D

GT1 diagnostic
--------------


,num_GT1_components,GT_in_largest_fragment_fraction,GT_in_top2_fragments_fraction,GT_in_le_1pct_fragment_fraction,GT_in_le_5pct_fragment_fraction,median_raw_fragments
0,248,0.830645,0.983871,0.0,0.004032,4.0



GT2+ separation diagnostic
--------------------------


,num_GT2plus_components,median_GT_count,mean_GT_count,median_GT_bearing_fragments,mean_GT_bearing_fragments,fraction_with_GT_in_2plus_fragments,fraction_with_GT_in_3plus_fragments,fraction_all_GT_separated,median_separated_GT_fraction,median_GT_fraction_in_largest_fragment,median_GT_fraction_in_top2_fragments,median_GT_fraction_in_le_1pct_fragments,median_GT_fraction_in_le_5pct_fragments
0,269,3.0,7.32342,2.0,2.888476,0.814126,0.301115,0.420074,0.5,0.5,1.0,0.0,0.0



Raw candidate-fragment GT occupancy
-----------------------------------


...


GT2+ separation distribution
----------------------------


,original_gt_count,num_gt_bearing_fragments,num_components
0,2,1,27
1,2,2,83
2,3,1,9
3,3,2,22
4,3,3,20
5,4,1,4
6,4,2,11
7,4,3,4
8,4,4,7
9,5,1,1



GT2+ cases with highest number of GT-bearing fragments
------------------------------------------------------


,dataset_index,original_component_id,original_gt_count,num_raw_fragments,num_gt_bearing_fragments,max_gt_in_one_fragment,separated_gt_fraction,gt_fraction_in_largest_fragment,gt_fraction_in_top2_fragments,gt_fraction_in_le_1pct_fragments,gt_fraction_in_le_5pct_fragments
616,1192,13,197,173,90,20,0.898477,0.101523,0.111675,0.715736,0.888325
615,1192,9,32,40,21,3,0.906250,0.093750,0.093750,0.406250,0.906250
619,1192,20,21,19,17,3,0.857143,0.095238,0.238095,0.666667,0.904762
620,1192,24,18,20,13,3,0.833333,0.166667,0.333333,0.111111,0.666667
581,1183,7,20,15,11,6,0.700000,0.050000,0.350000,0.050000,0.650000
564,1157,7,12,22,11,2,0.833333,0.000000,0.166667,0.166667,0.833333
613,1192,3,13,15,8,3,0.769231,0.153846,0.384615,0.076923,0.615385
622,1192,30,9,15,8,2,0.777778,0.000000,0.222222,0.222222,0.666667
618,1192,18,8,13,7,2,0.750000,0.125000,0.375000,0.125000,0.375000
245,253,84,13,67,6,7,0.461538,0.538462,0.692308,0.076923,0.230769



Consistency checks
------------------
Candidate components analysed        : 699
GT0 candidate components             : 182
GT1 candidate components             : 248
GT2+ candidate components            : 269
Mapped GT points inside candidates   : 2218
Expected Step-6D candidate GT points : 2218
Exact GT occupancy reproduction      : YES

GT source reused from Step 6D       : YES — gt_lookup_6dc
GT coordinate convention reused      : YES — round + clip
GT used to generate raw proposals     : NO
GT used for diagnostic evaluation     : YES
Fragment filtering                    : NO
Fragment merging                      : NO
Split accepted/rejected               : NO
Size threshold selected               : NO
Count rule modified                   : NO
Density routing used                  : NO
Test set accessed                     : NO

Step 7C Part B: PASS


### **Finding 8 — Step 7C Part B:**

Raw Marigold depth partitioning provides genuine instance-separation information within candidate A3c regions.

Among GT2+ components, 81.4% place annotated objects into at least two distinct depth-derived fragments, and 42.0% completely separate all contained GT points.

For GT1 regions, 98.4% of GT points remain within the two largest fragments.

Most raw fragments are GT-empty and extremely small, indicating substantial connectivity noise; however, small fragments cannot simply be discarded because 9.0% and 17.0% of mapped GT points occur in fragments occupying ≤1% and ≤5% of their parent region respectively.

In [30]:
# ============================================================
# Step 7C Part C — Depth-cluster consolidation diagnostic
# ============================================================
#
# Purpose
# -------
# Determine whether the useful object-separation signal found
# in Step 7C Part B comes primarily from:
#
#   A. the TWO Marigold depth populations produced by k-means
#
# or
#
#   B. the many disconnected spatial islands created after
#      applying connected-component analysis separately to
#      those depth populations.
#
# Step 7B used:
#
#       Marigold depth
#           ↓
#       k-means, k=2
#           ↓
#       depth cluster 0 / depth cluster 1
#           ↓
#       4-connected pieces
#           ↓
#       many raw fragments
#
# In this diagnostic we conceptually CONSOLIDATE all spatial
# pieces belonging to the same depth cluster:
#
#       many raw fragments
#           ↓
#       at most TWO depth groups per candidate
#
# IMPORTANT
# ---------
# This does NOT yet modify the actual semantic masks or count.
#
# No:
#   - size threshold
#   - fragment deletion
#   - GT-based routing
#   - density routing
#   - count modification
#
# GT is used only for post-hoc diagnostic evaluation.
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 7C PART C — DEPTH-CLUSTER CONSOLIDATION DIAGNOSTIC")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "raw_split_proposals_7b_df" in globals(), (
    "raw_split_proposals_7b_df unavailable. "
    "Run Step 7B first."
)

assert "mapped_gt_points_7c_df" in globals(), (
    "mapped_gt_points_7c_df unavailable. "
    "Run Step 7C Part B first."
)

assert "component_gt_split_diag_7c_df" in globals(), (
    "component_gt_split_diag_7c_df unavailable. "
    "Run Step 7C Part B first."
)

assert "component_fragment_diag_7c_df" in globals(), (
    "component_fragment_diag_7c_df unavailable. "
    "Run Step 7C Part A first."
)


assert len(
    component_gt_split_diag_7c_df
) == 699


# ------------------------------------------------------------
# 2. Candidate raw proposal metadata including depth cluster
# ------------------------------------------------------------

candidate_raw_meta_7cc_df = (
    raw_split_proposals_7b_df[
        raw_split_proposals_7b_df[
            "split_candidate"
        ]
    ]
    [
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
            "depth_cluster",
            "proposal_area",
            "original_component_area",
            "proposal_area_fraction_of_original",
            "split_attempted",
            "split_status",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 3. Audit whether every frozen candidate received the
#    intended binary depth partition
# ------------------------------------------------------------

candidate_attempt_summary_7cc_df = (
    candidate_raw_meta_7cc_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ],
        as_index=False,
    )
    .agg(
        split_attempted=(
            "split_attempted",
            "max",
        ),

        num_raw_fragments=(
            "proposal_subcomponent_id",
            "count",
        ),

        num_depth_clusters_present=(
            "depth_cluster",
            lambda s:
                int(
                    len(
                        set(
                            int(v)
                            for v in s
                            if int(v) >= 0
                        )
                    )
                ),
        ),
    )
)


print("\nCandidate split-attempt audit")
print("-----------------------------")

print(
    "Candidate components:",
    len(
        candidate_attempt_summary_7cc_df
    ),
)

print(
    "Split attempted:",
    int(
        candidate_attempt_summary_7cc_df[
            "split_attempted"
        ]
        .sum()
    ),
)

print(
    "Components with 2 depth clusters:",
    int(
        (
            candidate_attempt_summary_7cc_df[
                "num_depth_clusters_present"
            ]
            ==
            2
        )
        .sum()
    ),
)

print(
    "Components with <2 depth clusters:",
    int(
        (
            candidate_attempt_summary_7cc_df[
                "num_depth_clusters_present"
            ]
            <
            2
        )
        .sum()
    ),
)


# ------------------------------------------------------------
# 4. Build consolidated depth-group table
#
# Each row now represents ONE k-means depth cluster within
# one original candidate.
#
# All disconnected spatial pieces belonging to that same
# depth cluster are combined conceptually.
#
# No pixels are deleted.
# ------------------------------------------------------------

valid_depth_fragments_7cc_df = (
    candidate_raw_meta_7cc_df[
        candidate_raw_meta_7cc_df[
            "depth_cluster"
        ]
        >=
        0
    ]
    .copy()
)


consolidated_depth_groups_7cc_df = (
    valid_depth_fragments_7cc_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ],
        as_index=False,
    )
    .agg(
        num_spatial_fragments=(
            "proposal_subcomponent_id",
            "count",
        ),

        consolidated_area=(
            "proposal_area",
            "sum",
        ),

        original_component_area=(
            "original_component_area",
            "first",
        ),
    )
)


consolidated_depth_groups_7cc_df[
    "consolidated_area_fraction"
] = (
    consolidated_depth_groups_7cc_df[
        "consolidated_area"
    ]
    /
    consolidated_depth_groups_7cc_df[
        "original_component_area"
    ]
)


# ------------------------------------------------------------
# 5. Rank the two consolidated depth groups by area
#
# Rank 1 = larger depth population
# Rank 2 = smaller depth population
# ------------------------------------------------------------

consolidated_depth_groups_7cc_df[
    "depth_group_area_rank"
] = (
    consolidated_depth_groups_7cc_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ]
    )[
        "consolidated_area"
    ]
    .rank(
        method="first",
        ascending=False,
    )
    .astype(int)
)


# ------------------------------------------------------------
# 6. Integrity check:
#    consolidated depth groups must preserve all candidate
#    semantic foreground exactly.
# ------------------------------------------------------------

consolidated_area_check_7cc_df = (
    consolidated_depth_groups_7cc_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ],
        as_index=False,
    )
    .agg(
        consolidated_area_sum=(
            "consolidated_area",
            "sum",
        ),

        original_component_area=(
            "original_component_area",
            "first",
        ),

        num_depth_groups=(
            "depth_cluster",
            "count",
        ),
    )
)


assert (
    consolidated_area_check_7cc_df[
        "consolidated_area_sum"
    ]
    ==
    consolidated_area_check_7cc_df[
        "original_component_area"
    ]
).all()


# ------------------------------------------------------------
# 7. Attach depth-cluster identity to each mapped GT point
#
# mapped_gt_points_7c_df already tells us which raw proposal
# fragment contains each GT point.
# ------------------------------------------------------------

proposal_to_depth_cluster_7cc_df = (
    candidate_raw_meta_7cc_df[
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
            "depth_cluster",
        ]
    ]
    .drop_duplicates()
)


mapped_gt_depth_7cc_df = (
    mapped_gt_points_7c_df
    .merge(
        proposal_to_depth_cluster_7cc_df,
        on=[
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    mapped_gt_depth_7cc_df[
        "depth_cluster"
    ]
    .notna()
    .all()
)


assert (
    mapped_gt_depth_7cc_df[
        "depth_cluster"
    ]
    >=
    0
).all()


# ------------------------------------------------------------
# 8. Count GT points in each consolidated depth group
# ------------------------------------------------------------

depth_group_gt_counts_7cc_df = (
    mapped_gt_depth_7cc_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ],
        as_index=False,
    )
    .agg(
        gt_points_in_depth_group=(
            "gt_point_index",
            "count",
        )
    )
)


consolidated_depth_groups_7cc_df = (
    consolidated_depth_groups_7cc_df
    .merge(
        depth_group_gt_counts_7cc_df,
        on=[
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ],
        how="left",
        validate="one_to_one",
    )
)


consolidated_depth_groups_7cc_df[
    "gt_points_in_depth_group"
] = (
    consolidated_depth_groups_7cc_df[
        "gt_points_in_depth_group"
    ]
    .fillna(0)
    .astype(int)
)


# ------------------------------------------------------------
# 9. Attach original GT count / GT group
# ------------------------------------------------------------

component_gt_lookup_7cc_df = (
    component_gt_split_diag_7c_df[
        [
            "dataset_index",
            "original_component_id",
            "gt_group",
            "original_gt_count",
        ]
    ]
    .copy()
)


consolidated_depth_groups_7cc_df = (
    consolidated_depth_groups_7cc_df
    .merge(
        component_gt_lookup_7cc_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    consolidated_depth_groups_7cc_df[
        "original_gt_count"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 10. One row per ORIGINAL candidate:
#
# compare raw-fragment separation with consolidated
# two-depth-group separation.
# ------------------------------------------------------------

component_depth_consolidation_rows_7cc = []


for _, comp_row in (
    component_gt_split_diag_7c_df
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .iterrows()
):

    dataset_index = int(
        comp_row[
            "dataset_index"
        ]
    )

    original_component_id = int(
        comp_row[
            "original_component_id"
        ]
    )

    gt_group = str(
        comp_row[
            "gt_group"
        ]
    )

    original_gt_count = int(
        comp_row[
            "original_gt_count"
        ]
    )


    depth_df = (
        consolidated_depth_groups_7cc_df[
            (
                consolidated_depth_groups_7cc_df[
                    "dataset_index"
                ]
                ==
                dataset_index
            )
            &
            (
                consolidated_depth_groups_7cc_df[
                    "original_component_id"
                ]
                ==
                original_component_id
            )
        ]
        .sort_values(
            "depth_group_area_rank"
        )
        .reset_index(
            drop=True
        )
    )


    num_depth_groups = int(
        len(
            depth_df
        )
    )


    # Normally 2.
    # Keep diagnostic robust to any rare degenerate case.
    gt_bearing_depth_groups = int(
        (
            depth_df[
                "gt_points_in_depth_group"
            ]
            >
            0
        )
        .sum()
    )


    gt_mapped_depth = int(
        depth_df[
            "gt_points_in_depth_group"
        ]
        .sum()
    )


    assert (
        gt_mapped_depth
        ==
        original_gt_count
    )


    if original_gt_count == 0:

        max_gt_in_depth_group = 0

        depth_separated_gt_fraction = (
            np.nan
        )

        both_depth_groups_gt_supported = (
            False
        )

        gt_fraction_larger_depth_group = (
            np.nan
        )

        gt_fraction_smaller_depth_group = (
            np.nan
        )


    else:

        max_gt_in_depth_group = int(
            depth_df[
                "gt_points_in_depth_group"
            ]
            .max()
        )


        depth_separated_gt_fraction = float(
            1.0
            -
            (
                max_gt_in_depth_group
                /
                original_gt_count
            )
        )


        both_depth_groups_gt_supported = bool(
            gt_bearing_depth_groups
            >=
            2
        )


        # rank 1 = larger consolidated depth population
        gt_larger = int(
            depth_df.loc[
                depth_df[
                    "depth_group_area_rank"
                ]
                ==
                1,
                "gt_points_in_depth_group",
            ]
            .sum()
        )


        gt_smaller = int(
            depth_df.loc[
                depth_df[
                    "depth_group_area_rank"
                ]
                ==
                2,
                "gt_points_in_depth_group",
            ]
            .sum()
        )


        gt_fraction_larger_depth_group = float(
            gt_larger
            /
            original_gt_count
        )


        gt_fraction_smaller_depth_group = float(
            gt_smaller
            /
            original_gt_count
        )


    # Area fractions of larger / smaller depth groups
    larger_area_fraction = float(
        depth_df.loc[
            depth_df[
                "depth_group_area_rank"
            ]
            ==
            1,
            "consolidated_area_fraction",
        ]
        .sum()
    )


    smaller_area_fraction = float(
        depth_df.loc[
            depth_df[
                "depth_group_area_rank"
            ]
            ==
            2,
            "consolidated_area_fraction",
        ]
        .sum()
    )


    component_depth_consolidation_rows_7cc.append(
        {
            "dataset_index":
                dataset_index,

            "original_component_id":
                original_component_id,

            "gt_group":
                gt_group,

            "original_gt_count":
                original_gt_count,

            "num_raw_fragments":
                int(
                    comp_row[
                        "num_raw_fragments"
                    ]
                ),

            "num_raw_GT_bearing_fragments":
                int(
                    comp_row[
                        "num_gt_bearing_fragments"
                    ]
                ),

            "num_consolidated_depth_groups":
                num_depth_groups,

            "num_GT_bearing_depth_groups":
                gt_bearing_depth_groups,

            "max_GT_in_one_depth_group":
                max_gt_in_depth_group,

            "depth_separated_GT_fraction":
                depth_separated_gt_fraction,

            "both_depth_groups_GT_supported":
                both_depth_groups_gt_supported,

            "larger_depth_group_area_fraction":
                larger_area_fraction,

            "smaller_depth_group_area_fraction":
                smaller_area_fraction,

            "GT_fraction_larger_depth_group":
                gt_fraction_larger_depth_group,

            "GT_fraction_smaller_depth_group":
                gt_fraction_smaller_depth_group,
        }
    )


component_depth_consolidation_7cc_df = (
    pd.DataFrame(
        component_depth_consolidation_rows_7cc
    )
)


assert len(
    component_depth_consolidation_7cc_df
) == 699


# ------------------------------------------------------------
# 11. GT1 diagnostic
#
# If k-means depth partition is geometrically meaningful,
# a GT1 component may still be divided into two depth groups,
# but only one group should contain its single GT point.
#
# We also inspect whether the GT tends to lie in the larger
# depth population.
# ------------------------------------------------------------

gt1_depth_7cc_df = (
    component_depth_consolidation_7cc_df[
        component_depth_consolidation_7cc_df[
            "gt_group"
        ]
        ==
        "GT1"
    ]
    .copy()
)


gt1_depth_summary_7cc_df = (
    pd.DataFrame(
        [
            {
                "num_GT1_components":
                    int(
                        len(
                            gt1_depth_7cc_df
                        )
                    ),

                "GT_in_larger_depth_group_fraction":
                    float(
                        (
                            gt1_depth_7cc_df[
                                "GT_fraction_larger_depth_group"
                            ]
                            ==
                            1.0
                        )
                        .mean()
                    ),

                "GT_in_smaller_depth_group_fraction":
                    float(
                        (
                            gt1_depth_7cc_df[
                                "GT_fraction_smaller_depth_group"
                            ]
                            ==
                            1.0
                        )
                        .mean()
                    ),

                "median_larger_depth_area_fraction":
                    float(
                        gt1_depth_7cc_df[
                            "larger_depth_group_area_fraction"
                        ]
                        .median()
                    ),

                "median_smaller_depth_area_fraction":
                    float(
                        gt1_depth_7cc_df[
                            "smaller_depth_group_area_fraction"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


# ------------------------------------------------------------
# 12. GT2+ diagnostic
#
# Key question:
#
# After ALL disconnected islands are consolidated back into
# the two original k-means depth populations, do multiple
# GT objects still occupy BOTH depth groups?
# ------------------------------------------------------------

gt2_depth_7cc_df = (
    component_depth_consolidation_7cc_df[
        component_depth_consolidation_7cc_df[
            "gt_group"
        ]
        ==
        "GT2+"
    ]
    .copy()
)


gt2_depth_summary_7cc_df = (
    pd.DataFrame(
        [
            {
                "num_GT2plus_components":
                    int(
                        len(
                            gt2_depth_7cc_df
                        )
                    ),

                "fraction_both_depth_groups_GT_supported":
                    float(
                        gt2_depth_7cc_df[
                            "both_depth_groups_GT_supported"
                        ]
                        .mean()
                    ),

                "median_depth_separated_GT_fraction":
                    float(
                        gt2_depth_7cc_df[
                            "depth_separated_GT_fraction"
                        ]
                        .median()
                    ),

                "mean_depth_separated_GT_fraction":
                    float(
                        gt2_depth_7cc_df[
                            "depth_separated_GT_fraction"
                        ]
                        .mean()
                    ),

                "median_GT_fraction_larger_depth_group":
                    float(
                        gt2_depth_7cc_df[
                            "GT_fraction_larger_depth_group"
                        ]
                        .median()
                    ),

                "median_GT_fraction_smaller_depth_group":
                    float(
                        gt2_depth_7cc_df[
                            "GT_fraction_smaller_depth_group"
                        ]
                        .median()
                    ),

                "median_larger_depth_area_fraction":
                    float(
                        gt2_depth_7cc_df[
                            "larger_depth_group_area_fraction"
                        ]
                        .median()
                    ),

                "median_smaller_depth_area_fraction":
                    float(
                        gt2_depth_7cc_df[
                            "smaller_depth_group_area_fraction"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


# ------------------------------------------------------------
# 13. Direct comparison:
#     raw spatial-fragment separation
#     versus
#     consolidated two-depth-group separation
# ------------------------------------------------------------

raw_vs_depth_7cc_df = (
    gt2_depth_7cc_df[
        [
            "dataset_index",
            "original_component_id",
            "original_gt_count",
            "num_raw_GT_bearing_fragments",
            "num_GT_bearing_depth_groups",
            "depth_separated_GT_fraction",
        ]
    ]
    .copy()
)


raw_gt2_reference_7cc_df = (
    component_gt_split_diag_7c_df[
        component_gt_split_diag_7c_df[
            "gt_group"
        ]
        ==
        "GT2+"
    ]
    [
        [
            "dataset_index",
            "original_component_id",
            "separated_gt_fraction",
        ]
    ]
    .rename(
        columns={
            "separated_gt_fraction":
                "raw_spatial_separated_GT_fraction"
        }
    )
)


raw_vs_depth_7cc_df = (
    raw_vs_depth_7cc_df
    .merge(
        raw_gt2_reference_7cc_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


raw_vs_depth_summary_7cc_df = (
    pd.DataFrame(
        [
            {
                "num_GT2plus_components":
                    len(
                        raw_vs_depth_7cc_df
                    ),

                "raw_fraction_with_2plus_GT_fragments":
                    float(
                        (
                            raw_vs_depth_7cc_df[
                                "num_raw_GT_bearing_fragments"
                            ]
                            >=
                            2
                        )
                        .mean()
                    ),

                "consolidated_fraction_both_depth_groups_GT":
                    float(
                        (
                            raw_vs_depth_7cc_df[
                                "num_GT_bearing_depth_groups"
                            ]
                            >=
                            2
                        )
                        .mean()
                    ),

                "median_raw_spatial_separated_GT_fraction":
                    float(
                        raw_vs_depth_7cc_df[
                            "raw_spatial_separated_GT_fraction"
                        ]
                        .median()
                    ),

                "median_consolidated_depth_separated_GT_fraction":
                    float(
                        raw_vs_depth_7cc_df[
                            "depth_separated_GT_fraction"
                        ]
                        .median()
                    ),
            }
        ]
    )
)


# ------------------------------------------------------------
# 14. GT2+ breakdown by original GT count
#
# This is diagnostic only.
#
# Useful to see whether the two-depth-group signal is mainly
# effective for lightly merged regions (e.g. 2-3 GT) or also
# persists in denser merged components.
# ------------------------------------------------------------

gt2_depth_by_count_7cc_df = (
    gt2_depth_7cc_df
    .groupby(
        "original_gt_count",
        as_index=False,
    )
    .agg(
        num_components=(
            "original_component_id",
            "count",
        ),

        fraction_both_depth_groups_GT=(
            "both_depth_groups_GT_supported",
            "mean",
        ),

        median_depth_separated_GT_fraction=(
            "depth_separated_GT_fraction",
            "median",
        ),

        mean_depth_separated_GT_fraction=(
            "depth_separated_GT_fraction",
            "mean",
        ),

        median_smaller_depth_area_fraction=(
            "smaller_depth_group_area_fraction",
            "median",
        ),
    )
    .sort_values(
        "original_gt_count"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 15. Consolidated depth-group GT occupancy
#
# Across all consolidated depth groups:
#
#   GT0 group  : no GT points
#   GT1 group  : exactly one GT point
#   GT2+ group : multiple GT points
# ------------------------------------------------------------

consolidated_depth_groups_7cc_df[
    "depth_group_GT_class"
] = np.select(
    [
        consolidated_depth_groups_7cc_df[
            "gt_points_in_depth_group"
        ]
        ==
        0,

        consolidated_depth_groups_7cc_df[
            "gt_points_in_depth_group"
        ]
        ==
        1,

        consolidated_depth_groups_7cc_df[
            "gt_points_in_depth_group"
        ]
        >=
        2,
    ],
    [
        "GT0",
        "GT1",
        "GT2+",
    ],
    default="UNKNOWN",
)


assert (
    consolidated_depth_groups_7cc_df[
        "depth_group_GT_class"
    ]
    !=
    "UNKNOWN"
).all()


depth_group_GT_occupancy_7cc_df = (
    consolidated_depth_groups_7cc_df
    .groupby(
        "depth_group_GT_class",
        as_index=False,
    )
    .agg(
        num_depth_groups=(
            "depth_cluster",
            "count",
        ),

        median_area_fraction=(
            "consolidated_area_fraction",
            "median",
        ),

        mean_area_fraction=(
            "consolidated_area_fraction",
            "mean",
        ),

        median_num_spatial_fragments=(
            "num_spatial_fragments",
            "median",
        ),

        mean_num_spatial_fragments=(
            "num_spatial_fragments",
            "mean",
        ),
    )
)


depth_group_GT_occupancy_7cc_df[
    "fraction_all_depth_groups"
] = (
    depth_group_GT_occupancy_7cc_df[
        "num_depth_groups"
    ]
    /
    len(
        consolidated_depth_groups_7cc_df
    )
)


# ------------------------------------------------------------
# 16. Save outputs
# ------------------------------------------------------------

M1_7C_C_DEPTH_GROUP_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_consolidated_depth_groups.csv"
)

M1_7C_C_COMPONENT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_component_depth_consolidation.csv"
)

M1_7C_C_GT1_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_GT1_depth_consolidation_summary.csv"
)

M1_7C_C_GT2_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_GT2plus_depth_consolidation_summary.csv"
)

M1_7C_C_RAW_VS_DEPTH_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_raw_vs_depth_consolidation.csv"
)

M1_7C_C_GTCOUNT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_depth_consolidation_by_GT_count.csv"
)

M1_7C_C_OCCUPANCY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_C_depth_group_GT_occupancy.csv"
)


consolidated_depth_groups_7cc_df.to_csv(
    M1_7C_C_DEPTH_GROUP_FILE,
    index=False,
)

component_depth_consolidation_7cc_df.to_csv(
    M1_7C_C_COMPONENT_FILE,
    index=False,
)

gt1_depth_summary_7cc_df.to_csv(
    M1_7C_C_GT1_FILE,
    index=False,
)

gt2_depth_summary_7cc_df.to_csv(
    M1_7C_C_GT2_FILE,
    index=False,
)

raw_vs_depth_7cc_df.to_csv(
    M1_7C_C_RAW_VS_DEPTH_FILE,
    index=False,
)

gt2_depth_by_count_7cc_df.to_csv(
    M1_7C_C_GTCOUNT_FILE,
    index=False,
)

depth_group_GT_occupancy_7cc_df.to_csv(
    M1_7C_C_OCCUPANCY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 17. Report — candidate audit
# ------------------------------------------------------------

print("\nDepth-cluster structural audit")
print("------------------------------")

display(
    candidate_attempt_summary_7cc_df[
        [
            "num_raw_fragments",
            "num_depth_clusters_present",
        ]
    ]
    .describe()
)


# ------------------------------------------------------------
# 18. Report — GT1
# ------------------------------------------------------------

print("\nGT1 — consolidated depth-group diagnostic")
print("-----------------------------------------")

display(
    gt1_depth_summary_7cc_df
)


# ------------------------------------------------------------
# 19. Report — GT2+
# ------------------------------------------------------------

print("\nGT2+ — consolidated depth-group diagnostic")
print("------------------------------------------")

display(
    gt2_depth_summary_7cc_df
)


# ------------------------------------------------------------
# 20. Report — raw versus consolidated
# ------------------------------------------------------------

print("\nGT2+ — raw spatial fragments vs consolidated depth groups")
print("----------------------------------------------------------")

display(
    raw_vs_depth_summary_7cc_df
)


# ------------------------------------------------------------
# 21. Report — by GT count
# ------------------------------------------------------------

print("\nGT2+ consolidation by original GT count")
print("---------------------------------------")

display(
    gt2_depth_by_count_7cc_df
    .head(
        30
    )
)


# ------------------------------------------------------------
# 22. Report — depth-group GT occupancy
# ------------------------------------------------------------

print("\nConsolidated depth-group GT occupancy")
print("-------------------------------------")

display(
    depth_group_GT_occupancy_7cc_df
)


# ------------------------------------------------------------
# 23. Most informative GT2+ cases
# ------------------------------------------------------------

print("\nGT2+ cases with strongest two-depth separation")
print("----------------------------------------------")

display(
    gt2_depth_7cc_df
    .sort_values(
        [
            "depth_separated_GT_fraction",
            "original_gt_count",
        ],
        ascending=[
            False,
            False,
        ],
    )
    [
        [
            "dataset_index",
            "original_component_id",
            "original_gt_count",
            "num_raw_fragments",
            "num_raw_GT_bearing_fragments",
            "num_GT_bearing_depth_groups",
            "depth_separated_GT_fraction",
            "larger_depth_group_area_fraction",
            "smaller_depth_group_area_fraction",
            "GT_fraction_larger_depth_group",
            "GT_fraction_smaller_depth_group",
        ]
    ]
    .head(
        20
    )
)


# ------------------------------------------------------------
# 24. Protocol report
# ------------------------------------------------------------

print("\nStep-7A candidate rule reused       : YES")
print("Step-7B k=2 depth partition reused  : YES")
print("Disconnected fragments consolidated : YES — diagnostic only")
print("Pixels deleted                      : NO")
print("Pixels added                        : NO")
print("Size threshold used                 : NO")
print("GT used for consolidation           : NO")
print("GT used for diagnostic evaluation   : YES")
print("Split accepted/rejected             : NO")
print("Count rule modified                 : NO")
print("Density routing used                : NO")
print("Test set accessed                   : NO")

print("\nStep 7C Part C: PASS")

STEP 7C PART C — DEPTH-CLUSTER CONSOLIDATION DIAGNOSTIC

Candidate split-attempt audit
-----------------------------
Candidate components: 699
Split attempted: 699
Components with 2 depth clusters: 699
Components with <2 depth clusters: 0

Depth-cluster structural audit
------------------------------


,num_raw_fragments,num_depth_clusters_present
count,699.000000,699.0
mean,9.330472,2.0
std,19.046041,0.0
min,2.000000,2.0
25%,3.000000,2.0
50%,5.000000,2.0
75%,9.000000,2.0
max,359.000000,2.0



GT1 — consolidated depth-group diagnostic
-----------------------------------------


,num_GT1_components,GT_in_larger_depth_group_fraction,GT_in_smaller_depth_group_fraction,median_larger_depth_area_fraction,median_smaller_depth_area_fraction
0,248,0.83871,0.16129,0.673952,0.326048



GT2+ — consolidated depth-group diagnostic
------------------------------------------


...


Consolidated depth-group GT occupancy
-------------------------------------


,depth_group_GT_class,num_depth_groups,median_area_fraction,mean_area_fraction,median_num_spatial_fragments,mean_num_spatial_fragments,fraction_all_depth_groups
0,GT0,753,0.416131,0.424115,2.0,3.912351,0.538627
1,GT1,362,0.614332,0.616566,1.0,3.226519,0.258941
2,GT2+,283,0.552239,0.552808,4.0,8.508834,0.202432



GT2+ cases with strongest two-depth separation
----------------------------------------------


,dataset_index,original_component_id,original_gt_count,num_raw_fragments,num_raw_GT_bearing_fragments,num_GT_bearing_depth_groups,depth_separated_GT_fraction,larger_depth_group_area_fraction,smaller_depth_group_area_fraction,GT_fraction_larger_depth_group,GT_fraction_smaller_depth_group
695,1269,4,26,68,3,2,0.5,0.503127,0.496873,0.5,0.5
204,229,11,12,3,2,2,0.5,0.590759,0.409241,0.5,0.5
73,143,23,10,7,2,2,0.5,0.656556,0.343444,0.5,0.5
208,229,23,8,8,2,2,0.5,0.537797,0.462203,0.5,0.5
74,143,25,6,38,2,2,0.5,0.647315,0.352685,0.5,0.5
162,183,20,6,17,4,2,0.5,0.577437,0.422563,0.5,0.5
205,229,12,6,4,2,2,0.5,0.565008,0.434992,0.5,0.5
163,183,22,4,27,2,2,0.5,0.576622,0.423378,0.5,0.5
172,184,20,4,10,2,2,0.5,0.548435,0.451565,0.5,0.5
224,253,12,4,3,2,2,0.5,0.863529,0.136471,0.5,0.5



Step-7A candidate rule reused       : YES
Step-7B k=2 depth partition reused  : YES
Disconnected fragments consolidated : YES — diagnostic only
Pixels deleted                      : NO
Pixels added                        : NO
Size threshold used                 : NO
GT used for consolidation           : NO
GT used for diagnostic evaluation   : YES
Split accepted/rejected             : NO
Count rule modified                 : NO
Density routing used                : NO
Test set accessed                   : NO

Step 7C Part C: PASS


### **Finding 9 — Step 7C Part C:**

Consolidating disconnected raw fragments solely according to their two k-means depth-cluster identities substantially reduces the object-separation signal observed in Part B.

The proportion of GT2+ components with GT support in multiple groups falls from 81.4% for raw spatial fragments to 47.6% after depth-cluster consolidation, while the median separated-GT fraction falls from 0.50 to 0.00.

Therefore, the useful Marigold signal is not explained by coarse binary depth partition alone; it arises from the interaction between depth structure and spatial connectivity.

Full depth-cluster consolidation is consequently not supported as the refinement mechanism.

In [31]:
# ============================================================
# Step 7C Part D — Anchor–satellite spatial-noise diagnostic
# ============================================================
#
# Purpose
# -------
# Step 7C Part C showed that globally consolidating all
# disconnected pieces belonging to the same k-means depth
# cluster destroys too much useful instance separation.
#
# Therefore we now examine the disconnected pieces within
# each depth cluster more carefully.
#
# Inference-only structural definition:
#
#   For each original candidate component
#     and for each k-means depth cluster:
#
#       largest connected fragment = ANCHOR
#       all other connected fragments = SATELLITES
#
# We then measure:
#
#   - satellite relative area
#   - satellite area relative to its depth cluster
#   - minimum spatial distance to the same-depth anchor
#   - centroid distance to the anchor
#
# GT is attached ONLY AFTER those quantities are calculated.
#
# Questions
# ---------
# 1. Are most satellites GT-empty?
#
# 2. Are GT-empty satellites smaller than GT-bearing
#    satellites?
#
# 3. Are GT-empty satellites spatially closer to their
#    same-depth anchor?
#
# 4. How much genuine GT support resides in satellites?
#
# 5. Does a plausible inference-time distinction exist
#    between connectivity noise and spatially meaningful
#    satellite fragments?
#
# IMPORTANT
# ---------
# No fragment is merged or removed in this step.
#
# No:
#   - size threshold
#   - distance threshold
#   - fragment deletion
#   - fragment merging
#   - GT-based routing
#   - density routing
#   - count modification
#
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from scipy import ndimage

from sklearn.metrics import roc_auc_score


print("=" * 120)
print("STEP 7C PART D — ANCHOR–SATELLITE SPATIAL-NOISE DIAGNOSTIC")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "raw_split_proposals_7b_df" in globals(), (
    "raw_split_proposals_7b_df unavailable. "
    "Run Step 7B first."
)

assert "all_candidate_fragments_7c_df" in globals(), (
    "all_candidate_fragments_7c_df unavailable. "
    "Run Step 7C Part B first."
)

assert "image_split_summary_7b_df" in globals(), (
    "image_split_summary_7b_df unavailable. "
    "Run Step 7B first."
)

assert "a3c_align_6d_df" in globals(), (
    "a3c_align_6d_df unavailable. "
    "Run Step 6D Part B first."
)

assert "component_gt_split_diag_7c_df" in globals(), (
    "component_gt_split_diag_7c_df unavailable. "
    "Run Step 7C Part B first."
)


# ------------------------------------------------------------
# 2. Candidate raw-fragment metadata
# ------------------------------------------------------------

fragment_meta_7cd_df = (
    raw_split_proposals_7b_df[
        raw_split_proposals_7b_df[
            "split_candidate"
        ]
    ]
    [
        [
            "dataset_index",
            "image_id",
            "original_component_id",
            "proposal_subcomponent_id",
            "depth_cluster",
            "proposal_area",
            "original_component_area",
            "proposal_area_fraction_of_original",
        ]
    ]
    .copy()
)


assert len(
    fragment_meta_7cd_df
) == 6522


assert (
    fragment_meta_7cd_df[
        "depth_cluster"
    ]
    >= 0
).all()


# ------------------------------------------------------------
# 3. Define ANCHOR versus SATELLITE
#
# For each:
#
#   dataset_index
#   original_component_id
#   depth_cluster
#
# the largest connected raw fragment is the anchor.
#
# Ties are resolved deterministically using proposal ID.
#
# No GT is involved.
# ------------------------------------------------------------

fragment_meta_7cd_df = (
    fragment_meta_7cd_df
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
            "depth_cluster",
            "proposal_area",
            "proposal_subcomponent_id",
        ],
        ascending=[
            True,
            True,
            True,
            False,
            True,
        ],
    )
    .reset_index(
        drop=True
    )
)


fragment_meta_7cd_df[
    "rank_within_depth_cluster"
] = (
    fragment_meta_7cd_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ]
    )
    .cumcount()
    +
    1
)


fragment_meta_7cd_df[
    "is_depth_anchor"
] = (
    fragment_meta_7cd_df[
        "rank_within_depth_cluster"
    ]
    ==
    1
)


fragment_meta_7cd_df[
    "is_depth_satellite"
] = (
    ~fragment_meta_7cd_df[
        "is_depth_anchor"
    ]
)


# ------------------------------------------------------------
# 4. Depth-cluster totals
# ------------------------------------------------------------

depth_cluster_totals_7cd_df = (
    fragment_meta_7cd_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ],
        as_index=False,
    )
    .agg(
        depth_cluster_area=(
            "proposal_area",
            "sum",
        ),

        num_fragments_in_depth_cluster=(
            "proposal_subcomponent_id",
            "count",
        ),
    )
)


fragment_meta_7cd_df = (
    fragment_meta_7cd_df
    .merge(
        depth_cluster_totals_7cd_df,
        on=[
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ],
        how="left",
        validate="many_to_one",
    )
)


fragment_meta_7cd_df[
    "area_fraction_of_depth_cluster"
] = (
    fragment_meta_7cd_df[
        "proposal_area"
    ]
    /
    fragment_meta_7cd_df[
        "depth_cluster_area"
    ]
)


# ------------------------------------------------------------
# 5. Anchor ID lookup
# ------------------------------------------------------------

anchor_lookup_7cd_df = (
    fragment_meta_7cd_df[
        fragment_meta_7cd_df[
            "is_depth_anchor"
        ]
    ]
    [
        [
            "dataset_index",
            "original_component_id",
            "depth_cluster",
            "proposal_subcomponent_id",
            "proposal_area",
        ]
    ]
    .rename(
        columns={
            "proposal_subcomponent_id":
                "anchor_proposal_id",

            "proposal_area":
                "anchor_area",
        }
    )
)


assert len(
    anchor_lookup_7cd_df
) == (
    699 * 2
)


fragment_meta_7cd_df = (
    fragment_meta_7cd_df
    .merge(
        anchor_lookup_7cd_df,
        on=[
            "dataset_index",
            "original_component_id",
            "depth_cluster",
        ],
        how="left",
        validate="many_to_one",
    )
)


# ------------------------------------------------------------
# 6. Spatial measurements
#
# For every raw fragment calculate:
#
#   minimum Euclidean pixel distance to its same-depth anchor
#
#   centroid distance to its same-depth anchor
#
# Both are normalized by sqrt(original component area).
#
# Normalization provides a local scale reference and avoids
# using absolute image-scale thresholds.
#
# Anchors have distance = 0.
#
# No GT is used here.
# ------------------------------------------------------------

spatial_rows_7cd = []


# Candidate lookup by image for speed
fragment_groups_by_image_7cd = {
    int(dataset_index): df.copy()
    for dataset_index, df
    in fragment_meta_7cd_df.groupby(
        "dataset_index"
    )
}


for _, image_row in (
    image_split_summary_7b_df
    .sort_values(
        "dataset_index"
    )
    .iterrows()
):

    dataset_index = int(
        image_row[
            "dataset_index"
        ]
    )


    image_frag_df = (
        fragment_groups_by_image_7cd[
            dataset_index
        ]
    )


    # --------------------------------------------------------
    # Load Step-7B proposal map
    # --------------------------------------------------------

    proposal_map_path = Path(
        image_row[
            "proposal_map_file"
        ]
    )


    with np.load(
        proposal_map_path,
        allow_pickle=False,
    ) as data:

        proposal_labels = (
            np.asarray(
                data[
                    "proposal_labels"
                ]
            )
            .astype(
                np.int32
            )
        )


    # --------------------------------------------------------
    # Recover frozen A3c component labels so that processing
    # can be cropped to each original component bounding box.
    # --------------------------------------------------------

    align_row_df = (
        a3c_align_6d_df[
            a3c_align_6d_df[
                "dataset_index"
            ]
            ==
            dataset_index
        ]
    )


    assert len(
        align_row_df
    ) == 1


    align_row = (
        align_row_df.iloc[0]
    )


    with np.load(
        Path(
            align_row[
                "cache_file"
            ]
        ),
        allow_pickle=False,
    ) as data:

        a3c_mask = (
            np.asarray(
                data[
                    "mask"
                ]
            )
            .astype(bool)
        )


    assert (
        a3c_mask.shape
        ==
        proposal_labels.shape
    )


    structure_4_7cd = (
        ndimage.generate_binary_structure(
            rank=2,
            connectivity=1,
        )
    )


    original_labels, num_original = (
        ndimage.label(
            a3c_mask,
            structure=
                structure_4_7cd,
        )
    )


    assert (
        int(
            num_original
        )
        ==
        int(
            align_row[
                "num_components"
            ]
        )
    )


    component_slices = (
        ndimage.find_objects(
            original_labels
        )
    )


    # --------------------------------------------------------
    # Process only Step-7A candidate original components
    # --------------------------------------------------------

    for original_component_id, comp_df in (
        image_frag_df
        .groupby(
            "original_component_id"
        )
    ):

        original_component_id = int(
            original_component_id
        )


        component_slice = (
            component_slices[
                original_component_id
                -
                1
            ]
        )


        assert (
            component_slice
            is not None
        )


        label_crop = (
            proposal_labels[
                component_slice
            ]
        )


        original_area = int(
            comp_df[
                "original_component_area"
            ]
            .iloc[0]
        )


        scale_px = float(
            np.sqrt(
                max(
                    original_area,
                    1,
                )
            )
        )


        # ----------------------------------------------------
        # Process the two depth clusters independently
        # ----------------------------------------------------

        for depth_cluster, cluster_df in (
            comp_df
            .groupby(
                "depth_cluster"
            )
        ):

            depth_cluster = int(
                depth_cluster
            )


            anchor_row = (
                cluster_df[
                    cluster_df[
                        "is_depth_anchor"
                    ]
                ]
            )


            assert len(
                anchor_row
            ) == 1


            anchor_id = int(
                anchor_row[
                    "proposal_subcomponent_id"
                ]
                .iloc[0]
            )


            anchor_mask = (
                label_crop
                ==
                anchor_id
            )


            assert (
                anchor_mask.sum()
                >
                0
            )


            # ------------------------------------------------
            # Distance transform:
            #
            # for every non-anchor pixel:
            # distance to nearest anchor pixel
            # ------------------------------------------------

            distance_to_anchor = (
                ndimage.distance_transform_edt(
                    ~anchor_mask
                )
            )


            # Anchor centroid
            anchor_y, anchor_x = (
                np.where(
                    anchor_mask
                )
            )


            anchor_centroid_y = float(
                anchor_y.mean()
            )

            anchor_centroid_x = float(
                anchor_x.mean()
            )


            # ------------------------------------------------
            # Every fragment in this depth cluster
            # ------------------------------------------------

            for _, fragment_row in (
                cluster_df
                .iterrows()
            ):

                proposal_id = int(
                    fragment_row[
                        "proposal_subcomponent_id"
                    ]
                )


                fragment_mask = (
                    label_crop
                    ==
                    proposal_id
                )


                assert (
                    fragment_mask.sum()
                    ==
                    int(
                        fragment_row[
                            "proposal_area"
                        ]
                    )
                )


                if (
                    bool(
                        fragment_row[
                            "is_depth_anchor"
                        ]
                    )
                ):

                    min_distance_px = 0.0

                    centroid_distance_px = 0.0


                else:

                    # Minimum pixel-to-anchor distance
                    min_distance_px = float(
                        distance_to_anchor[
                            fragment_mask
                        ]
                        .min()
                    )


                    fragment_y, fragment_x = (
                        np.where(
                            fragment_mask
                        )
                    )


                    fragment_centroid_y = float(
                        fragment_y.mean()
                    )

                    fragment_centroid_x = float(
                        fragment_x.mean()
                    )


                    centroid_distance_px = float(
                        np.hypot(
                            fragment_centroid_y
                            -
                            anchor_centroid_y,

                            fragment_centroid_x
                            -
                            anchor_centroid_x,
                        )
                    )


                spatial_rows_7cd.append(
                    {
                        "dataset_index":
                            dataset_index,

                        "original_component_id":
                            original_component_id,

                        "depth_cluster":
                            depth_cluster,

                        "proposal_subcomponent_id":
                            proposal_id,

                        "min_distance_to_anchor_px":
                            min_distance_px,

                        "centroid_distance_to_anchor_px":
                            centroid_distance_px,

                        "min_distance_to_anchor_norm":
                            float(
                                min_distance_px
                                /
                                scale_px
                            ),

                        "centroid_distance_to_anchor_norm":
                            float(
                                centroid_distance_px
                                /
                                scale_px
                            ),
                    }
                )


fragment_spatial_7cd_df = (
    pd.DataFrame(
        spatial_rows_7cd
    )
)


assert len(
    fragment_spatial_7cd_df
) == len(
    fragment_meta_7cd_df
)


# ------------------------------------------------------------
# 7. Attach spatial measurements
# ------------------------------------------------------------

fragment_structure_7cd_df = (
    fragment_meta_7cd_df
    .merge(
        fragment_spatial_7cd_df,
        on=[
            "dataset_index",
            "original_component_id",
            "depth_cluster",
            "proposal_subcomponent_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    fragment_structure_7cd_df[
        "min_distance_to_anchor_norm"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 8. Attach GT occupancy AFTER structural measurements
#
# This is the first point where GT enters Part D.
# ------------------------------------------------------------

fragment_gt_lookup_7cd_df = (
    all_candidate_fragments_7c_df[
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
            "gt_points_in_fragment",
        ]
    ]
    .copy()
)


fragment_structure_7cd_df = (
    fragment_structure_7cd_df
    .merge(
        fragment_gt_lookup_7cd_df,
        on=[
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert (
    fragment_structure_7cd_df[
        "gt_points_in_fragment"
    ]
    .notna()
    .all()
)


fragment_structure_7cd_df[
    "gt_points_in_fragment"
] = (
    fragment_structure_7cd_df[
        "gt_points_in_fragment"
    ]
    .astype(int)
)


fragment_structure_7cd_df[
    "gt_supported_fragment"
] = (
    fragment_structure_7cd_df[
        "gt_points_in_fragment"
    ]
    >
    0
)


fragment_structure_7cd_df[
    "fragment_GT_class"
] = np.select(
    [
        fragment_structure_7cd_df[
            "gt_points_in_fragment"
        ]
        ==
        0,

        fragment_structure_7cd_df[
            "gt_points_in_fragment"
        ]
        ==
        1,

        fragment_structure_7cd_df[
            "gt_points_in_fragment"
        ]
        >=
        2,
    ],
    [
        "GT0",
        "GT1",
        "GT2+",
    ],
    default="UNKNOWN",
)


# ------------------------------------------------------------
# 9. Anchor versus satellite inventory
# ------------------------------------------------------------

anchor_satellite_inventory_7cd_df = (
    fragment_structure_7cd_df
    .groupby(
        "is_depth_anchor",
        as_index=False,
    )
    .agg(
        num_fragments=(
            "proposal_subcomponent_id",
            "count",
        ),

        GT_supported_fragments=(
            "gt_supported_fragment",
            "sum",
        ),

        total_GT_points=(
            "gt_points_in_fragment",
            "sum",
        ),

        total_pixels=(
            "proposal_area",
            "sum",
        ),

        median_area_fraction_parent=(
            "proposal_area_fraction_of_original",
            "median",
        ),

        median_area_fraction_depth_cluster=(
            "area_fraction_of_depth_cluster",
            "median",
        ),
    )
)


anchor_satellite_inventory_7cd_df[
    "GT_supported_fragment_fraction"
] = (
    anchor_satellite_inventory_7cd_df[
        "GT_supported_fragments"
    ]
    /
    anchor_satellite_inventory_7cd_df[
        "num_fragments"
    ]
)


anchor_satellite_inventory_7cd_df[
    "fraction_all_GT_points"
] = (
    anchor_satellite_inventory_7cd_df[
        "total_GT_points"
    ]
    /
    fragment_structure_7cd_df[
        "gt_points_in_fragment"
    ]
    .sum()
)


# ------------------------------------------------------------
# 10. Satellite-only diagnostic
# ------------------------------------------------------------

satellites_7cd_df = (
    fragment_structure_7cd_df[
        fragment_structure_7cd_df[
            "is_depth_satellite"
        ]
    ]
    .copy()
)


assert len(
    satellites_7cd_df
) == (
    len(
        fragment_structure_7cd_df
    )
    -
    (699 * 2)
)


# ------------------------------------------------------------
# 11. Satellite summary by GT support
# ------------------------------------------------------------

satellite_by_gt_7cd_df = (
    satellites_7cd_df
    .groupby(
        "gt_supported_fragment",
        as_index=False,
    )
    .agg(
        num_satellites=(
            "proposal_subcomponent_id",
            "count",
        ),

        total_GT_points=(
            "gt_points_in_fragment",
            "sum",
        ),

        median_area_pixels=(
            "proposal_area",
            "median",
        ),

        median_area_fraction_parent=(
            "proposal_area_fraction_of_original",
            "median",
        ),

        median_area_fraction_depth_cluster=(
            "area_fraction_of_depth_cluster",
            "median",
        ),

        median_min_distance_px=(
            "min_distance_to_anchor_px",
            "median",
        ),

        median_min_distance_norm=(
            "min_distance_to_anchor_norm",
            "median",
        ),

        median_centroid_distance_norm=(
            "centroid_distance_to_anchor_norm",
            "median",
        ),
    )
)


satellite_by_gt_7cd_df[
    "fraction_all_satellites"
] = (
    satellite_by_gt_7cd_df[
        "num_satellites"
    ]
    /
    len(
        satellites_7cd_df
    )
)


# ------------------------------------------------------------
# 12. GT-point support:
#     anchors versus satellites
# ------------------------------------------------------------

total_gt_points_in_candidates_7cd = int(
    fragment_structure_7cd_df[
        "gt_points_in_fragment"
    ]
    .sum()
)


gt_points_in_anchors_7cd = int(
    fragment_structure_7cd_df.loc[
        fragment_structure_7cd_df[
            "is_depth_anchor"
        ],
        "gt_points_in_fragment",
    ]
    .sum()
)


gt_points_in_satellites_7cd = int(
    satellites_7cd_df[
        "gt_points_in_fragment"
    ]
    .sum()
)


assert (
    gt_points_in_anchors_7cd
    +
    gt_points_in_satellites_7cd
    ==
    total_gt_points_in_candidates_7cd
)


GT_anchor_satellite_7cd_df = (
    pd.DataFrame(
        [
            {
                "location":
                    "anchor",

                "num_GT_points":
                    gt_points_in_anchors_7cd,

                "fraction_candidate_GT_points":
                    float(
                        gt_points_in_anchors_7cd
                        /
                        total_gt_points_in_candidates_7cd
                    ),
            },

            {
                "location":
                    "satellite",

                "num_GT_points":
                    gt_points_in_satellites_7cd,

                "fraction_candidate_GT_points":
                    float(
                        gt_points_in_satellites_7cd
                        /
                        total_gt_points_in_candidates_7cd
                    ),
            },
        ]
    )
)


# ------------------------------------------------------------
# 13. Diagnostic discrimination
#
# Can inference-available satellite properties distinguish:
#
#     GT-bearing satellite
#     versus
#     GT-empty satellite ?
#
# AUC is diagnostic only.
#
# NO threshold is selected.
# ------------------------------------------------------------

y_satellite_gt_7cd = (
    satellites_7cd_df[
        "gt_supported_fragment"
    ]
    .astype(int)
    .to_numpy()
)


assert len(
    np.unique(
        y_satellite_gt_7cd
    )
) == 2


diagnostic_features_7cd = {
    "area_fraction_parent":
        satellites_7cd_df[
            "proposal_area_fraction_of_original"
        ].to_numpy(),

    "area_fraction_depth_cluster":
        satellites_7cd_df[
            "area_fraction_of_depth_cluster"
        ].to_numpy(),

    "min_distance_to_anchor_norm":
        satellites_7cd_df[
            "min_distance_to_anchor_norm"
        ].to_numpy(),

    "centroid_distance_to_anchor_norm":
        satellites_7cd_df[
            "centroid_distance_to_anchor_norm"
        ].to_numpy(),
}


auc_rows_7cd = []


for feature_name, feature_values in (
    diagnostic_features_7cd.items()
):

    auc_value = float(
        roc_auc_score(
            y_satellite_gt_7cd,
            feature_values,
        )
    )


    auc_rows_7cd.append(
        {
            "feature":
                feature_name,

            "AUC_GT_supported_vs_GT0_satellite":
                auc_value,
        }
    )


satellite_auc_7cd_df = (
    pd.DataFrame(
        auc_rows_7cd
    )
    .sort_values(
        "AUC_GT_supported_vs_GT0_satellite",
        ascending=False,
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 14. GT0 vs GT1 vs GT2+ satellite descriptive comparison
# ------------------------------------------------------------

satellite_GT_class_summary_7cd_df = (
    satellites_7cd_df
    .groupby(
        "fragment_GT_class",
        as_index=False,
    )
    .agg(
        num_satellites=(
            "proposal_subcomponent_id",
            "count",
        ),

        median_area_fraction_parent=(
            "proposal_area_fraction_of_original",
            "median",
        ),

        median_area_fraction_depth_cluster=(
            "area_fraction_of_depth_cluster",
            "median",
        ),

        median_min_distance_norm=(
            "min_distance_to_anchor_norm",
            "median",
        ),

        median_centroid_distance_norm=(
            "centroid_distance_to_anchor_norm",
            "median",
        ),
    )
)


# ------------------------------------------------------------
# 15. Parent-component GT grouping
#
# Useful to determine whether GT-bearing satellites occur
# primarily in GT2+ parent components.
# ------------------------------------------------------------

component_parent_gt_7cd_df = (
    component_gt_split_diag_7c_df[
        [
            "dataset_index",
            "original_component_id",
            "gt_group",
            "original_gt_count",
        ]
    ]
    .copy()
)


satellites_7cd_df = (
    satellites_7cd_df
    .merge(
        component_parent_gt_7cd_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


satellite_by_parent_GT_7cd_df = (
    satellites_7cd_df
    .groupby(
        "gt_group",
        as_index=False,
    )
    .agg(
        num_satellites=(
            "proposal_subcomponent_id",
            "count",
        ),

        GT_supported_satellites=(
            "gt_supported_fragment",
            "sum",
        ),

        GT_points_in_satellites=(
            "gt_points_in_fragment",
            "sum",
        ),

        median_area_fraction_parent=(
            "proposal_area_fraction_of_original",
            "median",
        ),

        median_min_distance_norm=(
            "min_distance_to_anchor_norm",
            "median",
        ),
    )
)


satellite_by_parent_GT_7cd_df[
    "GT_supported_satellite_fraction"
] = (
    satellite_by_parent_GT_7cd_df[
        "GT_supported_satellites"
    ]
    /
    satellite_by_parent_GT_7cd_df[
        "num_satellites"
    ]
)


# ------------------------------------------------------------
# 16. Area-distance quartile diagnostic
#
# We do NOT select a threshold.
#
# Instead we divide satellites by empirical quartiles of:
#
#   relative area
#   normalized distance
#
# and observe GT support.
#
# This is purely descriptive and provides evidence about
# whether small/near satellites behave differently.
# ------------------------------------------------------------

satellites_7cd_df[
    "area_quartile"
] = pd.qcut(
    satellites_7cd_df[
        "proposal_area_fraction_of_original"
    ],
    q=4,
    labels=[
        "Q1_smallest",
        "Q2",
        "Q3",
        "Q4_largest",
    ],
    duplicates="drop",
)


satellites_7cd_df[
    "distance_quartile"
] = pd.qcut(
    satellites_7cd_df[
        "min_distance_to_anchor_norm"
    ],
    q=4,
    labels=[
        "Q1_nearest",
        "Q2",
        "Q3",
        "Q4_farthest",
    ],
    duplicates="drop",
)


area_distance_quartile_7cd_df = (
    satellites_7cd_df
    .groupby(
        [
            "area_quartile",
            "distance_quartile",
        ],
        observed=True,
        as_index=False,
    )
    .agg(
        num_satellites=(
            "proposal_subcomponent_id",
            "count",
        ),

        GT_supported_satellites=(
            "gt_supported_fragment",
            "sum",
        ),

        GT_points=(
            "gt_points_in_fragment",
            "sum",
        ),

        median_area_fraction_parent=(
            "proposal_area_fraction_of_original",
            "median",
        ),

        median_distance_norm=(
            "min_distance_to_anchor_norm",
            "median",
        ),
    )
)


area_distance_quartile_7cd_df[
    "GT_supported_fraction"
] = (
    area_distance_quartile_7cd_df[
        "GT_supported_satellites"
    ]
    /
    area_distance_quartile_7cd_df[
        "num_satellites"
    ]
)


# ------------------------------------------------------------
# 17. Save outputs
# ------------------------------------------------------------

M1_7C_D_FRAGMENT_STRUCTURE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_anchor_satellite_fragment_structure.csv"
)

M1_7C_D_INVENTORY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_anchor_satellite_inventory.csv"
)

M1_7C_D_SATELLITE_GT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_satellite_by_GT_support.csv"
)

M1_7C_D_GT_LOCATION_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_GT_anchor_vs_satellite.csv"
)

M1_7C_D_AUC_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_satellite_feature_AUC.csv"
)

M1_7C_D_GT_CLASS_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_satellite_GT_class_summary.csv"
)

M1_7C_D_PARENT_GT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_satellite_by_parent_GT_group.csv"
)

M1_7C_D_QUARTILE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_D_area_distance_quartile_diagnostic.csv"
)


fragment_structure_7cd_df.to_csv(
    M1_7C_D_FRAGMENT_STRUCTURE_FILE,
    index=False,
)

anchor_satellite_inventory_7cd_df.to_csv(
    M1_7C_D_INVENTORY_FILE,
    index=False,
)

satellite_by_gt_7cd_df.to_csv(
    M1_7C_D_SATELLITE_GT_FILE,
    index=False,
)

GT_anchor_satellite_7cd_df.to_csv(
    M1_7C_D_GT_LOCATION_FILE,
    index=False,
)

satellite_auc_7cd_df.to_csv(
    M1_7C_D_AUC_FILE,
    index=False,
)

satellite_GT_class_summary_7cd_df.to_csv(
    M1_7C_D_GT_CLASS_FILE,
    index=False,
)

satellite_by_parent_GT_7cd_df.to_csv(
    M1_7C_D_PARENT_GT_FILE,
    index=False,
)

area_distance_quartile_7cd_df.to_csv(
    M1_7C_D_QUARTILE_FILE,
    index=False,
)


# ------------------------------------------------------------
# 18. Report — anchor/satellite inventory
# ------------------------------------------------------------

print("\nAnchor versus satellite inventory")
print("---------------------------------")

display(
    anchor_satellite_inventory_7cd_df
)


# ------------------------------------------------------------
# 19. Report — satellite GT support
# ------------------------------------------------------------

print("\nSatellite structure by GT support")
print("---------------------------------")

display(
    satellite_by_gt_7cd_df
)


# ------------------------------------------------------------
# 20. Report — candidate GT points in anchors/satellites
# ------------------------------------------------------------

print("\nGT points: anchors versus satellites")
print("------------------------------------")

display(
    GT_anchor_satellite_7cd_df
)


# ------------------------------------------------------------
# 21. Report — diagnostic AUC
# ------------------------------------------------------------

print("\nInference-available satellite-feature discrimination")
print("-----------------------------------------------------")

display(
    satellite_auc_7cd_df
)


# ------------------------------------------------------------
# 22. Report — fragment GT class
# ------------------------------------------------------------

print("\nSatellite summary by fragment GT occupancy")
print("------------------------------------------")

display(
    satellite_GT_class_summary_7cd_df
)


# ------------------------------------------------------------
# 23. Report — parent GT group
# ------------------------------------------------------------

print("\nSatellite behaviour by parent-component GT group")
print("-----------------------------------------------")

display(
    satellite_by_parent_GT_7cd_df
)


# ------------------------------------------------------------
# 24. Report — area × distance quartiles
# ------------------------------------------------------------

print("\nSatellite area × spatial-distance quartile diagnostic")
print("------------------------------------------------------")

display(
    area_distance_quartile_7cd_df
)


# ------------------------------------------------------------
# 25. Global consistency
# ------------------------------------------------------------

print("\nGlobal consistency")
print("------------------")

print(
    "Candidate raw fragments       :",
    len(
        fragment_structure_7cd_df
    ),
)

print(
    "Depth-cluster anchors         :",
    int(
        fragment_structure_7cd_df[
            "is_depth_anchor"
        ]
        .sum()
    ),
)

print(
    "Depth-cluster satellites      :",
    int(
        fragment_structure_7cd_df[
            "is_depth_satellite"
        ]
        .sum()
    ),
)

print(
    "Candidate GT points           :",
    total_gt_points_in_candidates_7cd,
)

print(
    "GT points in anchors          :",
    gt_points_in_anchors_7cd,
)

print(
    "GT points in satellites       :",
    gt_points_in_satellites_7cd,
)


# ------------------------------------------------------------
# 26. Protocol report
# ------------------------------------------------------------

print("\nAnchor definition uses GT          : NO")
print("Satellite definition uses GT       : NO")
print("Spatial measurements use GT        : NO")
print("GT attached after measurements     : YES")
print("Size threshold selected            : NO")
print("Distance threshold selected        : NO")
print("Fragments deleted                  : NO")
print("Fragments merged                   : NO")
print("Split accepted/rejected            : NO")
print("Count rule modified                : NO")
print("Density routing used               : NO")
print("Test set accessed                  : NO")

print("\nStep 7C Part D: PASS")

STEP 7C PART D — ANCHOR–SATELLITE SPATIAL-NOISE DIAGNOSTIC

Anchor versus satellite inventory
---------------------------------


,is_depth_anchor,num_fragments,GT_supported_fragments,total_GT_points,total_pixels,median_area_fraction_parent,median_area_fraction_depth_cluster,GT_supported_fragment_fraction,fraction_all_GT_points
0,False,5124,401,561,230677,0.001199,0.003130,0.078259,0.252931
1,True,1398,624,1657,1534901,0.443222,0.991818,0.446352,0.747069



Satellite structure by GT support
---------------------------------


,gt_supported_fragment,num_satellites,total_GT_points,median_area_pixels,median_area_fraction_parent,median_area_fraction_depth_cluster,median_min_distance_px,median_min_distance_norm,median_centroid_distance_norm,fraction_all_satellites
0,False,4723,0,1.0,0.001024,0.002695,8.062258,0.212238,0.688481,0.921741
1,True,401,561,52.0,0.023486,0.091603,16.278821,0.334554,0.771176,0.078259



GT points: anchors versus satellites
------------------------------------


...


Satellite behaviour by parent-component GT group
-----------------------------------------------


,gt_group,num_satellites,GT_supported_satellites,GT_points_in_satellites,median_area_fraction_parent,median_min_distance_norm,GT_supported_satellite_fraction
0,GT0,794,0,0,0.001980,0.186935,0.000000
1,GT1,796,6,6,0.001645,0.169086,0.007538
2,GT2+,3534,395,555,0.000960,0.249537,0.111771



Satellite area × spatial-distance quartile diagnostic
------------------------------------------------------


,area_quartile,distance_quartile,num_satellites,GT_supported_satellites,GT_points,median_area_fraction_parent,median_distance_norm,GT_supported_fraction
0,Q1_smallest,Q1_nearest,534,6,6,0.000144,0.019991,0.011236
1,Q1_smallest,Q2,179,0,0,0.000153,0.118255,0.000000
2,Q1_smallest,Q3,248,0,0,0.000144,0.369493,0.000000
3,Q1_smallest,Q4_farthest,320,0,0,0.000153,0.773083,0.000000
4,Q2,Q1_nearest,381,1,1,0.000666,0.038222,0.002625
5,Q2,Q2,229,0,0,0.000757,0.110896,0.000000
6,Q2,Q3,308,4,4,0.000666,0.370330,0.012987
7,Q2,Q4_farthest,363,2,2,0.000619,0.797230,0.005510
8,Q3,Q1_nearest,223,5,9,0.001807,0.051681,0.022422
9,Q3,Q2,483,13,21,0.002160,0.108465,0.026915



Global consistency
------------------
Candidate raw fragments       : 6522
Depth-cluster anchors         : 1398
Depth-cluster satellites      : 5124
Candidate GT points           : 2218
GT points in anchors          : 1657
GT points in satellites       : 561

Anchor definition uses GT          : NO
Satellite definition uses GT       : NO
Spatial measurements use GT        : NO
GT attached after measurements     : YES
Size threshold selected            : NO
Distance threshold selected        : NO
Fragments deleted                  : NO
Fragments merged                   : NO
Split accepted/rejected            : NO
Count rule modified                : NO
Density routing used               : NO
Test set accessed                  : NO

Step 7C Part D: PASS


### **Finding 10:**

Within each Marigold depth cluster, disconnected satellite fragments are predominantly GT-empty connectivity artefacts, while GT-supported satellites are substantially larger.

Relative satellite area provides strong discrimination between GT-supported and GT-empty satellites (AUC ≈ 0.86), whereas spatial distance to the same-depth anchor provides only weak discrimination (AUC ≈ 0.54–0.59).

Although anchors contain 74.7% of candidate GT points, satellites retain 25.3%, predominantly within GT2+ parent components.

Therefore, anchor-only consolidation would discard useful multi-object structure, while **a size-aware satellite-retention strategy is empirically better motivated** than a distance-based consolidation rule.

In [32]:
# ============================================================
# Step 7C Part E — Controlled satellite-retention
#                  sensitivity analysis
# ============================================================
#
# Purpose
# -------
# Step 7C Part D showed that satellite SIZE is substantially
# more informative than distance-to-anchor for distinguishing
# GT-supported satellites from connectivity noise.
#
# We now evaluate controlled inference-time retention rules:
#
#   ALWAYS retain each depth-cluster anchor
#
#   retain a satellite if:
#
#       satellite relative area >= threshold
#
# Two size normalisations are compared:
#
#   PARENT:
#       satellite area / original A3c component area
#
#   DEPTH:
#       satellite area / same-depth-cluster area
#
# This is a DEVELOPMENT-SET sensitivity analysis.
#
# GT is used to evaluate candidate rules and may later be used
# to select/freeze one rule.
#
# IMPORTANT
# ---------
# This step DOES NOT yet modify semantic masks or counting.
#
# No:
#   - distance threshold
#   - density routing
#   - count modification
#   - test-set access
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 7C PART E — CONTROLLED SATELLITE-RETENTION SENSITIVITY ANALYSIS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "fragment_structure_7cd_df" in globals(), (
    "fragment_structure_7cd_df unavailable. "
    "Run Step 7C Part D first."
)

assert "component_gt_split_diag_7c_df" in globals(), (
    "component_gt_split_diag_7c_df unavailable. "
    "Run Step 7C Part B first."
)

assert "M1_TABLES_DIR" in globals()


assert len(
    fragment_structure_7cd_df
) == 6522


assert (
    fragment_structure_7cd_df[
        "is_depth_anchor"
    ].sum()
    ==
    1398
)


assert (
    fragment_structure_7cd_df[
        "is_depth_satellite"
    ].sum()
    ==
    5124
)


# ------------------------------------------------------------
# 2. Attach original parent GT information
# ------------------------------------------------------------

parent_gt_lookup_7ce_df = (
    component_gt_split_diag_7c_df[
        [
            "dataset_index",
            "original_component_id",
            "gt_group",
            "original_gt_count",
        ]
    ]
    .copy()
)


fragments_7ce_df = (
    fragment_structure_7cd_df
    .merge(
        parent_gt_lookup_7ce_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="many_to_one",
    )
)


assert (
    fragments_7ce_df[
        "original_gt_count"
    ]
    .notna()
    .all()
)


# ------------------------------------------------------------
# 3. Check total candidate GT support
# ------------------------------------------------------------

TOTAL_RAW_FRAGMENTS_7CE = int(
    len(
        fragments_7ce_df
    )
)


TOTAL_CANDIDATE_GT_7CE = int(
    fragments_7ce_df[
        "gt_points_in_fragment"
    ].sum()
)


assert TOTAL_RAW_FRAGMENTS_7CE == 6522
assert TOTAL_CANDIDATE_GT_7CE == 2218


# ------------------------------------------------------------
# 4. Candidate threshold grids
#
# These are DEVELOPMENT sensitivity values.
#
# They are deliberately broad and monotonic.
#
# We are NOT declaring any one threshold final here.
# ------------------------------------------------------------

PARENT_THRESHOLDS_7CE = [
    0.001,
    0.0025,
    0.005,
    0.010,
    0.020,
    0.050,
]


DEPTH_THRESHOLDS_7CE = [
    0.0025,
    0.005,
    0.010,
    0.020,
    0.050,
    0.100,
]


print("\nSensitivity grids")
print("-----------------")

print(
    "Parent-relative thresholds:",
    PARENT_THRESHOLDS_7CE,
)

print(
    "Depth-cluster-relative thresholds:",
    DEPTH_THRESHOLDS_7CE,
)


# ------------------------------------------------------------
# 5. Deterministic image folds
#
# These folds are NOT used to train anything.
#
# They only provide a robustness/stability view so that a rule
# is not chosen solely because of one subset of dev100.
# ------------------------------------------------------------

unique_images_7ce = np.array(
    sorted(
        fragments_7ce_df[
            "dataset_index"
        ]
        .unique()
    )
)


assert len(
    unique_images_7ce
) == 100


image_to_fold_7ce = {
    int(dataset_index):
        int(i % 5)

    for i, dataset_index
    in enumerate(
        unique_images_7ce
    )
}


fragments_7ce_df[
    "diagnostic_fold"
] = (
    fragments_7ce_df[
        "dataset_index"
    ]
    .map(
        image_to_fold_7ce
    )
    .astype(int)
)


# ------------------------------------------------------------
# 6. Evaluation helper
# ------------------------------------------------------------

def evaluate_retention_policy_7ce(
    keep_mask,
    policy_family,
    threshold,
):

    """
    Evaluate one satellite-retention rule.

    Anchors and selected satellites are treated as retained
    spatial proposals.

    GT is used ONLY to evaluate the retained proposals.

    No count prediction is produced.
    """

    policy_df = (
        fragments_7ce_df
        .copy()
    )


    policy_df[
        "retained"
    ] = np.asarray(
        keep_mask,
        dtype=bool,
    )


    # Every depth anchor MUST be preserved
    assert (
        policy_df.loc[
            policy_df[
                "is_depth_anchor"
            ],
            "retained",
        ]
        .all()
    )


    retained_df = (
        policy_df[
            policy_df[
                "retained"
            ]
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Fragment-level totals
    # --------------------------------------------------------

    retained_fragments = int(
        len(
            retained_df
        )
    )


    retained_satellites = int(
        retained_df[
            "is_depth_satellite"
        ]
        .sum()
    )


    fragment_reduction_fraction = float(
        1.0
        -
        (
            retained_fragments
            /
            TOTAL_RAW_FRAGMENTS_7CE
        )
    )


    GT_supported_retained_fragments = int(
        (
            retained_df[
                "gt_points_in_fragment"
            ]
            >
            0
        )
        .sum()
    )


    retained_fragment_GT_supported_fraction = float(
        GT_supported_retained_fragments
        /
        retained_fragments
    )


    # --------------------------------------------------------
    # GT-point retention
    # --------------------------------------------------------

    retained_GT_points = int(
        retained_df[
            "gt_points_in_fragment"
        ]
        .sum()
    )


    candidate_GT_recall = float(
        retained_GT_points
        /
        TOTAL_CANDIDATE_GT_7CE
    )


    # --------------------------------------------------------
    # Per-original-component retained structure
    # --------------------------------------------------------

    retained_component_df = (
        retained_df
        .groupby(
            [
                "dataset_index",
                "original_component_id",
            ],
            as_index=False,
        )
        .agg(
            retained_fragments=(
                "proposal_subcomponent_id",
                "count",
            ),

            retained_satellites=(
                "is_depth_satellite",
                "sum",
            ),

            retained_GT_points=(
                "gt_points_in_fragment",
                "sum",
            ),

            retained_GT_bearing_fragments=(
                "gt_points_in_fragment",
                lambda s:
                    int(
                        (
                            s
                            >
                            0
                        )
                        .sum()
                    ),
            ),
        )
    )


    # All candidate components retain their two anchors,
    # therefore every candidate must remain present.
    component_eval_df = (
        parent_gt_lookup_7ce_df
        .merge(
            retained_component_df,
            on=[
                "dataset_index",
                "original_component_id",
            ],
            how="left",
            validate="one_to_one",
        )
    )


    assert len(
        component_eval_df
    ) == 699


    assert (
        component_eval_df[
            "retained_fragments"
        ]
        .notna()
        .all()
    )


    # --------------------------------------------------------
    # GT1 behaviour
    # --------------------------------------------------------

    gt1_df = (
        component_eval_df[
            component_eval_df[
                "gt_group"
            ]
            ==
            "GT1"
        ]
    )


    assert len(
        gt1_df
    ) == 248


    GT1_point_recall = float(
        (
            gt1_df[
                "retained_GT_points"
            ]
            ==
            1
        )
        .mean()
    )


    GT1_median_retained_fragments = float(
        gt1_df[
            "retained_fragments"
        ]
        .median()
    )


    GT1_mean_retained_fragments = float(
        gt1_df[
            "retained_fragments"
        ]
        .mean()
    )


    GT1_median_retained_satellites = float(
        gt1_df[
            "retained_satellites"
        ]
        .median()
    )


    # --------------------------------------------------------
    # GT0 behaviour
    # --------------------------------------------------------

    gt0_df = (
        component_eval_df[
            component_eval_df[
                "gt_group"
            ]
            ==
            "GT0"
        ]
    )


    assert len(
        gt0_df
    ) == 182


    GT0_median_retained_fragments = float(
        gt0_df[
            "retained_fragments"
        ]
        .median()
    )


    GT0_mean_retained_fragments = float(
        gt0_df[
            "retained_fragments"
        ]
        .mean()
    )


    GT0_median_retained_satellites = float(
        gt0_df[
            "retained_satellites"
        ]
        .median()
    )


    # --------------------------------------------------------
    # GT2+ behaviour
    # --------------------------------------------------------

    gt2_df = (
        component_eval_df[
            component_eval_df[
                "gt_group"
            ]
            ==
            "GT2+"
        ]
    )


    assert len(
        gt2_df
    ) == 269


    GT2_GT_point_recall = float(
        gt2_df[
            "retained_GT_points"
        ]
        .sum()
        /
        gt2_df[
            "original_gt_count"
        ]
        .sum()
    )


    GT2_fraction_all_GT_retained = float(
        (
            gt2_df[
                "retained_GT_points"
            ]
            ==
            gt2_df[
                "original_gt_count"
            ]
        )
        .mean()
    )


    GT2_fraction_with_2plus_GT_fragments = float(
        (
            gt2_df[
                "retained_GT_bearing_fragments"
            ]
            >=
            2
        )
        .mean()
    )


    GT2_fraction_with_3plus_GT_fragments = float(
        (
            gt2_df[
                "retained_GT_bearing_fragments"
            ]
            >=
            3
        )
        .mean()
    )


    GT2_fraction_all_GT_separated_and_retained = float(
        (
            (
                gt2_df[
                    "retained_GT_points"
                ]
                ==
                gt2_df[
                    "original_gt_count"
                ]
            )
            &
            (
                gt2_df[
                    "retained_GT_bearing_fragments"
                ]
                ==
                gt2_df[
                    "original_gt_count"
                ]
            )
        )
        .mean()
    )


    GT2_median_retained_fragments = float(
        gt2_df[
            "retained_fragments"
        ]
        .median()
    )


    GT2_mean_retained_fragments = float(
        gt2_df[
            "retained_fragments"
        ]
        .mean()
    )


    # --------------------------------------------------------
    # 5-fold stability diagnostics
    #
    # Evaluate:
    #
    #   candidate GT-point recall
    #   GT2+ separation
    #
    # separately on each image fold.
    # --------------------------------------------------------

    fold_rows = []


    for fold_id in range(5):

        fold_images = set(
            unique_images_7ce[
                np.arange(
                    len(
                        unique_images_7ce
                    )
                )
                %
                5
                ==
                fold_id
            ]
        )


        fold_policy_df = (
            policy_df[
                policy_df[
                    "dataset_index"
                ]
                .isin(
                    fold_images
                )
            ]
        )


        fold_retained_df = (
            fold_policy_df[
                fold_policy_df[
                    "retained"
                ]
            ]
        )


        fold_total_GT = int(
            fold_policy_df[
                "gt_points_in_fragment"
            ]
            .sum()
        )


        fold_retained_GT = int(
            fold_retained_df[
                "gt_points_in_fragment"
            ]
            .sum()
        )


        if fold_total_GT > 0:

            fold_GT_recall = float(
                fold_retained_GT
                /
                fold_total_GT
            )

        else:

            fold_GT_recall = np.nan


        fold_gt2_components = (
            gt2_df[
                gt2_df[
                    "dataset_index"
                ]
                .isin(
                    fold_images
                )
            ]
        )


        if len(
            fold_gt2_components
        ) > 0:

            fold_GT2_sep = float(
                (
                    fold_gt2_components[
                        "retained_GT_bearing_fragments"
                    ]
                    >=
                    2
                )
                .mean()
            )

        else:

            fold_GT2_sep = np.nan


        fold_rows.append(
            {
                "fold":
                    fold_id,

                "GT_recall":
                    fold_GT_recall,

                "GT2_sep":
                    fold_GT2_sep,
            }
        )


    fold_df = pd.DataFrame(
        fold_rows
    )


    # --------------------------------------------------------
    # Output row
    # --------------------------------------------------------

    return {
        "policy_family":
            policy_family,

        "threshold":
            threshold,

        "retained_fragments":
            retained_fragments,

        "retained_satellites":
            retained_satellites,

        "fragment_reduction_fraction":
            fragment_reduction_fraction,

        "retained_fragment_GT_supported_fraction":
            retained_fragment_GT_supported_fraction,

        "retained_GT_points":
            retained_GT_points,

        "candidate_GT_recall":
            candidate_GT_recall,

        "GT1_point_recall":
            GT1_point_recall,

        "GT1_median_retained_fragments":
            GT1_median_retained_fragments,

        "GT1_mean_retained_fragments":
            GT1_mean_retained_fragments,

        "GT1_median_retained_satellites":
            GT1_median_retained_satellites,

        "GT0_median_retained_fragments":
            GT0_median_retained_fragments,

        "GT0_mean_retained_fragments":
            GT0_mean_retained_fragments,

        "GT0_median_retained_satellites":
            GT0_median_retained_satellites,

        "GT2_GT_point_recall":
            GT2_GT_point_recall,

        "GT2_fraction_all_GT_retained":
            GT2_fraction_all_GT_retained,

        "GT2_fraction_with_2plus_GT_fragments":
            GT2_fraction_with_2plus_GT_fragments,

        "GT2_fraction_with_3plus_GT_fragments":
            GT2_fraction_with_3plus_GT_fragments,

        "GT2_fraction_all_GT_separated_and_retained":
            GT2_fraction_all_GT_separated_and_retained,

        "GT2_median_retained_fragments":
            GT2_median_retained_fragments,

        "GT2_mean_retained_fragments":
            GT2_mean_retained_fragments,

        "fold_mean_GT_recall":
            float(
                fold_df[
                    "GT_recall"
                ]
                .mean()
            ),

        "fold_std_GT_recall":
            float(
                fold_df[
                    "GT_recall"
                ]
                .std(
                    ddof=0
                )
            ),

        "fold_mean_GT2_sep":
            float(
                fold_df[
                    "GT2_sep"
                ]
                .mean()
            ),

        "fold_std_GT2_sep":
            float(
                fold_df[
                    "GT2_sep"
                ]
                .std(
                    ddof=0
                )
            ),
    }


# ------------------------------------------------------------
# 7. Evaluate RAW baseline
#
# Retain everything.
# ------------------------------------------------------------

sensitivity_rows_7ce = []


raw_keep_mask_7ce = np.ones(
    len(
        fragments_7ce_df
    ),
    dtype=bool,
)


sensitivity_rows_7ce.append(
    evaluate_retention_policy_7ce(
        keep_mask=
            raw_keep_mask_7ce,

        policy_family=
            "RAW_ALL",

        threshold=
            0.0,
    )
)


# ------------------------------------------------------------
# 8. Evaluate ANCHOR-ONLY baseline
# ------------------------------------------------------------

anchor_only_mask_7ce = (
    fragments_7ce_df[
        "is_depth_anchor"
    ]
    .to_numpy(
        dtype=bool
    )
)


sensitivity_rows_7ce.append(
    evaluate_retention_policy_7ce(
        keep_mask=
            anchor_only_mask_7ce,

        policy_family=
            "ANCHOR_ONLY",

        threshold=
            np.nan,
    )
)


# ------------------------------------------------------------
# 9. Parent-relative satellite retention
#
# Anchors always retained.
#
# Satellite retained if:
#
#   proposal_area / parent_component_area >= threshold
# ------------------------------------------------------------

for threshold in (
    PARENT_THRESHOLDS_7CE
):

    keep_mask = (
        fragments_7ce_df[
            "is_depth_anchor"
        ]
        |
        (
            fragments_7ce_df[
                "is_depth_satellite"
            ]
            &
            (
                fragments_7ce_df[
                    "proposal_area_fraction_of_original"
                ]
                >=
                threshold
            )
        )
    )


    sensitivity_rows_7ce.append(
        evaluate_retention_policy_7ce(
            keep_mask=
                keep_mask.to_numpy(
                    dtype=bool
                ),

            policy_family=
                "PARENT_RELATIVE",

            threshold=
                float(
                    threshold
                ),
        )
    )


# ------------------------------------------------------------
# 10. Depth-cluster-relative satellite retention
#
# Anchors always retained.
#
# Satellite retained if:
#
#   proposal_area / depth_cluster_area >= threshold
# ------------------------------------------------------------

for threshold in (
    DEPTH_THRESHOLDS_7CE
):

    keep_mask = (
        fragments_7ce_df[
            "is_depth_anchor"
        ]
        |
        (
            fragments_7ce_df[
                "is_depth_satellite"
            ]
            &
            (
                fragments_7ce_df[
                    "area_fraction_of_depth_cluster"
                ]
                >=
                threshold
            )
        )
    )


    sensitivity_rows_7ce.append(
        evaluate_retention_policy_7ce(
            keep_mask=
                keep_mask.to_numpy(
                    dtype=bool
                ),

            policy_family=
                "DEPTH_RELATIVE",

            threshold=
                float(
                    threshold
                ),
        )
    )


# ------------------------------------------------------------
# 11. Full sensitivity table
# ------------------------------------------------------------

satellite_retention_sensitivity_7ce_df = (
    pd.DataFrame(
        sensitivity_rows_7ce
    )
)


# Deterministic display ordering
policy_order_7ce = {
    "RAW_ALL":
        0,

    "PARENT_RELATIVE":
        1,

    "DEPTH_RELATIVE":
        2,

    "ANCHOR_ONLY":
        3,
}


satellite_retention_sensitivity_7ce_df[
    "_policy_order"
] = (
    satellite_retention_sensitivity_7ce_df[
        "policy_family"
    ]
    .map(
        policy_order_7ce
    )
)


satellite_retention_sensitivity_7ce_df = (
    satellite_retention_sensitivity_7ce_df
    .sort_values(
        [
            "_policy_order",
            "threshold",
        ],
        na_position="last",
    )
    .drop(
        columns=[
            "_policy_order"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 12. Compact trade-off table
# ------------------------------------------------------------

tradeoff_cols_7ce = [
    "policy_family",
    "threshold",

    "retained_fragments",
    "fragment_reduction_fraction",

    "candidate_GT_recall",
    "GT1_point_recall",

    "GT2_GT_point_recall",
    "GT2_fraction_all_GT_retained",
    "GT2_fraction_with_2plus_GT_fragments",
    "GT2_fraction_all_GT_separated_and_retained",

    "GT0_mean_retained_fragments",
    "GT1_mean_retained_fragments",
    "GT2_mean_retained_fragments",

    "fold_std_GT_recall",
    "fold_std_GT2_sep",
]


satellite_retention_tradeoff_7ce_df = (
    satellite_retention_sensitivity_7ce_df[
        tradeoff_cols_7ce
    ]
    .copy()
)


# ------------------------------------------------------------
# 13. Fragment efficiency
#
# Diagnostic quantity only:
#
# How many GT-supported retained fragments remain per total
# retained fragment?
#
# Higher means the retained proposal set is less dominated by
# GT-empty fragmentation.
# ------------------------------------------------------------

efficiency_7ce_df = (
    satellite_retention_sensitivity_7ce_df[
        [
            "policy_family",
            "threshold",
            "retained_fragments",
            "fragment_reduction_fraction",
            "retained_fragment_GT_supported_fraction",
            "candidate_GT_recall",
            "GT2_fraction_with_2plus_GT_fragments",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# 14. Preserve raw-reference values explicitly
# ------------------------------------------------------------

raw_reference_7ce = (
    satellite_retention_sensitivity_7ce_df[
        satellite_retention_sensitivity_7ce_df[
            "policy_family"
        ]
        ==
        "RAW_ALL"
    ]
    .iloc[0]
)


RAW_GT_RECALL_7CE = float(
    raw_reference_7ce[
        "candidate_GT_recall"
    ]
)


RAW_GT2_SEP_7CE = float(
    raw_reference_7ce[
        "GT2_fraction_with_2plus_GT_fragments"
    ]
)


assert np.isclose(
    RAW_GT_RECALL_7CE,
    1.0,
)


# Expected from Step 7C Part B:
assert np.isclose(
    RAW_GT2_SEP_7CE,
    0.8141263940520446,
    atol=1e-9,
)


# ------------------------------------------------------------
# 15. Add relative preservation columns
# ------------------------------------------------------------

satellite_retention_sensitivity_7ce_df[
    "GT_recall_loss_vs_raw"
] = (
    RAW_GT_RECALL_7CE
    -
    satellite_retention_sensitivity_7ce_df[
        "candidate_GT_recall"
    ]
)


satellite_retention_sensitivity_7ce_df[
    "GT2_separation_preservation_vs_raw"
] = (
    satellite_retention_sensitivity_7ce_df[
        "GT2_fraction_with_2plus_GT_fragments"
    ]
    /
    RAW_GT2_SEP_7CE
)


# ------------------------------------------------------------
# 16. Development Pareto diagnostic
#
# A policy is dominated if another policy has:
#
#   >= GT recall
#   >= GT2 separation
#   >= fragment reduction
#
# with at least one strict improvement.
#
# This does NOT choose a winner.
# It only removes obviously inferior trade-offs.
# ------------------------------------------------------------

pareto_flags_7ce = []


for i, row_i in (
    satellite_retention_sensitivity_7ce_df
    .iterrows()
):

    dominated = False


    for j, row_j in (
        satellite_retention_sensitivity_7ce_df
        .iterrows()
    ):

        if i == j:
            continue


        no_worse = (
            (
                row_j[
                    "candidate_GT_recall"
                ]
                >=
                row_i[
                    "candidate_GT_recall"
                ]
            )
            and
            (
                row_j[
                    "GT2_fraction_with_2plus_GT_fragments"
                ]
                >=
                row_i[
                    "GT2_fraction_with_2plus_GT_fragments"
                ]
            )
            and
            (
                row_j[
                    "fragment_reduction_fraction"
                ]
                >=
                row_i[
                    "fragment_reduction_fraction"
                ]
            )
        )


        strictly_better = (
            (
                row_j[
                    "candidate_GT_recall"
                ]
                >
                row_i[
                    "candidate_GT_recall"
                ]
            )
            or
            (
                row_j[
                    "GT2_fraction_with_2plus_GT_fragments"
                ]
                >
                row_i[
                    "GT2_fraction_with_2plus_GT_fragments"
                ]
            )
            or
            (
                row_j[
                    "fragment_reduction_fraction"
                ]
                >
                row_i[
                    "fragment_reduction_fraction"
                ]
            )
        )


        if (
            no_worse
            and
            strictly_better
        ):

            dominated = True

            break


    pareto_flags_7ce.append(
        not dominated
    )


satellite_retention_sensitivity_7ce_df[
    "pareto_nondominated"
] = (
    pareto_flags_7ce
)


pareto_7ce_df = (
    satellite_retention_sensitivity_7ce_df[
        satellite_retention_sensitivity_7ce_df[
            "pareto_nondominated"
        ]
    ]
    [
        [
            "policy_family",
            "threshold",
            "retained_fragments",
            "fragment_reduction_fraction",
            "candidate_GT_recall",
            "GT1_point_recall",
            "GT2_GT_point_recall",
            "GT2_fraction_with_2plus_GT_fragments",
            "GT2_fraction_all_GT_retained",
            "GT0_mean_retained_fragments",
            "GT1_mean_retained_fragments",
            "GT2_mean_retained_fragments",
            "fold_std_GT_recall",
            "fold_std_GT2_sep",
        ]
    ]
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 17. Save outputs
# ------------------------------------------------------------

M1_7C_E_FULL_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_E_satellite_retention_sensitivity.csv"
)

M1_7C_E_TRADEOFF_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_E_satellite_retention_tradeoff.csv"
)

M1_7C_E_EFFICIENCY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_E_fragment_efficiency.csv"
)

M1_7C_E_PARETO_FILE = (
    M1_TABLES_DIR
    /
    "M1_7C_E_pareto_nondominated_policies.csv"
)


satellite_retention_sensitivity_7ce_df.to_csv(
    M1_7C_E_FULL_FILE,
    index=False,
)

satellite_retention_tradeoff_7ce_df.to_csv(
    M1_7C_E_TRADEOFF_FILE,
    index=False,
)

efficiency_7ce_df.to_csv(
    M1_7C_E_EFFICIENCY_FILE,
    index=False,
)

pareto_7ce_df.to_csv(
    M1_7C_E_PARETO_FILE,
    index=False,
)


# ------------------------------------------------------------
# 18. Report — main trade-off
# ------------------------------------------------------------

print("\nSatellite-retention trade-off")
print("-----------------------------")

display(
    satellite_retention_sensitivity_7ce_df[
        [
            "policy_family",
            "threshold",

            "retained_fragments",
            "fragment_reduction_fraction",

            "candidate_GT_recall",
            "GT_recall_loss_vs_raw",

            "GT1_point_recall",

            "GT2_GT_point_recall",
            "GT2_fraction_all_GT_retained",
            "GT2_fraction_with_2plus_GT_fragments",
            "GT2_separation_preservation_vs_raw",

            "GT0_mean_retained_fragments",
            "GT1_mean_retained_fragments",
            "GT2_mean_retained_fragments",

            "retained_fragment_GT_supported_fraction",

            "fold_std_GT_recall",
            "fold_std_GT2_sep",
        ]
    ]
)


# ------------------------------------------------------------
# 19. Report — Pareto diagnostic
# ------------------------------------------------------------

print("\nNon-dominated development trade-offs")
print("------------------------------------")

display(
    pareto_7ce_df
)


# ------------------------------------------------------------
# 20. Report — structural baselines
# ------------------------------------------------------------

print("\nReference baselines")
print("-------------------")

display(
    satellite_retention_sensitivity_7ce_df[
        satellite_retention_sensitivity_7ce_df[
            "policy_family"
        ]
        .isin(
            [
                "RAW_ALL",
                "ANCHOR_ONLY",
            ]
        )
    ]
    [
        [
            "policy_family",
            "retained_fragments",
            "fragment_reduction_fraction",
            "candidate_GT_recall",
            "GT1_point_recall",
            "GT2_GT_point_recall",
            "GT2_fraction_with_2plus_GT_fragments",
            "GT0_mean_retained_fragments",
            "GT1_mean_retained_fragments",
            "GT2_mean_retained_fragments",
        ]
    ]
)


# ------------------------------------------------------------
# 21. Global consistency
# ------------------------------------------------------------

print("\nGlobal consistency")
print("------------------")

print(
    "Raw candidate fragments:",
    TOTAL_RAW_FRAGMENTS_7CE,
)

print(
    "Candidate GT points:",
    TOTAL_CANDIDATE_GT_7CE,
)

print(
    "Policies evaluated:",
    len(
        satellite_retention_sensitivity_7ce_df
    ),
)

print(
    "Parent-relative policies:",
    len(
        PARENT_THRESHOLDS_7CE
    ),
)

print(
    "Depth-relative policies:",
    len(
        DEPTH_THRESHOLDS_7CE
    ),
)

print(
    "Non-dominated policies:",
    int(
        satellite_retention_sensitivity_7ce_df[
            "pareto_nondominated"
        ]
        .sum()
    ),
)


# ------------------------------------------------------------
# 22. Protocol report
# ------------------------------------------------------------

print("\nDevelopment set used for sensitivity : YES")
print("GT used to evaluate candidate rules   : YES")
print("GT used as inference input            : NO")
print("Inference feature                     : fragment relative area")
print("Distance feature used                 : NO")
print("Anchors always retained               : YES")
print("Threshold finally selected            : NO")
print("Semantic pixels actually deleted      : NO — simulation only")
print("Semantic pixels actually merged       : NO")
print("Count rule modified                   : NO")
print("Density routing used                  : NO")
print("Test set accessed                     : NO")

print("\nStep 7C Part E: PASS")

STEP 7C PART E — CONTROLLED SATELLITE-RETENTION SENSITIVITY ANALYSIS

Sensitivity grids
-----------------
Parent-relative thresholds: [0.001, 0.0025, 0.005, 0.01, 0.02, 0.05]
Depth-cluster-relative thresholds: [0.0025, 0.005, 0.01, 0.02, 0.05, 0.1]

Satellite-retention trade-off
-----------------------------


,policy_family,threshold,retained_fragments,fragment_reduction_fraction,candidate_GT_recall,GT_recall_loss_vs_raw,GT1_point_recall,GT2_GT_point_recall,GT2_fraction_all_GT_retained,GT2_fraction_with_2plus_GT_fragments,GT2_separation_preservation_vs_raw,GT0_mean_retained_fragments,GT1_mean_retained_fragments,GT2_mean_retained_fragments,retained_fragment_GT_supported_fraction,fold_std_GT_recall,fold_std_GT2_sep
0,RAW_ALL,0.0000,6522,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,0.814126,1.000000,6.362637,5.209677,15.137546,0.157160,0.000000,0.083282
1,PARENT_RELATIVE,0.0010,4210,0.354492,0.994590,0.005410,1.000000,0.993909,0.985130,0.814126,1.000000,4.994505,4.048387,8.539033,0.240618,0.006938,0.083282
2,PARENT_RELATIVE,0.0025,3146,0.517633,0.973399,0.026601,1.000000,0.970051,0.977695,0.814126,1.000000,4.016484,3.060484,6.156134,0.309282,0.039459,0.083282
3,PARENT_RELATIVE,0.0050,2675,0.589850,0.943192,0.056808,1.000000,0.936041,0.947955,0.810409,0.995434,3.346154,2.810484,5.089219,0.349533,0.080381,0.078655
4,PARENT_RELATIVE,0.0100,2332,0.642441,0.909829,0.090171,1.000000,0.898477,0.903346,0.806691,0.990868,2.956044,2.625000,4.249071,0.379503,0.120289,0.081352
5,PARENT_RELATIVE,0.0200,2070,0.682613,0.876465,0.123535,1.000000,0.860914,0.877323,0.799257,0.981735,2.703297,2.491935,3.568773,0.403865,0.147242,0.081581
6,PARENT_RELATIVE,0.0500,1798,0.724318,0.840397,0.159603,0.995968,0.820812,0.788104,0.739777,0.908676,2.428571,2.278226,2.940520,0.429366,0.162234,0.075060
7,DEPTH_RELATIVE,0.0025,4215,0.353726,0.995041,0.004959,1.000000,0.994416,0.977695,0.814126,1.000000,4.972527,4.149194,8.479554,0.240569,0.005288,0.083282
8,DEPTH_RELATIVE,0.0050,3460,0.469488,0.987376,0.012624,1.000000,0.985787,0.966543,0.810409,0.995434,4.148352,3.544355,6.788104,0.288439,0.014247,0.078655
9,DEPTH_RELATIVE,0.0100,2941,0.549065,0.963030,0.036970,1.000000,0.958376,0.955390,0.810409,0.995434,3.576923,3.157258,5.602230,0.325740,0.050792,0.078655



Non-dominated development trade-offs
------------------------------------


,policy_family,threshold,retained_fragments,fragment_reduction_fraction,candidate_GT_recall,GT1_point_recall,GT2_GT_point_recall,GT2_fraction_with_2plus_GT_fragments,GT2_fraction_all_GT_retained,GT0_mean_retained_fragments,GT1_mean_retained_fragments,GT2_mean_retained_fragments,fold_std_GT_recall,fold_std_GT2_sep
0,RAW_ALL,0.0000,6522,0.000000,1.000000,1.000000,1.000000,0.814126,1.000000,6.362637,5.209677,15.137546,0.000000,0.083282
1,PARENT_RELATIVE,0.0010,4210,0.354492,0.994590,1.000000,0.993909,0.814126,0.985130,4.994505,4.048387,8.539033,0.006938,0.083282
2,PARENT_RELATIVE,0.0025,3146,0.517633,0.973399,1.000000,0.970051,0.814126,0.977695,4.016484,3.060484,6.156134,0.039459,0.083282
3,PARENT_RELATIVE,0.0050,2675,0.589850,0.943192,1.000000,0.936041,0.810409,0.947955,3.346154,2.810484,5.089219,0.080381,0.078655
4,PARENT_RELATIVE,0.0100,2332,0.642441,0.909829,1.000000,0.898477,0.806691,0.903346,2.956044,2.625000,4.249071,0.120289,0.081352
5,PARENT_RELATIVE,0.0200,2070,0.682613,0.876465,1.000000,0.860914,0.799257,0.877323,2.703297,2.491935,3.568773,0.147242,0.081581
6,PARENT_RELATIVE,0.0500,1798,0.724318,0.840397,0.995968,0.820812,0.739777,0.788104,2.428571,2.278226,2.940520,0.162234,0.075060
7,DEPTH_RELATIVE,0.0025,4215,0.353726,0.995041,1.000000,0.994416,0.814126,0.977695,4.972527,4.149194,8.479554,0.005288,0.083282
8,DEPTH_RELATIVE,0.0050,3460,0.469488,0.987376,1.000000,0.985787,0.810409,0.966543,4.148352,3.544355,6.788104,0.014247,0.078655
9,DEPTH_RELATIVE,0.0100,2941,0.549065,0.963030,1.000000,0.958376,0.810409,0.955390,3.576923,3.157258,5.602230,0.050792,0.078655



Reference baselines
-------------------


,policy_family,retained_fragments,fragment_reduction_fraction,candidate_GT_recall,GT1_point_recall,GT2_GT_point_recall,GT2_fraction_with_2plus_GT_fragments,GT0_mean_retained_fragments,GT1_mean_retained_fragments,GT2_mean_retained_fragments
0,RAW_ALL,6522,0.000000,1.000000,1.000000,1.000000,0.814126,6.362637,5.209677,15.137546
13,ANCHOR_ONLY,1398,0.785649,0.747069,0.975806,0.718274,0.423792,2.000000,2.000000,2.000000



Global consistency
------------------
Raw candidate fragments: 6522
Candidate GT points: 2218
Policies evaluated: 14
Parent-relative policies: 6
Depth-relative policies: 6
Non-dominated policies: 14

Development set used for sensitivity : YES
GT used to evaluate candidate rules   : YES
GT used as inference input            : NO
Inference feature                     : fragment relative area
Distance feature used                 : NO
Anchors always retained               : YES
Threshold finally selected            : NO
Semantic pixels actually deleted      : NO — simulation only
Semantic pixels actually merged       : NO
Count rule modified                   : NO
Density routing used                  : NO
Test set accessed                     : NO

Step 7C Part E: PASS


### **Finding 11:**

Size-aware retention of disconnected Marigold satellite fragments can substantially suppress raw geometric fragmentation while preserving most object-supporting structure.

Using satellite area relative to the parent A3c component, a threshold of 0.001 removes 35.4% of raw fragments while retaining 99.5% of candidate GT points and preserving the original GT2+ multi-fragment separation rate.

A threshold of 0.0025 removes 51.8% of fragments while retaining 97.3% of GT points and still preserving the same 81.4% GT2+ separation rate.

More aggressive thresholds yield diminishing reductions in fragmentation and progressively greater loss of GT support.

Parent-relative area provides slightly more favourable trade-offs than depth-cluster-relative area across comparable operating regions.

## **Step 7D — count-level impact of geometry-guided refinement.**

In [34]:
# ============================================================
# Step 7D Part A — Freeze geometry variants and define
#                  direct-fragment count interpretation
# ============================================================
#
# Purpose
# -------
# Freeze a small, controlled set of geometry variants for
# downstream count-level evaluation.
#
# This step converts the Step-7C geometric structures into
# COMPONENT-LEVEL COUNT CONTRIBUTIONS, but DOES NOT yet
# compare them with FSC-147 GT image counts.
#
#
# Frozen variants
# ---------------
#
# G0 — NO_GEOMETRY
#
#      Every original A3c component contributes:
#
#          1
#
#
# G1 — RAW_SPLIT
#
#      Non-candidate component:
#
#          1
#
#      Step-7A candidate:
#
#          number of raw Step-7B spatial fragments
#
#
# G2 — PARENT_0.001
#
#      Non-candidate component:
#
#          1
#
#      Candidate:
#
#          all two depth-cluster anchors
#          +
#          satellites satisfying:
#
#          satellite_area / parent_component_area >= 0.001
#
#
# G3 — PARENT_0.0025
#
#      Same as G2, but threshold = 0.0025
#
#
# IMPORTANT INTERPRETATION
# ------------------------
#
# G1/G2/G3 deliberately treat retained geometric fragments
# as countable units.
#
# This is a CONTROLLED STRESS-TEST ABLATION.
#
# It does NOT assume or claim that:
#
#     retained fragment == true object instance
#
# Step 7C already showed this equivalence is not generally
# valid.
#
# The purpose of Step 7D is to test empirically whether this
# simple interpretation improves or worsens counting.
#
#
# No:
#   - new semantic evidence
#   - new Marigold inference
#   - new candidate selection
#   - new threshold tuning
#   - density routing
#   - GT-based inference decision
#   - test-set access
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 7D PART A — FREEZE GEOMETRY VARIANTS + DIRECT-FRAGMENT COUNT INTERPRETATION")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "fragment_structure_7cd_df" in globals(), (
    "fragment_structure_7cd_df unavailable. "
    "Run Step 7C Part D first."
)

assert "component_fragment_diag_7c_df" in globals(), (
    "component_fragment_diag_7c_df unavailable. "
    "Run Step 7C Part A first."
)

assert "a3c_component_geom_df" in globals(), (
    "a3c_component_geom_df unavailable. "
    "Run Step 6D Part C first."
)

assert "image_split_summary_7b_df" in globals(), (
    "image_split_summary_7b_df unavailable. "
    "Run Step 7B first."
)

assert "M1_TABLES_DIR" in globals()


# Frozen global identities established previously
assert len(
    a3c_component_geom_df
) == 2554


assert len(
    component_fragment_diag_7c_df
) == 699


assert len(
    fragment_structure_7cd_df
) == 6522


# ------------------------------------------------------------
# 2. Freeze Step-7D geometry variants
#
# DO NOT alter these thresholds during Step 7D Part B.
#
# Any later fine tuning must be explicitly reported as a
# separate experiment.
# ------------------------------------------------------------

GEOMETRY_VARIANTS_7D = {
    "G0_NO_GEOMETRY": {
        "candidate_mode":
            "original_component",

        "parent_relative_threshold":
            None,
    },

    "G1_RAW_SPLIT": {
        "candidate_mode":
            "all_raw_fragments",

        "parent_relative_threshold":
            None,
    },

    "G2_PARENT_0.001": {
        "candidate_mode":
            "anchor_plus_parent_threshold",

        "parent_relative_threshold":
            0.001,
    },

    "G3_PARENT_0.0025": {
        "candidate_mode":
            "anchor_plus_parent_threshold",

        "parent_relative_threshold":
            0.0025,
    },
}


print("\nFrozen Step-7D variants")
print("-----------------------")

for variant_name, config in (
    GEOMETRY_VARIANTS_7D.items()
):

    print(
        f"{variant_name:22s}",
        config,
    )


# ------------------------------------------------------------
# 3. Build authoritative original A3c component inventory
#
# IMPORTANT:
# Step 6D Part C defines component area as:
#
#     area_pixels
#
# NOT:
#
#     component_area
#
# Step 7A also uses `area_pixels` as the frozen component-area
# field. Therefore this exact schema is reused here.
# ------------------------------------------------------------

required_component_cols_7d = {
    "dataset_index",
    "component_id",
    "area_pixels",
}


assert required_component_cols_7d.issubset(
    set(
        a3c_component_geom_df.columns
    )
), (
    "Required component columns missing from "
    "a3c_component_geom_df.\n"
    f"Required: {required_component_cols_7d}\n"
    f"Found: {list(a3c_component_geom_df.columns)}"
)


original_components_7d_df = (
    a3c_component_geom_df[
        [
            "dataset_index",
            "component_id",
            "area_pixels",
        ]
    ]
    .rename(
        columns={
            "component_id":
                "original_component_id",

            "area_pixels":
                "original_component_area",
        }
    )
    .copy()
)


assert len(
    original_components_7d_df
) == 2554


assert (
    original_components_7d_df[
        [
            "dataset_index",
            "original_component_id",
        ]
    ]
    .duplicated()
    .sum()
    ==
    0
)


assert (
    original_components_7d_df[
        "original_component_area"
    ]
    >
    0
).all()


print("\nOriginal A3c component schema")
print("-----------------------------")

print(
    "Area source column           : area_pixels"
)

print(
    "Renamed for Step 7D          : original_component_area"
)

print(
    "Original components          :",
    len(
        original_components_7d_df
    ),
)

print(
    "Total original foreground px :",
    int(
        original_components_7d_df[
            "original_component_area"
        ]
        .sum()
    ),
)


# ------------------------------------------------------------
# 4. Frozen Step-7A candidate identity
# ------------------------------------------------------------

candidate_keys_7d_df = (
    component_fragment_diag_7c_df[
        [
            "dataset_index",
            "original_component_id",
        ]
    ]
    .drop_duplicates()
    .copy()
)


assert len(
    candidate_keys_7d_df
) == 699


candidate_keys_7d_df[
    "split_candidate"
] = True


original_components_7d_df = (
    original_components_7d_df
    .merge(
        candidate_keys_7d_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


original_components_7d_df[
    "split_candidate"
] = (
    original_components_7d_df[
        "split_candidate"
    ]
    .fillna(False)
    .astype(bool)
)


assert (
    original_components_7d_df[
        "split_candidate"
    ]
    .sum()
    ==
    699
)


assert (
    (
        ~original_components_7d_df[
            "split_candidate"
        ]
    )
    .sum()
    ==
    1855
)


# ------------------------------------------------------------
# 5. Candidate fragment table
#
# Structural information only.
#
# No GT columns are used in the inference rule.
# ------------------------------------------------------------

required_fragment_cols_7d = {
    "dataset_index",
    "original_component_id",
    "proposal_subcomponent_id",
    "proposal_area",
    "original_component_area",
    "proposal_area_fraction_of_original",
    "is_depth_anchor",
    "is_depth_satellite",
}


assert required_fragment_cols_7d.issubset(
    set(
        fragment_structure_7cd_df.columns
    )
), (
    "Required Step-7C fragment columns missing.\n"
    f"Required: {required_fragment_cols_7d}\n"
    f"Found: {list(fragment_structure_7cd_df.columns)}"
)


candidate_fragments_7d_df = (
    fragment_structure_7cd_df[
        [
            "dataset_index",
            "original_component_id",
            "proposal_subcomponent_id",
            "proposal_area",
            "original_component_area",
            "proposal_area_fraction_of_original",
            "is_depth_anchor",
            "is_depth_satellite",
        ]
    ]
    .copy()
)


assert len(
    candidate_fragments_7d_df
) == 6522


# ------------------------------------------------------------
# 6. Cross-check parent component areas
#
# The original-component area recorded in Step 7B/7C must
# exactly match Step 6D's authoritative `area_pixels`.
# ------------------------------------------------------------

candidate_parent_area_check_7d_df = (
    candidate_fragments_7d_df[
        [
            "dataset_index",
            "original_component_id",
            "original_component_area",
        ]
    ]
    .drop_duplicates()
    .merge(
        original_components_7d_df[
            [
                "dataset_index",
                "original_component_id",
                "original_component_area",
            ]
        ]
        .rename(
            columns={
                "original_component_area":
                    "step6d_original_component_area"
            }
        ),
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


assert len(
    candidate_parent_area_check_7d_df
) == 699


assert (
    candidate_parent_area_check_7d_df[
        "original_component_area"
    ]
    ==
    candidate_parent_area_check_7d_df[
        "step6d_original_component_area"
    ]
).all(), (
    "Step-7B/7C candidate parent areas do not exactly match "
    "Step-6D area_pixels."
)


print(
    "Candidate parent-area identity : 699/699 exact"
)


# ------------------------------------------------------------
# 7. Structural integrity:
#    every candidate must have exactly two depth anchors
#    available from the k=2 depth partition.
# ------------------------------------------------------------

anchor_count_per_candidate_7d = (
    candidate_fragments_7d_df[
        candidate_fragments_7d_df[
            "is_depth_anchor"
        ]
    ]
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .size()
)


assert len(
    anchor_count_per_candidate_7d
) == 699


assert (
    anchor_count_per_candidate_7d
    ==
    2
).all()


# ------------------------------------------------------------
# 8. Count retained candidate fragments for each variant
#
# IMPORTANT:
#
# GT is NOT referenced anywhere here.
# ------------------------------------------------------------

candidate_variant_rows_7d = []


for (
    dataset_index,
    original_component_id
), comp_frag_df in (
    candidate_fragments_7d_df
    .groupby(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
):

    dataset_index = int(
        dataset_index
    )

    original_component_id = int(
        original_component_id
    )


    raw_fragment_count = int(
        len(
            comp_frag_df
        )
    )


    anchor_count = int(
        comp_frag_df[
            "is_depth_anchor"
        ]
        .sum()
    )


    assert anchor_count == 2


    # --------------------------------------------------------
    # G0
    #
    # Candidate remains one original semantic component.
    # --------------------------------------------------------

    G0_count = 1


    # --------------------------------------------------------
    # G1
    #
    # Every raw Step-7B fragment treated as one countable unit.
    # --------------------------------------------------------

    G1_count = raw_fragment_count


    # --------------------------------------------------------
    # G2
    #
    # Anchors +
    # parent-relative satellites >= 0.001
    # --------------------------------------------------------

    G2_keep = (
        comp_frag_df[
            "is_depth_anchor"
        ]
        |
        (
            comp_frag_df[
                "is_depth_satellite"
            ]
            &
            (
                comp_frag_df[
                    "proposal_area_fraction_of_original"
                ]
                >=
                0.001
            )
        )
    )


    G2_count = int(
        G2_keep.sum()
    )


    # --------------------------------------------------------
    # G3
    #
    # Anchors +
    # parent-relative satellites >= 0.0025
    # --------------------------------------------------------

    G3_keep = (
        comp_frag_df[
            "is_depth_anchor"
        ]
        |
        (
            comp_frag_df[
                "is_depth_satellite"
            ]
            &
            (
                comp_frag_df[
                    "proposal_area_fraction_of_original"
                ]
                >=
                0.0025
            )
        )
    )


    G3_count = int(
        G3_keep.sum()
    )


    # --------------------------------------------------------
    # Structural monotonicity
    # --------------------------------------------------------

    assert (
        G0_count
        <=
        G3_count
        <=
        G2_count
        <=
        G1_count
    )


    candidate_variant_rows_7d.append(
        {
            "dataset_index":
                dataset_index,

            "original_component_id":
                original_component_id,

            "raw_fragment_count":
                raw_fragment_count,

            "anchor_count":
                anchor_count,

            "G0_candidate_contribution":
                G0_count,

            "G1_candidate_contribution":
                G1_count,

            "G2_candidate_contribution":
                G2_count,

            "G3_candidate_contribution":
                G3_count,
        }
    )


candidate_variant_counts_7d_df = (
    pd.DataFrame(
        candidate_variant_rows_7d
    )
    .sort_values(
        [
            "dataset_index",
            "original_component_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert len(
    candidate_variant_counts_7d_df
) == 699


# ------------------------------------------------------------
# 9. Attach candidate contributions to ALL original A3c
#    components.
#
# Non-candidates remain exactly one component under every
# geometry variant.
# ------------------------------------------------------------

component_geometry_counts_7d_df = (
    original_components_7d_df
    .merge(
        candidate_variant_counts_7d_df,
        on=[
            "dataset_index",
            "original_component_id",
        ],
        how="left",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# G0 contribution
#
# ALL original components contribute exactly 1.
# ------------------------------------------------------------

component_geometry_counts_7d_df[
    "G0_count_contribution"
] = 1


# ------------------------------------------------------------
# G1 contribution
# ------------------------------------------------------------

component_geometry_counts_7d_df[
    "G1_count_contribution"
] = np.where(
    component_geometry_counts_7d_df[
        "split_candidate"
    ],

    component_geometry_counts_7d_df[
        "G1_candidate_contribution"
    ],

    1,
).astype(int)


# ------------------------------------------------------------
# G2 contribution
# ------------------------------------------------------------

component_geometry_counts_7d_df[
    "G2_count_contribution"
] = np.where(
    component_geometry_counts_7d_df[
        "split_candidate"
    ],

    component_geometry_counts_7d_df[
        "G2_candidate_contribution"
    ],

    1,
).astype(int)


# ------------------------------------------------------------
# G3 contribution
# ------------------------------------------------------------

component_geometry_counts_7d_df[
    "G3_count_contribution"
] = np.where(
    component_geometry_counts_7d_df[
        "split_candidate"
    ],

    component_geometry_counts_7d_df[
        "G3_candidate_contribution"
    ],

    1,
).astype(int)


# ------------------------------------------------------------
# 10. Sanity checks
# ------------------------------------------------------------

assert (
    component_geometry_counts_7d_df[
        "G0_count_contribution"
    ]
    ==
    1
).all()


noncandidate_mask_7d = (
    ~component_geometry_counts_7d_df[
        "split_candidate"
    ]
)


for col in [
    "G1_count_contribution",
    "G2_count_contribution",
    "G3_count_contribution",
]:

    assert (
        component_geometry_counts_7d_df.loc[
            noncandidate_mask_7d,
            col,
        ]
        ==
        1
    ).all()


candidate_mask_7d = (
    component_geometry_counts_7d_df[
        "split_candidate"
    ]
)


assert (
    component_geometry_counts_7d_df.loc[
        candidate_mask_7d,
        "G1_count_contribution",
    ]
    >=
    component_geometry_counts_7d_df.loc[
        candidate_mask_7d,
        "G2_count_contribution",
    ]
).all()


assert (
    component_geometry_counts_7d_df.loc[
        candidate_mask_7d,
        "G2_count_contribution",
    ]
    >=
    component_geometry_counts_7d_df.loc[
        candidate_mask_7d,
        "G3_count_contribution",
    ]
).all()


assert (
    component_geometry_counts_7d_df.loc[
        candidate_mask_7d,
        "G3_count_contribution",
    ]
    >=
    2
).all()


# ------------------------------------------------------------
# 11. Aggregate to IMAGE LEVEL
#
# These are geometry-derived semantic-component counts.
#
# They are NOT yet final B0/B2 predictions.
# ------------------------------------------------------------

image_geometry_counts_7d_df = (
    component_geometry_counts_7d_df
    .groupby(
        "dataset_index",
        as_index=False,
    )
    .agg(
        original_A3c_components=(
            "original_component_id",
            "count",
        ),

        num_split_candidates=(
            "split_candidate",
            "sum",
        ),

        G0_geometry_count=(
            "G0_count_contribution",
            "sum",
        ),

        G1_geometry_count=(
            "G1_count_contribution",
            "sum",
        ),

        G2_geometry_count=(
            "G2_count_contribution",
            "sum",
        ),

        G3_geometry_count=(
            "G3_count_contribution",
            "sum",
        ),
    )
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    image_geometry_counts_7d_df
) == 100


# Frozen A3c identity
assert (
    image_geometry_counts_7d_df[
        "original_A3c_components"
    ]
    .sum()
    ==
    2554
)


assert (
    image_geometry_counts_7d_df[
        "G0_geometry_count"
    ]
    .sum()
    ==
    2554
)


# ------------------------------------------------------------
# 12. Geometry-induced count increments relative to G0
# ------------------------------------------------------------

for variant in [
    "G1",
    "G2",
    "G3",
]:

    image_geometry_counts_7d_df[
        f"{variant}_increment_vs_G0"
    ] = (
        image_geometry_counts_7d_df[
            f"{variant}_geometry_count"
        ]
        -
        image_geometry_counts_7d_df[
            "G0_geometry_count"
        ]
    )


# ------------------------------------------------------------
# 13. Global structural summary
# ------------------------------------------------------------

variant_summary_rows_7d = []


for variant, description in [
    (
        "G0",
        "No geometric refinement",
    ),

    (
        "G1",
        "All raw Step-7B fragments",
    ),

    (
        "G2",
        "Anchors + parent-relative satellites >= 0.001",
    ),

    (
        "G3",
        "Anchors + parent-relative satellites >= 0.0025",
    ),
]:

    total_count = int(
        image_geometry_counts_7d_df[
            f"{variant}_geometry_count"
        ]
        .sum()
    )


    increment = int(
        total_count
        -
        2554
    )


    variant_summary_rows_7d.append(
        {
            "variant":
                variant,

            "description":
                description,

            "total_geometry_count":
                total_count,

            "increment_vs_G0":
                increment,

            "relative_count_vs_G0":
                float(
                    total_count
                    /
                    2554
                ),

            "mean_count_per_image":
                float(
                    image_geometry_counts_7d_df[
                        f"{variant}_geometry_count"
                    ]
                    .mean()
                ),

            "median_count_per_image":
                float(
                    image_geometry_counts_7d_df[
                        f"{variant}_geometry_count"
                    ]
                    .median()
                ),

            "max_count_per_image":
                int(
                    image_geometry_counts_7d_df[
                        f"{variant}_geometry_count"
                    ]
                    .max()
                ),
        }
    )


geometry_variant_summary_7d_df = (
    pd.DataFrame(
        variant_summary_rows_7d
    )
)


# ------------------------------------------------------------
# 14. Candidate-only structural summary
# ------------------------------------------------------------

candidate_contribution_summary_7d_df = (
    candidate_variant_counts_7d_df[
        [
            "G0_candidate_contribution",
            "G1_candidate_contribution",
            "G2_candidate_contribution",
            "G3_candidate_contribution",
        ]
    ]
    .describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.90,
            0.95,
            0.99,
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index":
                "variant_contribution"
        }
    )
)


# ------------------------------------------------------------
# 15. Save frozen Step-7D Part-A artifacts
# ------------------------------------------------------------

M1_7D_A_COMPONENT_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_A_component_geometry_count_contributions.csv"
)

M1_7D_A_IMAGE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_A_image_geometry_counts.csv"
)

M1_7D_A_VARIANT_SUMMARY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_A_geometry_variant_summary.csv"
)

M1_7D_A_CANDIDATE_SUMMARY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_A_candidate_contribution_summary.csv"
)


component_geometry_counts_7d_df.to_csv(
    M1_7D_A_COMPONENT_FILE,
    index=False,
)

image_geometry_counts_7d_df.to_csv(
    M1_7D_A_IMAGE_FILE,
    index=False,
)

geometry_variant_summary_7d_df.to_csv(
    M1_7D_A_VARIANT_SUMMARY_FILE,
    index=False,
)

candidate_contribution_summary_7d_df.to_csv(
    M1_7D_A_CANDIDATE_SUMMARY_FILE,
    index=False,
)


# ------------------------------------------------------------
# 16. Report — frozen variants
# ------------------------------------------------------------

print("\nFrozen geometry variants")
print("------------------------")

display(
    geometry_variant_summary_7d_df
)


# ------------------------------------------------------------
# 17. Report — candidate contribution distributions
# ------------------------------------------------------------

print("\nCandidate count-contribution distributions")
print("------------------------------------------")

display(
    candidate_contribution_summary_7d_df
)


# ------------------------------------------------------------
# 18. Report — image-level structural counts
# ------------------------------------------------------------

print("\nImage-level geometry counts — first 20 images")
print("---------------------------------------------")

display(
    image_geometry_counts_7d_df[
        [
            "dataset_index",
            "original_A3c_components",
            "num_split_candidates",

            "G0_geometry_count",
            "G1_geometry_count",
            "G2_geometry_count",
            "G3_geometry_count",

            "G1_increment_vs_G0",
            "G2_increment_vs_G0",
            "G3_increment_vs_G0",
        ]
    ]
    .head(
        20
    )
)


# ------------------------------------------------------------
# 19. Images with largest raw geometry inflation
# ------------------------------------------------------------

print("\nImages with largest G1 count inflation")
print("--------------------------------------")

display(
    image_geometry_counts_7d_df
    .sort_values(
        "G1_increment_vs_G0",
        ascending=False,
    )
    [
        [
            "dataset_index",
            "original_A3c_components",
            "num_split_candidates",

            "G0_geometry_count",
            "G1_geometry_count",
            "G2_geometry_count",
            "G3_geometry_count",

            "G1_increment_vs_G0",
            "G2_increment_vs_G0",
            "G3_increment_vs_G0",
        ]
    ]
    .head(
        20
    )
)


# ------------------------------------------------------------
# 20. Global consistency
# ------------------------------------------------------------

print("\nGlobal consistency")
print("------------------")

print(
    "Original A3c components      :",
    int(
        len(
            original_components_7d_df
        )
    ),
)

print(
    "Frozen split candidates      :",
    int(
        original_components_7d_df[
            "split_candidate"
        ]
        .sum()
    ),
)

print(
    "Untouched non-candidates     :",
    int(
        (
            ~original_components_7d_df[
                "split_candidate"
            ]
        )
        .sum()
    ),
)

print(
    "Raw candidate fragments      :",
    int(
        len(
            candidate_fragments_7d_df
        )
    ),
)

print(
    "G0 total geometry count      :",
    int(
        image_geometry_counts_7d_df[
            "G0_geometry_count"
        ]
        .sum()
    ),
)

print(
    "G1 total geometry count      :",
    int(
        image_geometry_counts_7d_df[
            "G1_geometry_count"
        ]
        .sum()
    ),
)

print(
    "G2 total geometry count      :",
    int(
        image_geometry_counts_7d_df[
            "G2_geometry_count"
        ]
        .sum()
    ),
)

print(
    "G3 total geometry count      :",
    int(
        image_geometry_counts_7d_df[
            "G3_geometry_count"
        ]
        .sum()
    ),
)


# ------------------------------------------------------------
# 21. Protocol report
# ------------------------------------------------------------

print("\nStep-7A candidate rule reused        : YES — frozen 75th percentile")
print("Step-7B raw geometry reused          : YES")
print("Step-7C G2 threshold reused          : YES — parent >= 0.001")
print("Step-7C G3 threshold reused          : YES — parent >= 0.0025")

print("\nStep-6D component-area field reused  : YES — area_pixels")
print("Candidate parent-area identity       : checked exactly")

print("\nGT used to construct G0/G1/G2/G3    : NO")
print("GT used in Step 7D Part A evaluation: NO")
print("New threshold tuning performed       : NO")
print("Non-candidate components modified    : NO")

print("\nFragments interpreted as instances   : YES — experimental stress-test only")
print("Claim fragment == true object        : NO")

print("\nB0 prediction modified               : NO")
print("B2 recovery modified                 : NO")
print("Final image count evaluated          : NO")
print("MAE/RMSE evaluated                   : NO")
print("Density routing used                 : NO")
print("Test set accessed                    : NO")

print("\nGeometry variants frozen for Step 7D Part B:")
print("  G0 = original A3c components")
print("  G1 = all raw geometric fragments")
print("  G2 = anchors + parent-relative satellites >= 0.001")
print("  G3 = anchors + parent-relative satellites >= 0.0025")

print("\nStep 7D Part A: PASS")

STEP 7D PART A — FREEZE GEOMETRY VARIANTS + DIRECT-FRAGMENT COUNT INTERPRETATION

Frozen Step-7D variants
-----------------------
G0_NO_GEOMETRY         {'candidate_mode': 'original_component', 'parent_relative_threshold': None}
G1_RAW_SPLIT           {'candidate_mode': 'all_raw_fragments', 'parent_relative_threshold': None}
G2_PARENT_0.001        {'candidate_mode': 'anchor_plus_parent_threshold', 'parent_relative_threshold': 0.001}
G3_PARENT_0.0025       {'candidate_mode': 'anchor_plus_parent_threshold', 'parent_relative_threshold': 0.0025}

Original A3c component schema
-----------------------------
Area source column           : area_pixels
Renamed for Step 7D          : original_component_area
Original components          : 2554
Total original foreground px : 2105914
Candidate parent-area identity : 699/699 exact


/tmp/ipykernel_3524/1150923898.py:355: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .fillna(False)



Frozen geometry variants
------------------------


,variant,description,total_geometry_count,increment_vs_G0,relative_count_vs_G0,mean_count_per_image,median_count_per_image,max_count_per_image
0,G0,No geometric refinement,2554,0,1.000000,25.54,20.5,107
1,G1,All raw Step-7B fragments,8377,5823,3.279953,83.77,54.5,1202
2,G2,Anchors + parent-relative satellites >= 0.001,6065,3511,2.374706,60.65,41.0,726
3,G3,Anchors + parent-relative satellites >= 0.0025,5001,2447,1.958105,50.01,39.0,476



Candidate count-contribution distributions
------------------------------------------


...


Image-level geometry counts — first 20 images
---------------------------------------------


,dataset_index,original_A3c_components,num_split_candidates,G0_geometry_count,G1_geometry_count,G2_geometry_count,G3_geometry_count,G1_increment_vs_G0,G2_increment_vs_G0,G3_increment_vs_G0
0,9,6,2,6,28,16,15,22,10,9
1,14,32,9,32,99,66,61,67,34,29
2,20,26,7,26,67,48,44,41,22,18
3,31,7,2,7,11,9,9,4,2,2
4,40,1,1,1,5,3,2,4,2,1
5,49,8,3,8,78,23,19,70,15,11
6,52,18,5,18,37,33,32,19,15,14
7,59,8,3,8,28,17,16,20,9,8
8,67,9,3,9,35,28,27,26,19,18
9,87,10,3,10,55,23,21,45,13,11



Images with largest G1 count inflation
--------------------------------------


,dataset_index,original_A3c_components,num_split_candidates,G0_geometry_count,G1_geometry_count,G2_geometry_count,G3_geometry_count,G1_increment_vs_G0,G2_increment_vs_G0,G3_increment_vs_G0
32,253,107,27,107,1202,726,476,1095,619,369
93,1192,53,14,53,393,289,231,340,236,178
71,862,26,7,26,282,166,71,256,140,45
28,227,51,13,51,257,217,120,206,166,69
15,143,50,13,50,200,109,86,150,59,36
92,1187,97,25,97,244,232,219,147,135,122
24,183,42,11,42,168,102,77,126,60,35
55,534,24,7,24,149,84,47,125,60,23
97,1257,98,25,98,221,206,161,123,108,63
62,618,27,7,27,142,65,52,115,38,25



Global consistency
------------------
Original A3c components      : 2554
Frozen split candidates      : 699
Untouched non-candidates     : 1855
Raw candidate fragments      : 6522
G0 total geometry count      : 2554
G1 total geometry count      : 8377
G2 total geometry count      : 6065
G3 total geometry count      : 5001

Step-7A candidate rule reused        : YES — frozen 75th percentile
Step-7B raw geometry reused          : YES
Step-7C G2 threshold reused          : YES — parent >= 0.001
Step-7C G3 threshold reused          : YES — parent >= 0.0025

Step-6D component-area field reused  : YES — area_pixels
Candidate parent-area identity       : checked exactly

GT used to construct G0/G1/G2/G3    : NO
GT used in Step 7D Part A evaluation: NO
New threshold tuning performed       : NO
Non-candidate components modified    : NO

Fragments interpreted as instances   : YES — experimental stress-test only
Claim fragment == true object        : NO

B0 prediction modified               : N

In [35]:
# ============================================================
# Step 7D Part B — Count-level evaluation of frozen
#                  geometry variants G0/G1/G2/G3
# ============================================================
#
# Purpose
# -------
# Evaluate the four geometry variants frozen in Step 7D Part A
# as DIRECT image-level counters on the frozen FSC-147 dev100.
#
#
# Frozen variants
# ---------------
#
# G0:
#     original A3c connected-component count
#
# G1:
#     all raw Marigold spatial fragments
#
# G2:
#     anchors + parent-relative satellites >= 0.001
#
# G3:
#     anchors + parent-relative satellites >= 0.0025
#
#
# Evaluation
# ----------
#
# Overall:
#   - MAE
#   - RMSE
#   - signed error
#   - under / exact / over image counts
#   - mean prediction
#   - total prediction
#
# Density strata:
#   S1 ... S6
#   - n
#   - mean GT
#   - mean prediction
#   - MAE
#   - RMSE
#   - signed error
#   - under / exact / over
#
#
# IMPORTANT INTERPRETATION
# ------------------------
#
# This is a DIRECT-FRAGMENT-AS-INSTANCE stress test.
#
# It does NOT yet:
#
#   - modify B0
#   - modify B2
#   - combine geometry with C1 recovery
#   - use density routing
#   - tune any thresholds
#
# Therefore the result answers:
#
#   "Can this geometric refinement itself act as a counter?"
#
# NOT:
#
#   "Does Marigold improve the final B0/B2 architecture?"
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 7D PART B — COUNT-LEVEL EVALUATION OF FROZEN GEOMETRY VARIANTS")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "image_geometry_counts_7d_df" in globals(), (
    "image_geometry_counts_7d_df unavailable. "
    "Run Step 7D Part A first."
)

assert "GEOMETRY_VARIANTS_7D" in globals(), (
    "GEOMETRY_VARIANTS_7D unavailable. "
    "Run Step 7D Part A first."
)

assert "dev100_df" in globals(), (
    "dev100_df unavailable. "
    "Run the frozen dev100 loading section first."
)

assert "M1_TABLES_DIR" in globals()


assert len(
    image_geometry_counts_7d_df
) == 100


assert len(
    dev100_df
) == 100


# ------------------------------------------------------------
# 2. Validate authoritative dev100 schema
# ------------------------------------------------------------

required_dev_cols_7db = {
    "dataset_index",
    "image_id",
    "category",
    "gt_count",
    "density_stratum",
}


assert required_dev_cols_7db.issubset(
    set(
        dev100_df.columns
    )
), (
    "Required dev100 columns missing.\n"
    f"Required: {required_dev_cols_7db}\n"
    f"Found: {list(dev100_df.columns)}"
)


assert (
    dev100_df[
        "dataset_index"
    ]
    .duplicated()
    .sum()
    ==
    0
)


# ------------------------------------------------------------
# 3. Frozen density-stratum identities
# ------------------------------------------------------------

STRATUM_ORDER_7DB = [
    "S1_0_20",
    "S2_20_40",
    "S3_40_60",
    "S4_60_80",
    "S5_80_95",
    "S6_95_100",
]


EXPECTED_STRATUM_COUNTS_7DB = {
    "S1_0_20":
        20,

    "S2_20_40":
        20,

    "S3_40_60":
        20,

    "S4_60_80":
        20,

    "S5_80_95":
        10,

    "S6_95_100":
        10,
}


actual_stratum_counts_7db = (
    dev100_df[
        "density_stratum"
    ]
    .value_counts()
    .to_dict()
)


assert (
    actual_stratum_counts_7db
    ==
    EXPECTED_STRATUM_COUNTS_7DB
), (
    "Frozen dev100 stratum counts changed.\n"
    f"Expected: {EXPECTED_STRATUM_COUNTS_7DB}\n"
    f"Found: {actual_stratum_counts_7db}"
)


# ------------------------------------------------------------
# 4. Build authoritative evaluation table
#
# Merge ONLY on dataset_index.
#
# dev100 supplies:
#   GT + image identity + density stratum
#
# Step 7D Part A supplies:
#   frozen G0/G1/G2/G3 counts
# ------------------------------------------------------------

geometry_count_eval_7db_df = (
    dev100_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "gt_count",
            "density_stratum",
        ]
    ]
    .merge(
        image_geometry_counts_7d_df[
            [
                "dataset_index",

                "original_A3c_components",
                "num_split_candidates",

                "G0_geometry_count",
                "G1_geometry_count",
                "G2_geometry_count",
                "G3_geometry_count",

                "G1_increment_vs_G0",
                "G2_increment_vs_G0",
                "G3_increment_vs_G0",
            ]
        ],
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    geometry_count_eval_7db_df
) == 100


assert (
    geometry_count_eval_7db_df[
        [
            "G0_geometry_count",
            "G1_geometry_count",
            "G2_geometry_count",
            "G3_geometry_count",
        ]
    ]
    .notna()
    .all()
    .all()
)


# ------------------------------------------------------------
# 5. Frozen identity checks
# ------------------------------------------------------------

assert (
    geometry_count_eval_7db_df[
        "G0_geometry_count"
    ]
    ==
    geometry_count_eval_7db_df[
        "original_A3c_components"
    ]
).all()


assert (
    geometry_count_eval_7db_df[
        "G1_geometry_count"
    ]
    >=
    geometry_count_eval_7db_df[
        "G2_geometry_count"
    ]
).all()


assert (
    geometry_count_eval_7db_df[
        "G2_geometry_count"
    ]
    >=
    geometry_count_eval_7db_df[
        "G3_geometry_count"
    ]
).all()


assert (
    geometry_count_eval_7db_df[
        "G3_geometry_count"
    ]
    >=
    geometry_count_eval_7db_df[
        "G0_geometry_count"
    ]
).all()


# ------------------------------------------------------------
# 6. Add error columns for each frozen geometry variant
#
# signed_error convention:
#
#       prediction - GT
#
# Negative = undercount
# Positive = overcount
# ------------------------------------------------------------

VARIANT_COUNT_COLS_7DB = {
    "G0":
        "G0_geometry_count",

    "G1":
        "G1_geometry_count",

    "G2":
        "G2_geometry_count",

    "G3":
        "G3_geometry_count",
}


for variant, pred_col in (
    VARIANT_COUNT_COLS_7DB.items()
):

    signed_col = (
        f"{variant}_signed_error"
    )

    abs_col = (
        f"{variant}_abs_error"
    )

    squared_col = (
        f"{variant}_squared_error"
    )

    ratio_col = (
        f"{variant}_pred_gt_ratio"
    )


    geometry_count_eval_7db_df[
        signed_col
    ] = (
        geometry_count_eval_7db_df[
            pred_col
        ]
        -
        geometry_count_eval_7db_df[
            "gt_count"
        ]
    )


    geometry_count_eval_7db_df[
        abs_col
    ] = (
        geometry_count_eval_7db_df[
            signed_col
        ]
        .abs()
    )


    geometry_count_eval_7db_df[
        squared_col
    ] = (
        geometry_count_eval_7db_df[
            signed_col
        ]
        ** 2
    )


    geometry_count_eval_7db_df[
        ratio_col
    ] = (
        geometry_count_eval_7db_df[
            pred_col
        ]
        /
        geometry_count_eval_7db_df[
            "gt_count"
        ]
    )


# ------------------------------------------------------------
# 7. Overall metric helper
# ------------------------------------------------------------

def overall_metrics_7db(
    df,
    variant,
    pred_col,
):

    signed = (
        df[
            f"{variant}_signed_error"
        ]
        .to_numpy(
            dtype=float
        )
    )


    abs_error = (
        np.abs(
            signed
        )
    )


    squared_error = (
        signed
        ** 2
    )


    prediction = (
        df[
            pred_col
        ]
        .to_numpy(
            dtype=float
        )
    )


    gt = (
        df[
            "gt_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    return {
        "variant":
            variant,

        "description":
            {
                "G0":
                    "Original A3c components",

                "G1":
                    "All raw Marigold fragments",

                "G2":
                    "Anchors + parent satellites >= 0.001",

                "G3":
                    "Anchors + parent satellites >= 0.0025",
            }[
                variant
            ],

        "num_images":
            int(
                len(
                    df
                )
            ),

        "total_GT":
            int(
                gt.sum()
            ),

        "total_prediction":
            int(
                prediction.sum()
            ),

        "mean_GT":
            float(
                gt.mean()
            ),

        "mean_prediction":
            float(
                prediction.mean()
            ),

        "MAE":
            float(
                abs_error.mean()
            ),

        "RMSE":
            float(
                np.sqrt(
                    squared_error.mean()
                )
            ),

        "mean_signed_error":
            float(
                signed.mean()
            ),

        "median_signed_error":
            float(
                np.median(
                    signed
                )
            ),

        "mean_pred_GT_ratio":
            float(
                np.mean(
                    prediction
                    /
                    gt
                )
            ),

        "median_pred_GT_ratio":
            float(
                np.median(
                    prediction
                    /
                    gt
                )
            ),

        "under_images":
            int(
                (
                    signed
                    <
                    0
                )
                .sum()
            ),

        "exact_images":
            int(
                (
                    signed
                    ==
                    0
                )
                .sum()
            ),

        "over_images":
            int(
                (
                    signed
                    >
                    0
                )
                .sum()
            ),
    }


# ------------------------------------------------------------
# 8. Overall metrics G0/G1/G2/G3
# ------------------------------------------------------------

overall_rows_7db = []


for variant, pred_col in (
    VARIANT_COUNT_COLS_7DB.items()
):

    overall_rows_7db.append(
        overall_metrics_7db(
            df=
                geometry_count_eval_7db_df,

            variant=
                variant,

            pred_col=
                pred_col,
        )
    )


geometry_overall_metrics_7db_df = (
    pd.DataFrame(
        overall_rows_7db
    )
)


# ------------------------------------------------------------
# 9. Relative change versus G0
#
# Descriptive only.
#
# Negative delta MAE/RMSE = improvement versus G0.
# Positive delta = worsening.
# ------------------------------------------------------------

g0_overall_7db = (
    geometry_overall_metrics_7db_df[
        geometry_overall_metrics_7db_df[
            "variant"
        ]
        ==
        "G0"
    ]
    .iloc[0]
)


geometry_overall_metrics_7db_df[
    "delta_MAE_vs_G0"
] = (
    geometry_overall_metrics_7db_df[
        "MAE"
    ]
    -
    float(
        g0_overall_7db[
            "MAE"
        ]
    )
)


geometry_overall_metrics_7db_df[
    "delta_RMSE_vs_G0"
] = (
    geometry_overall_metrics_7db_df[
        "RMSE"
    ]
    -
    float(
        g0_overall_7db[
            "RMSE"
        ]
    )
)


geometry_overall_metrics_7db_df[
    "delta_signed_error_vs_G0"
] = (
    geometry_overall_metrics_7db_df[
        "mean_signed_error"
    ]
    -
    float(
        g0_overall_7db[
            "mean_signed_error"
        ]
    )
)


# ------------------------------------------------------------
# 10. Density-stratum metrics
# ------------------------------------------------------------

stratum_rows_7db = []


for stratum in (
    STRATUM_ORDER_7DB
):

    stratum_df = (
        geometry_count_eval_7db_df[
            geometry_count_eval_7db_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
        .copy()
    )


    assert len(
        stratum_df
    ) == (
        EXPECTED_STRATUM_COUNTS_7DB[
            stratum
        ]
    )


    for variant, pred_col in (
        VARIANT_COUNT_COLS_7DB.items()
    ):

        signed = (
            stratum_df[
                f"{variant}_signed_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        prediction = (
            stratum_df[
                pred_col
            ]
            .to_numpy(
                dtype=float
            )
        )


        gt = (
            stratum_df[
                "gt_count"
            ]
            .to_numpy(
                dtype=float
            )
        )


        stratum_rows_7db.append(
            {
                "density_stratum":
                    stratum,

                "variant":
                    variant,

                "num_images":
                    int(
                        len(
                            stratum_df
                        )
                    ),

                "mean_GT":
                    float(
                        gt.mean()
                    ),

                "mean_prediction":
                    float(
                        prediction.mean()
                    ),

                "MAE":
                    float(
                        np.abs(
                            signed
                        )
                        .mean()
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                signed
                                ** 2
                            )
                        )
                    ),

                "mean_signed_error":
                    float(
                        signed.mean()
                    ),

                "median_signed_error":
                    float(
                        np.median(
                            signed
                        )
                    ),

                "mean_pred_GT_ratio":
                    float(
                        np.mean(
                            prediction
                            /
                            gt
                        )
                    ),

                "under_images":
                    int(
                        (
                            signed
                            <
                            0
                        )
                        .sum()
                    ),

                "exact_images":
                    int(
                        (
                            signed
                            ==
                            0
                        )
                        .sum()
                    ),

                "over_images":
                    int(
                        (
                            signed
                            >
                            0
                        )
                        .sum()
                    ),
            }
        )


geometry_stratum_metrics_7db_df = (
    pd.DataFrame(
        stratum_rows_7db
    )
)


geometry_stratum_metrics_7db_df[
    "density_stratum"
] = pd.Categorical(
    geometry_stratum_metrics_7db_df[
        "density_stratum"
    ],
    categories=
        STRATUM_ORDER_7DB,
    ordered=True,
)


geometry_stratum_metrics_7db_df = (
    geometry_stratum_metrics_7db_df
    .sort_values(
        [
            "density_stratum",
            "variant",
        ]
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 11. Wide MAE / RMSE / signed-error tables
#
# Easier to compare G0/G1/G2/G3 across S1-S6.
# ------------------------------------------------------------

stratum_MAE_wide_7db_df = (
    geometry_stratum_metrics_7db_df
    .pivot(
        index=
            "density_stratum",

        columns=
            "variant",

        values=
            "MAE",
    )
    .reset_index()
)


stratum_RMSE_wide_7db_df = (
    geometry_stratum_metrics_7db_df
    .pivot(
        index=
            "density_stratum",

        columns=
            "variant",

        values=
            "RMSE",
    )
    .reset_index()
)


stratum_signed_wide_7db_df = (
    geometry_stratum_metrics_7db_df
    .pivot(
        index=
            "density_stratum",

        columns=
            "variant",

        values=
            "mean_signed_error",
    )
    .reset_index()
)


# ------------------------------------------------------------
# 12. Per-image comparison versus G0
#
# Determine whether each geometry treatment reduces absolute
# error relative to original A3c count.
# ------------------------------------------------------------

for variant in [
    "G1",
    "G2",
    "G3",
]:

    geometry_count_eval_7db_df[
        f"{variant}_abs_error_delta_vs_G0"
    ] = (
        geometry_count_eval_7db_df[
            f"{variant}_abs_error"
        ]
        -
        geometry_count_eval_7db_df[
            "G0_abs_error"
        ]
    )


    geometry_count_eval_7db_df[
        f"{variant}_improves_vs_G0"
    ] = (
        geometry_count_eval_7db_df[
            f"{variant}_abs_error_delta_vs_G0"
        ]
        <
        0
    )


    geometry_count_eval_7db_df[
        f"{variant}_ties_G0"
    ] = (
        geometry_count_eval_7db_df[
            f"{variant}_abs_error_delta_vs_G0"
        ]
        ==
        0
    )


    geometry_count_eval_7db_df[
        f"{variant}_worsens_vs_G0"
    ] = (
        geometry_count_eval_7db_df[
            f"{variant}_abs_error_delta_vs_G0"
        ]
        >
        0
    )


# ------------------------------------------------------------
# 13. Image-level improve/tie/worsen summary
# ------------------------------------------------------------

comparison_rows_7db = []


for variant in [
    "G1",
    "G2",
    "G3",
]:

    comparison_rows_7db.append(
        {
            "variant":
                variant,

            "improves_images":
                int(
                    geometry_count_eval_7db_df[
                        f"{variant}_improves_vs_G0"
                    ]
                    .sum()
                ),

            "ties_images":
                int(
                    geometry_count_eval_7db_df[
                        f"{variant}_ties_G0"
                    ]
                    .sum()
                ),

            "worsens_images":
                int(
                    geometry_count_eval_7db_df[
                        f"{variant}_worsens_vs_G0"
                    ]
                    .sum()
                ),

            "median_abs_error_delta_vs_G0":
                float(
                    geometry_count_eval_7db_df[
                        f"{variant}_abs_error_delta_vs_G0"
                    ]
                    .median()
                ),

            "mean_abs_error_delta_vs_G0":
                float(
                    geometry_count_eval_7db_df[
                        f"{variant}_abs_error_delta_vs_G0"
                    ]
                    .mean()
                ),
        }
    )


geometry_vs_G0_comparison_7db_df = (
    pd.DataFrame(
        comparison_rows_7db
    )
)


# ------------------------------------------------------------
# 14. Stratum-specific improve/tie/worsen
# ------------------------------------------------------------

stratum_comparison_rows_7db = []


for stratum in (
    STRATUM_ORDER_7DB
):

    stratum_df = (
        geometry_count_eval_7db_df[
            geometry_count_eval_7db_df[
                "density_stratum"
            ]
            ==
            stratum
        ]
    )


    for variant in [
        "G1",
        "G2",
        "G3",
    ]:

        delta = (
            stratum_df[
                f"{variant}_abs_error_delta_vs_G0"
            ]
        )


        stratum_comparison_rows_7db.append(
            {
                "density_stratum":
                    stratum,

                "variant":
                    variant,

                "improves":
                    int(
                        (
                            delta
                            <
                            0
                        )
                        .sum()
                    ),

                "ties":
                    int(
                        (
                            delta
                            ==
                            0
                        )
                        .sum()
                    ),

                "worsens":
                    int(
                        (
                            delta
                            >
                            0
                        )
                        .sum()
                    ),

                "mean_abs_error_delta_vs_G0":
                    float(
                        delta.mean()
                    ),

                "median_abs_error_delta_vs_G0":
                    float(
                        delta.median()
                    ),
            }
        )


geometry_stratum_vs_G0_7db_df = (
    pd.DataFrame(
        stratum_comparison_rows_7db
    )
)


# ------------------------------------------------------------
# 15. Largest improvements / worsenings
#
# Diagnostic only.
# ------------------------------------------------------------

extreme_rows_7db = []


for variant in [
    "G1",
    "G2",
    "G3",
]:

    base_cols = [
        "dataset_index",
        "image_id",
        "category",
        "density_stratum",
        "gt_count",
        "G0_geometry_count",
        f"{variant}_geometry_count",
        "G0_abs_error",
        f"{variant}_abs_error",
        f"{variant}_abs_error_delta_vs_G0",
        "num_split_candidates",
    ]


    best = (
        geometry_count_eval_7db_df
        .sort_values(
            f"{variant}_abs_error_delta_vs_G0",
            ascending=True,
        )
        [
            base_cols
        ]
        .head(
            10
        )
        .copy()
    )


    best[
        "variant"
    ] = variant


    best[
        "case_type"
    ] = "largest_improvement"


    worst = (
        geometry_count_eval_7db_df
        .sort_values(
            f"{variant}_abs_error_delta_vs_G0",
            ascending=False,
        )
        [
            base_cols
        ]
        .head(
            10
        )
        .copy()
    )


    worst[
        "variant"
    ] = variant


    worst[
        "case_type"
    ] = "largest_worsening"


    extreme_rows_7db.append(
        best
    )

    extreme_rows_7db.append(
        worst
    )


geometry_extreme_cases_7db_df = (
    pd.concat(
        extreme_rows_7db,
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# 16. Save outputs
# ------------------------------------------------------------

M1_7D_B_IMAGE_EVAL_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_B_geometry_count_image_evaluation.csv"
)

M1_7D_B_OVERALL_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_B_geometry_count_overall_metrics.csv"
)

M1_7D_B_STRATUM_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_B_geometry_count_stratum_metrics.csv"
)

M1_7D_B_COMPARISON_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_B_geometry_vs_G0_comparison.csv"
)

M1_7D_B_STRATUM_COMPARISON_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_B_geometry_stratum_vs_G0_comparison.csv"
)

M1_7D_B_EXTREMES_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_B_geometry_extreme_cases.csv"
)


geometry_count_eval_7db_df.to_csv(
    M1_7D_B_IMAGE_EVAL_FILE,
    index=False,
)

geometry_overall_metrics_7db_df.to_csv(
    M1_7D_B_OVERALL_FILE,
    index=False,
)

geometry_stratum_metrics_7db_df.to_csv(
    M1_7D_B_STRATUM_FILE,
    index=False,
)

geometry_vs_G0_comparison_7db_df.to_csv(
    M1_7D_B_COMPARISON_FILE,
    index=False,
)

geometry_stratum_vs_G0_7db_df.to_csv(
    M1_7D_B_STRATUM_COMPARISON_FILE,
    index=False,
)

geometry_extreme_cases_7db_df.to_csv(
    M1_7D_B_EXTREMES_FILE,
    index=False,
)


# ------------------------------------------------------------
# 17. Report — overall
# ------------------------------------------------------------

print("\nOverall count-level metrics")
print("---------------------------")

display(
    geometry_overall_metrics_7db_df[
        [
            "variant",
            "description",

            "total_GT",
            "total_prediction",

            "mean_GT",
            "mean_prediction",

            "MAE",
            "RMSE",
            "mean_signed_error",

            "under_images",
            "exact_images",
            "over_images",

            "delta_MAE_vs_G0",
            "delta_RMSE_vs_G0",
        ]
    ]
)


# ------------------------------------------------------------
# 18. Report — image-level comparison with G0
# ------------------------------------------------------------

print("\nPer-image absolute-error comparison versus G0")
print("---------------------------------------------")

display(
    geometry_vs_G0_comparison_7db_df
)


# ------------------------------------------------------------
# 19. Report — MAE by density stratum
# ------------------------------------------------------------

print("\nMAE by density stratum")
print("----------------------")

display(
    stratum_MAE_wide_7db_df
)


# ------------------------------------------------------------
# 20. Report — RMSE by density stratum
# ------------------------------------------------------------

print("\nRMSE by density stratum")
print("-----------------------")

display(
    stratum_RMSE_wide_7db_df
)


# ------------------------------------------------------------
# 21. Report — signed error by density stratum
# ------------------------------------------------------------

print("\nMean signed error by density stratum")
print("------------------------------------")

display(
    stratum_signed_wide_7db_df
)


# ------------------------------------------------------------
# 22. Report — improve/tie/worsen by stratum
# ------------------------------------------------------------

print("\nPer-stratum comparison versus G0")
print("--------------------------------")

display(
    geometry_stratum_vs_G0_7db_df
)


# ------------------------------------------------------------
# 23. Report — largest G3 improvements
#
# G3 is the more conservative direct-fragment variant and is
# therefore especially useful to inspect.
# ------------------------------------------------------------

print("\nG3 — largest improvements versus G0")
print("-----------------------------------")

display(
    geometry_extreme_cases_7db_df[
        (
            geometry_extreme_cases_7db_df[
                "variant"
            ]
            ==
            "G3"
        )
        &
        (
            geometry_extreme_cases_7db_df[
                "case_type"
            ]
            ==
            "largest_improvement"
        )
    ]
)


print("\nG3 — largest worsenings versus G0")
print("---------------------------------")

display(
    geometry_extreme_cases_7db_df[
        (
            geometry_extreme_cases_7db_df[
                "variant"
            ]
            ==
            "G3"
        )
        &
        (
            geometry_extreme_cases_7db_df[
                "case_type"
            ]
            ==
            "largest_worsening"
        )
    ]
)


# ------------------------------------------------------------
# 24. Consistency
# ------------------------------------------------------------

print("\nGlobal consistency")
print("------------------")

print(
    "Images evaluated            :",
    len(
        geometry_count_eval_7db_df
    ),
)

print(
    "Total GT count              :",
    int(
        geometry_count_eval_7db_df[
            "gt_count"
        ]
        .sum()
    ),
)

print(
    "G0 total prediction         :",
    int(
        geometry_count_eval_7db_df[
            "G0_geometry_count"
        ]
        .sum()
    ),
)

print(
    "G1 total prediction         :",
    int(
        geometry_count_eval_7db_df[
            "G1_geometry_count"
        ]
        .sum()
    ),
)

print(
    "G2 total prediction         :",
    int(
        geometry_count_eval_7db_df[
            "G2_geometry_count"
        ]
        .sum()
    ),
)

print(
    "G3 total prediction         :",
    int(
        geometry_count_eval_7db_df[
            "G3_geometry_count"
        ]
        .sum()
    ),
)


print("\nDensity strata")

for stratum in (
    STRATUM_ORDER_7DB
):

    print(
        f"  {stratum:<10s}:",
        int(
            (
                geometry_count_eval_7db_df[
                    "density_stratum"
                ]
                ==
                stratum
            )
            .sum()
        ),
    )


# ------------------------------------------------------------
# 25. Protocol report
# ------------------------------------------------------------

print("\nGeometry variants reused frozen     : YES")
print("New geometry threshold tuning       : NO")
print("GT used as inference input          : NO")
print("GT used for evaluation              : YES — dev100 only")

print("\nDirect fragments treated as count   : YES — stress-test ablation")
print("Claim fragment == true instance     : NO")

print("\nB0 modified                         : NO")
print("B2 modified                         : NO")
print("Semantic evidence regenerated       : NO")
print("Marigold regenerated                : NO")
print("Density routing used                : NO")

print("\nMAE/RMSE evaluated                  : YES")
print("S1-S6 evaluated                     : YES")
print("Test set accessed                   : NO")

print("\nStep 7D Part B: PASS")

STEP 7D PART B — COUNT-LEVEL EVALUATION OF FROZEN GEOMETRY VARIANTS

Overall count-level metrics
---------------------------


,variant,description,total_GT,total_prediction,mean_GT,mean_prediction,MAE,RMSE,mean_signed_error,under_images,exact_images,over_images,delta_MAE_vs_G0,delta_RMSE_vs_G0
0,G0,Original A3c components,10045,2554,100.45,25.54,79.67,263.541363,-74.91,68,5,27,0.00,0.000000
1,G1,All raw Marigold fragments,10045,8377,100.45,83.77,85.56,241.916391,-16.68,25,1,74,5.89,-21.624972
2,G2,Anchors + parent satellites >= 0.001,10045,6065,100.45,60.65,71.70,240.928620,-39.80,31,2,67,-7.97,-22.612743
3,G3,Anchors + parent satellites >= 0.0025,10045,5001,100.45,50.01,72.26,246.961576,-50.44,39,1,60,-7.41,-16.579787



Per-image absolute-error comparison versus G0
---------------------------------------------


,variant,improves_images,ties_images,worsens_images,median_abs_error_delta_vs_G0,mean_abs_error_delta_vs_G0
0,G1,36,0,64,9.5,5.89
1,G2,48,0,52,3.5,-7.97
2,G3,52,0,48,-1.0,-7.41



MAE by density stratum
----------------------


...


G3 — largest improvements versus G0
-----------------------------------


,dataset_index,image_id,category,density_stratum,gt_count,G0_geometry_count,G1_geometry_count,G0_abs_error,G1_abs_error,G1_abs_error_delta_vs_G0,num_split_candidates,variant,case_type,G2_geometry_count,G2_abs_error,G2_abs_error_delta_vs_G0,G3_geometry_count,G3_abs_error,G3_abs_error_delta_vs_G0
40,253,975.jpg,birds,S6_95_100,718,107,NaN,611,NaN,NaN,27,G3,largest_improvement,NaN,NaN,NaN,476.0,242.0,-369.0
41,1192,6969.jpg,birds,S6_95_100,949,53,NaN,896,NaN,NaN,14,G3,largest_improvement,NaN,NaN,NaN,231.0,718.0,-178.0
42,1187,6931.jpg,birds,S6_95_100,355,97,NaN,258,NaN,NaN,25,G3,largest_improvement,NaN,NaN,NaN,219.0,136.0,-122.0
43,1257,7436.jpg,pills,S6_95_100,283,98,NaN,185,NaN,NaN,25,G3,largest_improvement,NaN,NaN,NaN,161.0,122.0,-63.0
44,184,840.jpg,books,S6_95_100,637,55,NaN,582,NaN,NaN,14,G3,largest_improvement,NaN,NaN,NaN,104.0,533.0,-49.0
45,1256,7013.jpg,pills,S5_80_95,139,84,NaN,55,NaN,NaN,22,G3,largest_improvement,NaN,NaN,NaN,131.0,8.0,-47.0
46,177,832.jpg,bottle caps,S5_80_95,116,66,NaN,50,NaN,NaN,17,G3,largest_improvement,NaN,NaN,NaN,111.0,5.0,-45.0
47,176,831.jpg,bottle caps,S5_80_95,117,57,NaN,60,NaN,NaN,15,G3,largest_improvement,NaN,NaN,NaN,92.0,25.0,-35.0
48,183,838.jpg,books,S5_80_95,113,42,NaN,71,NaN,NaN,11,G3,largest_improvement,NaN,NaN,NaN,77.0,36.0,-35.0
49,143,764.jpg,birds,S5_80_95,85,50,NaN,35,NaN,NaN,13,G3,largest_improvement,NaN,NaN,NaN,86.0,1.0,-34.0



G3 — largest worsenings versus G0
---------------------------------


,dataset_index,image_id,category,density_stratum,gt_count,G0_geometry_count,G1_geometry_count,G0_abs_error,G1_abs_error,G1_abs_error_delta_vs_G0,num_split_candidates,variant,case_type,G2_geometry_count,G2_abs_error,G2_abs_error_delta_vs_G0,G3_geometry_count,G3_abs_error,G3_abs_error_delta_vs_G0
50,227,933.jpg,birds,S4_60_80,67,51,NaN,16,NaN,NaN,13,G3,largest_worsening,NaN,NaN,NaN,120.0,53.0,37.0
51,148,769.jpg,books,S2_20_40,13,30,NaN,17,NaN,NaN,8,G3,largest_worsening,NaN,NaN,NaN,67.0,54.0,37.0
52,1183,7305.jpg,birds,S3_40_60,31,31,NaN,0,NaN,NaN,8,G3,largest_worsening,NaN,NaN,NaN,60.0,29.0,29.0
53,822,5078.jpg,chicken wings,S2_20_40,20,33,NaN,13,NaN,NaN,9,G3,largest_worsening,NaN,NaN,NaN,60.0,40.0,27.0
54,231,941.jpg,birds,S2_20_40,13,12,NaN,1,NaN,NaN,4,G3,largest_worsening,NaN,NaN,NaN,40.0,27.0,26.0
55,1169,7421.jpg,horses,S1_0_20,10,24,NaN,14,NaN,NaN,7,G3,largest_worsening,NaN,NaN,NaN,50.0,40.0,26.0
56,331,1976.jpg,bottle caps,S3_40_60,32,35,NaN,3,NaN,NaN,9,G3,largest_worsening,NaN,NaN,NaN,60.0,28.0,25.0
57,835,5091.jpg,chicken wings,S2_20_40,15,44,NaN,29,NaN,NaN,12,G3,largest_worsening,NaN,NaN,NaN,69.0,54.0,25.0
58,1034,6096.jpg,chicken wings,S2_20_40,18,37,NaN,19,NaN,NaN,9,G3,largest_worsening,NaN,NaN,NaN,61.0,43.0,24.0
59,564,3777.jpg,donuts tray,S2_20_40,14,20,NaN,6,NaN,NaN,6,G3,largest_worsening,NaN,NaN,NaN,43.0,29.0,23.0



Global consistency
------------------
Images evaluated            : 100
Total GT count              : 10045
G0 total prediction         : 2554
G1 total prediction         : 8377
G2 total prediction         : 6065
G3 total prediction         : 5001

Density strata
  S1_0_20   : 20
  S2_20_40  : 20
  S3_40_60  : 20
  S4_60_80  : 20
  S5_80_95  : 10
  S6_95_100 : 10

Geometry variants reused frozen     : YES
New geometry threshold tuning       : NO
GT used as inference input          : NO
GT used for evaluation              : YES — dev100 only

Direct fragments treated as count   : YES — stress-test ablation
Claim fragment == true instance     : NO

B0 modified                         : NO
B2 modified                         : NO
Semantic evidence regenerated       : NO
Marigold regenerated                : NO
Density routing used                : NO

MAE/RMSE evaluated                  : YES
S1-S6 evaluated                     : YES
Test set accessed                   : NO

Step 7D Part

### **Finding 12:**

Direct counting of Marigold-refined semantic fragments provides useful corrective information but is not reliable as a universal standalone counting rule.

Raw splitting (G1) substantially reduces the severe undercount bias of unrefined A3c components but causes excessive over-fragmentation and worsens MAE.

Size-filtered variants improve the overall count, with G2 reducing MAE from 79.67 to 71.70 and RMSE from 263.54 to 240.93.

The benefit is strongly regime-dependent: geometry consistently improves the high-density S5–S6 subsets but degrades sparse S1–S3 images.

This indicates that Marigold-derived multiplicity is valuable primarily when semantic regions represent merged-object structure, rather than as a global replacement for the semantic component count.

In [36]:
# ============================================================
# Step 7D Part C — Inference-available selective-geometry
#                  diagnostic
# ============================================================
#
# Purpose
# -------
# Step 7D Part B showed that geometry:
#
#   - helps many difficult / strongly undercounted images
#   - hurts many easier images
#
# We now ask:
#
#   Can the images that BENEFIT from geometry be identified
#   using quantities available at inference time?
#
#
# IMPORTANT
# ---------
# All structural features are constructed BEFORE GT labels
# are attached.
#
# Inference-available features include:
#
#   - number of original A3c components
#   - number / fraction of Step-7A candidates
#   - total and candidate A3c foreground area
#   - candidate-area fraction
#   - raw geometry count increment
#   - filtered geometry count increment
#   - geometry inflation ratios
#   - fraction of raw geometric increment retained after
#     filtering
#
#
# GT is then used ONLY to determine:
#
#   G2 improves G0?  YES / NO
#   G3 improves G0?  YES / NO
#
#
# This step DOES NOT:
#
#   - choose a routing threshold
#   - switch between G0/G2/G3
#   - modify B0
#   - modify B2
#   - use GT density strata as an inference feature
#   - tune geometry thresholds
#
# ============================================================

import numpy as np
import pandas as pd

from scipy.stats import spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline


print("=" * 120)
print("STEP 7D PART C — INFERENCE-AVAILABLE SELECTIVE-GEOMETRY DIAGNOSTIC")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "geometry_count_eval_7db_df" in globals(), (
    "geometry_count_eval_7db_df unavailable. "
    "Run Step 7D Part B first."
)

assert "image_geometry_counts_7d_df" in globals(), (
    "image_geometry_counts_7d_df unavailable. "
    "Run Step 7D Part A first."
)

assert "component_geometry_counts_7d_df" in globals(), (
    "component_geometry_counts_7d_df unavailable. "
    "Run Step 7D Part A first."
)

assert "M1_TABLES_DIR" in globals()


assert len(
    geometry_count_eval_7db_df
) == 100


assert len(
    image_geometry_counts_7d_df
) == 100


assert len(
    component_geometry_counts_7d_df
) == 2554


# ------------------------------------------------------------
# 2. Construct inference-only component-area statistics
#
# NO GT columns are referenced here.
# ------------------------------------------------------------

component_structure_7dc_df = (
    component_geometry_counts_7d_df[
        [
            "dataset_index",
            "original_component_id",
            "original_component_area",
            "split_candidate",
        ]
    ]
    .copy()
)


# ------------------------------------------------------------
# Per-image total A3c area
# ------------------------------------------------------------

image_area_total_7dc_df = (
    component_structure_7dc_df
    .groupby(
        "dataset_index",
        as_index=False,
    )
    .agg(
        total_A3c_area=(
            "original_component_area",
            "sum",
        ),

        median_A3c_component_area=(
            "original_component_area",
            "median",
        ),

        max_A3c_component_area=(
            "original_component_area",
            "max",
        ),
    )
)


# ------------------------------------------------------------
# Candidate-only area structure
# ------------------------------------------------------------

candidate_component_structure_7dc_df = (
    component_structure_7dc_df[
        component_structure_7dc_df[
            "split_candidate"
        ]
    ]
    .copy()
)


image_candidate_area_7dc_df = (
    candidate_component_structure_7dc_df
    .groupby(
        "dataset_index",
        as_index=False,
    )
    .agg(
        candidate_area_sum=(
            "original_component_area",
            "sum",
        ),

        candidate_area_mean=(
            "original_component_area",
            "mean",
        ),

        candidate_area_median=(
            "original_component_area",
            "median",
        ),

        candidate_area_max=(
            "original_component_area",
            "max",
        ),
    )
)


# ------------------------------------------------------------
# 3. Start inference-only image feature table
# ------------------------------------------------------------

inference_features_7dc_df = (
    image_geometry_counts_7d_df[
        [
            "dataset_index",
            "original_A3c_components",
            "num_split_candidates",

            "G0_geometry_count",
            "G1_geometry_count",
            "G2_geometry_count",
            "G3_geometry_count",

            "G1_increment_vs_G0",
            "G2_increment_vs_G0",
            "G3_increment_vs_G0",
        ]
    ]
    .merge(
        image_area_total_7dc_df,
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
    .merge(
        image_candidate_area_7dc_df,
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    inference_features_7dc_df
) == 100


assert (
    inference_features_7dc_df
    .notna()
    .all()
    .all()
)


# ------------------------------------------------------------
# 4. Derived inference-only features
# ------------------------------------------------------------

inference_features_7dc_df[
    "candidate_fraction"
] = (
    inference_features_7dc_df[
        "num_split_candidates"
    ]
    /
    inference_features_7dc_df[
        "original_A3c_components"
    ]
)


inference_features_7dc_df[
    "candidate_area_fraction"
] = (
    inference_features_7dc_df[
        "candidate_area_sum"
    ]
    /
    inference_features_7dc_df[
        "total_A3c_area"
    ]
)


# Geometry inflation relative to original semantic count
inference_features_7dc_df[
    "G1_increment_per_G0"
] = (
    inference_features_7dc_df[
        "G1_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "G0_geometry_count"
    ]
)


inference_features_7dc_df[
    "G2_increment_per_G0"
] = (
    inference_features_7dc_df[
        "G2_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "G0_geometry_count"
    ]
)


inference_features_7dc_df[
    "G3_increment_per_G0"
] = (
    inference_features_7dc_df[
        "G3_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "G0_geometry_count"
    ]
)


# Increment per candidate
inference_features_7dc_df[
    "G1_increment_per_candidate"
] = (
    inference_features_7dc_df[
        "G1_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "num_split_candidates"
    ]
)


inference_features_7dc_df[
    "G2_increment_per_candidate"
] = (
    inference_features_7dc_df[
        "G2_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "num_split_candidates"
    ]
)


inference_features_7dc_df[
    "G3_increment_per_candidate"
] = (
    inference_features_7dc_df[
        "G3_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "num_split_candidates"
    ]
)


# How much of raw geometric fragmentation survives filtering?
#
# 1 = all raw increment survives
# 0 = none survives
inference_features_7dc_df[
    "G2_raw_increment_retention"
] = np.where(
    inference_features_7dc_df[
        "G1_increment_vs_G0"
    ]
    >
    0,

    inference_features_7dc_df[
        "G2_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "G1_increment_vs_G0"
    ],

    0.0,
)


inference_features_7dc_df[
    "G3_raw_increment_retention"
] = np.where(
    inference_features_7dc_df[
        "G1_increment_vs_G0"
    ]
    >
    0,

    inference_features_7dc_df[
        "G3_increment_vs_G0"
    ]
    /
    inference_features_7dc_df[
        "G1_increment_vs_G0"
    ],

    0.0,
)


# ------------------------------------------------------------
# 5. Explicitly define inference feature set
#
# GT count and density_stratum are deliberately excluded.
# ------------------------------------------------------------

INFERENCE_FEATURES_7DC = [
    "original_A3c_components",
    "num_split_candidates",

    "candidate_fraction",

    "total_A3c_area",
    "median_A3c_component_area",
    "max_A3c_component_area",

    "candidate_area_sum",
    "candidate_area_mean",
    "candidate_area_median",
    "candidate_area_max",
    "candidate_area_fraction",

    "G1_increment_vs_G0",
    "G2_increment_vs_G0",
    "G3_increment_vs_G0",

    "G1_increment_per_G0",
    "G2_increment_per_G0",
    "G3_increment_per_G0",

    "G1_increment_per_candidate",
    "G2_increment_per_candidate",
    "G3_increment_per_candidate",

    "G2_raw_increment_retention",
    "G3_raw_increment_retention",
]


assert "gt_count" not in INFERENCE_FEATURES_7DC
assert "density_stratum" not in INFERENCE_FEATURES_7DC


# ------------------------------------------------------------
# 6. Freeze feature table BEFORE attaching GT evaluation
# ------------------------------------------------------------

inference_feature_snapshot_7dc_df = (
    inference_features_7dc_df[
        [
            "dataset_index",
            *
            INFERENCE_FEATURES_7DC,
        ]
    ]
    .copy()
)


assert (
    inference_feature_snapshot_7dc_df
    .notna()
    .all()
    .all()
)


print("\nInference-only feature construction")
print("-----------------------------------")

print(
    "Images:",
    len(
        inference_feature_snapshot_7dc_df
    ),
)

print(
    "Features:",
    len(
        INFERENCE_FEATURES_7DC
    ),
)

print(
    "GT count used as feature       : NO"
)

print(
    "Density stratum used as feature: NO"
)


# ------------------------------------------------------------
# 7. Attach GT-derived improvement labels AFTER feature freeze
#
# Label:
#
#   1 = geometry reduces absolute count error relative to G0
#   0 = geometry does not improve G0
#
# Part B had no ties, but use <= robustly.
# ------------------------------------------------------------

evaluation_labels_7dc_df = (
    geometry_count_eval_7db_df[
        [
            "dataset_index",

            "G0_abs_error",

            "G2_abs_error",
            "G3_abs_error",

            "G2_abs_error_delta_vs_G0",
            "G3_abs_error_delta_vs_G0",
        ]
    ]
    .copy()
)


evaluation_labels_7dc_df[
    "G2_improves"
] = (
    evaluation_labels_7dc_df[
        "G2_abs_error_delta_vs_G0"
    ]
    <
    0
).astype(int)


evaluation_labels_7dc_df[
    "G3_improves"
] = (
    evaluation_labels_7dc_df[
        "G3_abs_error_delta_vs_G0"
    ]
    <
    0
).astype(int)


selective_geometry_diag_7dc_df = (
    inference_feature_snapshot_7dc_df
    .merge(
        evaluation_labels_7dc_df,
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    selective_geometry_diag_7dc_df
) == 100


print("\nGT-derived development labels")
print("-----------------------------")

print(
    "G2 improves images:",
    int(
        selective_geometry_diag_7dc_df[
            "G2_improves"
        ]
        .sum()
    ),
)

print(
    "G2 non-improves:",
    int(
        (
            1
            -
            selective_geometry_diag_7dc_df[
                "G2_improves"
            ]
        )
        .sum()
    ),
)

print(
    "G3 improves images:",
    int(
        selective_geometry_diag_7dc_df[
            "G3_improves"
        ]
        .sum()
    ),
)

print(
    "G3 non-improves:",
    int(
        (
            1
            -
            selective_geometry_diag_7dc_df[
                "G3_improves"
            ]
        )
        .sum()
    ),
)


# ------------------------------------------------------------
# 8. Univariate discrimination
#
# AUC:
#
#   0.5 = no discrimination
#
# For interpretability we report:
#
#   raw_AUC
#
# and:
#
#   oriented_AUC = max(AUC, 1-AUC)
#
# Direction:
#
#   HIGHER -> improvement
#   LOWER  -> improvement
#
# No threshold is selected.
# ------------------------------------------------------------

univariate_rows_7dc = []


for target_name in [
    "G2",
    "G3",
]:

    y = (
        selective_geometry_diag_7dc_df[
            f"{target_name}_improves"
        ]
        .to_numpy(
            dtype=int
        )
    )


    delta = (
        selective_geometry_diag_7dc_df[
            f"{target_name}_abs_error_delta_vs_G0"
        ]
        .to_numpy(
            dtype=float
        )
    )


    for feature in (
        INFERENCE_FEATURES_7DC
    ):

        x = (
            selective_geometry_diag_7dc_df[
                feature
            ]
            .to_numpy(
                dtype=float
            )
        )


        raw_auc = float(
            roc_auc_score(
                y,
                x,
            )
        )


        if raw_auc >= 0.5:

            oriented_auc = raw_auc

            direction = (
                "HIGHER -> geometry improvement"
            )

        else:

            oriented_auc = (
                1.0
                -
                raw_auc
            )

            direction = (
                "LOWER -> geometry improvement"
            )


        rho, p_value = (
            spearmanr(
                x,
                delta,
            )
        )


        univariate_rows_7dc.append(
            {
                "target_variant":
                    target_name,

                "feature":
                    feature,

                "raw_AUC":
                    raw_auc,

                "oriented_AUC":
                    float(
                        oriented_auc
                    ),

                "direction":
                    direction,

                # delta < 0 is better.
                #
                # Therefore:
                #
                # negative rho:
                #   larger feature -> more improvement
                #
                # positive rho:
                #   larger feature -> more worsening
                "spearman_rho_vs_abs_error_delta":
                    float(
                        rho
                    ),

                "spearman_p":
                    float(
                        p_value
                    ),
            }
        )


univariate_feature_diag_7dc_df = (
    pd.DataFrame(
        univariate_rows_7dc
    )
    .sort_values(
        [
            "target_variant",
            "oriented_AUC",
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 9. Descriptive feature comparison:
#
# improve vs non-improve
# ------------------------------------------------------------

feature_group_rows_7dc = []


for target_name in [
    "G2",
    "G3",
]:

    for feature in (
        INFERENCE_FEATURES_7DC
    ):

        improve_values = (
            selective_geometry_diag_7dc_df.loc[
                selective_geometry_diag_7dc_df[
                    f"{target_name}_improves"
                ]
                ==
                1,
                feature,
            ]
        )


        nonimprove_values = (
            selective_geometry_diag_7dc_df.loc[
                selective_geometry_diag_7dc_df[
                    f"{target_name}_improves"
                ]
                ==
                0,
                feature,
            ]
        )


        feature_group_rows_7dc.append(
            {
                "target_variant":
                    target_name,

                "feature":
                    feature,

                "improve_median":
                    float(
                        improve_values.median()
                    ),

                "nonimprove_median":
                    float(
                        nonimprove_values.median()
                    ),

                "improve_mean":
                    float(
                        improve_values.mean()
                    ),

                "nonimprove_mean":
                    float(
                        nonimprove_values.mean()
                    ),
            }
        )


feature_group_summary_7dc_df = (
    pd.DataFrame(
        feature_group_rows_7dc
    )
)


# ------------------------------------------------------------
# 10. Compact multivariate feature set
#
# Avoid feeding all highly redundant features into the
# diagnostic model.
#
# These six quantities represent distinct aspects:
#
#   semantic-component quantity
#   candidate prevalence
#   candidate area dominance
#   raw fragmentation severity
#   filtered geometry inflation
#   survival of raw fragments after filtering
#
# ------------------------------------------------------------

MULTIVARIATE_FEATURES_G2_7DC = [
    "original_A3c_components",
    "candidate_fraction",
    "candidate_area_fraction",
    "G1_increment_per_G0",
    "G2_increment_per_G0",
    "G2_raw_increment_retention",
]


MULTIVARIATE_FEATURES_G3_7DC = [
    "original_A3c_components",
    "candidate_fraction",
    "candidate_area_fraction",
    "G1_increment_per_G0",
    "G3_increment_per_G0",
    "G3_raw_increment_retention",
]


# ------------------------------------------------------------
# 11. 5-fold out-of-fold logistic diagnostic
#
# IMPORTANT:
#
# This is NOT a frozen production router.
#
# It only tests whether the inference features jointly contain
# predictive information about when geometry helps.
# ------------------------------------------------------------

multivariate_rows_7dc = []

oof_prediction_rows_7dc = []


for target_name, feature_names in [
    (
        "G2",
        MULTIVARIATE_FEATURES_G2_7DC,
    ),

    (
        "G3",
        MULTIVARIATE_FEATURES_G3_7DC,
    ),
]:

    X = (
        selective_geometry_diag_7dc_df[
            feature_names
        ]
        .to_numpy(
            dtype=float
        )
    )


    y = (
        selective_geometry_diag_7dc_df[
            f"{target_name}_improves"
        ]
        .to_numpy(
            dtype=int
        )
    )


    cv = StratifiedKFold(
        n_splits=5,
        shuffle=True,
        random_state=0,
    )


    model = Pipeline(
        [
            (
                "scale",
                StandardScaler(),
            ),

            (
                "logreg",
                LogisticRegression(
                    max_iter=5000,
                    random_state=0,
                    class_weight="balanced",
                ),
            ),
        ]
    )


    oof_prob = (
        cross_val_predict(
            model,
            X,
            y,
            cv=cv,
            method="predict_proba",
        )
        [
            :,
            1
        ]
    )


    oof_auc = float(
        roc_auc_score(
            y,
            oof_prob,
        )
    )


    multivariate_rows_7dc.append(
        {
            "target_variant":
                target_name,

            "num_features":
                len(
                    feature_names
                ),

            "features":
                " | ".join(
                    feature_names
                ),

            "OOF_ROC_AUC":
                oof_auc,

            "positive_images":
                int(
                    y.sum()
                ),

            "negative_images":
                int(
                    len(y)
                    -
                    y.sum()
                ),
        }
    )


    for dataset_index, true_label, prob in zip(
        selective_geometry_diag_7dc_df[
            "dataset_index"
        ],
        y,
        oof_prob,
    ):

        oof_prediction_rows_7dc.append(
            {
                "dataset_index":
                    int(
                        dataset_index
                    ),

                "target_variant":
                    target_name,

                "true_improves":
                    int(
                        true_label
                    ),

                "OOF_probability_improves":
                    float(
                        prob
                    ),
            }
        )


multivariate_diag_7dc_df = (
    pd.DataFrame(
        multivariate_rows_7dc
    )
)


multivariate_oof_7dc_df = (
    pd.DataFrame(
        oof_prediction_rows_7dc
    )
)


# ------------------------------------------------------------
# 12. Quartile diagnostic for strongest inference features
#
# We do NOT choose thresholds.
#
# For each variant:
# take top 3 univariate features by oriented AUC and inspect
# improvement rate across empirical quartiles.
# ------------------------------------------------------------

quartile_rows_7dc = []


for target_name in [
    "G2",
    "G3",
]:

    top_features = (
        univariate_feature_diag_7dc_df[
            univariate_feature_diag_7dc_df[
                "target_variant"
            ]
            ==
            target_name
        ]
        .head(
            3
        )
        [
            "feature"
        ]
        .tolist()
    )


    for feature in (
        top_features
    ):

        temp_df = (
            selective_geometry_diag_7dc_df[
                [
                    "dataset_index",
                    feature,
                    f"{target_name}_improves",
                    f"{target_name}_abs_error_delta_vs_G0",
                ]
            ]
            .copy()
        )


        temp_df[
            "feature_quartile"
        ] = pd.qcut(
            temp_df[
                feature
            ],
            q=4,
            labels=[
                "Q1_lowest",
                "Q2",
                "Q3",
                "Q4_highest",
            ],
            duplicates="drop",
        )


        quartile_summary = (
            temp_df
            .groupby(
                "feature_quartile",
                observed=True,
                as_index=False,
            )
            .agg(
                num_images=(
                    "dataset_index",
                    "count",
                ),

                feature_median=(
                    feature,
                    "median",
                ),

                improvement_rate=(
                    f"{target_name}_improves",
                    "mean",
                ),

                mean_abs_error_delta_vs_G0=(
                    f"{target_name}_abs_error_delta_vs_G0",
                    "mean",
                ),

                median_abs_error_delta_vs_G0=(
                    f"{target_name}_abs_error_delta_vs_G0",
                    "median",
                ),
            )
        )


        quartile_summary[
            "target_variant"
        ] = (
            target_name
        )


        quartile_summary[
            "feature"
        ] = (
            feature
        )


        quartile_rows_7dc.append(
            quartile_summary
        )


top_feature_quartiles_7dc_df = (
    pd.concat(
        quartile_rows_7dc,
        ignore_index=True,
    )
)


# ------------------------------------------------------------
# 13. Save outputs
# ------------------------------------------------------------

M1_7D_C_FEATURE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_inference_only_image_features.csv"
)

M1_7D_C_DIAGNOSTIC_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_selective_geometry_diagnostic.csv"
)

M1_7D_C_UNIVARIATE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_univariate_feature_diagnostic.csv"
)

M1_7D_C_GROUP_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_feature_improve_vs_nonimprove.csv"
)

M1_7D_C_MULTIVARIATE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_multivariate_OOF_diagnostic.csv"
)

M1_7D_C_OOF_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_multivariate_OOF_predictions.csv"
)

M1_7D_C_QUARTILE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_C_top_feature_quartile_diagnostic.csv"
)


inference_feature_snapshot_7dc_df.to_csv(
    M1_7D_C_FEATURE_FILE,
    index=False,
)

selective_geometry_diag_7dc_df.to_csv(
    M1_7D_C_DIAGNOSTIC_FILE,
    index=False,
)

univariate_feature_diag_7dc_df.to_csv(
    M1_7D_C_UNIVARIATE_FILE,
    index=False,
)

feature_group_summary_7dc_df.to_csv(
    M1_7D_C_GROUP_FILE,
    index=False,
)

multivariate_diag_7dc_df.to_csv(
    M1_7D_C_MULTIVARIATE_FILE,
    index=False,
)

multivariate_oof_7dc_df.to_csv(
    M1_7D_C_OOF_FILE,
    index=False,
)

top_feature_quartiles_7dc_df.to_csv(
    M1_7D_C_QUARTILE_FILE,
    index=False,
)


# ------------------------------------------------------------
# 14. Report — strongest univariate G2 features
# ------------------------------------------------------------

print("\nG2 — strongest inference-available univariate features")
print("------------------------------------------------------")

display(
    univariate_feature_diag_7dc_df[
        univariate_feature_diag_7dc_df[
            "target_variant"
        ]
        ==
        "G2"
    ]
    .head(
        12
    )
)


# ------------------------------------------------------------
# 15. Report — strongest univariate G3 features
# ------------------------------------------------------------

print("\nG3 — strongest inference-available univariate features")
print("------------------------------------------------------")

display(
    univariate_feature_diag_7dc_df[
        univariate_feature_diag_7dc_df[
            "target_variant"
        ]
        ==
        "G3"
    ]
    .head(
        12
    )
)


# ------------------------------------------------------------
# 16. Report — multivariate out-of-fold diagnostic
# ------------------------------------------------------------

print("\nMultivariate 5-fold out-of-fold diagnostic")
print("------------------------------------------")

display(
    multivariate_diag_7dc_df
)


# ------------------------------------------------------------
# 17. Report — top-feature quartiles
# ------------------------------------------------------------

print("\nTop-feature quartile diagnostic")
print("-------------------------------")

display(
    top_feature_quartiles_7dc_df
)


# ------------------------------------------------------------
# 18. Compact structural comparison
#
# Show medians of a few especially interpretable features
# for geometry-helpful vs geometry-harmful images.
# ------------------------------------------------------------

COMPACT_FEATURES_7DC = [
    "original_A3c_components",
    "num_split_candidates",
    "candidate_fraction",
    "candidate_area_fraction",
    "G1_increment_per_G0",
    "G2_increment_per_G0",
    "G3_increment_per_G0",
    "G2_raw_increment_retention",
    "G3_raw_increment_retention",
]


compact_rows_7dc = []


for target_name in [
    "G2",
    "G3",
]:

    for feature in (
        COMPACT_FEATURES_7DC
    ):

        improve_mask = (
            selective_geometry_diag_7dc_df[
                f"{target_name}_improves"
            ]
            ==
            1
        )


        compact_rows_7dc.append(
            {
                "target_variant":
                    target_name,

                "feature":
                    feature,

                "median_if_improves":
                    float(
                        selective_geometry_diag_7dc_df.loc[
                            improve_mask,
                            feature,
                        ]
                        .median()
                    ),

                "median_if_not_improves":
                    float(
                        selective_geometry_diag_7dc_df.loc[
                            ~improve_mask,
                            feature,
                        ]
                        .median()
                    ),
            }
        )


compact_structural_comparison_7dc_df = (
    pd.DataFrame(
        compact_rows_7dc
    )
)


print("\nCompact improve vs non-improve structural comparison")
print("-----------------------------------------------------")

display(
    compact_structural_comparison_7dc_df
)


# ------------------------------------------------------------
# 19. Protocol report
# ------------------------------------------------------------

print("\nInference features constructed before GT labels : YES")
print("GT count used as inference feature              : NO")
print("Density stratum used as inference feature       : NO")
print("Semantic category used as inference feature     : NO")

print("\nGT used for post-hoc improvement labels         : YES — dev100")
print("Univariate diagnostic performed                 : YES")
print("5-fold OOF multivariate diagnostic performed    : YES")

print("\nRouting rule selected                            : NO")
print("Routing threshold selected                       : NO")
print("Geometry threshold changed                       : NO")
print("G2/G3 count actually switched                   : NO")

print("\nB0 modified                                      : NO")
print("B2 modified                                      : NO")
print("Test set accessed                                : NO")

print("\nStep 7D Part C: PASS")

STEP 7D PART C — INFERENCE-AVAILABLE SELECTIVE-GEOMETRY DIAGNOSTIC

Inference-only feature construction
-----------------------------------
Images: 100
Features: 22
GT count used as feature       : NO
Density stratum used as feature: NO

GT-derived development labels
-----------------------------
G2 improves images: 48
G2 non-improves: 52
G3 improves images: 52
G3 non-improves: 48

G2 — strongest inference-available univariate features
------------------------------------------------------


,target_variant,feature,raw_AUC,oriented_AUC,direction,spearman_rho_vs_abs_error_delta,spearman_p
0,G2,candidate_area_fraction,0.691106,0.691106,HIGHER -> geometry improvement,-0.156178,0.120738
1,G2,max_A3c_component_area,0.676683,0.676683,HIGHER -> geometry improvement,-0.161404,0.108651
2,G2,candidate_area_max,0.676683,0.676683,HIGHER -> geometry improvement,-0.161404,0.108651
3,G2,candidate_area_sum,0.664263,0.664263,HIGHER -> geometry improvement,-0.204944,0.040812
4,G2,G1_increment_per_G0,0.608574,0.608574,HIGHER -> geometry improvement,-0.146664,0.145368
5,G2,G2_raw_increment_retention,0.407452,0.592548,LOWER -> geometry improvement,0.132565,0.188578
6,G2,candidate_area_mean,0.587740,0.587740,HIGHER -> geometry improvement,-0.060988,0.546659
7,G2,G2_increment_per_G0,0.587540,0.587540,HIGHER -> geometry improvement,-0.102093,0.312143
8,G2,G3_raw_increment_retention,0.412861,0.587139,LOWER -> geometry improvement,0.157790,0.116905
9,G2,total_A3c_area,0.586138,0.586138,HIGHER -> geometry improvement,-0.140030,0.164666



G3 — strongest inference-available univariate features
------------------------------------------------------


,target_variant,feature,raw_AUC,oriented_AUC,direction,spearman_rho_vs_abs_error_delta,spearman_p
22,G3,max_A3c_component_area,0.661058,0.661058,HIGHER -> geometry improvement,-0.110146,0.275302
23,G3,candidate_area_max,0.661058,0.661058,HIGHER -> geometry improvement,-0.110146,0.275302
24,G3,candidate_area_fraction,0.659856,0.659856,HIGHER -> geometry improvement,-0.086690,0.391102
25,G3,G1_increment_per_G0,0.642228,0.642228,HIGHER -> geometry improvement,-0.211613,0.034556
26,G3,G3_raw_increment_retention,0.363982,0.636018,LOWER -> geometry improvement,0.224275,0.024883
27,G3,candidate_area_sum,0.634215,0.634215,HIGHER -> geometry improvement,-0.133278,0.186191
28,G3,G2_increment_per_G0,0.626402,0.626402,HIGHER -> geometry improvement,-0.186897,0.062615
29,G3,G1_increment_per_candidate,0.615986,0.615986,HIGHER -> geometry improvement,-0.212817,0.033518
30,G3,G2_raw_increment_retention,0.392428,0.607572,LOWER -> geometry improvement,0.126965,0.208108
31,G3,G2_increment_per_candidate,0.598958,0.598958,HIGHER -> geometry improvement,-0.197665,0.048690



Multivariate 5-fold out-of-fold diagnostic
------------------------------------------


...


Top-feature quartile diagnostic
-------------------------------


,feature_quartile,num_images,feature_median,improvement_rate,mean_abs_error_delta_vs_G0,median_abs_error_delta_vs_G0,target_variant,feature
0,Q1_lowest,25,0.680344,0.36,1.20,5.0,G2,candidate_area_fraction
1,Q2,25,0.825632,0.36,-19.80,9.0,G2,candidate_area_fraction
2,Q3,25,0.910071,0.40,-0.68,8.0,G2,candidate_area_fraction
3,Q4_highest,25,0.976449,0.80,-12.60,-5.0,G2,candidate_area_fraction
4,Q1_lowest,25,3052.000000,0.40,-3.08,6.0,G2,max_A3c_component_area
5,Q2,25,6212.000000,0.24,-10.96,11.0,G2,max_A3c_component_area
6,Q3,25,9443.000000,0.48,0.72,3.0,G2,max_A3c_component_area
7,Q4_highest,25,17810.000000,0.80,-18.56,-7.0,G2,max_A3c_component_area
8,Q1_lowest,25,3052.000000,0.40,-3.08,6.0,G2,candidate_area_max
9,Q2,25,6212.000000,0.24,-10.96,11.0,G2,candidate_area_max



Compact improve vs non-improve structural comparison
-----------------------------------------------------


,target_variant,feature,median_if_improves,median_if_not_improves
0,G2,original_A3c_components,19.000000,22.500000
1,G2,num_split_candidates,5.000000,6.000000
2,G2,candidate_fraction,0.276820,0.275000
3,G2,candidate_area_fraction,0.917466,0.836130
4,G2,G1_increment_per_G0,1.720779,1.181818
5,G2,G2_increment_per_G0,1.082270,0.854895
6,G2,G3_increment_per_G0,0.833333,0.707143
7,G2,G2_raw_increment_retention,0.562060,0.730249
8,G2,G3_raw_increment_retention,0.486842,0.555212
9,G3,original_A3c_components,19.000000,21.500000



Inference features constructed before GT labels : YES
GT count used as inference feature              : NO
Density stratum used as inference feature       : NO
Semantic category used as inference feature     : NO

GT used for post-hoc improvement labels         : YES — dev100
Univariate diagnostic performed                 : YES
5-fold OOF multivariate diagnostic performed    : YES

Routing rule selected                            : NO
Routing threshold selected                       : NO
Geometry threshold changed                       : NO
G2/G3 count actually switched                   : NO

B0 modified                                      : NO
B2 modified                                      : NO
Test set accessed                                : NO

Step 7D Part C: PASS


### **Finding 13:**

The usefulness of Marigold refinement is partially predictable from inference-available semantic and geometric structure.

Candidate-area dominance is the strongest individual indicator, with candidate-area fraction achieving AUC 0.69 for G2 improvement, while large candidate components and raw geometry inflation provide additional signal.

Images in the highest candidate-area-fraction quartile show an 80% geometry-improvement rate for both G2 and G3.

A six-feature 5-fold out-of-fold logistic diagnostic achieves AUC 0.67 for G2 and 0.72 for G3, indicating that selective geometry activation may be feasible without using GT density or category.

However, the signal is moderate rather than decisive, so a final routing rule is not yet justified.

In [38]:
# ============================================================
# Step 7D Part D — Simple interpretable routing sensitivity
# ============================================================
#
# Purpose
# -------
# Test whether Marigold geometry can be activated selectively
# using simple INFERENCE-AVAILABLE rules.
#
#
# Rule families
# -------------
#
# A. Single-feature candidate-area gate
#
#       use geometry if:
#
#       candidate_area_fraction >= threshold
#
#       thresholds:
#           0.85
#           0.90
#           0.95
#
#
# B. Conservative two-feature AND gate
#
#       use geometry if:
#
#       candidate_area_fraction >= 0.90
#
#       AND
#
#       G1_increment_per_G0 >= threshold
#
#       inflation thresholds:
#           1.0
#           1.5
#           2.0
#
#
# Geometry choices tested:
#
#       G2
#       G3
#
#
# IMPORTANT
# ---------
#
# These are DEVELOPMENT-SET sensitivity experiments.
#
# The rules use ONLY inference-visible quantities.
#
# GT is used only AFTER routing decisions have been generated
# to evaluate MAE / RMSE / signed error.
#
#
# No:
#   - learned router
#   - density routing
#   - category routing
#   - GT-count routing
#   - new Marigold threshold tuning
#   - B0 modification
#   - B2 modification
#   - test-set access
#
# ============================================================

import numpy as np
import pandas as pd


print("=" * 120)
print("STEP 7D PART D — SIMPLE INTERPRETABLE ROUTING SENSITIVITY")
print("=" * 120)


# ------------------------------------------------------------
# 1. Guards
# ------------------------------------------------------------

assert "selective_geometry_diag_7dc_df" in globals(), (
    "selective_geometry_diag_7dc_df unavailable. "
    "Run Step 7D Part C first."
)

assert "geometry_count_eval_7db_df" in globals(), (
    "geometry_count_eval_7db_df unavailable. "
    "Run Step 7D Part B first."
)

assert "M1_TABLES_DIR" in globals()


assert len(
    selective_geometry_diag_7dc_df
) == 100


assert len(
    geometry_count_eval_7db_df
) == 100


# ------------------------------------------------------------
# 2. Frozen coarse routing grids
#
# These values are deliberately sparse.
#
# We are NOT performing fine-grained threshold optimisation.
# ------------------------------------------------------------

CANDIDATE_AREA_THRESHOLDS_7DD = [
    0.85,
    0.90,
    0.95,
]


RAW_INFLATION_THRESHOLDS_7DD = [
    1.0,
    1.5,
    2.0,
]


GEOMETRY_TARGETS_7DD = [
    "G2",
    "G3",
]


# ------------------------------------------------------------
# 3. Build routing table
#
# IMPORTANT DATA PROVENANCE
# -------------------------
#
# Part C's selective_geometry_diag_7dc_df was intentionally
# created from an inference-feature snapshot containing only:
#
#     dataset_index + INFERENCE_FEATURES_7DC
#
# Therefore the absolute frozen geometry-count columns
# G0/G2/G3 are NOT present in that dataframe.
#
# Correct sources:
#
#   Part C inference features:
#       selective_geometry_diag_7dc_df
#
#   Frozen Part-A geometry counts:
#       image_geometry_counts_7d_df
#
#   GT metadata for evaluation:
#       geometry_count_eval_7db_df
#
# ------------------------------------------------------------

required_routing_feature_cols_7dd = {
    "dataset_index",
    "candidate_area_fraction",
    "G1_increment_per_G0",
}

assert required_routing_feature_cols_7dd.issubset(
    set(
        selective_geometry_diag_7dc_df.columns
    )
), (
    "Required Part-C routing features missing.\n"
    f"Required: {required_routing_feature_cols_7dd}\n"
    f"Found: {list(selective_geometry_diag_7dc_df.columns)}"
)


required_geometry_count_cols_7dd = {
    "dataset_index",
    "G0_geometry_count",
    "G2_geometry_count",
    "G3_geometry_count",
}

assert required_geometry_count_cols_7dd.issubset(
    set(
        image_geometry_counts_7d_df.columns
    )
), (
    "Required frozen Part-A geometry counts missing.\n"
    f"Required: {required_geometry_count_cols_7dd}\n"
    f"Found: {list(image_geometry_counts_7d_df.columns)}"
)


# ------------------------------------------------------------
# 3A. Freeze inference-visible routing features
#
# Do NOT bring GT-derived improvement labels into the routing
# table.
# ------------------------------------------------------------

routing_features_7dd_df = (
    selective_geometry_diag_7dc_df[
        [
            "dataset_index",
            "candidate_area_fraction",
            "G1_increment_per_G0",
        ]
    ]
    .copy()
)


assert len(
    routing_features_7dd_df
) == 100


assert (
    routing_features_7dd_df[
        "dataset_index"
    ]
    .nunique()
    ==
    100
)


# ------------------------------------------------------------
# 3B. Attach frozen G0/G2/G3 count variants
#
# These are inference outputs established in Step 7D Part A.
# No GT is involved.
# ------------------------------------------------------------

routing_base_7dd_df = (
    routing_features_7dd_df
    .merge(
        image_geometry_counts_7d_df[
            [
                "dataset_index",
                "G0_geometry_count",
                "G2_geometry_count",
                "G3_geometry_count",
            ]
        ],
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        "dataset_index"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    routing_base_7dd_df
) == 100


assert (
    routing_base_7dd_df[
        [
            "candidate_area_fraction",
            "G1_increment_per_G0",
            "G0_geometry_count",
            "G2_geometry_count",
            "G3_geometry_count",
        ]
    ]
    .notna()
    .all()
    .all()
)


# Frozen structural identities
assert (
    routing_base_7dd_df[
        "G2_geometry_count"
    ]
    >=
    routing_base_7dd_df[
        "G3_geometry_count"
    ]
).all()


assert (
    routing_base_7dd_df[
        "G3_geometry_count"
    ]
    >=
    routing_base_7dd_df[
        "G0_geometry_count"
    ]
).all()


# ------------------------------------------------------------
# 3C. Attach GT metadata only AFTER the inference-time routing
#     quantities and frozen geometry variants are assembled.
# ------------------------------------------------------------

routing_eval_7dd_df = (
    routing_base_7dd_df
    .merge(
        geometry_count_eval_7db_df[
            [
                "dataset_index",
                "image_id",
                "category",
                "density_stratum",
                "gt_count",
            ]
        ],
        on="dataset_index",
        how="left",
        validate="one_to_one",
    )
)


assert len(
    routing_eval_7dd_df
) == 100


assert (
    routing_eval_7dd_df[
        [
            "image_id",
            "category",
            "density_stratum",
            "gt_count",
        ]
    ]
    .notna()
    .all()
    .all()
)


print("\nRouting-table provenance")
print("------------------------")
print(
    "Inference routing features : selective_geometry_diag_7dc_df"
)
print(
    "Frozen geometry counts     : image_geometry_counts_7d_df"
)
print(
    "GT evaluation metadata     : geometry_count_eval_7db_df"
)
print(
    "Images aligned             :",
    len(
        routing_eval_7dd_df
    ),
)

# ------------------------------------------------------------
# 4. Metric helper
# ------------------------------------------------------------

def evaluate_prediction_7dd(
    df,
    pred,
    policy_name,
    policy_family,
    geometry_variant,
    rule_description,
):

    gt = (
        df[
            "gt_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    pred = np.asarray(
        pred,
        dtype=float,
    )


    signed = (
        pred
        -
        gt
    )


    abs_error = np.abs(
        signed
    )


    squared_error = (
        signed
        ** 2
    )


    return {
        "policy_name":
            policy_name,

        "policy_family":
            policy_family,

        "geometry_variant":
            geometry_variant,

        "rule_description":
            rule_description,

        "num_geometry_images":
            int(
                np.sum(
                    pred
                    !=
                    df[
                        "G0_geometry_count"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )
            ),

        "geometry_activation_fraction":
            float(
                np.mean(
                    pred
                    !=
                    df[
                        "G0_geometry_count"
                    ]
                    .to_numpy(
                        dtype=float
                    )
                )
            ),

        "total_prediction":
            int(
                pred.sum()
            ),

        "mean_prediction":
            float(
                pred.mean()
            ),

        "MAE":
            float(
                abs_error.mean()
            ),

        "RMSE":
            float(
                np.sqrt(
                    squared_error.mean()
                )
            ),

        "mean_signed_error":
            float(
                signed.mean()
            ),

        "median_signed_error":
            float(
                np.median(
                    signed
                )
            ),

        "under_images":
            int(
                (
                    signed
                    <
                    0
                )
                .sum()
            ),

        "exact_images":
            int(
                (
                    signed
                    ==
                    0
                )
                .sum()
            ),

        "over_images":
            int(
                (
                    signed
                    >
                    0
                )
                .sum()
            ),
    }


# ------------------------------------------------------------
# 5. Reference policies
# ------------------------------------------------------------

policy_rows_7dd = []

policy_prediction_map_7dd = {}


# G0 reference
g0_pred_7dd = (
    routing_eval_7dd_df[
        "G0_geometry_count"
    ]
    .to_numpy(
        dtype=float
    )
)


policy_prediction_map_7dd[
    "G0_GLOBAL"
] = g0_pred_7dd


policy_rows_7dd.append(
    evaluate_prediction_7dd(
        df=
            routing_eval_7dd_df,

        pred=
            g0_pred_7dd,

        policy_name=
            "G0_GLOBAL",

        policy_family=
            "REFERENCE",

        geometry_variant=
            "G0",

        rule_description=
            "Always use G0",
    )
)


# Global G2 / G3 references
for geometry_variant in (
    GEOMETRY_TARGETS_7DD
):

    pred_col = (
        f"{geometry_variant}_geometry_count"
    )


    pred = (
        routing_eval_7dd_df[
            pred_col
        ]
        .to_numpy(
            dtype=float
        )
    )


    policy_name = (
        f"{geometry_variant}_GLOBAL"
    )


    policy_prediction_map_7dd[
        policy_name
    ] = pred


    policy_rows_7dd.append(
        evaluate_prediction_7dd(
            df=
                routing_eval_7dd_df,

            pred=
                pred,

            policy_name=
                policy_name,

            policy_family=
                "REFERENCE",

            geometry_variant=
                geometry_variant,

            rule_description=
                f"Always use {geometry_variant}",
        )
    )


# ------------------------------------------------------------
# 6. Rule Family A:
#    candidate_area_fraction threshold
# ------------------------------------------------------------

for geometry_variant in (
    GEOMETRY_TARGETS_7DD
):

    geometry_pred = (
        routing_eval_7dd_df[
            f"{geometry_variant}_geometry_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    for threshold in (
        CANDIDATE_AREA_THRESHOLDS_7DD
    ):

        route_mask = (
            routing_eval_7dd_df[
                "candidate_area_fraction"
            ]
            .to_numpy(
                dtype=float
            )
            >=
            threshold
        )


        routed_pred = np.where(
            route_mask,
            geometry_pred,
            g0_pred_7dd,
        )


        policy_name = (
            f"{geometry_variant}_AREA_{threshold:.2f}"
        )


        policy_prediction_map_7dd[
            policy_name
        ] = routed_pred


        policy_rows_7dd.append(
            evaluate_prediction_7dd(
                df=
                    routing_eval_7dd_df,

                pred=
                    routed_pred,

                policy_name=
                    policy_name,

                policy_family=
                    "AREA_ONLY",

                geometry_variant=
                    geometry_variant,

                rule_description=(
                    f"use {geometry_variant} if "
                    f"candidate_area_fraction >= {threshold:.2f}; "
                    f"else G0"
                ),
            )
        )


# ------------------------------------------------------------
# 7. Rule Family B:
#
# candidate_area_fraction >= 0.90
#
# AND
#
# G1_increment_per_G0 >= inflation threshold
# ------------------------------------------------------------

FIXED_AREA_THRESHOLD_7DD = 0.90


for geometry_variant in (
    GEOMETRY_TARGETS_7DD
):

    geometry_pred = (
        routing_eval_7dd_df[
            f"{geometry_variant}_geometry_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    for inflation_threshold in (
        RAW_INFLATION_THRESHOLDS_7DD
    ):

        route_mask = (
            (
                routing_eval_7dd_df[
                    "candidate_area_fraction"
                ]
                .to_numpy(
                    dtype=float
                )
                >=
                FIXED_AREA_THRESHOLD_7DD
            )
            &
            (
                routing_eval_7dd_df[
                    "G1_increment_per_G0"
                ]
                .to_numpy(
                    dtype=float
                )
                >=
                inflation_threshold
            )
        )


        routed_pred = np.where(
            route_mask,
            geometry_pred,
            g0_pred_7dd,
        )


        policy_name = (
            f"{geometry_variant}_AREA090_RAWINF_{inflation_threshold:.1f}"
        )


        policy_prediction_map_7dd[
            policy_name
        ] = routed_pred


        policy_rows_7dd.append(
            evaluate_prediction_7dd(
                df=
                    routing_eval_7dd_df,

                pred=
                    routed_pred,

                policy_name=
                    policy_name,

                policy_family=
                    "AREA_AND_RAW_INFLATION",

                geometry_variant=
                    geometry_variant,

                rule_description=(
                    f"use {geometry_variant} if "
                    f"candidate_area_fraction >= 0.90 AND "
                    f"G1_increment_per_G0 >= {inflation_threshold:.1f}; "
                    f"else G0"
                ),
            )
        )


# ------------------------------------------------------------
# 8. Non-deployable oracle references
#
# Per image:
# choose whichever of G0 and G2/G3 has lower absolute error.
#
# This is ONLY an upper-bound diagnostic.
# ------------------------------------------------------------

gt_7dd = (
    routing_eval_7dd_df[
        "gt_count"
    ]
    .to_numpy(
        dtype=float
    )
)


for geometry_variant in (
    GEOMETRY_TARGETS_7DD
):

    geometry_pred = (
        routing_eval_7dd_df[
            f"{geometry_variant}_geometry_count"
        ]
        .to_numpy(
            dtype=float
        )
    )


    g0_abs = np.abs(
        g0_pred_7dd
        -
        gt_7dd
    )


    geometry_abs = np.abs(
        geometry_pred
        -
        gt_7dd
    )


    oracle_use_geometry = (
        geometry_abs
        <
        g0_abs
    )


    oracle_pred = np.where(
        oracle_use_geometry,
        geometry_pred,
        g0_pred_7dd,
    )


    policy_name = (
        f"ORACLE_G0_{geometry_variant}"
    )


    policy_prediction_map_7dd[
        policy_name
    ] = oracle_pred


    policy_rows_7dd.append(
        evaluate_prediction_7dd(
            df=
                routing_eval_7dd_df,

            pred=
                oracle_pred,

            policy_name=
                policy_name,

            policy_family=
                "ORACLE_NON_DEPLOYABLE",

            geometry_variant=
                geometry_variant,

            rule_description=(
                f"GT oracle: choose lower-error of G0 and {geometry_variant}"
            ),
        )
    )


# ------------------------------------------------------------
# 9. Build policy summary
# ------------------------------------------------------------

routing_policy_summary_7dd_df = (
    pd.DataFrame(
        policy_rows_7dd
    )
)


# ------------------------------------------------------------
# 10. Add deltas vs G0
# ------------------------------------------------------------

g0_row_7dd = (
    routing_policy_summary_7dd_df[
        routing_policy_summary_7dd_df[
            "policy_name"
        ]
        ==
        "G0_GLOBAL"
    ]
    .iloc[0]
)


routing_policy_summary_7dd_df[
    "delta_MAE_vs_G0"
] = (
    routing_policy_summary_7dd_df[
        "MAE"
    ]
    -
    float(
        g0_row_7dd[
            "MAE"
        ]
    )
)


routing_policy_summary_7dd_df[
    "delta_RMSE_vs_G0"
] = (
    routing_policy_summary_7dd_df[
        "RMSE"
    ]
    -
    float(
        g0_row_7dd[
            "RMSE"
        ]
    )
)


routing_policy_summary_7dd_df[
    "delta_signed_error_vs_G0"
] = (
    routing_policy_summary_7dd_df[
        "mean_signed_error"
    ]
    -
    float(
        g0_row_7dd[
            "mean_signed_error"
        ]
    )
)


# ------------------------------------------------------------
# 11. Per-image policy table
# ------------------------------------------------------------

routing_image_eval_7dd_df = (
    routing_eval_7dd_df[
        [
            "dataset_index",
            "image_id",
            "category",
            "density_stratum",
            "gt_count",

            "candidate_area_fraction",
            "G1_increment_per_G0",

            "G0_geometry_count",
            "G2_geometry_count",
            "G3_geometry_count",
        ]
    ]
    .copy()
)


for policy_name, pred in (
    policy_prediction_map_7dd.items()
):

    routing_image_eval_7dd_df[
        f"{policy_name}_prediction"
    ] = (
        pred.astype(int)
    )


    routing_image_eval_7dd_df[
        f"{policy_name}_abs_error"
    ] = np.abs(
        pred
        -
        gt_7dd
    )


# ------------------------------------------------------------
# 12. Per-stratum diagnostic
#
# Density stratum is used ONLY for POST-HOC evaluation.
# It is NOT part of any routing rule.
# ------------------------------------------------------------

stratum_rows_7dd = []


for policy_name, pred in (
    policy_prediction_map_7dd.items()
):

    temp_df = (
        routing_eval_7dd_df[
            [
                "density_stratum",
                "gt_count",
            ]
        ]
        .copy()
    )


    temp_df[
        "prediction"
    ] = pred


    temp_df[
        "signed_error"
    ] = (
        temp_df[
            "prediction"
        ]
        -
        temp_df[
            "gt_count"
        ]
    )


    for stratum, sdf in (
        temp_df.groupby(
            "density_stratum"
        )
    ):

        signed = (
            sdf[
                "signed_error"
            ]
            .to_numpy(
                dtype=float
            )
        )


        stratum_rows_7dd.append(
            {
                "policy_name":
                    policy_name,

                "density_stratum":
                    stratum,

                "num_images":
                    int(
                        len(
                            sdf
                        )
                    ),

                "MAE":
                    float(
                        np.abs(
                            signed
                        )
                        .mean()
                    ),

                "RMSE":
                    float(
                        np.sqrt(
                            np.mean(
                                signed
                                ** 2
                            )
                        )
                    ),

                "mean_signed_error":
                    float(
                        signed.mean()
                    ),
            }
        )


routing_stratum_eval_7dd_df = (
    pd.DataFrame(
        stratum_rows_7dd
    )
)


# ------------------------------------------------------------
# 13. Simple selection summary
#
# IMPORTANT:
#
# We identify the best DEVELOPMENT policy descriptively,
# but DO NOT freeze it yet.
# ------------------------------------------------------------

deployable_policy_mask_7dd = (
    routing_policy_summary_7dd_df[
        "policy_family"
    ]
    !=
    "ORACLE_NON_DEPLOYABLE"
)


deployable_summary_7dd_df = (
    routing_policy_summary_7dd_df[
        deployable_policy_mask_7dd
    ]
    .sort_values(
        [
            "MAE",
            "RMSE",
        ]
    )
    .reset_index(
        drop=True
    )
)


oracle_summary_7dd_df = (
    routing_policy_summary_7dd_df[
        routing_policy_summary_7dd_df[
            "policy_family"
        ]
        ==
        "ORACLE_NON_DEPLOYABLE"
    ]
    .sort_values(
        "MAE"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# 14. Save outputs
# ------------------------------------------------------------

M1_7D_D_POLICY_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_D_routing_policy_sensitivity.csv"
)

M1_7D_D_IMAGE_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_D_routing_image_evaluation.csv"
)

M1_7D_D_STRATUM_FILE = (
    M1_TABLES_DIR
    /
    "M1_7D_D_routing_stratum_evaluation.csv"
)


routing_policy_summary_7dd_df.to_csv(
    M1_7D_D_POLICY_FILE,
    index=False,
)

routing_image_eval_7dd_df.to_csv(
    M1_7D_D_IMAGE_FILE,
    index=False,
)

routing_stratum_eval_7dd_df.to_csv(
    M1_7D_D_STRATUM_FILE,
    index=False,
)


# ------------------------------------------------------------
# 15. Report — references
# ------------------------------------------------------------

print("\nReference policies")
print("------------------")

display(
    routing_policy_summary_7dd_df[
        routing_policy_summary_7dd_df[
            "policy_family"
        ]
        ==
        "REFERENCE"
    ]
    [
        [
            "policy_name",
            "num_geometry_images",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "under_images",
            "exact_images",
            "over_images",
            "delta_MAE_vs_G0",
            "delta_RMSE_vs_G0",
        ]
    ]
)


# ------------------------------------------------------------
# 16. Report — Area-only routing sensitivity
# ------------------------------------------------------------

print("\nArea-only routing sensitivity")
print("-----------------------------")

display(
    routing_policy_summary_7dd_df[
        routing_policy_summary_7dd_df[
            "policy_family"
        ]
        ==
        "AREA_ONLY"
    ]
    .sort_values(
        [
            "geometry_variant",
            "MAE",
        ]
    )
    [
        [
            "policy_name",
            "geometry_variant",
            "num_geometry_images",
            "geometry_activation_fraction",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_G0",
            "delta_RMSE_vs_G0",
        ]
    ]
)


# ------------------------------------------------------------
# 17. Report — Two-feature routing sensitivity
# ------------------------------------------------------------

print("\nArea + raw-inflation routing sensitivity")
print("----------------------------------------")

display(
    routing_policy_summary_7dd_df[
        routing_policy_summary_7dd_df[
            "policy_family"
        ]
        ==
        "AREA_AND_RAW_INFLATION"
    ]
    .sort_values(
        [
            "geometry_variant",
            "MAE",
        ]
    )
    [
        [
            "policy_name",
            "geometry_variant",
            "num_geometry_images",
            "geometry_activation_fraction",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_G0",
            "delta_RMSE_vs_G0",
        ]
    ]
)


# ------------------------------------------------------------
# 18. Report — deployable policies ranked descriptively
# ------------------------------------------------------------

print("\nDeployable policies — development ranking")
print("-----------------------------------------")

display(
    deployable_summary_7dd_df[
        [
            "policy_name",
            "policy_family",
            "geometry_variant",
            "num_geometry_images",
            "geometry_activation_fraction",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_G0",
            "delta_RMSE_vs_G0",
        ]
    ]
)


# ------------------------------------------------------------
# 19. Report — Oracle upper bounds
# ------------------------------------------------------------

print("\nOracle references — NON-DEPLOYABLE")
print("----------------------------------")

display(
    oracle_summary_7dd_df[
        [
            "policy_name",
            "geometry_variant",
            "num_geometry_images",
            "MAE",
            "RMSE",
            "mean_signed_error",
            "delta_MAE_vs_G0",
            "delta_RMSE_vs_G0",
        ]
    ]
)


# ------------------------------------------------------------
# 20. Stratum diagnostic for top few deployable policies
#
# Show:
#   G0
#   global G2/G3
#   top 3 routed deployable policies
# ------------------------------------------------------------

top_routed_names_7dd = (
    deployable_summary_7dd_df[
        ~deployable_summary_7dd_df[
            "policy_family"
        ]
        .eq(
            "REFERENCE"
        )
    ]
    .head(
        3
    )
    [
        "policy_name"
    ]
    .tolist()
)


stratum_display_names_7dd = [
    "G0_GLOBAL",
    "G2_GLOBAL",
    "G3_GLOBAL",
    *
    top_routed_names_7dd,
]


print("\nPost-hoc density-stratum MAE for key policies")
print("---------------------------------------------")

display(
    routing_stratum_eval_7dd_df[
        routing_stratum_eval_7dd_df[
            "policy_name"
        ]
        .isin(
            stratum_display_names_7dd
        )
    ]
    .pivot(
        index=
            "density_stratum",

        columns=
            "policy_name",

        values=
            "MAE",
    )
)


# ------------------------------------------------------------
# 21. Protocol report
# ------------------------------------------------------------

print("\nDevelopment routing sensitivity performed       : YES")
print("Inference features used                         : candidate_area_fraction, G1_increment_per_G0")
print("GT count used as routing input                  : NO")
print("Density stratum used as routing input           : NO")
print("Semantic category used as routing input         : NO")

print("\nLearned router used                              : NO")
print("Fine-grained threshold search                    : NO")
print("Candidate-area thresholds                       :", CANDIDATE_AREA_THRESHOLDS_7DD)
print("Raw-inflation thresholds                        :", RAW_INFLATION_THRESHOLDS_7DD)

print("\nGeometry variants tested                         : G2, G3")
print("Geometry threshold changed                       : NO")

print("\nOracle evaluated                                 : YES — diagnostic only")
print("Oracle deployable                               : NO")

print("\nFinal routing rule frozen                        : NO")
print("B0 modified                                      : NO")
print("B2 modified                                      : NO")
print("Test set accessed                                : NO")

print("\nStep 7D Part D: PASS")

STEP 7D PART D — SIMPLE INTERPRETABLE ROUTING SENSITIVITY

Routing-table provenance
------------------------
Inference routing features : selective_geometry_diag_7dc_df
Frozen geometry counts     : image_geometry_counts_7d_df
GT evaluation metadata     : geometry_count_eval_7db_df
Images aligned             : 100

Reference policies
------------------


,policy_name,num_geometry_images,MAE,RMSE,mean_signed_error,under_images,exact_images,over_images,delta_MAE_vs_G0,delta_RMSE_vs_G0
0,G0_GLOBAL,0,79.67,263.541363,-74.91,68,5,27,0.00,0.000000
1,G2_GLOBAL,100,71.70,240.928620,-39.80,31,2,67,-7.97,-22.612743
2,G3_GLOBAL,100,72.26,246.961576,-50.44,39,1,60,-7.41,-16.579787



Area-only routing sensitivity
-----------------------------


,policy_name,geometry_variant,num_geometry_images,geometry_activation_fraction,MAE,RMSE,mean_signed_error,delta_MAE_vs_G0,delta_RMSE_vs_G0
3,G2_AREA_0.85,G2,57,0.57,69.60,242.624236,-54.02,-10.07,-20.917127
4,G2_AREA_0.90,G2,39,0.39,75.33,254.574174,-64.53,-4.34,-8.967189
5,G2_AREA_0.95,G2,23,0.23,76.31,255.855995,-69.67,-3.36,-7.685368
6,G3_AREA_0.85,G3,57,0.57,72.54,248.554582,-59.76,-7.13,-14.986781
7,G3_AREA_0.90,G3,39,0.39,75.92,256.281330,-66.26,-3.75,-7.260033
8,G3_AREA_0.95,G3,23,0.23,76.94,257.469493,-70.72,-2.73,-6.071870



Area + raw-inflation routing sensitivity
----------------------------------------


...


Oracle references — NON-DEPLOYABLE
----------------------------------


,policy_name,geometry_variant,num_geometry_images,MAE,RMSE,mean_signed_error,delta_MAE_vs_G0,delta_RMSE_vs_G0
0,ORACLE_G0_G2,G2,48,60.89,239.506555,-53.73,-18.78,-24.034808
1,ORACLE_G0_G3,G3,52,65.14,246.432628,-59.12,-14.53,-17.108735



Post-hoc density-stratum MAE for key policies
---------------------------------------------


policy_name,G0_GLOBAL,G2_AREA090_RAWINF_1.5,G2_AREA_0.85,G2_GLOBAL,G3_AREA_0.85,G3_GLOBAL
density_stratum,,,,,,
S1_0_20,5.25,7.65,9.85,13.00,9.2,11.80
S2_20_40,8.30,7.90,15.85,23.65,14.2,20.45
S3_40_60,11.20,10.65,15.85,22.65,15.0,19.95
S4_60_80,26.50,17.90,18.75,31.60,17.4,18.20
S5_80_95,74.60,74.60,64.90,43.40,68.4,47.60
S6_95_100,619.60,576.10,510.50,491.80,545.4,534.20



Development routing sensitivity performed       : YES
Inference features used                         : candidate_area_fraction, G1_increment_per_G0
GT count used as routing input                  : NO
Density stratum used as routing input           : NO
Semantic category used as routing input         : NO

Learned router used                              : NO
Fine-grained threshold search                    : NO
Candidate-area thresholds                       : [0.85, 0.9, 0.95]
Raw-inflation thresholds                        : [1.0, 1.5, 2.0]

Geometry variants tested                         : G2, G3
Geometry threshold changed                       : NO

Oracle evaluated                                 : YES — diagnostic only
Oracle deployable                               : NO

Final routing rule frozen                        : NO
B0 modified                                      : NO
B2 modified                                      : NO
Test set accessed                            

### **Finding 14:**

Selective application of Marigold refinement improves upon both the unrefined semantic-component counter and globally applied geometry.

A simple inference-time rule using candidate semantic-area dominance performs best among the tested routing policies: applying G2 only when candidate components occupy at least 85% of the A3c foreground reduces MAE from 79.67 (G0) and 71.70 (global G2) to 69.60.

Adding raw-fragmentation inflation as a second routing condition does not improve performance.

This supports a simple and interpretable selective-geometry mechanism rather than global geometry application or a more complex learned router.